# Trading AI: Notebook 10, every strategy and liquidity (model v1.5)

This adds two parts to the brain the StreixAnalyst server loads. Both are learned from years of candles on 30 markets (forex, gold, silver, indices, oil and 10 crypto pairs), on every timeframe from 1 minute to 1 week.

**1. The strategy head.** Every strategy a person can pick in the app (smart money and ICT setups, price action, the indicator strategies such as Supertrend, MACD, RSI and Bollinger, scalping, and fixed-time calls: 45 in all) is run on every market and timeframe exactly the way the app runs it. That means entry on the next candle's open, the strategy's own stop moved past nearby liquidity, its target, the market's costs and slippage, and the stop counted first when one candle touches both. From those trades the brain learns:
- **The odds of each live signal**: the chance a signal like this ends in profit, from the market's context when it fires (structure, higher-timeframe trend, momentum, volatility, liquidity, session), the trade's shape, and how that strategy had been doing on that market just before.
- **What works where**: every strategy's results by market and timeframe, after costs. The strategy picker in the app shows them.

**2. The liquidity head.** For every pool of liquidity near price (untaken swing highs and lows, equal highs and lows, the previous day's high and low, and the Asian range), it learns:
- **Sweep**: the chance it's taken within the next 24 candles.
- **Reversal**: after a sweep, the chance price closes back inside and runs 1 ATR the other way (a stop hunt that turns, not a breakout).
- **Depth**: how far past the pool such hunts reach. The app puts stops beyond that point, so a normal hunt doesn't stop the trade out.

**Honest testing.** Every part is trained on the first 70% of each market's history and judged on the last 30%, which it never saw. Each part is compared with a simple baseline: the strategy's own historical rate, the distance alone, or the chart rule the app uses now. **The app only uses a part that beats its baseline there** (the `use` flags in section 6). The evidence tables are shown either way. None of this guarantees profit. It makes the analyst's numbers better informed and more honest.

*Research tool, not financial advice. Paper-trade first.*

**How to run**
1. Runtime → Run all. Allow Google Drive.
2. With `PRESET = "standard"` it takes about 45–75 minutes the first time (downloading and training). Later runs reuse the Drive cache.
3. If you uploaded a brain from Notebook 7 or 8 before, `REUSE_BRAIN = True` keeps its main models and adds the two heads, which saves about 30 minutes.
4. Download `guru_brain_v15.joblib` from your Drive (`trading_ai/guru_v15/`), then upload it in the app: Admin → System & model.

## 0. Settings and setup

In [ ]:
# ---- Settings you can change ----
PRESET = "standard"              # "quick" (about 15 min, for a check) or "standard" (about 45-75 min)
REUSE_BRAIN = True               # keep the main models from your last brain (Notebook 8 or 7) when one is found
USE_DRIVE = True
SCALP_DATA = True                # also 5-minute and 1-minute history, for the scalping strategies
APP_VERSION = "1.5"              # recorded in the file for reference


In [ ]:
import os
# Where things are saved (Google Drive if mounted, otherwise this Colab session only)
SAVE_DIR = "trading_ai"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        SAVE_DIR = "/content/drive/MyDrive/trading_ai"
    except Exception as e:
        print("Drive not mounted, saving inside this session only:", e)

GURU_DIR = os.path.join(SAVE_DIR, "guru_v15")
CACHE_DIR = os.path.join(SAVE_DIR, "guru_cache")
BRAIN_PATH = os.path.join(GURU_DIR, "guru_brain_v15.joblib")
PREVIOUS = [os.path.join(SAVE_DIR, "guru_v14", "guru_brain_v14_scalp.joblib"),
            os.path.join(SAVE_DIR, "guru_v13", "guru_brain_v13.joblib")]
for d in (GURU_DIR, CACHE_DIR):
    os.makedirs(d, exist_ok=True)
print("Saving to", GURU_DIR)


## 0b. The engine code

The engine (the same files as `backend/guru` on the server, including the strategies and the two new heads) is written into this notebook by the cells below. Nothing needs to be downloaded from GitHub. Run these cells once.

In [ ]:
%%writefile structure_engine.py
"""Market structure engine (Notebook 1), packaged as a module.
Shared by training (Notebook 2), the backtester (Notebook 3) and the live web app."""
import math
import numpy as np
import pandas as pd

ENGINE_DEFAULTS = dict(swing_left=3, swing_right=3, atr_period=14, zone_tol_atr=0.6, zone_min_touches=2,
                       max_zones=6, equal_tol_atr=0.15, min_fvg_atr=0.25, pattern_trend_bars=5)


def load_yfinance(ticker, interval="15m", period="59d"):
    import yfinance as yf
    df = yf.Ticker(ticker).history(period=period, interval=interval, auto_adjust=False)
    if df is None or df.empty:
        raise ValueError(f"No data returned for {ticker} ({interval}, {period}). "
                         "Check the ticker, shorten the period, or use SOURCE='csv'.")
    df = df.rename(columns=str.lower)[["open", "high", "low", "close", "volume"]]
    df.index.name = "time"
    return df.dropna().astype(float)


def load_csv(path):
    """Reads most broker/charting exports (MT5 <DATE>/<TIME> columns, TradingView, Dukascopy)."""
    raw = pd.read_csv(path, sep=None, engine="python")
    raw.columns = [str(c).strip().strip("<>").lower() for c in raw.columns]
    if "date" in raw.columns and "time" in raw.columns:
        raw["time"] = raw["date"].astype(str) + " " + raw["time"].astype(str)
        raw = raw.drop(columns="date")
    if "tickvol" in raw.columns:   # MT5: tick volume is the useful one for FX
        raw = raw.drop(columns=[c for c in ("vol", "volume") if c in raw.columns]).rename(columns={"tickvol": "volume"})
    raw = raw.rename(columns={"date": "time", "datetime": "time", "timestamp": "time", "gmt time": "time",
                              "tick_volume": "volume", "vol": "volume"})
    t = raw["time"].astype(str).str.strip().str.replace(r"\s*(GMT|UTC).*$", "", regex=True)
    if t.str.fullmatch(r"\d{9,11}").all():                                         # TradingView: unix seconds
        raw["time"] = pd.to_datetime(t.astype("int64"), unit="s")
    else:
        t = t.str.replace(r"^(\d{4})\.(\d{1,2})\.(\d{1,2})", r"\1-\2-\3", regex=True)   # MT5: 2024.01.02
        t = t.str.replace(r"^(\d{1,2})\.(\d{1,2})\.(\d{4})", r"\3-\2-\1", regex=True)   # Dukascopy: 02.01.2024
        raw["time"] = pd.to_datetime(t)
    df = raw.set_index("time").sort_index()
    if "volume" not in df.columns:
        df["volume"] = 0.0
    return df[["open", "high", "low", "close", "volume"]].astype(float).dropna()


def make_demo_data(n=3000, seed=7, start_price=1.0850, freq="15min"):
    """Synthetic trending/ranging prices. For testing the code offline only."""
    rng = np.random.default_rng(seed)
    regime_len = 150
    regimes = rng.choice([-1, 0, 1], size=n // regime_len + 1)
    drift = np.repeat(regimes, regime_len)[:n] * 0.00025
    rets = drift + rng.normal(0, 0.0011, n)
    close = start_price * np.exp(np.cumsum(rets))
    open_ = np.r_[start_price, close[:-1]]
    wicks = np.abs(rng.normal(0, 0.0006, (n, 2))) * close[:, None]
    high = np.maximum(open_, close) + wicks[:, 0]
    low = np.minimum(open_, close) - wicks[:, 1]
    idx = pd.date_range("2026-01-05", periods=n, freq=freq, name="time")
    return pd.DataFrame({"open": open_, "high": high, "low": low, "close": close,
                         "volume": rng.integers(100, 1000, n).astype(float)}, index=idx)


def add_atr(df, n=14):
    prev_close = df["close"].shift()
    tr = pd.concat([df["high"] - df["low"],
                    (df["high"] - prev_close).abs(),
                    (df["low"] - prev_close).abs()], axis=1).max(axis=1)
    df["atr"] = tr.ewm(alpha=1 / n, adjust=False).mean()
    return df


def find_pivots(df, left=3, right=3):
    """Raw fractal pivots, in the order they happened. confirmed_idx = first bar where the pivot is knowable."""
    h, l = df["high"].to_numpy(), df["low"].to_numpy()
    pivots = []
    for i in range(left, len(df) - right):
        if h[i] > h[i - left:i].max() and h[i] >= h[i + 1:i + right + 1].max():
            pivots.append(dict(idx=i, time=str(df.index[i]), type="high", price=float(h[i]), confirmed_idx=i + right))
        if l[i] < l[i - left:i].min() and l[i] <= l[i + 1:i + right + 1].min():
            pivots.append(dict(idx=i, time=str(df.index[i]), type="low", price=float(l[i]), confirmed_idx=i + right))
    return pivots


def alternate_swings(pivots):
    """Force high/low/high/low order: of two same-type swings in a row, keep the more extreme one."""
    out = []
    for p in pivots:
        p = dict(p)
        if out and out[-1]["type"] == p["type"]:
            more_extreme = p["price"] > out[-1]["price"] if p["type"] == "high" else p["price"] < out[-1]["price"]
            if more_extreme:
                out[-1] = p
        else:
            out.append(p)
    return out


def label_structure(swings):
    last_high = last_low = None
    for s in swings:
        if s["type"] == "high":
            s["label"] = "H" if last_high is None else ("HH" if s["price"] > last_high else "LH")
            last_high = s["price"]
        else:
            s["label"] = "L" if last_low is None else ("HL" if s["price"] > last_low else "LL")
            last_low = s["price"]
    return swings


def detect_structure_breaks(df, pivots):
    c, h, l = (df[k].to_numpy() for k in ("close", "high", "low"))
    by_conf = {}
    for p in pivots:
        by_conf.setdefault(p["confirmed_idx"], []).append(p)
    trend, hi, lo = 0, None, None
    events, sweeps = [], []
    trend_arr = np.zeros(len(df), dtype=int)
    for i in range(len(df)):
        for p in by_conf.get(i, []):
            if p["type"] == "high":
                hi = dict(p, swept=False)
            else:
                lo = dict(p, swept=False)
        if hi is not None:
            if c[i] > hi["price"]:
                events.append(dict(kind="CHoCH" if trend == -1 else "BOS", direction="bullish", level=hi["price"],
                                   from_idx=hi["idx"], idx=i, time=str(df.index[i])))
                trend, hi = 1, None
            elif h[i] > hi["price"] and not hi["swept"]:
                sweeps.append(dict(side="buy-side", level=hi["price"], wick=float(h[i]),
                                   from_idx=hi["idx"], idx=i, time=str(df.index[i])))
                hi["swept"] = True
        if lo is not None:
            if c[i] < lo["price"]:
                events.append(dict(kind="CHoCH" if trend == 1 else "BOS", direction="bearish", level=lo["price"],
                                   from_idx=lo["idx"], idx=i, time=str(df.index[i])))
                trend, lo = -1, None
            elif l[i] < lo["price"] and not lo["swept"]:
                sweeps.append(dict(side="sell-side", level=lo["price"], wick=float(l[i]),
                                   from_idx=lo["idx"], idx=i, time=str(df.index[i])))
                lo["swept"] = True
        trend_arr[i] = trend
    return events, sweeps, trend_arr


def find_liquidity(df, swings, equal_tol_atr=0.15):
    h, l, atr = (df[k].to_numpy() for k in ("high", "low", "atr"))
    # highest high / lowest low strictly AFTER each bar
    fut_max = np.r_[np.maximum.accumulate(h[::-1])[::-1][1:], -np.inf]
    fut_min = np.r_[np.minimum.accumulate(l[::-1])[::-1][1:], np.inf]
    highs = [s for s in swings if s["type"] == "high"]
    lows = [s for s in swings if s["type"] == "low"]
    for s in highs:
        s["taken"] = bool(fut_max[s["idx"]] > s["price"])
    for s in lows:
        s["taken"] = bool(fut_min[s["idx"]] < s["price"])
    pools = []
    for a, b in zip(highs, highs[1:]):
        if abs(a["price"] - b["price"]) <= equal_tol_atr * atr[b["idx"]]:
            lvl = max(a["price"], b["price"])
            pools.append(dict(kind="EQH", side="buy-side", level=lvl, idx_a=a["idx"], idx_b=b["idx"],
                              taken=bool(fut_max[b["idx"]] > lvl)))
    for a, b in zip(lows, lows[1:]):
        if abs(a["price"] - b["price"]) <= equal_tol_atr * atr[b["idx"]]:
            lvl = min(a["price"], b["price"])
            pools.append(dict(kind="EQL", side="sell-side", level=lvl, idx_a=a["idx"], idx_b=b["idx"],
                              taken=bool(fut_min[b["idx"]] < lvl)))
    price = float(df["close"].iloc[-1])
    buy_side = sorted([s for s in highs if not s["taken"] and s["price"] > price], key=lambda s: s["price"])
    sell_side = sorted([s for s in lows if not s["taken"] and s["price"] < price], key=lambda s: -s["price"])
    return dict(pools=pools, buy_side=buy_side, sell_side=sell_side)


def build_zones(df, swings, tol_atr=0.6, min_touches=2, max_zones=6, lookback_swings=200):
    """Density clustering: repeatedly take the price band (width = tol) holding the most recency-weighted
    swings, turn it into a zone, remove those swings, repeat. Keeps the best zones on each side of price."""
    swings = swings[-lookback_swings:]
    if not swings:
        return []
    atr = float(df["atr"].median())
    tol, pad = tol_atr * atr, 0.1 * atr
    n, price = len(df), float(df["close"].iloc[-1])
    prices = np.array([s["price"] for s in swings])
    idxs = np.array([s["idx"] for s in swings])
    weights = 0.5 + idxs / n                       # recent touches count more
    left = np.ones(len(prices), bool)
    zones = []
    while left.sum() >= min_touches:
        near = (np.abs(prices[:, None] - prices[None, :]) <= tol / 2) & left[None, :] & left[:, None]
        best = int(np.argmax((near * weights[None, :]).sum(1)))
        members = near[best]
        if members.sum() < min_touches:
            break
        left &= ~members
        bottom, top = prices[members].min() - pad, prices[members].max() + pad
        if any(bottom <= z["top"] and top >= z["bottom"] for z in zones):
            continue                               # overlaps a stronger zone already kept
        dist_atr = max(0.0, bottom - price, price - top) / atr
        score = weights[members].sum() / (1 + dist_atr / 20)
        zones.append(dict(bottom=float(bottom), top=float(top), touches=int(members.sum()),
                          first_idx=int(idxs[members].min()), last_idx=int(idxs[members].max()),
                          score=round(float(score), 3),
                          role="resistance" if bottom > price else "support" if top < price else "at zone"))
    zones.sort(key=lambda z: -z["score"])
    per_side = max(1, max_zones // 2)
    return ([z for z in zones if z["role"] == "at zone"]
            + [z for z in zones if z["role"] == "support"][:per_side]
            + [z for z in zones if z["role"] == "resistance"][:per_side])


def find_fvgs(df, min_atr=0.25):
    h, l, atr = (df[k].to_numpy() for k in ("high", "low", "atr"))
    gaps = []
    for i in range(2, len(df)):
        if l[i] > h[i - 2] and l[i] - h[i - 2] >= min_atr * atr[i]:
            gaps.append(dict(direction="bullish", top=float(l[i]), bottom=float(h[i - 2]), idx=i - 1))
        elif h[i] < l[i - 2] and l[i - 2] - h[i] >= min_atr * atr[i]:
            gaps.append(dict(direction="bearish", top=float(l[i - 2]), bottom=float(h[i]), idx=i - 1))
    for g in gaps:   # search forward in growing chunks: fast even on years of data
        j, step, g["filled_idx"] = g["idx"] + 2, 64, None
        while j < len(df):
            seg = l[j:j + step] <= g["bottom"] if g["direction"] == "bullish" else h[j:j + step] >= g["top"]
            hit = np.flatnonzero(seg)
            if len(hit):
                g["filled_idx"] = int(j + hit[0])
                break
            j, step = j + step, step * 2
    return gaps


PATTERN_BIAS = {
    "Hammer": "bullish", "Inverted Hammer": "bullish", "Bullish Engulfing": "bullish", "Bullish Harami": "bullish",
    "Piercing Line": "bullish", "Morning Star": "bullish", "Three White Soldiers": "bullish",
    "Bullish Marubozu": "bullish", "Tweezer Bottom": "bullish",
    "Hanging Man": "bearish", "Shooting Star": "bearish", "Bearish Engulfing": "bearish", "Bearish Harami": "bearish",
    "Dark Cloud Cover": "bearish", "Evening Star": "bearish", "Three Black Crows": "bearish",
    "Bearish Marubozu": "bearish", "Tweezer Top": "bearish",
    "Doji": "neutral", "Inside Bar": "neutral",
}


def detect_candles(df, trend_bars=5):
    o, h, l, c = (df[k] for k in ("open", "high", "low", "close"))
    rng = (h - l).replace(0, np.nan)
    body = (c - o).abs()
    top, bot = np.maximum(o, c), np.minimum(o, c)
    upper, lower = h - top, bot - l
    bull, bear = c > o, c < o
    avg_body = body.rolling(14, min_periods=5).mean()
    long_body = body > avg_body
    small_body = body <= 0.3 * rng
    down_before = c.shift(1) < c.shift(trend_bars + 1)
    up_before = c.shift(1) > c.shift(trend_bars + 1)

    def b(s, k=1):   # boolean from k bars ago
        return s.shift(k, fill_value=False).astype(bool)

    def v(s, k=1):   # value from k bars ago
        return s.shift(k)

    hammer = (lower >= 2 * body) & (upper <= 0.15 * rng) & (body >= 0.05 * rng)
    inverted = (upper >= 2 * body) & (lower <= 0.15 * rng) & (body >= 0.05 * rng)
    mid_1 = (v(o) + v(c)) / 2
    mid_2 = (v(o, 2) + v(c, 2)) / 2

    pats = {
        "Doji": body <= 0.08 * rng,
        "Hammer": hammer & down_before,
        "Hanging Man": hammer & up_before,
        "Inverted Hammer": inverted & down_before,
        "Shooting Star": inverted & up_before,
        "Bullish Engulfing": bull & b(bear) & long_body & (c >= v(o)) & (o <= v(c)) & (body > 1.2 * v(body)) & down_before,
        "Bearish Engulfing": bear & b(bull) & long_body & (c <= v(o)) & (o >= v(c)) & (body > 1.2 * v(body)) & up_before,
        "Bullish Harami": b(bear) & b(long_body) & bull & (top <= v(o)) & (bot >= v(c)) & (body <= 0.5 * v(body)) & b(down_before),
        "Bearish Harami": b(bull) & b(long_body) & bear & (top <= v(c)) & (bot >= v(o)) & (body <= 0.5 * v(body)) & b(up_before),
        "Piercing Line": b(bear) & b(long_body) & bull & (o <= v(c)) & (c > mid_1) & (c < v(o)) & b(down_before),
        "Dark Cloud Cover": b(bull) & b(long_body) & bear & (o >= v(c)) & (c < mid_1) & (c > v(o)) & b(up_before),
        "Morning Star": b(bear, 2) & b(long_body, 2) & b(small_body) & bull & (c > mid_2) & b(down_before, 2),
        "Evening Star": b(bull, 2) & b(long_body, 2) & b(small_body) & bear & (c < mid_2) & b(up_before, 2),
        "Three White Soldiers": bull & b(bull) & b(bull, 2) & (c > v(c)) & (v(c) > v(c, 2))
                                & long_body & b(long_body) & b(long_body, 2) & (upper <= 0.3 * body),
        "Three Black Crows": bear & b(bear) & b(bear, 2) & (c < v(c)) & (v(c) < v(c, 2))
                             & long_body & b(long_body) & b(long_body, 2) & (lower <= 0.3 * body),
        "Bullish Marubozu": bull & (body >= 0.9 * rng) & long_body,
        "Bearish Marubozu": bear & (body >= 0.9 * rng) & long_body,
        "Tweezer Bottom": b(bear) & bull & ((l - v(l)).abs() <= 0.05 * rng) & down_before,
        "Tweezer Top": b(bull) & bear & ((h - v(h)).abs() <= 0.05 * rng) & up_before,
        "Inside Bar": (h <= v(h)) & (l >= v(l)),
    }
    flags = pd.DataFrame({k: s.fillna(False).astype(bool) for k, s in pats.items()}, index=df.index)
    events = [dict(idx=int(i), time=str(df.index[i]), name=name, bias=PATTERN_BIAS[name])
              for name in flags.columns for i in np.flatnonzero(flags[name].to_numpy())]
    events.sort(key=lambda e: e["idx"])
    return flags, events


def _decimals(price):
    return int(max(2, 5 - math.floor(math.log10(abs(price))))) if price else 2


def summarize(df, A, recent_bars=3):
    price, atr = float(df["close"].iloc[-1]), float(df["atr"].iloc[-1])
    dp, n = _decimals(price), len(df)
    r = lambda v: round(float(v), dp)
    dist = lambda lvl: round(abs(lvl - price) / atr, 2)
    zone = lambda z: dict(bottom=r(z["bottom"]), top=r(z["top"]), touches=z["touches"],
                          distance_atr=dist((z["bottom"] + z["top"]) / 2))
    ev = A["events"][-1] if A["events"] else None
    sw = A["sweeps"][-1] if A["sweeps"] else None
    liq = A["liquidity"]
    sup = sorted([z for z in A["zones"] if z["role"] == "support"], key=lambda z: price - z["top"])
    res = sorted([z for z in A["zones"] if z["role"] == "resistance"], key=lambda z: z["bottom"] - price)
    open_fvgs = sorted([g for g in A["fvgs"] if g["filled_idx"] is None],
                       key=lambda g: abs((g["top"] + g["bottom"]) / 2 - price))[:3]
    return {
        "time": str(df.index[-1]), "price": r(price), "atr": r(atr),
        "structure": {
            "trend": {1: "bullish", -1: "bearish", 0: "undetermined"}[int(A["trend"][-1])],
            "last_break": None if ev is None else dict(type=ev["kind"], direction=ev["direction"],
                                                       level=r(ev["level"]), bars_ago=n - 1 - ev["idx"]),
            "recent_swings": [w["label"] for w in A["swings"][-6:]],
        },
        "support_zones": [zone(z) for z in sup[:3]],
        "resistance_zones": [zone(z) for z in res[:3]],
        "price_inside_zone": [zone(z) for z in A["zones"] if z["role"] == "at zone"],
        "liquidity": {
            "buy_side_above": [dict(level=r(w["price"]), distance_atr=dist(w["price"])) for w in liq["buy_side"][:3]],
            "sell_side_below": [dict(level=r(w["price"]), distance_atr=dist(w["price"])) for w in liq["sell_side"][:3]],
            "untaken_equal_highs": [r(p["level"]) for p in liq["pools"] if p["kind"] == "EQH" and not p["taken"]][-3:],
            "untaken_equal_lows": [r(p["level"]) for p in liq["pools"] if p["kind"] == "EQL" and not p["taken"]][-3:],
            "last_sweep": None if sw is None else dict(side=sw["side"], level=r(sw["level"]), bars_ago=n - 1 - sw["idx"]),
        },
        "open_fvgs": [dict(direction=g["direction"], bottom=r(g["bottom"]), top=r(g["top"])) for g in open_fvgs],
        "recent_candle_patterns": [dict(name=p["name"], bias=p["bias"], bars_ago=n - 1 - p["idx"])
                                   for p in A["patterns"] if p["idx"] >= n - recent_bars],
    }


def analyze(df, cfg=None):
    cfg = {**ENGINE_DEFAULTS, **(cfg or {})}
    df = add_atr(df[["open", "high", "low", "close", "volume"]].astype(float).copy(), cfg["atr_period"])
    pivots = find_pivots(df, cfg["swing_left"], cfg["swing_right"])
    swings = label_structure(alternate_swings(pivots))
    events, sweeps, trend = detect_structure_breaks(df, pivots)
    liquidity = find_liquidity(df, swings, cfg["equal_tol_atr"])
    zones = build_zones(df, swings, cfg["zone_tol_atr"], cfg["zone_min_touches"], cfg["max_zones"])
    fvgs = find_fvgs(df, cfg["min_fvg_atr"])
    flags, patterns = detect_candles(df, cfg["pattern_trend_bars"])
    df["trend"] = trend
    return df, dict(pivots=pivots, swings=swings, events=events, sweeps=sweeps, trend=trend,
                    liquidity=liquidity, zones=zones, fvgs=fvgs, pattern_flags=flags, patterns=patterns)


In [ ]:
%%writefile market_data.py
"""market_data.py: candles for any symbol on any timeframe (1m to 1M), cached and topped up to now.

Sources are tried in order. The first one that works wins and is remembered for that symbol.
  * Forex, metals, indices, energies, soft commodities, US stocks (and some crypto CFDs): Dukascopy
  * Crypto pairs (BTCUSDT, SOL/USDT, BINANCE:PEPEUSDT ...): Binance, Binance.US, KuCoin, Coinbase, Yahoo, Dukascopy
  * Anything else: Yahoo Finance
A prefix forces a source: DUKASCOPY:BTCUSD, BINANCE:SOLUSDT, BINANCEUS:, KUCOIN:, COINBASE:, YAHOO:AAPL.
TradingView-style prefixes (OANDA:EURUSD, BYBIT:BTCUSDT, NASDAQ:AAPL) are understood too.

Timeframes: 1m 5m 15m 30m 1h 4h 1D 3D 1W 1M  ("1m" = one minute, "1M" = one month).
Every function returns candles with a UTC index and float columns open, high, low, close, volume.
The candle that is still forming is returned separately: analysis runs on closed candles only.
"""
import json
import math
import os
import re
import time
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd

try:
    import requests
except ImportError:          # pragma: no cover - requests ships with Colab
    requests = None

TFS = ("1m", "5m", "15m", "30m", "1h", "4h", "1D", "3D", "1W", "1M")
INTRADAY = ("1m", "5m", "15m", "30m", "1h", "4h")
TF_MIN = {"1m": 1, "5m": 5, "15m": 15, "30m": 30, "1h": 60, "4h": 240, "1D": 1440, "3D": 4320, "1W": 10080,
          "1M": 43830}
TF_LABEL = {"1m": "1 minute", "5m": "5 minutes", "15m": "15 minutes", "30m": "30 minutes", "1h": "1 hour",
            "4h": "4 hours", "1D": "1 day", "3D": "3 days", "1W": "1 week", "1M": "1 month"}
_TF_ALIASES = {"1min": "1m", "m1": "1m", "1": "1m", "5min": "5m", "m5": "5m", "5": "5m", "15min": "15m",
               "m15": "15m", "15": "15m", "30min": "30m", "m30": "30m", "30": "30m", "60m": "1h", "60": "1h",
               "60min": "1h", "1hr": "1h", "1hour": "1h", "h1": "1h", "1h": "1h", "240": "4h", "4hr": "4h",
               "4hour": "4h", "h4": "4h", "4h": "4h", "1d": "1D", "d1": "1D", "d": "1D", "1day": "1D",
               "day": "1D", "daily": "1D", "3d": "3D", "3day": "3D", "1w": "1W", "w1": "1W", "w": "1W",
               "1wk": "1W", "week": "1W", "weekly": "1W", "1mo": "1M", "1mon": "1M", "1month": "1M", "mn": "1M",
               "mn1": "1M", "month": "1M", "monthly": "1M"}


class DataError(Exception):
    pass


class SourceBlocked(DataError):
    """The source refuses this server (e.g. Binance answers 451 to US data centres) or can't be reached."""


class SymbolNotFound(DataError):
    pass


# ───────────────────────────── timeframes ─────────────────────────────
def norm_tf(tf):
    """'1m' = minute, '1M' = month. Also accepts 15min, H1, 4H, D, 1d, W, 1wk, MN, monthly ..."""
    t = str(tf).strip()
    if t in TFS:
        return t
    low = t.lower().replace(" ", "")
    if low in _TF_ALIASES:
        return _TF_ALIASES[low]
    raise ValueError(f"Unknown timeframe {tf!r}. Use one of {', '.join(TFS)} ('1m' = minute, '1M' = month).")


def tf_delta(tf):
    """Fixed length of a candle (None for months, which vary)."""
    return None if tf == "1M" else pd.Timedelta(minutes=TF_MIN[tf])


def close_times(index, tf):
    """Time each candle closes (= when its values became known)."""
    if tf == "1M":
        return pd.DatetimeIndex(index) + pd.offsets.MonthBegin(1)
    return pd.DatetimeIndex(index) + tf_delta(tf)


def utcnow():
    return pd.Timestamp.now(tz="UTC")


def epoch_s(index):
    """Epoch seconds as int64, whatever pandas' internal time resolution is."""
    idx = pd.DatetimeIndex(index)
    if idx.tz is None:
        idx = idx.tz_localize("UTC")
    return np.asarray((idx - pd.Timestamp("1970-01-01", tz="UTC")) // pd.Timedelta(seconds=1), dtype=np.int64)


def to_utc_index(index):
    idx = pd.DatetimeIndex(index)
    idx = idx.tz_convert("UTC") if idx.tz is not None else idx.tz_localize("UTC")
    idx.name = "time"
    return idx


# ───────────────────────────── symbols ─────────────────────────────
FIAT = {"USD", "EUR", "GBP", "JPY", "CHF", "CAD", "AUD", "NZD", "SEK", "NOK", "DKK", "PLN", "HUF", "CZK", "TRY",
        "ZAR", "MXN", "SGD", "HKD", "CNH", "RUB", "ILS", "BRL", "RON", "THB"}
G10 = {"USD", "EUR", "GBP", "JPY", "CHF", "CAD", "AUD", "NZD"}
FX_MAJORS = {"EURUSD", "GBPUSD", "USDJPY", "USDCHF", "USDCAD", "AUDUSD", "NZDUSD"}
STABLE = ("USDT", "USDC", "FDUSD", "BUSD", "TUSD", "DAI", "USD")
CRYPTO_QUOTES = ("FDUSD", "USDT", "USDC", "BUSD", "TUSD", "DAI", "USD", "BTC", "ETH", "BNB", "EUR", "TRY")
KNOWN_CRYPTO = set("""BTC ETH SOL BNB XRP ADA DOGE AVAX DOT LINK LTC BCH TRX MATIC POL TON SHIB NEAR UNI ATOM ETC XLM FIL
APT ARB OP SUI PEPE INJ AAVE XMR EOS DASH COMP YFI MKR BAT ENJ ICP HBAR VET ALGO SAND MANA AXS FTM SEI TIA WIF
BONK FLOKI JUP RNDR RENDER FET GRT IMX STX RUNE KAS TAO ENA ONDO WLD TRUMP HYPE PENGU CRV LDO CAKE GALA CHZ THETA
EGLD XTZ ZEC NEO KAVA 1INCH SNX ZIL QNT FLOW MINA CFX ORDI BLUR JTO PYTH DYDX GMX ZRO NOT BOME MEME PEOPLE
LUNA LUNC USTC BTT HOT IOTA ONE ROSE CELO ANKR AR APE GMT MASK LRC ENS SUSHI BAL OCEAN XEC XDC KSM VIRTUAL
POPCAT BRETT MOG SPX NEIRO GOAT MOODENG ACT PNUT FARTCOIN AI16Z AIXBT S BERA IP KAITO PI XAUT PAXG""".split())
DUKA_CRYPTO = {"BTC": "BTC", "ETH": "ETH", "LTC": "LTC", "BCH": "BCH", "XRP": "XRP", "ADA": "ADA", "XLM": "XLM",
               "LINK": "LNK", "MATIC": "MAT", "POL": "MAT", "UNI": "UNI", "TRX": "TRX", "AAVE": "AVE", "EOS": "EOS",
               "XMR": "XMR", "DASH": "DSH", "COMP": "CMP", "YFI": "YFI", "BAT": "BAT", "ENJ": "ENJ", "MKR": "MKR"}
CRYPTO_MAJORS = {"BTC", "ETH"}

# canonical name: (Dukascopy attribute without "INSTRUMENT_", display, asset class, sub-class, Yahoo ticker)
_DUKA = {
    "XAUUSD": ("FX_METALS_XAU_USD", "XAU/USD", "metal", "gold", "GC=F"),
    "XAGUSD": ("FX_METALS_XAG_USD", "XAG/USD", "metal", "silver", "SI=F"),
    "XPTUSD": ("CMD_METALS_XPT_CMD_USD", "XPT/USD", "metal", "metal", "PL=F"),
    "XPDUSD": ("CMD_METALS_XPD_CMD_USD", "XPD/USD", "metal", "metal", "PA=F"),
    "COPPER": ("CMD_METALS_COPPER_CMD_USD", "COPPER", "metal", "metal", "HG=F"),
    "US30": ("IDX_AMERICA_E_D_J_IND", "US30", "index", "index", "^DJI"),
    "NAS100": ("IDX_AMERICA_E_NQ_100", "NAS100", "index", "index", "^NDX"),
    "US500": ("IDX_AMERICA_E_SANDP_500", "US500", "index", "index", "^GSPC"),
    "US2000": ("IDX_AMERICA_USSC2000_IDX_USD", "US2000", "index", "index", "^RUT"),
    "DXY": ("IDX_AMERICA_DOLLAR_IDX_USD", "DXY", "index", "index", "DX-Y.NYB"),
    "VIX": ("IDX_AMERICA_VOL_IDX_USD", "VIX", "index", "index", "^VIX"),
    "GER40": ("IDX_EUROPE_E_DAAX", "GER40", "index", "index", "^GDAXI"),
    "UK100": ("IDX_EUROPE_E_FUTSEE_100", "UK100", "index", "index", "^FTSE"),
    "FRA40": ("IDX_EUROPE_E_CAAC_40", "FRA40", "index", "index", "^FCHI"),
    "EU50": ("IDX_EUROPE_E_DJE50XX", "EU50", "index", "index", "^STOXX50E"),
    "ESP35": ("IDX_EUROPE_E_IBC_MAC", "ESP35", "index", "index", "^IBEX"),
    "SWI20": ("IDX_EUROPE_E_SWMI", "SWI20", "index", "index", "^SSMI"),
    "JP225": ("IDX_ASIA_E_N225JAP", "JP225", "index", "index", "^N225"),
    "HK50": ("IDX_ASIA_E_H_KONG", "HK50", "index", "index", "^HSI"),
    "AUS200": ("IDX_ASIA_E_XJO_ASX", "AUS200", "index", "index", "^AXJO"),
    "CHINA50": ("IDX_ASIA_CHI_IDX_USD", "CHINA50", "index", "index", None),
    "USOIL": ("CMD_ENERGY_E_LIGHT", "USOIL", "energy", "energy", "CL=F"),
    "UKOIL": ("CMD_ENERGY_E_BRENT", "UKOIL", "energy", "energy", "BZ=F"),
    "NATGAS": ("CMD_ENERGY_GAS_CMD_USD", "NATGAS", "energy", "energy", "NG=F"),
    "COCOA": ("CMD_AGRICULTURAL_COCOA_CMD_USD", "COCOA", "commodity", "commodity", "CC=F"),
    "COFFEE": ("CMD_AGRICULTURAL_COFFEE_CMD_USX", "COFFEE", "commodity", "commodity", "KC=F"),
    "COTTON": ("CMD_AGRICULTURAL_COTTON_CMD_USX", "COTTON", "commodity", "commodity", "CT=F"),
    "SOYBEAN": ("CMD_AGRICULTURAL_SOYBEAN_CMD_USX", "SOYBEAN", "commodity", "commodity", "ZS=F"),
    "SUGAR": ("CMD_AGRICULTURAL_SUGAR_CMD_USD", "SUGAR", "commodity", "commodity", None),
}
_QUOTE = {'GER40': 'EUR', 'UK100': 'GBP', 'FRA40': 'EUR', 'EU50': 'EUR', 'ESP35': 'EUR', 'SWI20': 'CHF', 'JP225': 'JPY', 'HK50': 'HKD', 'AUS200': 'AUD', 'COFFEE': 'USX', 'COTTON': 'USX', 'SOYBEAN': 'USX'}       # price currency where it isn't USD (USX = US cents)
_ALIASES = {"GOLD": "XAUUSD", "SILVER": "XAGUSD", "PLATINUM": "XPTUSD", "PALLADIUM": "XPDUSD", "XCUUSD": "COPPER",
            "HG": "COPPER", "DJ30": "US30", "DJI": "US30", "DOW": "US30", "WS30": "US30", "US30CASH": "US30",
            "DJIA": "US30", "YM": "US30", "US100": "NAS100", "USTEC": "NAS100", "NDX": "NAS100", "NASDAQ": "NAS100",
            "NAS": "NAS100", "NQ": "NAS100", "NASDAQ100": "NAS100", "USTECH": "NAS100", "US100CASH": "NAS100",
            "SPX500": "US500", "SPX": "US500", "SP500": "US500", "S&P500": "US500", "ES": "US500", "US500CASH": "US500",
            "RUSSELL2000": "US2000", "RUT": "US2000", "RTY": "US2000", "US2K": "US2000", "USDX": "DXY",
            "USDINDEX": "DXY", "DOLLARINDEX": "DXY", "DE40": "GER40", "DAX": "GER40", "GER30": "GER40", "DE30": "GER40",
            "DAX40": "GER40", "FTSE": "UK100", "FTSE100": "UK100", "CAC40": "FRA40", "CAC": "FRA40", "F40": "FRA40",
            "STOXX50": "EU50", "EUSTX50": "EU50", "SX5E": "EU50", "ESTX50": "EU50", "SPA35": "ESP35", "IBEX": "ESP35",
            "IBEX35": "ESP35", "SMI": "SWI20", "CH20": "SWI20", "JPN225": "JP225", "NIKKEI": "JP225", "N225": "JP225",
            "NIKKEI225": "JP225", "HSI": "HK50", "HANGSENG": "HK50", "HK33": "HK50", "ASX200": "AUS200",
            "AU200": "AUS200", "CN50": "CHINA50", "CHI50": "CHINA50", "WTI": "USOIL", "XTIUSD": "USOIL", "CL": "USOIL",
            "OIL": "USOIL", "CRUDE": "USOIL", "WTIUSD": "USOIL", "USOUSD": "USOIL", "BRENT": "UKOIL",
            "XBRUSD": "UKOIL", "BRN": "UKOIL", "UKOUSD": "UKOIL", "XNGUSD": "NATGAS", "NGAS": "NATGAS", "NG": "NATGAS",
            "GAS": "NATGAS"}
_SOURCES = ("dukascopy", "binance", "binanceus", "kucoin", "coinbase", "yahoo")
_PREFIX_SOURCE = {"DUKASCOPY": "dukascopy", "DUKA": "dukascopy", "BINANCE": "binance", "BINANCEUS": "binanceus",
                  "KUCOIN": "kucoin", "COINBASE": "coinbase", "YAHOO": "yahoo", "YF": "yahoo"}
_PREFIX_CRYPTO = {"BYBIT", "OKX", "BITGET", "MEXC", "GATEIO", "BINGX", "KRAKEN", "BITSTAMP", "HTX", "CRYPTO", "CRYPTOCAP"}
_PREFIX_STOCK = {"NASDAQ", "NYSE", "AMEX", "BATS", "ARCA"}


def parse_symbol(symbol):
    """Work out what a symbol is and where its candles can come from.
    Returns dict(display, key, asset_class, sub, base, quote, candidates=[(source, native), ...], forced)."""
    raw = str(symbol).strip()
    if not raw:
        raise ValueError("Empty symbol")
    prefix = None
    if ":" in raw:
        prefix, raw = raw.split(":", 1)
        prefix = prefix.strip().upper()
    forced = _PREFIX_SOURCE.get(prefix)
    up = raw.strip().upper().replace(" ", "")
    if "=" in up or up.startswith("^"):                       # a Yahoo ticker like EURUSD=X, GC=F, ^GSPC
        return dict(display=up, key=up, asset_class="other", sub="other", base=up, quote="USD",
                    candidates=[("yahoo", up)], forced="yahoo")
    up = re.sub(r"([._\-]?PERP|\.P)$", "", up)                 # BYBIT:BTCUSDT.P / BTC-PERP style perpetual names
    clean = re.sub(r"[/\-_.]", "", up)
    clean = _ALIASES.get(clean, clean)

    def out(display, cls, sub, base, quote, cands):
        if forced:
            cands = [c for c in cands if c[0] == forced] or [(forced, cands[0][1] if cands else clean)]
        return dict(display=display, key=f"{display}", asset_class=cls, sub=sub, base=base, quote=quote,
                    candidates=cands, forced=forced)

    if clean in _DUKA and prefix not in _PREFIX_CRYPTO:
        attr, disp, cls, sub, yh = _DUKA[clean]
        cands = td_candidates(clean, cls, clean, "USD") + [("dukascopy", attr)] + ([("yahoo", yh)] if yh else [])
        return out(disp, cls, sub, clean, _QUOTE.get(clean, "USD"), cands)
    if len(clean) == 6 and clean[:3] in FIAT and clean[3:] in FIAT and prefix not in _PREFIX_CRYPTO:
        a, b = clean[:3], clean[3:]
        sub = "fx_major" if clean in FX_MAJORS else ("fx_cross" if a in G10 and b in G10 else "fx_exotic")
        cands = td_candidates(clean, "fx", a, b) + [("dukascopy", f"FX_MAJORS_{a}_{b}|FX_CROSSES_{a}_{b}"),
                                                    ("yahoo", f"{clean}=X")]
        return out(f"{a}/{b}", "fx", sub, a, b, cands)
    base = quote = None
    for q in CRYPTO_QUOTES:
        if clean.endswith(q) and len(clean) > len(q):
            b = clean[: -len(q)]
            if b in KNOWN_CRYPTO or (q in STABLE and b not in FIAT and len(b) >= 2 and prefix not in _PREFIX_STOCK
                                     and (prefix in _PREFIX_CRYPTO or forced in ("binance", "binanceus", "kucoin",
                                                                                 "coinbase") or q != "USD")):
                base, quote = b, q
                break
    if base is None and clean in KNOWN_CRYPTO:
        base, quote = clean, "USDT"
    if base is not None:
        bq = "USDT" if quote == "USD" else quote
        cb_q = "USD" if quote in ("USDT", "USD", "USDC", "FDUSD", "BUSD", "TUSD", "DAI") else quote
        cands = [("binance", f"{base}{bq}"), ("binanceus", f"{base}{bq}"), ("kucoin", f"{base}-{bq}"),
                 ("coinbase", f"{base}-{cb_q}"), ("yahoo", f"{base}-{'USD' if cb_q == 'USD' else cb_q}")]
        if base in DUKA_CRYPTO and quote in STABLE:
            cands.append(("dukascopy", f"VCCY_{DUKA_CRYPTO[base]}_USD"))
            if forced == "dukascopy":
                cands = [("dukascopy", f"VCCY_{DUKA_CRYPTO[base]}_USD")]
        sub = "crypto_major" if base in CRYPTO_MAJORS else "crypto_alt"
        return out(f"{base}/{quote}", "crypto", sub, base, quote, cands)
    if re.fullmatch(r"[A-Z]{1,5}", clean):                     # looks like a US stock ticker
        return out(clean, "stock", "stock", clean, "USD",
                   td_candidates(clean, "stock", clean, "USD") + [("dukascopy", f"US_{clean}_US_USD"), ("yahoo", clean)])
    return out(raw.upper(), "other", "other", clean, "USD", [("yahoo", raw.strip())])


# ───────────────────────────── cleaning / resampling ─────────────────────────────
def clean(df):
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=["open", "high", "low", "close", "volume"], dtype=float,
                            index=pd.DatetimeIndex([], tz="UTC", name="time"))
    df = df.copy()
    df.columns = [str(c).lower() for c in df.columns]
    if "volume" not in df.columns:
        df["volume"] = 0.0
    df = df[["open", "high", "low", "close", "volume"]].apply(pd.to_numeric, errors="coerce").astype(float)
    df.index = to_utc_index(df.index)
    df = df[~df.index.duplicated(keep="last")].sort_index()
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["open", "high", "low", "close"])
    df = df[(df[["open", "high", "low", "close"]] > 0).all(axis=1)]
    px = df[["open", "high", "low", "close"]]
    df["high"] = px.max(axis=1)
    df["low"] = px.min(axis=1)
    df["volume"] = df["volume"].fillna(0.0).clip(lower=0.0)
    flat = (df["high"] <= df["low"]) & (df["volume"] <= 0)        # holiday filler candles
    return df[~flat]


def _agg(df, key):
    g = df.groupby(key)
    out = pd.DataFrame({"open": g["open"].first(), "high": g["high"].max(), "low": g["low"].min(),
                        "close": g["close"].last(), "volume": g["volume"].sum(), "_n": g["close"].count()})
    out.index = to_utc_index(out.index)
    return out


def trading_days(df):
    """Daily candles for markets that close at weekends: a Saturday/Sunday stub is merged into Monday,
    as a broker's daily chart does."""
    if len(df) == 0:
        return df
    d = df.index.floor("1D")
    shift = np.where(d.dayofweek == 5, 2, np.where(d.dayofweek == 6, 1, 0))
    key = d + pd.to_timedelta(shift, unit="D")
    return _agg(df, key).drop(columns="_n")


def resample(df, tf, base_tf=None):
    """Candles of `tf` from smaller candles. Bins are UTC-aligned: 4h at 00/04/08..., days at 00:00,
    3D aligned to the epoch (like exchanges), weeks start Monday, months on the 1st."""
    if len(df) == 0:
        return df
    idx = df.index
    if tf in INTRADAY:
        key = idx.floor(f"{TF_MIN[tf]}min")
    elif tf == "1D":
        key = idx.floor("1D")
    elif tf == "3D":
        key = idx.floor("3D")
    elif tf == "1W":
        d = idx.floor("1D")
        key = d - pd.to_timedelta(d.dayofweek, unit="D")
    elif tf == "1M":
        d = idx.floor("1D")
        key = d - pd.to_timedelta(d.day - 1, unit="D")
    else:
        raise ValueError(tf)
    out = _agg(df, key)
    if base_tf in INTRADAY and tf in INTRADAY:                      # drop stub bins (e.g. FX Sunday open)
        expected = TF_MIN[tf] / TF_MIN[base_tf]
        out = out[out["_n"] >= max(1.0, 0.25 * expected)]
    return out.drop(columns="_n")


def split_forming(df, tf, now=None):
    """(closed candles, forming candle or None)."""
    if len(df) == 0:
        return df, None
    now = now or utcnow()
    end = close_times(df.index[-1:], tf)[0]
    if end > now:
        return df.iloc[:-1], df.iloc[-1]
    return df, None


def heuristic_decimals(price):
    """About six significant digits, at least two decimals (EURUSD 5, USDJPY 3, BTC 2, PEPE 10)."""
    p = abs(float(price))
    return int(min(10, max(2, 5 - math.floor(math.log10(p))))) if p > 0 else 2


def infer_decimals(prices):
    """Decimals the feed actually quotes (EURUSD 1.08432 → 5), capped near the heuristic for noisy floats."""
    x = np.asarray(prices, dtype=float)[-300:]
    x = x[np.isfinite(x) & (x > 0)]
    if len(x) == 0:
        return None
    heur = heuristic_decimals(np.median(x))
    for d in range(0, 11):
        s = x * 10.0 ** d
        if np.all(np.abs(s - np.round(s)) < 1e-6):
            return int(max(min(d, heur + 2), 2))
    return None                                                      # feed has float noise: caller decides


def price_decimals(prices, asset_class=None, quote=None):
    d = infer_decimals(prices)
    if d is not None:
        return d
    if asset_class == "fx":
        return 3 if quote in ("JPY", "HUF", "THB", "RUB", "CZK") else 5
    x = np.asarray(prices, dtype=float)
    x = x[np.isfinite(x) & (x > 0)]
    return heuristic_decimals(np.median(x)) if len(x) else 2


# ───────────────────────────── HTTP ─────────────────────────────
UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0 Safari/537.36"
_HTTP = {"session": None}


def set_http(session):
    """Swap the HTTP client (used by tests)."""
    _HTTP["session"] = session


def _session():
    if _HTTP["session"] is None:
        if requests is None:
            raise DataError("The 'requests' package is not installed")
        _HTTP["session"] = requests.Session()
    return _HTTP["session"]


def _get_json(url, params=None, timeout=20, tries=3):
    last = None
    for k in range(tries):
        try:
            r = _session().get(url, params=params, headers={"User-Agent": UA, "Accept": "application/json"},
                               timeout=timeout)
        except Exception as e:                                       # DNS / firewall / timeout
            last = e
            time.sleep(0.5 * (k + 1))
            continue
        code = int(getattr(r, "status_code", 200))
        if code in (451, 403):
            raise SourceBlocked(f"HTTP {code}: refused from this server's location")
        if code in (429, 418):
            last = DataError(f"HTTP {code}: rate limited")
            time.sleep(2.0 * (k + 1))
            continue
        if code >= 500:
            last = DataError(f"HTTP {code}")
            time.sleep(1.0)
            continue
        if code in (400, 404):
            raise SymbolNotFound(f"HTTP {code}: {str(getattr(r, 'text', ''))[:160]}")
        try:
            return r.json()
        except Exception as e:
            last = DataError(f"bad JSON: {e}")
            continue
    if isinstance(last, DataError):
        raise last
    raise SourceBlocked(f"unreachable: {last}")


def _ms(ts):
    return int(pd.Timestamp(ts).timestamp() * 1000)


# ───────────────────────────── sources ─────────────────────────────
BINANCE_IV = {"1m": "1m", "5m": "5m", "15m": "15m", "30m": "30m", "1h": "1h", "4h": "4h", "1D": "1d", "3D": "3d",
              "1W": "1w", "1M": "1M"}
KUCOIN_IV = {"1m": "1min", "5m": "5min", "15m": "15min", "30m": "30min", "1h": "1hour", "4h": "4hour",
             "1D": "1day", "1W": "1week"}
COINBASE_GRAN = {"1m": 60, "5m": 300, "15m": 900, "1h": 3600, "1D": 86400}
YAHOO_IV = {"1m": ("1m", "7d"), "5m": ("5m", "60d"), "15m": ("15m", "60d"), "30m": ("30m", "60d"),
            "1h": ("1h", "730d"), "1D": ("1d", None), "1W": ("1wk", None), "1M": ("1mo", None)}
DUKA_IV = {"1m": "INTERVAL_MIN_1", "5m": "INTERVAL_MIN_5", "15m": "INTERVAL_MIN_15", "30m": "INTERVAL_MIN_30",
           "1h": "INTERVAL_HOUR_1", "4h": "INTERVAL_HOUR_4", "1D": "INTERVAL_DAY_1"}
BINANCE_URL = {"binance": "https://api.binance.com", "binanceus": "https://api.binance.us"}

# which candle size each source downloads to build a timeframe it doesn't offer itself
NATIVE = {
    "dukascopy": {"1m": "1m", "5m": "5m", "15m": "15m", "30m": "30m", "1h": "1h", "4h": "4h", "1D": "1D",
                  "3D": "1D", "1W": "1D", "1M": "1D"},
    "binance": {t: t for t in TFS},
    "binanceus": {t: t for t in TFS},
    "kucoin": {"1m": "1m", "5m": "5m", "15m": "15m", "30m": "30m", "1h": "1h", "4h": "4h", "1D": "1D",
               "3D": "1D", "1W": "1W", "1M": "1D"},
    "coinbase": {"1m": "1m", "5m": "5m", "15m": "15m", "30m": "15m", "1h": "1h", "4h": "1h", "1D": "1D",
                 "3D": "1D", "1W": "1D", "1M": "1D"},
    "yahoo": {"1m": "1m", "5m": "5m", "15m": "15m", "30m": "30m", "1h": "1h", "4h": "1h", "1D": "1D",
              "3D": "1D", "1W": "1W", "1M": "1M"},
}


def _empty():
    return clean(None)


# ───────────────────────── Twelve Data (paid plan for most metals and indices) ─────────────────────────
# Off unless TWELVEDATA_ENABLED=1 and TWELVEDATA_API_KEY is set. While it is off, nothing changes: the free sources answer.
TD_IV = {"1m": "1min", "5m": "5min", "15m": "15min", "30m": "30min", "1h": "1h", "4h": "4h", "1D": "1day",
         "1W": "1week", "1M": "1month"}
# Twelve Data names for metals and indices. Check each one with a test call. A name Twelve Data rejects falls back to
# the free sources, so the chart keeps working.
TD_SYMBOL = {"XAUUSD": "XAU/USD", "XAGUSD": "XAG/USD", "XPTUSD": "XPT/USD", "XPDUSD": "XPD/USD",
             "US30": "DJI", "NAS100": "NDX", "US500": "SPX", "US2000": "RUT", "VIX": "VIX",
             "UK100": "FTSE", "JP225": "N225", "HK50": "HSI"}
NATIVE["twelvedata"] = {"1m": "1m", "5m": "5m", "15m": "15m", "30m": "30m", "1h": "1h", "4h": "4h", "1D": "1D",
                        "3D": "1D", "1W": "1W", "1M": "1M"}


def twelvedata_on():
    return os.environ.get("TWELVEDATA_ENABLED", "0").strip().lower() in ("1", "true", "yes", "on")


def td_candidates(clean_name, asset_class, base, quote):
    """Twelve Data goes first for forex, metals, indices and stocks, and only when it is switched on."""
    if not twelvedata_on():
        return []
    if asset_class == "fx":
        return [("twelvedata", f"{base}/{quote}")]
    if asset_class == "stock":
        return [("twelvedata", clean_name)]
    if asset_class in ("metal", "index") and clean_name in TD_SYMBOL:
        return [("twelvedata", TD_SYMBOL[clean_name])]
    return []


def fetch_twelvedata(native, tf, start, end):
    """Twelve Data time series. Needs TWELVEDATA_API_KEY. Any error falls through to the next source."""
    key = os.environ.get("TWELVEDATA_API_KEY", "").strip()
    if not key:
        raise DataError("Twelve Data is on but TWELVEDATA_API_KEY is empty")
    s = pd.Timestamp(start).tz_convert("UTC").strftime("%Y-%m-%d %H:%M:%S")
    e = pd.Timestamp(end).tz_convert("UTC").strftime("%Y-%m-%d %H:%M:%S")
    js = _get_json("https://api.twelvedata.com/time_series",
                   dict(symbol=native, interval=TD_IV[tf], start_date=s, end_date=e, outputsize=5000,
                        timezone="UTC", order="ASC", format="JSON", apikey=key))
    if not isinstance(js, dict) or js.get("status") == "error" or "values" not in js:
        msg = js.get("message") if isinstance(js, dict) else js
        raise SymbolNotFound(f"Twelve Data: {str(msg)[:160]}")
    values = js.get("values") or []
    if not values:
        return _empty()
    df = pd.DataFrame(values)
    df.index = pd.to_datetime(df.pop("datetime"), utc=True)
    df = df.rename(columns=str.lower)
    for col in ("open", "high", "low", "close", "volume"):
        if col not in df.columns:
            df[col] = 0.0
    return clean(df[["open", "high", "low", "close", "volume"]].astype(float))


def fetch_dukascopy(native, tf, start, end):
    import dukascopy_python as dk
    from dukascopy_python import instruments as ins
    attr = None
    for name in native.split("|"):
        if hasattr(ins, "INSTRUMENT_" + name):
            attr = "INSTRUMENT_" + name
            break
    if attr is None:
        raise SymbolNotFound(f"Dukascopy has no instrument {native.split('|')[0]}")
    interval = getattr(dk, DUKA_IV[tf])
    s = pd.Timestamp(start).tz_convert("UTC").to_pydatetime()
    e = pd.Timestamp(end).tz_convert("UTC").to_pydatetime()
    df = dk.fetch(getattr(ins, attr), interval, dk.OFFER_SIDE_BID, s, e, max_retries=3)
    if df is None or len(df) == 0:
        return _empty()
    df.columns = [str(c).lower() for c in df.columns]
    return clean(df)


def fetch_binance(native, tf, start, end, source="binance"):
    base, rows = BINANCE_URL[source], []
    cur, end_ms = _ms(start), _ms(end)
    while cur <= end_ms:
        data = _get_json(f"{base}/api/v3/klines", dict(symbol=native, interval=BINANCE_IV[tf], startTime=cur,
                                                        endTime=end_ms, limit=1000))
        if isinstance(data, dict):
            raise SymbolNotFound(str(data)[:160])
        if not data:
            break
        rows.extend(data)
        nxt = int(data[-1][0]) + 1
        if len(data) < 1000 or nxt <= cur:
            break
        cur = nxt
    if not rows:
        return _empty()
    df = pd.DataFrame([r[:6] for r in rows], columns=["t", "open", "high", "low", "close", "volume"])
    df.index = pd.to_datetime(df.pop("t").astype("int64"), unit="ms", utc=True)
    return clean(df)


def fetch_kucoin(native, tf, start, end):
    rows, start_s, end_s = [], int(_ms(start) // 1000), int(_ms(end) // 1000)
    while end_s > start_s:
        js = _get_json("https://api.kucoin.com/api/v1/market/candles",
                       dict(type=KUCOIN_IV[tf], symbol=native, startAt=start_s, endAt=end_s))
        if str(js.get("code")) != "200000":
            raise SymbolNotFound(f"KuCoin: {js.get('msg', js.get('code'))}")
        data = js.get("data") or []
        if not data:
            break
        rows.extend(data)
        oldest = min(int(r[0]) for r in data)
        if oldest <= start_s or oldest - 1 >= end_s:
            break
        end_s = oldest - 1
    if not rows:
        return _empty()
    df = pd.DataFrame([[int(r[0]), r[1], r[3], r[4], r[2], r[5]] for r in rows],      # t, open, high, low, close, vol
                      columns=["t", "open", "high", "low", "close", "volume"])
    df.index = pd.to_datetime(df.pop("t"), unit="s", utc=True)
    return clean(df)


def fetch_coinbase(native, tf, start, end):
    gran = COINBASE_GRAN[tf]
    rows, cur, end_ts = [], pd.Timestamp(start), pd.Timestamp(end)
    step = pd.Timedelta(seconds=gran * 300)
    while cur < end_ts:
        nxt = min(cur + step, end_ts)
        data = _get_json(f"https://api.exchange.coinbase.com/products/{native}/candles",
                         dict(granularity=gran, start=cur.strftime("%Y-%m-%dT%H:%M:%SZ"),
                              end=nxt.strftime("%Y-%m-%dT%H:%M:%SZ")))
        if isinstance(data, dict):
            raise SymbolNotFound(f"Coinbase: {data.get('message', data)}")
        rows.extend(data or [])
        cur = nxt
        time.sleep(0.12)                                             # public limit: ~10 requests/second
    if not rows:
        return _empty()
    df = pd.DataFrame([[int(r[0]), r[3], r[2], r[1], r[4], r[5]] for r in rows],      # t, open, high, low, close, vol
                      columns=["t", "open", "high", "low", "close", "volume"])
    df.index = pd.to_datetime(df.pop("t"), unit="s", utc=True)
    return clean(df)


def fetch_yahoo(native, tf, start, end):
    import yfinance as yf
    iv, period = YAHOO_IV[tf]
    t = yf.Ticker(native)
    if period is None:
        h = t.history(start=pd.Timestamp(start).strftime("%Y-%m-%d"),
                      end=(pd.Timestamp(end) + pd.Timedelta(days=2)).strftime("%Y-%m-%d"), interval=iv,
                      auto_adjust=False)
    else:
        h = t.history(period=period, interval=iv, auto_adjust=False, prepost=False)
    if h is None or len(h) == 0:
        raise SymbolNotFound(f"Yahoo has no data for {native}")
    if isinstance(h.columns, pd.MultiIndex):
        h.columns = h.columns.get_level_values(0)
    h = h.rename(columns=str.lower)
    idx = pd.DatetimeIndex(h.index)
    if tf in ("1D", "1W", "1M"):                                     # daily+ bars: keep the calendar date
        idx = (idx.tz_localize(None) if idx.tz is not None else idx).normalize()
    h.index = to_utc_index(idx)
    return clean(h)


FETCH = {"dukascopy": fetch_dukascopy, "binance": lambda n, t, s, e: fetch_binance(n, t, s, e, "binance"),
         "binanceus": lambda n, t, s, e: fetch_binance(n, t, s, e, "binanceus"), "kucoin": fetch_kucoin,
         "coinbase": fetch_coinbase, "yahoo": fetch_yahoo, "twelvedata": fetch_twelvedata}


# ───────────────────────────── cache ─────────────────────────────
_STATE = {"dir": None}
_MEM, _CHECKED, _WORKING, _BLOCKED = {}, {}, {}, {}


def set_cache_dir(path):
    """Where candles are cached (e.g. a folder on Google Drive). None = memory only."""
    _STATE["dir"] = path
    if path:
        os.makedirs(path, exist_ok=True)


def reset_sources():
    """Forget which sources were blocked or chosen (e.g. after moving the notebook to another runtime)."""
    _BLOCKED.clear()
    _WORKING.clear()


def _cache_path(source, native, tf):
    if not _STATE["dir"]:
        return None
    safe = re.sub(r"[^A-Za-z0-9]+", "_", native).strip("_")
    return os.path.join(_STATE["dir"], source, f"{safe}__{tf}.csv.gz")


def _read_cache(path):
    if not path or not os.path.exists(path):
        return None
    try:
        df = pd.read_csv(path, index_col=0)
        df.index = to_utc_index(pd.to_datetime(df.index, utc=True))
        return clean(df)
    except Exception:
        return None


def _write_cache(path, df):
    if not path:
        return
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp.csv.gz"
    df.to_csv(tmp)
    os.replace(tmp, path)


def _meta_path(path):
    return path.replace(".csv.gz", ".meta.json") if path else None


def _series(source, native, tf, start, end, refresh_s):
    """Candles of `tf` (a timeframe this source offers) covering start..end, from memory/disk cache + top-up."""
    key = (source, native, tf)
    now = utcnow()
    path = _cache_path(source, native, tf)
    df, fetched_at = None, None
    if key in _MEM:
        df, fetched_at = _MEM[key]
    elif path:
        df = _read_cache(path)
        mp = _meta_path(path)
        if df is not None and mp and os.path.exists(mp):
            try:
                _CHECKED[key] = pd.Timestamp(json.load(open(mp))["checked_from"])
            except Exception:
                pass
    step = tf_delta(tf) or pd.Timedelta(days=31)
    checked = _CHECKED.get(key)
    have_start = df is not None and len(df) > 0 and (df.index[0] <= start + 3 * step or
                                                       (checked is not None and checked <= start + 3 * step))
    if have_start:
        if fetched_at is not None and (now - fetched_at).total_seconds() < refresh_s:
            return df
        new = FETCH[source](native, tf, df.index[-1] - 3 * step, end)
        df = clean(pd.concat([df, new]))
    else:
        if df is not None and len(df):                                 # extend backwards, then top up
            older = FETCH[source](native, tf, start, df.index[0] + step)
            newer = FETCH[source](native, tf, df.index[-1] - 3 * step, end)
            df = clean(pd.concat([older, df, newer]))
        else:
            df = clean(FETCH[source](native, tf, start, end))
        _CHECKED[key] = min(start, _CHECKED.get(key, start))
        if path:
            os.makedirs(os.path.dirname(path), exist_ok=True)
            json.dump({"checked_from": str(_CHECKED[key])}, open(_meta_path(path), "w"))
    _MEM[key] = (df, now)
    if path and len(df):
        _write_cache(path, df)
    return df


def start_for(tf, bars, asset_class, end=None):
    """How far back to ask for `bars` candles, allowing for weekends/holidays (or exchange hours for stocks)."""
    end = end or utcnow()
    m = TF_MIN[tf]
    if asset_class == "crypto" or tf in ("1W", "1M"):
        minutes = bars * m * 1.03 + 3 * m
    elif asset_class == "stock" and tf in INTRADAY:
        minutes = bars * m * 5.5 + 4 * 1440
    elif tf in INTRADAY:
        minutes = bars * m * 1.45 + 4 * 1440                       # + a weekend and a holiday
    else:
        minutes = bars * m * 1.48 + 10 * 1440
    return end - pd.Timedelta(minutes=minutes)


def daily_align(df):
    """Label each daily candle with its trading date, whichever convention the feed uses: a day that opens at
    00:00 UTC keeps its date, a forex day that opens at 22:00 UTC (New York close) belongs to the next date."""
    if len(df) == 0:
        return df
    return _agg(df, (df.index + pd.Timedelta(hours=12)).floor("1D")).drop(columns="_n")


def _transform(df, tf, base_tf, asset_class):
    if base_tf == "1D":
        df = daily_align(df)
    if asset_class != "crypto" and base_tf == "1D":
        df = trading_days(df)
    if tf != base_tf:
        df = resample(df, tf, base_tf)
    return df


def get_candles(symbol, tf="1h", bars=1000, start=None, end=None, refresh_s=None, min_bars=30, verbose=False):
    """Closed candles + the forming candle for any symbol/timeframe.
    Returns dict(symbol, tf, source, native, asset_class, sub, base, quote, df, forming, price, decimals,
    last_close_time, age_minutes)."""
    tf = norm_tf(tf)
    spec = parse_symbol(symbol)
    now = utcnow()
    pinned = end is not None                     # a moment in the past (a replay): nothing later may be used
    if pinned:
        end = pd.Timestamp(end)
        end = end.tz_localize("UTC") if end.tzinfo is None else end.tz_convert("UTC")
    else:
        end = now
    start = pd.Timestamp(start).tz_convert("UTC") if start is not None else start_for(tf, bars, spec["asset_class"], end)
    refresh_s = refresh_s if refresh_s is not None else max(15, min(300, TF_MIN[tf] * 6))
    cands = list(spec["candidates"])
    if spec["key"] in _WORKING and _WORKING[spec["key"]] in cands:
        cands.remove(_WORKING[spec["key"]])
        cands.insert(0, _WORKING[spec["key"]])
    errors, df, used = [], None, None
    for source, native in cands:
        blocked = _BLOCKED.get(source)
        if blocked and not spec["forced"]:
            msg, until = blocked
            if until is None or now < until:
                errors.append(f"{source}: skipped ({msg})")
                continue
            _BLOCKED.pop(source, None)
        try:
            base_tf = NATIVE[source][tf]
            bstart = start
            if base_tf == "1D" and tf in ("3D", "1W", "1M"):
                bstart = start - pd.Timedelta(days=40)
            raw = _series(source, native, base_tf, bstart, end, refresh_s)
            d = _transform(raw, tf, base_tf, spec["asset_class"])
            d = d[d.index >= start - (tf_delta(tf) or pd.Timedelta(days=31))]
            if len(d) < min_bars:
                raise SymbolNotFound(f"only {len(d)} candles")
            df, used = d, (source, native)
            break
        except SourceBlocked as e:
            # a refusal (451/403) lasts the session; a network failure is retried after 10 minutes
            _BLOCKED[source] = (str(e)[:80], None if str(e).startswith("HTTP") else now + pd.Timedelta(minutes=10))
            errors.append(f"{source}: {e}")
        except Exception as e:
            errors.append(f"{source}: {type(e).__name__}: {str(e)[:160]}")
        if verbose:
            print("  ", errors[-1])
    if df is None:
        raise DataError(f"No data for {symbol} {tf}. Tried → " + " | ".join(errors))
    _WORKING[spec["key"]] = used
    closed, forming = split_forming(df, tf, now)
    if pinned:
        # keep only candles that had closed by `end`: a candle still open at `end` holds prices from after it
        closed, forming = closed[close_times(closed.index, tf) <= end], None
        if not len(closed):
            raise DataError(f"No closed candles for {symbol} {tf} before {end:%Y-%m-%d %H:%M} UTC.")
    if bars and len(closed) > bars:
        closed = closed.iloc[-bars:]
    last_close = close_times(closed.index[-1:], tf)[0] if len(closed) else None
    price = float(forming["close"]) if forming is not None else float(closed["close"].iloc[-1])
    return dict(symbol=spec["display"], query=str(symbol), tf=tf, source=used[0], native=used[1],
                asset_class=spec["asset_class"], sub=spec["sub"], base=spec["base"], quote=spec["quote"],
                df=closed, forming=forming, price=price,
                decimals=price_decimals(np.r_[closed["close"].to_numpy()[-300:], price], spec["asset_class"],
                                        spec["quote"]),
                last_close_time=last_close,
                age_minutes=None if last_close is None else round((now - last_close).total_seconds() / 60, 1))


def history(symbol, tf, years=None, start=None, verbose=True):
    """Long history for training (closed candles only)."""
    tf = norm_tf(tf)
    end = utcnow()
    start = pd.Timestamp(start, tz="UTC") if start is not None else end - pd.Timedelta(days=int(365.25 * years))
    c = get_candles(symbol, tf, bars=None, start=start, end=end, refresh_s=3600, min_bars=200)
    if verbose:
        d = c["df"]
        print(f"  {c['symbol']:10s} {tf:>3s}: {len(d):>8,} candles {d.index[0]:%Y-%m-%d} → {d.index[-1]:%Y-%m-%d} "
              f"({c['source']})")
    return c


def seed_cache_from_csv(path, source, native, tf):
    """Import an earlier notebook's cached CSV (time index + OHLCV) into this cache, if not already there."""
    dest = _cache_path(source, native, tf)
    if not dest or os.path.exists(dest) or not os.path.exists(path):
        return False
    df = _read_cache(path)
    if df is None or len(df) < 1000:
        return False
    _write_cache(dest, df)
    json.dump({"checked_from": str(df.index[0])}, open(_meta_path(dest), "w"))
    return True


In [ ]:
%%writefile guru_levels.py
"""guru_levels.py: what an experienced discretionary trader marks on a chart, computed from candles.

Inputs are candles that went through structure_engine.analyze (they carry an `atr` column) and the engine's
analysis dict A (pivots, swings, events, sweeps, liquidity, zones, fvgs, patterns).
Every function returns plain dicts/lists with bar indices (positions in the candle frame) and prices.
"""
import math

import numpy as np
import pandas as pd

KILLZONES = (("Asian session", 20.0, 24.0), ("London killzone", 2.0, 5.0), ("New York AM killzone", 7.0, 10.0),
             ("London close", 10.0, 12.0), ("New York PM session", 13.5, 16.0))
SILVER_BULLETS = (("London silver bullet", 3.0, 4.0), ("NY AM silver bullet", 10.0, 11.0),
                  ("NY PM silver bullet", 14.0, 15.0))


# ───────────────────────────── indicators ─────────────────────────────
def rsi(close, n=14):
    d = close.diff()
    up = d.clip(lower=0).ewm(alpha=1 / n, adjust=False).mean()
    dn = (-d.clip(upper=0)).ewm(alpha=1 / n, adjust=False).mean()
    out = 100 - 100 / (1 + up / dn.replace(0, np.nan))
    return out.fillna(50.0)


def ema(close, n):
    return close.ewm(span=n, adjust=False).mean()


def adx(df, n=14):
    h, l, c = df["high"], df["low"], df["close"]
    up, dn = h.diff(), -l.diff()
    plus = pd.Series(np.where((up > dn) & (up > 0), up, 0.0), index=df.index)
    minus = pd.Series(np.where((dn > up) & (dn > 0), dn, 0.0), index=df.index)
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    atr = tr.ewm(alpha=1 / n, adjust=False).mean().replace(0, np.nan)
    pdi = 100 * plus.ewm(alpha=1 / n, adjust=False).mean() / atr
    mdi = 100 * minus.ewm(alpha=1 / n, adjust=False).mean() / atr
    dx = 100 * (pdi - mdi).abs() / (pdi + mdi).replace(0, np.nan)
    return dx.ewm(alpha=1 / n, adjust=False).mean().fillna(0.0), pdi.fillna(0.0), mdi.fillna(0.0)


def efficiency_ratio(close, n=50):
    move = (close - close.shift(n)).abs()
    path = close.diff().abs().rolling(n, min_periods=n).sum()
    return (move / path.replace(0, np.nan)).fillna(0.0)


def bb_width(close, n=20, k=2.0):
    mid = close.rolling(n, min_periods=n).mean()
    sd = close.rolling(n, min_periods=n).std()
    return (2 * k * sd / mid.replace(0, np.nan))


def pct_rank(s, window=250, min_periods=50):
    return s.rolling(window, min_periods=min_periods).rank(pct=True)


def _atr_ref(df, k=200):
    a = df["atr"].to_numpy()[-k:]
    a = a[np.isfinite(a) & (a > 0)]
    return float(np.median(a)) if len(a) else float((df["high"] - df["low"]).median())


# ───────────────────────────── order blocks ─────────────────────────────
def order_blocks(df, A, keep=3, max_scan=6):
    """Order block = the last opposite-colour candle before the impulse that broke structure.
    status: fresh (untouched) / tested (price came back into it) / breaker (a close through it: role flips)."""
    o, h, l, c = (df[k].to_numpy() for k in ("open", "high", "low", "close"))
    n = len(df)
    found = []
    for e in A["events"]:
        i, j0 = e["idx"], e["from_idx"]
        if i - j0 < 1:
            continue
        bull = e["direction"] == "bullish"
        origin = j0 + int(np.argmin(l[j0:i + 1]) if bull else np.argmax(h[j0:i + 1]))
        k = None
        for j in range(origin, max(origin - max_scan, -1), -1):
            if (c[j] < o[j]) if bull else (c[j] > o[j]):
                k = j
                break
        k = origin if k is None else k
        top, bot = float(h[k]), float(l[k])
        t_idx = b_idx = None
        if i + 1 < n:
            if bull:
                hit, brk = np.flatnonzero(l[i + 1:] <= top), np.flatnonzero(c[i + 1:] < bot)
            else:
                hit, brk = np.flatnonzero(h[i + 1:] >= bot), np.flatnonzero(c[i + 1:] > top)
            b_idx = i + 1 + int(brk[0]) if len(brk) else None
            t_idx = i + 1 + int(hit[0]) if len(hit) else None
        status = "breaker" if b_idx is not None else ("tested" if t_idx is not None else "fresh")
        if status == "breaker":
            role = "supply" if bull else "demand"            # a failed demand block becomes resistance
        else:
            role = "demand" if bull else "supply"
        found.append(dict(direction="bullish" if bull else "bearish", role=role, idx=int(k), event_idx=int(i),
                          event_kind=e["kind"], top=top, bottom=bot, status=status, touched_idx=t_idx,
                          broken_idx=b_idx))
    price = float(c[-1])
    uniq = {}
    for ob in found:                                          # one event can't produce two identical blocks
        uniq[(ob["idx"], ob["role"], ob["status"] == "breaker")] = ob
    found = sorted(uniq.values(), key=lambda b: b["event_idx"])
    demand = [b for b in found if b["role"] == "demand" and b["status"] != "breaker" and b["bottom"] < price]
    supply = [b for b in found if b["role"] == "supply" and b["status"] != "breaker" and b["top"] > price]
    breakers = [b for b in found if b["status"] == "breaker" and b["broken_idx"] is not None
                and n - 1 - b["broken_idx"] <= 200]
    pick = lambda xs: sorted(xs, key=lambda b: (b["status"] != "fresh", -b["event_idx"]))[:keep]
    return pick(demand) + pick(supply) + sorted(breakers, key=lambda b: -b["broken_idx"])[:2]


# ───────────────────────────── ranges, legs, fibs ─────────────────────────────
def dealing_range(df, A):
    """Current dealing range: last confirmed swing low → last swing high (stretched by any price beyond them).
    Above 50% = premium (sell zone for SMC traders), below = discount (buy zone). OTE = 62–79% retracement."""
    h, l, c = df["high"].to_numpy(), df["low"].to_numpy(), float(df["close"].iloc[-1])
    sw = A["swings"]
    highs = [s for s in sw if s["type"] == "high"]
    lows = [s for s in sw if s["type"] == "low"]
    if not highs or not lows:
        return None
    hs, ls = highs[-1], lows[-1]
    ti = hs["idx"] + int(np.argmax(h[hs["idx"]:]))
    bi = ls["idx"] + int(np.argmin(l[ls["idx"]:]))
    top, bot = max(hs["price"], float(h[ti])), min(ls["price"], float(l[bi]))
    if top <= bot:
        return None
    rng = top - bot
    trend = int(A["trend"][-1]) if len(A["trend"]) else 0
    leg = "up" if trend == 1 or (trend == 0 and bi < ti) else "down"
    pos = (c - bot) / rng
    zone = "equilibrium" if abs(pos - 0.5) < 0.05 else ("premium" if pos > 0.5 else "discount")
    return dict(top=float(top), bottom=float(bot), top_idx=int(ti), bottom_idx=int(bi), eq=float(bot + rng / 2),
                pos=float(pos), zone=zone, leg=leg,
                ote_long=(float(top - 0.79 * rng), float(top - 0.62 * rng)),
                ote_short=(float(bot + 0.62 * rng), float(bot + 0.79 * rng)))


def impulse_leg(df, A):
    """The latest impulse in the direction of structure: from the origin of the last break to its extreme."""
    h, l = df["high"].to_numpy(), df["low"].to_numpy()
    n = len(df)
    trend = int(A["trend"][-1]) if len(A["trend"]) else 0
    if trend == 0 or not A["events"]:
        return None
    want = "bullish" if trend == 1 else "bearish"
    evs = [e for e in A["events"] if e["direction"] == want]
    if not evs:
        return None
    e = evs[-1]
    j0, i = e["from_idx"], e["idx"]
    if trend == 1:
        oi = j0 + int(np.argmin(l[j0:i + 1]))
        xi = oi + int(np.argmax(h[oi:]))
        origin, ext = float(l[oi]), float(h[xi])
    else:
        oi = j0 + int(np.argmax(h[j0:i + 1]))
        xi = oi + int(np.argmin(l[oi:]))
        origin, ext = float(h[oi]), float(l[xi])
    L = ext - origin                       # signed: + for bullish legs
    if abs(L) <= 0:
        return None
    retr = {r: ext - r * L for r in (0.236, 0.382, 0.5, 0.618, 0.705, 0.786)}
    exts = {x: origin + x * L for x in (1.272, 1.618, 2.0)}
    return dict(direction=1 if trend == 1 else -1, origin=origin, origin_idx=int(oi), extreme=ext,
                extreme_idx=int(xi), event_idx=int(i), event_kind=e["kind"], length=float(abs(L)),
                retracements={str(k): float(v) for k, v in retr.items()},
                extensions={str(k): float(v) for k, v in exts.items()}, bars_old=int(n - 1 - i))


# ───────────────────────────── trendlines & chart patterns ─────────────────────────────
def fit_lines(df, A, kind, tol_atr=0.25, max_pivots=12, min_span=6):
    """Candidate trendlines through two swing pivots that price respected (no close beyond) between them.
    Lines broken more than 10 candles ago are dropped. Sorted best first (touches, then recency)."""
    c = df["close"].to_numpy()
    n = len(df)
    tol = tol_atr * _atr_ref(df)
    typ = "low" if kind == "support" else "high"
    piv = [p for p in A["pivots"] if p["type"] == typ][-max_pivots:]
    lines = []
    for a in range(len(piv)):
        for b in range(a + 1, len(piv)):
            pa, pb = piv[a], piv[b]
            span = pb["idx"] - pa["idx"]
            if span < min_span:
                continue
            slope = (pb["price"] - pa["price"]) / span
            x = np.arange(pa["idx"], n)
            line = pa["price"] + slope * (x - pa["idx"])
            seg = c[pa["idx"]:]
            viol = (seg < line - tol) if kind == "support" else (seg > line + tol)
            if viol[: span + 1].any():
                continue
            after = np.flatnonzero(viol[span + 1:])
            brk = None if len(after) == 0 else pb["idx"] + 1 + int(after[0])
            if brk is not None and n - 1 - brk > 10:
                continue
            end = brk if brk is not None else n - 1
            val = lambda i: pa["price"] + slope * (i - pa["idx"])
            touches = [p["idx"] for p in piv if pa["idx"] <= p["idx"] <= end and abs(p["price"] - val(p["idx"])) <= tol]
            lines.append(dict(kind=kind, i1=int(pa["idx"]), p1=float(pa["price"]), i2=int(n - 1),
                              p2=float(val(n - 1)), slope=float(slope), anchor2=int(pb["idx"]),
                              touches=len(touches), touch_idx=touches, broken_idx=brk,
                              status="broken" if brk is not None else "intact", span=int(end - pa["idx"])))
    lines.sort(key=lambda L: (-L["touches"], -L["anchor2"], -L["span"]))
    return lines


def line_value(L, i):
    return L["p1"] + L["slope"] * (i - L["i1"])


def trendlines(df, A, tol_atr=0.25):
    """Classic trendlines: rising support under higher lows, falling resistance over lower highs (+ channels)."""
    sup = fit_lines(df, A, "support", tol_atr)
    res = fit_lines(df, A, "resistance", tol_atr)
    h, l = df["high"].to_numpy(), df["low"].to_numpy()
    n, tol = len(df), tol_atr * _atr_ref(df)
    out = []
    up = next((L for L in sup if L["slope"] > 0), None)
    dn = next((L for L in res if L["slope"] < 0), None)
    for L, name in ((up, "Uptrend line"), (dn, "Downtrend line")):
        if L is None:
            continue
        L = dict(L, name=name)
        x = np.arange(L["i1"], n)
        base = L["p1"] + L["slope"] * (x - L["i1"])
        if L["kind"] == "support":                     # parallel channel line through the highest point above it
            off = float(np.max(h[L["i1"]:] - base))
            par_t = int(np.sum(np.abs(h[L["i1"]:] - (base + off)) <= tol))
        else:
            off = float(np.min(l[L["i1"]:] - base))
            par_t = int(np.sum(np.abs(l[L["i1"]:] - (base + off)) <= tol))
        if par_t >= 2 and L["status"] == "intact":
            L["channel_offset"] = off
        out.append(L)
    return out, sup, res


def chart_patterns(df, A, sup, res, tol_atr=0.25):
    """Triangles, wedges, channels, ranges, broadening formations (from the best support/resistance lines),
    double tops/bottoms and head & shoulders (from swings). Each has bias, breakout levels and a measured target."""
    c, h, l = df["close"].to_numpy(), df["high"].to_numpy(), df["low"].to_numpy()
    n, atr = len(df), _atr_ref(df)
    price = float(c[-1])
    pats = []
    s = next((L for L in sup if n - 1 - L["i1"] <= 160), None)
    r = next((L for L in res if n - 1 - L["i1"] <= 160), None)
    if s is not None and r is not None:
        start = max(s["i1"], r["i1"])
        span = n - 1 - start
        w0, w1 = line_value(r, start) - line_value(s, start), line_value(r, n - 1) - line_value(s, n - 1)
        if span >= 10 and w0 > 0.8 * atr and w1 > -0.2 * atr:
            s_ch, r_ch = s["slope"] * span / atr, r["slope"] * span / atr
            flat = 0.6
            if w1 < 0.8 * w0:
                if s_ch > flat and r_ch < -flat:
                    name, bias = "Symmetrical triangle", 0
                elif abs(r_ch) <= flat and s_ch > flat:
                    name, bias = "Ascending triangle", 1
                elif abs(s_ch) <= flat and r_ch < -flat:
                    name, bias = "Descending triangle", -1
                elif s_ch > flat and r_ch > flat:
                    name, bias = "Rising wedge", -1
                elif s_ch < -flat and r_ch < -flat:
                    name, bias = "Falling wedge", 1
                else:
                    name, bias = "Contracting range", 0
            elif w1 > 1.25 * w0:
                name, bias = "Broadening formation", 0
            else:
                if s_ch > flat and r_ch > flat:
                    name, bias = "Ascending channel", 1
                elif s_ch < -flat and r_ch < -flat:
                    name, bias = "Descending channel", -1
                else:
                    name, bias = "Range (rectangle)", 0
            up_lvl, dn_lvl = line_value(r, n - 1), line_value(s, n - 1)
            if r["status"] == "broken":
                status, bias_now = "broke up", 1
            elif s["status"] == "broken":
                status, bias_now = "broke down", -1
            else:
                status, bias_now = "forming", bias
            height = w0 if "triangle" in name.lower() or "wedge" in name.lower() else max(w1, 0.0)
            pats.append(dict(name=name, kind="lines", bias=int(bias), bias_now=int(bias_now), status=status,
                             start_idx=int(start), support=s, resistance=r, breakout_up=float(up_lvl),
                             breakout_down=float(dn_lvl), target_up=float(up_lvl + height),
                             target_down=float(dn_lvl - height), height=float(height)))
    sw = A["swings"]
    for typ in ("high", "low"):
        pts = [p for p in sw if p["type"] == typ]
        if len(pts) < 2:
            continue
        a, b = pts[-2], pts[-1]
        if not (abs(a["price"] - b["price"]) <= 0.35 * atr and b["idx"] - a["idx"] >= 5 and n - 1 - b["idx"] <= 80):
            continue
        if typ == "high":
            k = a["idx"] + int(np.argmin(l[a["idx"]:b["idx"] + 1]))
            neck, top = float(l[k]), max(a["price"], b["price"])
            height = top - neck
            if height < 1.2 * atr:
                continue
            after = c[b["idx"] + 1:]
            fail = np.flatnonzero(after > top + 0.25 * atr)
            brk = np.flatnonzero(after < neck)
            status = "failed" if len(fail) and (not len(brk) or fail[0] < brk[0]) else (
                "confirmed" if len(brk) else "forming")
            pats.append(dict(name="Double top", kind="double", bias=-1, bias_now=-1 if status != "failed" else 1,
                             status=status, points=[a["idx"], b["idx"]], prices=[a["price"], b["price"]],
                             neck=neck, neck_idx=int(k), trigger=neck, invalidation=float(top + 0.25 * atr),
                             target_down=float(neck - height), height=float(height),
                             break_idx=None if not len(brk) else int(b["idx"] + 1 + brk[0])))
        else:
            k = a["idx"] + int(np.argmax(h[a["idx"]:b["idx"] + 1]))
            neck, bot = float(h[k]), min(a["price"], b["price"])
            height = neck - bot
            if height < 1.2 * atr:
                continue
            after = c[b["idx"] + 1:]
            fail = np.flatnonzero(after < bot - 0.25 * atr)
            brk = np.flatnonzero(after > neck)
            status = "failed" if len(fail) and (not len(brk) or fail[0] < brk[0]) else (
                "confirmed" if len(brk) else "forming")
            pats.append(dict(name="Double bottom", kind="double", bias=1, bias_now=1 if status != "failed" else -1,
                             status=status, points=[a["idx"], b["idx"]], prices=[a["price"], b["price"]],
                             neck=neck, neck_idx=int(k), trigger=neck, invalidation=float(bot - 0.25 * atr),
                             target_up=float(neck + height), height=float(height),
                             break_idx=None if not len(brk) else int(b["idx"] + 1 + brk[0])))
    for typ in ("high", "low"):                                   # head & shoulders / inverse
        pts = [p for p in sw if p["type"] == typ]
        opp = [p for p in sw if p["type"] != typ]
        if len(pts) < 3:
            continue
        s1, hd, s2 = pts[-3], pts[-2], pts[-1]
        sign = 1 if typ == "high" else -1
        if n - 1 - s2["idx"] > 80:
            continue
        prom = sign * (hd["price"] - (max if sign == 1 else min)(s1["price"], s2["price"]))
        if prom < 0.5 * atr or abs(s1["price"] - s2["price"]) > max(0.6 * atr, 0.35 * abs(hd["price"] - s1["price"])):
            continue
        t1 = [p for p in opp if s1["idx"] < p["idx"] < hd["idx"]]
        t2 = [p for p in opp if hd["idx"] < p["idx"] < s2["idx"]]
        if not t1 or not t2:
            continue
        a, b = t1[-1], t2[-1]
        slope = (b["price"] - a["price"]) / max(1, b["idx"] - a["idx"])
        neck_now = a["price"] + slope * (n - 1 - a["idx"])
        neck_head = a["price"] + slope * (hd["idx"] - a["idx"])
        height = abs(hd["price"] - neck_head)
        after = np.arange(s2["idx"] + 1, n)
        nl = a["price"] + slope * (after - a["idx"])
        brk = np.flatnonzero(c[after] < nl) if sign == 1 else np.flatnonzero(c[after] > nl)
        status = "confirmed" if len(brk) else "forming"
        pats.append(dict(name="Head & shoulders" if sign == 1 else "Inverse head & shoulders", kind="hs",
                         bias=-sign, bias_now=-sign, status=status,
                         points=[s1["idx"], hd["idx"], s2["idx"]], prices=[s1["price"], hd["price"], s2["price"]],
                         neck_points=[a["idx"], b["idx"]], neck_prices=[a["price"], b["price"]],
                         neck_slope=float(slope), trigger=float(neck_now), height=float(height),
                         invalidation=float(hd["price"] + sign * 0.25 * atr),
                         **({"target_down": float(neck_now - height)} if sign == 1 else
                            {"target_up": float(neck_now + height)}),
                         break_idx=None if not len(brk) else int(after[brk[0]])))
    return pats


# ───────────────────────────── momentum, volume ─────────────────────────────
def rsi_divergences(df, A, r, lookback=60):
    """Regular divergence (price makes a new extreme, RSI doesn't) warns of a turn; hidden divergence
    (price holds a higher low / lower high, RSI overshoots) supports trend continuation."""
    n = len(df)
    rv = np.asarray(r, dtype=float)
    out = []
    sw = A["swings"]
    for typ in ("high", "low"):
        pts = [p for p in sw if p["type"] == typ]
        if len(pts) < 2:
            continue
        a, b = pts[-2], pts[-1]
        if n - 1 - b["idx"] > lookback:
            continue
        ra, rb = rv[a["idx"]], rv[b["idx"]]
        kind = None
        if typ == "high":
            if b["price"] > a["price"] and rb < ra - 2:
                kind, bias = "Bearish regular divergence", -1
            elif b["price"] < a["price"] and rb > ra + 2:
                kind, bias = "Bearish hidden divergence", -1
        else:
            if b["price"] < a["price"] and rb > ra + 2:
                kind, bias = "Bullish regular divergence", 1
            elif b["price"] > a["price"] and rb < ra - 2:
                kind, bias = "Bullish hidden divergence", 1
        if kind:
            out.append(dict(name=kind, bias=bias, i1=int(a["idx"]), p1=float(a["price"]), i2=int(b["idx"]),
                            p2=float(b["price"]), rsi1=float(ra), rsi2=float(rb), bars_ago=int(n - 1 - b["idx"])))
    return out


def volume_profile(df, window=300, bins=40):
    """Volume at price over the last `window` candles: POC (most traded price) and the 70% value area."""
    d = df.iloc[-window:]
    v = d["volume"].to_numpy()
    if len(d) < 20 or not np.isfinite(v).all() or v.sum() <= 0:
        return None
    lo, hi = float(d["low"].min()), float(d["high"].max())
    if hi <= lo:
        return None
    edges = np.linspace(lo, hi, bins + 1)
    hist = np.zeros(bins)
    i0 = np.clip(np.searchsorted(edges, d["low"].to_numpy(), "right") - 1, 0, bins - 1)
    i1 = np.clip(np.searchsorted(edges, d["high"].to_numpy(), "left") - 1, 0, bins - 1)
    for a, b, vol in zip(i0, i1, v):
        b = max(a, b)
        hist[a:b + 1] += vol / (b - a + 1)
    poc = int(np.argmax(hist))
    lo_i = hi_i = poc
    total, acc = hist.sum(), hist[poc]
    while acc < 0.7 * total and (lo_i > 0 or hi_i < bins - 1):
        down = hist[lo_i - 1] if lo_i > 0 else -1
        up = hist[hi_i + 1] if hi_i < bins - 1 else -1
        if up >= down:
            hi_i += 1
            acc += hist[hi_i]
        else:
            lo_i -= 1
            acc += hist[lo_i]
    mids = (edges[:-1] + edges[1:]) / 2
    return dict(poc=float(mids[poc]), vah=float(edges[hi_i + 1]), val=float(edges[lo_i]), start_idx=int(len(df) - len(d)),
                bins=[[float(m), float(x)] for m, x in zip(mids, hist)])


def vwap_daily(df):
    """Session VWAP (resets at 00:00 UTC) for intraday charts."""
    tp = (df["high"] + df["low"] + df["close"]) / 3
    v = df["volume"].where(df["volume"] > 0, 1.0)
    day = df.index.floor("1D")
    num = (tp * v).groupby(day).cumsum()
    den = v.groupby(day).cumsum()
    return num / den


# ───────────────────────────── time: sessions, killzones, period levels ─────────────────────────────
def sessions(df, now=None):
    """ICT killzones (New York time) and the latest completed Asian range, for intraday charts."""
    now = now or pd.Timestamp.now(tz="UTC")
    ny = now.tz_convert("America/New_York")
    hh = ny.hour + ny.minute / 60
    current = [k for k, a, b in KILLZONES if a <= hh < b]
    silver = [k for k, a, b in SILVER_BULLETS if a <= hh < b]
    nxt, mins = None, None
    for k, a, b in KILLZONES:
        d = (a - hh) % 24 * 60
        if d > 0 and (mins is None or d < mins):
            nxt, mins = k, d
    out = dict(ny_time=ny.strftime("%H:%M"), ny_weekday=ny.strftime("%A"), active=current, silver_bullet=silver,
               next_killzone=nxt, minutes_to_next=None if mins is None else int(round(mins)))
    if len(df) == 0:
        return out
    idx_ny = df.index.tz_convert("America/New_York")
    midnight = ny.normalize()
    mo = df.index[idx_ny >= midnight]
    out["midnight_open"] = float(df.loc[mo[0], "open"]) if len(mo) else None
    out["midnight_open_idx"] = int(np.flatnonzero(idx_ny >= midnight)[0]) if len(mo) else None
    for back in range(0, 4):
        end = midnight - pd.Timedelta(days=back)
        mask = (idx_ny >= end - pd.Timedelta(hours=4)) & (idx_ny < end)
        if mask.sum() >= 2:
            sel = df[mask]
            hi, lo = float(sel["high"].max()), float(sel["low"].min())
            pos = np.flatnonzero(mask)
            later = df.iloc[pos[-1] + 1:]
            out["asia"] = dict(high=hi, low=lo, i1=int(pos[0]), i2=int(pos[-1]),
                               high_swept=bool(len(later) and later["high"].max() > hi),
                               low_swept=bool(len(later) and later["low"].min() < lo))
            break
    return out


def period_levels(daily, forming=None):
    """Previous day / week / month highs & lows and the current day / week / month opens (from daily candles)."""
    if daily is None or len(daily) < 2:
        return {}
    d = daily
    cur = forming.name if forming is not None else d.index[-1] + pd.Timedelta(days=1)
    out = dict(pdh=float(d["high"].iloc[-1]), pdl=float(d["low"].iloc[-1]), pd_date=str(d.index[-1].date()),
               day_open=float(forming["open"]) if forming is not None else None)
    ws = cur.normalize() - pd.Timedelta(days=int(cur.dayofweek))
    pw = d[(d.index >= ws - pd.Timedelta(days=7)) & (d.index < ws)]
    if len(pw):
        out.update(pwh=float(pw["high"].max()), pwl=float(pw["low"].min()))
    tw = d[d.index >= ws]
    out["week_open"] = float(tw["open"].iloc[0]) if len(tw) else out["day_open"]
    ms = cur.normalize() - pd.Timedelta(days=int(cur.day) - 1)
    pm_start = (ms - pd.Timedelta(days=1)).normalize()
    pm_start = pm_start - pd.Timedelta(days=int(pm_start.day) - 1)
    pm = d[(d.index >= pm_start) & (d.index < ms)]
    if len(pm):
        out.update(pmh=float(pm["high"].max()), pml=float(pm["low"].min()))
    tm = d[d.index >= ms]
    out["month_open"] = float(tm["open"].iloc[0]) if len(tm) else out["day_open"]
    return out


def round_levels(price, ref_atr, n=2):
    """Psychological round numbers: the step is the 1-2-2.5-5 × 10^k size closest to a day's range."""
    if not (ref_atr and ref_atr > 0 and price > 0):
        return []
    k = math.floor(math.log10(ref_atr))
    cands = [m * 10 ** e for e in (k - 1, k, k + 1) for m in (1, 2, 2.5, 5)]
    step = min(cands, key=lambda s: abs(math.log(s / ref_atr)))
    base = math.floor(price / step) * step
    lv = [base + j * step for j in range(-n + 1, n + 1)]
    return [dict(price=float(round(x, 12)), step=float(step)) for x in lv if x > 0]


# ───────────────────────────── regime ─────────────────────────────
def market_phase(df, A):
    """Trend / range / compression / expansion, with the numbers behind it."""
    c = df["close"]
    ax, pdi, mdi = adx(df)
    er = efficiency_ratio(c, 50)
    bw = bb_width(c)
    bw_pct = pct_rank(bw, 250, 50)
    atr = df["atr"]
    atr_ratio = atr / atr.rolling(200, min_periods=50).mean()
    t = int(A["trend"][-1]) if len(A["trend"]) else 0
    a_, e_, b_, r_ = float(ax.iloc[-1]), float(er.iloc[-1]), bw_pct.iloc[-1], atr_ratio.iloc[-1]
    b_ = float(b_) if np.isfinite(b_) else 0.5
    r_ = float(r_) if np.isfinite(r_) else 1.0
    strong = a_ >= 25 or e_ >= 0.35
    if t == 1:
        label = "Strong uptrend" if strong else "Uptrend (choppy)"
    elif t == -1:
        label = "Strong downtrend" if strong else "Downtrend (choppy)"
    else:
        label = "Range / no clear structure"
    notes = []
    if b_ <= 0.12:
        notes.append("volatility squeeze: Bollinger width in its lowest 12% of the last 250 candles; a breakout is"
                     " building, direction unknown")
    if r_ >= 1.6:
        notes.append(f"volatility expansion: ATR is {r_:.1f}× its 200-candle average; stops need room and"
                     " position size should be smaller")
    elif r_ <= 0.7:
        notes.append(f"quiet market: ATR is only {r_:.1f}× its average")
    return dict(label=label, trend=t, adx=a_, er=e_, bbw_pct=b_, atr_ratio=r_, notes=notes,
                di_plus=float(pdi.iloc[-1]), di_minus=float(mdi.iloc[-1]))


In [ ]:
%%writefile guru_model.py
"""guru_model.py: the analyst's experience. Probability models trained on many markets and timeframes.

Everything here is point-in-time: a candle only ever sees what was known when it closed.

Three models (gradient-boosted trees, scikit-learn):
  * race       For a trade entered now, with the target `a` ATR away and the stop `b` ATR away:
               P(target first), P(stop first) within 100 candles. Works for any plan, including the user's own.
  * touch      P(price reaches a level `d` ATR away within the next 24 candles). Gives the probability of every
               level on the chart, the fill odds of limit/stop orders and the expected range.
  * direction  Up / down / neither: which of ±k ATR is hit first within the next 24 candles.
Plus two tables learned from history:
  * setup evidence   How the SMC/ICT setups (trend pullback to OTE, liquidity sweep → MSS → FVG) actually did,
                     against what their geometry alone predicts. This shows whether the setups add any edge.
  * level reactions  When price reaches a swing high/low, equal highs/lows, PDH/PDL or an FVG: how often it
                     breaks through vs rejects, per asset class and timeframe group.
"""
import bisect
import math
import time
import warnings

import numpy as np
import pandas as pd

import guru_levels as gl
import market_data as md
import structure_engine as se

VERSION = "6.1"          # 6.1: adds the calibrated bull/bear bar (range and lean views)
LADDER = {"1m": ("15m", "1h", "4h"), "5m": ("1h", "4h", "1D"), "15m": ("1h", "4h", "1D"),
          "30m": ("4h", "1D", "1W"), "1h": ("4h", "1D", "1W"), "4h": ("1D", "1W", "1M"), "1D": ("1W", "1M"),
          "3D": ("1W", "1M"), "1W": ("1M",), "1M": ()}
H_RACE, H_TOUCH, H_DIR, EXPIRY = 100, 24, 24, 24
B_GRID = np.array([0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0])
RR_GRID = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0])
D_GRID = np.array([0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 6.0, 8.0])
K_GRID = np.round(np.arange(1.0, 6.01, 0.25), 2)
WARMUP = 300

# round-trip cost (spread + fees) and extra slippage when a stop is hit, as fractions of price
COSTS = {"fx_major": (1.2e-4, 0.3e-4), "fx_cross": (2.5e-4, 0.5e-4), "fx_exotic": (15e-4, 3e-4),
         "gold": (2.0e-4, 0.5e-4), "silver": (6e-4, 1.5e-4), "metal": (6e-4, 1.5e-4), "index": (1.5e-4, 0.5e-4),
         "energy": (5e-4, 1e-4), "commodity": (10e-4, 2e-4), "crypto_major": (11e-4, 3e-4),
         "crypto_alt": (16e-4, 6e-4), "stock": (5e-4, 2e-4), "other": (5e-4, 2e-4)}

CLASS_COLS = ["cls_fx", "cls_metal", "cls_index", "cls_energy", "cls_crypto", "cls_stock"]
UNSIGNED = ["tf_log"] + CLASS_COLS + ["atr_pct", "atr_ratio", "vol_pct", "bbw_pct", "er50", "hour_sin", "hour_cos",
                                      "dow_sin", "dow_cos", "vol_z", "since_break", "break_choch"]
SIGNED = ["ret_1", "ret_4", "ret_16", "ret_64", "rsi_c", "ema20_d", "ema50_d", "ema200_d", "ema50_slope", "trend",
          "htf1_trend", "htf2_trend", "break_dir", "range_pos_c", "dealing_pos_c", "body"]
PAIRS = [("dist_bsl", "dist_ssl", "tgt_liq", "stp_liq"),
         ("since_bsl_sweep", "since_ssl_sweep", "since_tgt_sweep", "since_stp_sweep"),
         ("upper_wick", "lower_wick", "wick_against", "wick_with"),
         ("pdh_dist", "pdl_dist", "tgt_pd", "stp_pd")]
RAW_COLS = UNSIGNED + SIGNED + [p[0] for p in PAIRS] + [p[1] for p in PAIRS]
ALIGNED = UNSIGNED + SIGNED + [p[2] for p in PAIRS] + [p[3] for p in PAIRS]
GEO = ["a_atr", "b_atr", "implied", "log_rr"]
RACE_COLS = ALIGNED + GEO
TOUCH_COLS = ALIGNED + ["d_atr"]
DIR_COLS = ALIGNED
GEO_BASE = GEO + ["tf_log"] + CLASS_COLS
TOUCH_BASE = ["d_atr", "tf_log"] + CLASS_COLS
TF_GROUP = {"1m": "scalp", "5m": "scalp", "15m": "intraday", "30m": "intraday", "1h": "intraday", "4h": "swing",
            "1D": "swing", "3D": "position", "1W": "position", "1M": "position"}
LEVEL_TYPES = ("swing_high", "swing_low", "eqh", "eql", "pdh", "pdl", "fvg_bull", "fvg_bear", "random")


def cost_of(sub):
    return COSTS.get(sub, COSTS["other"])


def class_onehot(asset_class):
    return {"cls_fx": float(asset_class == "fx"), "cls_metal": float(asset_class == "metal"),
            "cls_index": float(asset_class == "index"),
            "cls_energy": float(asset_class in ("energy", "commodity")),
            "cls_crypto": float(asset_class == "crypto"), "cls_stock": float(asset_class in ("stock", "other"))}


# ───────────────────────────── point-in-time features ─────────────────────────────
def _structure_state(df, pivots, events, sweeps):
    """Per candle, using only swings confirmed by that candle's close: nearest untaken buy/sell-side liquidity,
    time since the last BOS/CHoCH and since each side was swept, position inside the dealing range."""
    n = len(df)
    h, l, c, a = (df[k].to_numpy(dtype=float) for k in ("high", "low", "close", "atr"))
    by_conf = {}
    for p in pivots:
        by_conf.setdefault(p["confirmed_idx"], []).append(p)
    ev_at = {e["idx"]: e for e in events}
    sw_at = {}
    for s in sweeps:
        sw_at.setdefault(s["idx"], []).append(s["side"])
    out = {k: np.full(n, np.nan) for k in ("dist_bsl", "dist_ssl", "dealing_pos")}
    for k in ("since_break", "break_dir", "break_choch", "since_bsl_sweep", "since_ssl_sweep"):
        out[k] = np.zeros(n)
    highs, lows = [], []
    top = bot = np.nan
    brk_i = bsl_i = ssl_i = -1
    bdir = bch = 0
    for i in range(n):
        for p in by_conf.get(i, ()):
            if p["type"] == "high":
                bisect.insort(highs, p["price"])
                top = p["price"]
            else:
                bisect.insort(lows, p["price"])
                bot = p["price"]
        if top == top:
            top = max(top, h[i])
        if bot == bot:
            bot = min(bot, l[i])
        if highs:
            k = bisect.bisect_left(highs, h[i])
            if k:
                del highs[:k]
            if len(highs) > 200:
                del highs[200:]
        if lows:
            k = bisect.bisect_right(lows, l[i])
            if k < len(lows):
                del lows[k:]
            if len(lows) > 200:
                del lows[:-200]
        e = ev_at.get(i)
        if e is not None:
            brk_i, bdir, bch = i, (1 if e["direction"] == "bullish" else -1), int(e["kind"] == "CHoCH")
        for side in sw_at.get(i, ()):
            if side == "buy-side":
                bsl_i = i
            else:
                ssl_i = i
        ai = a[i] if a[i] > 0 else np.nan
        if highs:
            out["dist_bsl"][i] = min(30.0, (highs[0] - c[i]) / ai)
        if lows:
            out["dist_ssl"][i] = min(30.0, (c[i] - lows[-1]) / ai)
        out["since_break"][i] = min(300, i - brk_i) if brk_i >= 0 else 300
        out["break_dir"][i], out["break_choch"][i] = bdir, bch
        out["since_bsl_sweep"][i] = min(300, i - bsl_i) if bsl_i >= 0 else 300
        out["since_ssl_sweep"][i] = min(300, i - ssl_i) if ssl_i >= 0 else 300
        if top == top and bot == bot and top > bot:
            out["dealing_pos"][i] = (c[i] - bot) / (top - bot)
    return out


def _known_series(values, known_at, query_times):
    """Value of a higher-timeframe series as known at each query time (last value whose candle had closed)."""
    s = pd.Series(np.asarray(values, dtype=float), index=pd.DatetimeIndex(known_at))
    s = s[~s.index.duplicated(keep="last")].sort_index()
    if len(s) == 0:
        return np.full(len(query_times), np.nan)
    pos = s.index.searchsorted(pd.DatetimeIndex(query_times), side="right") - 1
    vals = s.to_numpy()
    return np.where(pos >= 0, vals[np.clip(pos, 0, None)], np.nan)


def htf_trend_on(df, tf, hdf, htf):
    """Structure trend of the higher timeframe as known at the close of each candle of df."""
    if hdf is None or len(hdf) < 30:
        return np.full(len(df), np.nan)
    hd = hdf if "atr" in hdf.columns else se.add_atr(hdf[["open", "high", "low", "close", "volume"]].astype(float).copy())
    piv = se.find_pivots(hd, 3, 3)
    _, _, tr = se.detect_structure_breaks(hd, piv)
    return _known_series(tr, md.close_times(hd.index, htf), md.close_times(df.index, tf))


def daily_from_intraday(df, tf):
    """UTC daily candles built from intraday candles; stub days (< 25% of a normal day) are dropped."""
    key = df.index.floor("1D")
    g = df.groupby(key)
    d = pd.DataFrame({"open": g["open"].first(), "high": g["high"].max(), "low": g["low"].min(),
                      "close": g["close"].last(), "volume": g["volume"].sum(), "n": g["close"].count()})
    d = d[d["n"] >= 0.25 * 1440 / md.TF_MIN[tf]].drop(columns="n")
    d.index = md.to_utc_index(d.index)
    return d


def htf_frames_from(df, tf, asset_class):
    """Higher-timeframe candles built from df itself (training: keeps every timeframe consistent)."""
    out = {}
    for htf in LADDER[tf][:2]:
        try:
            if htf in md.INTRADAY:
                out[htf] = md.resample(df, htf, tf)
            else:
                d = daily_from_intraday(df, tf) if tf in md.INTRADAY else df
                if asset_class != "crypto":
                    d = md.trading_days(d)
                out[htf] = d if htf == "1D" else md.resample(d, htf, "1D")
        except Exception:
            out[htf] = None
    return out


def raw_features(df, tf, asset_class, htf=None, daily=None):
    """Feature frame (one row per candle, long-signed) + context (pivots, events, sweeps, trend, fvgs).
    htf: {timeframe: candles} for the ladder of `tf` (missing ones → NaN). daily: daily candles for PDH/PDL
    on intraday charts (None → built from df)."""
    if "atr" not in df.columns:
        df = se.add_atr(df[["open", "high", "low", "close", "volume"]].astype(float).copy())
    n = len(df)
    pivots = se.find_pivots(df, 3, 3)
    events, sweeps, trend = se.detect_structure_breaks(df, pivots)
    st = _structure_state(df, pivots, events, sweeps)
    cs, op, hi, lo, vol = df["close"], df["open"], df["high"], df["low"], df["volume"]
    atr = df["atr"].where(df["atr"] > 0)
    F = pd.DataFrame(index=df.index)
    F["tf_log"] = math.log(md.TF_MIN[tf])
    for k, val in class_onehot(asset_class).items():
        F[k] = val
    F["atr_pct"] = atr / cs * 100
    F["atr_ratio"] = atr / atr.rolling(200, min_periods=50).mean()
    F["vol_pct"] = (atr / cs).rolling(500, min_periods=100).rank(pct=True)
    F["bbw_pct"] = gl.pct_rank(gl.bb_width(cs), 500, 100)
    F["er50"] = gl.efficiency_ratio(cs, 50)
    idx = df.index
    if md.TF_MIN[tf] < 1440:
        hr = np.asarray(idx.hour + idx.minute / 60.0, dtype=float)
        F["hour_sin"], F["hour_cos"] = np.sin(2 * np.pi * hr / 24), np.cos(2 * np.pi * hr / 24)
    else:
        F["hour_sin"] = F["hour_cos"] = 0.0
    if md.TF_MIN[tf] <= 1440:
        dw = np.asarray(idx.dayofweek, dtype=float)
        F["dow_sin"], F["dow_cos"] = np.sin(2 * np.pi * dw / 7), np.cos(2 * np.pi * dw / 7)
    else:
        F["dow_sin"] = F["dow_cos"] = 0.0
    if float(vol.sum()) > 0:
        vm, vs = vol.rolling(96, min_periods=20).mean(), vol.rolling(96, min_periods=20).std()
        F["vol_z"] = ((vol - vm) / vs.replace(0, np.nan)).clip(-5, 5).fillna(0.0)
    else:
        F["vol_z"] = 0.0
    F["since_break"], F["break_choch"] = st["since_break"], st["break_choch"]
    for k in (1, 4, 16, 64):
        F[f"ret_{k}"] = ((cs - cs.shift(k)) / atr).clip(-30, 30)
    F["rsi_c"] = gl.rsi(cs) - 50.0
    for span in (20, 50, 200):
        e = gl.ema(cs, span)
        F[f"ema{span}_d"] = ((cs - e) / atr).clip(-30, 30)
        if span == 50:
            F["ema50_slope"] = ((e - e.shift(4)) / atr).clip(-10, 10)
    F["trend"] = np.asarray(trend, dtype=float)
    ladder = LADDER[tf]
    htf = htf or {}
    F["htf1_trend"] = htf_trend_on(df, tf, htf.get(ladder[0]), ladder[0]) if len(ladder) > 0 else np.nan
    F["htf2_trend"] = htf_trend_on(df, tf, htf.get(ladder[1]), ladder[1]) if len(ladder) > 1 else np.nan
    F["break_dir"] = st["break_dir"]
    lo96, hi96 = lo.rolling(96, min_periods=10).min(), hi.rolling(96, min_periods=10).max()
    F["range_pos_c"] = (cs - lo96) / (hi96 - lo96).replace(0, np.nan) - 0.5
    F["dealing_pos_c"] = st["dealing_pos"] - 0.5
    rng = (hi - lo).replace(0, np.nan)
    F["body"] = (cs - op) / rng
    F["upper_wick"] = (hi - np.maximum(op, cs)) / rng
    F["lower_wick"] = (np.minimum(op, cs) - lo) / rng
    F["dist_bsl"], F["dist_ssl"] = st["dist_bsl"], st["dist_ssl"]
    F["since_bsl_sweep"], F["since_ssl_sweep"] = st["since_bsl_sweep"], st["since_ssl_sweep"]
    if md.TF_MIN[tf] < 1440:
        d = daily if daily is not None and len(daily) > 2 else daily_from_intraday(df, tf)
        known = md.close_times(d.index, "1D")
        q = md.close_times(df.index, tf)
        pdh, pdl = _known_series(d["high"], known, q), _known_series(d["low"], known, q)
        F["pdh_dist"] = np.clip((pdh - cs.to_numpy()) / atr.to_numpy(), -50, 50)
        F["pdl_dist"] = np.clip((cs.to_numpy() - pdl) / atr.to_numpy(), -50, 50)
    else:
        F["pdh_dist"] = F["pdl_dist"] = np.nan
    F = F[RAW_COLS].replace([np.inf, -np.inf], np.nan)
    ctx = dict(df=df, pivots=pivots, events=events, sweeps=sweeps, trend=np.asarray(trend))
    return F, ctx


def align(F, s):
    """Features from the point of view of a trade/level in direction s (+1 long/up, -1 short/down).
    Signed features flip; paired features swap (e.g. distance to target-side vs stop-side liquidity)."""
    s = np.broadcast_to(np.asarray(s, dtype=float), (len(F),))
    out = {k: F[k].to_numpy(dtype=float) for k in UNSIGNED}
    for k in SIGNED:
        out[k] = F[k].to_numpy(dtype=float) * s
    for a_, b_, na, nb in PAIRS:
        A_, B_ = F[a_].to_numpy(dtype=float), F[b_].to_numpy(dtype=float)
        out[na] = np.where(s > 0, A_, B_)
        out[nb] = np.where(s > 0, B_, A_)
    return pd.DataFrame(out, index=F.index)[ALIGNED]


def flip_aligned(X):
    """Aligned features for direction s → the same features for direction -s."""
    Y = X.copy()
    for k in SIGNED:
        Y[k] = -X[k]
    for _, _, na, nb in PAIRS:
        Y[na], Y[nb] = X[nb], X[na]
    return Y


def align_row(row, sides):
    """One feature row (Series), repeated for several directions."""
    sides = np.atleast_1d(np.asarray(sides, dtype=float))
    F = pd.DataFrame([row[RAW_COLS].to_numpy(dtype=float)] * len(sides), columns=RAW_COLS)
    return align(F, sides).reset_index(drop=True)


# ───────────────────────────── setups (point-in-time) ─────────────────────────────
def find_setups(ctx, cost_frac, min_leg_atr=1.5, sweep_window=30):
    """The two SMC/ICT setups, detected candle by candle with only what was known at each close.
      pullback (OTE): after a BOS/CHoCH, limit at the 61.8% retracement of the impulse leg; stop 0.1 ATR beyond
                      the leg origin.
      sweep (MSS+FVG): a liquidity sweep, then a structure break the other way within 30 candles that leaves an
                      FVG; limit at the FVG edge; stop 0.1 ATR beyond the sweep extreme.
    Target for both: nearest untaken liquidity giving 2–6R, otherwise a fixed 2R."""
    df = ctx["df"]
    n = len(df)
    o, h, l, c, a = (df[k].to_numpy(dtype=float) for k in ("open", "high", "low", "close", "atr"))
    by_conf = {}
    for p in ctx["pivots"]:
        by_conf.setdefault(p["confirmed_idx"], []).append(p)
    ev_at = {}
    for e in ctx["events"]:
        ev_at.setdefault(e["idx"], []).append(e)
    fvgs = ctx.get("fvgs")
    if fvgs is None:
        fvgs = ctx["fvgs"] = se.find_fvgs(df, 0.25)
    fvg_at = {}
    for g in fvgs:
        fvg_at.setdefault(g["idx"], []).append(g)
    sw = {"sell-side": [s for s in ctx["sweeps"] if s["side"] == "sell-side"],
          "buy-side": [s for s in ctx["sweeps"] if s["side"] == "buy-side"]}
    sw_idx = {k: [s["idx"] for s in v] for k, v in sw.items()}
    highs, lows, out = [], [], []

    def target(d, entry, risk):
        lv = highs[bisect.bisect_right(highs, entry):] if d == 1 else lows[:bisect.bisect_left(lows, entry)][::-1]
        for x in lv:
            rr = abs(x - entry) / risk
            if rr >= 2.0:
                return (x, "liquidity") if rr <= 6.0 else (entry + d * 2.0 * risk, "fixed")
        return entry + d * 2.0 * risk, "fixed"

    for i in range(n):
        for p in by_conf.get(i, ()):
            if p["type"] == "high":
                bisect.insort(highs, p["price"])
            else:
                bisect.insort(lows, p["price"])
        if highs:
            k = bisect.bisect_left(highs, h[i])
            if k:
                del highs[:k]
        if lows:
            k = bisect.bisect_right(lows, l[i])
            if k < len(lows):
                del lows[k:]
        for e in ev_at.get(i, ()):
            ai = a[i]
            if not ai > 0:
                continue
            d = 1 if e["direction"] == "bullish" else -1
            j0 = e["from_idx"]
            # ── pullback to OTE
            if d == 1:
                oi = j0 + int(np.argmin(l[j0:i + 1]))
                origin, ext = l[oi], float(np.max(h[oi:i + 1]))
            else:
                oi = j0 + int(np.argmax(h[j0:i + 1]))
                origin, ext = h[oi], float(np.min(l[oi:i + 1]))
            L = abs(ext - origin)
            if L >= min_leg_atr * ai:
                entry = ext - d * 0.618 * L
                sl = origin - d * 0.1 * ai
                risk = d * (entry - sl)
                if d * (c[i] - entry) > 0 and risk >= 0.5 * ai and cost_frac * entry / risk <= 0.15:
                    tp, kind = target(d, entry, risk)
                    out.append(dict(family="pullback", idx=i, dir=d, entry=float(entry), sl=float(sl), tp=float(tp),
                                    risk=float(risk), target=kind, event_kind=e["kind"]))
            # ── liquidity sweep → MSS → FVG
            side = "sell-side" if d == 1 else "buy-side"
            k = bisect.bisect_left(sw_idx[side], i) - 1
            if k < 0 or i - sw_idx[side][k] > sweep_window:
                continue
            s0 = sw_idx[side][k]
            if (np.argmin(l[s0:i + 1]) if d == 1 else np.argmax(h[s0:i + 1])) != 0:
                continue
            gaps = [g for j in range(s0, i) for g in fvg_at.get(j, ())
                    if g["direction"] == e["direction"] and (g["filled_idx"] is None or g["filled_idx"] > i)]
            if not gaps:
                continue
            if d == 1:
                g = max(gaps, key=lambda g: g["top"])
                entry, sl = g["top"], float(np.min(l[s0:i + 1])) - 0.1 * ai
            else:
                g = min(gaps, key=lambda g: g["bottom"])
                entry, sl = g["bottom"], float(np.max(h[s0:i + 1])) + 0.1 * ai
            risk = d * (entry - sl)
            if d * (c[i] - entry) <= 0 or risk < 0.5 * ai or cost_frac * entry / risk > 0.15:
                continue
            tp, kind = target(d, entry, risk)
            out.append(dict(family="sweep", idx=i, dir=d, entry=float(entry), sl=float(sl), tp=float(tp),
                            risk=float(risk), target=kind, event_kind=e["kind"], sweep_idx=int(s0),
                            fvg_top=float(g["top"]), fvg_bottom=float(g["bottom"])))
    return out


def simulate(df, s, cost_frac, slip_frac, expiry=EXPIRY, max_hold=H_RACE):
    """Limit order at s['entry'] valid `expiry` candles (cancelled if the target trades first). After the fill:
    stop wins when stop and target share a candle; only the stop can trigger on the fill candle; gaps through the
    stop fill at the open; closed at market after `max_hold` candles. R is after costs."""
    o, h, l, c = (df[k].to_numpy(dtype=float) for k in ("open", "high", "low", "close"))
    n, d = len(c), s["dir"]
    entry, sl, tp, risk = s["entry"], s["sl"], s["tp"], s["risk"]
    last = min(s["idx"] + expiry, n - 1)
    fill = None
    for j in range(s["idx"] + 1, last + 1):
        if (l[j] <= entry) if d == 1 else (h[j] >= entry):
            fill = j
            break
        if (h[j] >= tp) if d == 1 else (l[j] <= tp):
            return dict(status="missed")
    if fill is None:
        return dict(status="pending" if s["idx"] + expiry > n - 1 else "expired")
    end = fill + max_hold
    cost = cost_frac * entry
    for k in range(fill, min(end, n - 1) + 1):
        if (l[k] <= sl) if d == 1 else (h[k] >= sl):
            px = o[k] if k > fill and ((o[k] < sl) if d == 1 else (o[k] > sl)) else sl
            return dict(status="closed", outcome="sl", fill=fill, exit=k,
                        r=float(d * (px - entry) / risk - (cost + slip_frac * entry) / risk))
        if k > fill and ((h[k] >= tp) if d == 1 else (l[k] <= tp)):
            return dict(status="closed", outcome="tp", fill=fill, exit=k, r=float(d * (tp - entry) / risk - cost / risk))
    if end <= n - 1:
        return dict(status="closed", outcome="time", fill=fill, exit=end,
                    r=float(d * (c[end] - entry) / risk - cost / risk))
    return dict(status="open", fill=fill)


def hist_factors(af, rr, tf, ny_hour=None):
    """Confluence factors that can be checked in history too (af = aligned features of the setup candle)."""
    g = lambda k: af.get(k, np.nan)
    f = {"htf1_with": g("htf1_trend") > 0, "htf2_with": g("htf2_trend") > 0, "structure_with": g("trend") > 0,
         "discount_premium": g("dealing_pos_c") < 0, "recent_stop_side_sweep": g("since_stp_sweep") <= 30,
         "rr_2_5": rr >= 2.5}
    if md.TF_MIN[tf] < 240 and ny_hour is not None:
        f["killzone"] = (2 <= ny_hour < 5) or (7 <= ny_hour < 10) or (13.5 <= ny_hour < 16)
    return f


def grade_of(score):
    return "A" if score >= 5 else ("B" if score >= 3 else "C")


# ───────────────────────────── level reactions ─────────────────────────────
def _first_touch(arr, start, level, above):
    j, step, n = start, 64, len(arr)
    while j < n:
        seg = arr[j:j + step] >= level if above else arr[j:j + step] <= level
        hit = np.flatnonzero(seg)
        if len(hit):
            return j + int(hit[0])
        j, step = j + step, step * 2
    return None


def level_events(ctx, tf, horizon=H_TOUCH):
    """For every liquidity level as it became known: first touch, then break (close 0.25 ATR beyond) vs reject
    (1 ATR move away) within `horizon` candles."""
    df = ctx["df"]
    o, h, l, c, a = (df[k].to_numpy(dtype=float) for k in ("open", "high", "low", "close", "atr"))
    n = len(df)
    lv = []
    tol_eq = 0.15
    last_hi, last_lo = [], []
    for p in sorted(ctx["pivots"], key=lambda p: p["confirmed_idx"]):
        k0 = p["confirmed_idx"]
        if k0 >= n - 2 or not a[k0] > 0:
            continue
        if p["type"] == "high":
            eq = any(abs(p["price"] - x) <= tol_eq * a[k0] for x in last_hi[-5:])
            lv.append(("eqh" if eq else "swing_high", p["price"], k0, 1))
            last_hi.append(p["price"])
        else:
            eq = any(abs(p["price"] - x) <= tol_eq * a[k0] for x in last_lo[-5:])
            lv.append(("eql" if eq else "swing_low", p["price"], k0, -1))
            last_lo.append(p["price"])
    for g in ctx.get("fvgs") or []:
        k0 = g["idx"] + 1
        if k0 < n - 2:
            lv.append(("fvg_bull", g["top"], k0, -1) if g["direction"] == "bullish" else ("fvg_bear", g["bottom"], k0, 1))
    if md.TF_MIN[tf] < 1440:
        d = daily_from_intraday(df, tf)
        ends = md.close_times(d.index, "1D")
        kpos = df.index.searchsorted(ends)
        for (dh, dl), k0 in zip(d[["high", "low"]].to_numpy(), kpos):
            if 1 < k0 < n - 2:
                lv.append(("pdh", float(dh), int(k0) - 1, 1))
                lv.append(("pdl", float(dl), int(k0) - 1, -1))
    rng = np.random.default_rng(len(df))                      # baseline: random levels near price
    for k0 in rng.integers(WARMUP, max(WARMUP + 1, n - 2), size=max(50, len(lv) // 4)):
        if a[k0] > 0:
            side = 1 if rng.random() < 0.5 else -1
            lv.append(("random", float(c[k0] + side * rng.uniform(0.5, 3.0) * a[k0]), int(k0), side))
    rows = []
    for typ, price, k0, side in lv:
        t = _first_touch(h if side == 1 else l, k0 + 1, price, side == 1)
        if t is None or t >= n - 1:
            continue
        at = a[t] if a[t] > 0 else a[k0]
        seg = slice(t, min(n, t + horizon))
        if side == 1:
            br = np.flatnonzero(c[seg] >= price + 0.25 * at)
            rj = np.flatnonzero(l[seg] <= price - 1.0 * at)
        else:
            br = np.flatnonzero(c[seg] <= price - 0.25 * at)
            rj = np.flatnonzero(h[seg] >= price + 1.0 * at)
        tb = br[0] if len(br) else None
        tr = rj[0] if len(rj) else None
        if tb is None and tr is None:
            res = "unresolved"
        elif tr is None or (tb is not None and tb <= tr):
            res = "break"
        else:
            res = "reject"
        rows.append((typ, res))
    return rows


# ───────────────────────────── training samples ─────────────────────────────
def make_samples(F, ctx, meta, rng, n_rows, test_frac=0.3):
    """Random entry points with random stop/target/level geometry, labelled by what price did next."""
    df = ctx["df"]
    n = len(df)
    o, h, l, c, a = (df[k].to_numpy(dtype=float) for k in ("open", "high", "low", "close", "atr"))
    lo_, hi_ = min(WARMUP, max(150, n // 5)), n - H_RACE - 2
    if hi_ - lo_ < 150:
        return None
    rows = np.sort(rng.choice(np.arange(lo_, hi_), size=min(n_rows, hi_ - lo_), replace=False))
    split = int(n * (1 - test_frac))
    rows = rows[(rows >= split) | (rows < split - H_RACE)]                # purge: no label crosses the split
    rows = rows[np.isfinite(a[rows]) & (a[rows] > 0)]
    m = len(rows)
    if m < 50:
        return None
    test = rows >= split
    a0, c0, e0 = a[rows], c[rows], o[rows + 1]
    W = rows[:, None] + 1 + np.arange(H_RACE)[None, :]
    HH, LL = h[W], l[W]
    t_idx = df.index[rows]
    base = dict(test=test, t=t_idx, bar=rows, sid=meta["sid"], tf=meta["tf"], asset_class=meta["asset_class"])
    # race
    s = rng.choice([-1.0, 1.0], m)
    b = rng.choice(B_GRID, m)
    rr = rng.choice(RR_GRID, m)
    A_ = b * rr
    up, dn = (HH - e0[:, None]) / a0[:, None], (e0[:, None] - LL) / a0[:, None]
    fav, adv = np.where(s[:, None] > 0, up, dn), np.where(s[:, None] > 0, dn, up)
    t_tp = (np.maximum.accumulate(fav, 1) < A_[:, None]).sum(1)
    t_sl = (np.maximum.accumulate(adv, 1) < b[:, None]).sum(1)
    sl_first = (t_sl < H_RACE) & (t_sl <= t_tp)
    tp_first = (t_tp < H_RACE) & ~sl_first
    j = np.minimum(t_sl, H_RACE - 1)
    adv_open = s * (e0 - o[rows + 1 + j]) / a0
    r_sl = np.where((j >= 1) & (adv_open > b), -adv_open / b, -1.0)
    r_to = s * (c[rows + H_RACE] - e0) / (b * a0)
    R = np.where(tp_first, rr, np.where(sl_first, r_sl, r_to))
    cost_r = (meta["cost"] * e0 + np.where(sl_first, meta["slip"] * e0, 0.0)) / (b * a0)
    Fr = F.iloc[rows]
    race = align(Fr, s).assign(a_atr=A_, b_atr=b, implied=b / (A_ + b), log_rr=np.log(rr), y_tp=tp_first,
                               y_sl=sl_first, R=R, cost_r=cost_r, **base)
    # touch
    s2 = rng.choice([-1.0, 1.0], m)
    dd = rng.choice(D_GRID, m)
    up2 = (HH[:, :H_TOUCH] - c0[:, None]) / a0[:, None]
    dn2 = (c0[:, None] - LL[:, :H_TOUCH]) / a0[:, None]
    reach = np.where(s2 > 0, up2.max(1), dn2.max(1))
    touch = align(Fr, s2).assign(d_atr=dd, y=reach >= dd, **base)
    # direction (labels assigned later, once the barrier k is chosen on all data)
    upr = np.maximum.accumulate(up2[:, :H_DIR], 1)
    dnr = np.maximum.accumulate(dn2[:, :H_DIR], 1)
    t_up = np.stack([(upr < k).sum(1) for k in K_GRID], 1).astype(np.int16)
    t_dn = np.stack([(dnr < k).sum(1) for k in K_GRID], 1).astype(np.int16)
    dirs = align(Fr, 1.0).assign(**base)
    to32 = lambda X: X.astype({k: np.float32 for k in X.columns if X[k].dtype == np.float64})
    return dict(race=to32(race), touch=to32(touch), dir=to32(dirs), t_up=t_up, t_dn=t_dn)


def series_dataset(df, tf, asset_class, sub, sid, rng, n_rows, htf=None):
    """Features, random-entry samples, historical setups and level reactions for one market/timeframe."""
    df = se.add_atr(df[["open", "high", "low", "close", "volume"]].astype(float).copy())
    htf = htf if htf is not None else htf_frames_from(df, tf, asset_class)
    F, ctx = raw_features(df, tf, asset_class, htf=htf)
    ctx["fvgs"] = se.find_fvgs(df, 0.25)
    cost, slip = cost_of(sub)
    meta = dict(sid=sid, tf=tf, asset_class=asset_class, cost=cost, slip=slip)
    S = make_samples(F, ctx, meta, rng, n_rows)
    setups = []
    split = int(len(df) * 0.7)
    ny = df.index.tz_convert("America/New_York")
    for s in find_setups(ctx, cost):
        if s["idx"] < WARMUP:
            continue
        res = simulate(df, s, cost, slip)
        if res["status"] in ("pending", "open"):
            continue
        i, d = s["idx"], s["dir"]
        ai = float(df["atr"].iat[i])
        af = align(F.iloc[[i]], d).iloc[0]
        rr = abs(s["tp"] - s["entry"]) / s["risk"]
        fac = hist_factors(af, rr, tf, ny[i].hour + ny[i].minute / 60)
        rec = dict(sid=sid, tf=tf, asset_class=asset_class, family=s["family"], t=df.index[i], bar=i, test=i >= split,
                   dir=d, a_atr=abs(s["tp"] - s["entry"]) / ai, b_atr=s["risk"] / ai,
                   dist_atr=abs(float(df["close"].iat[i]) - s["entry"]) / ai, rr=rr,
                   cost_r=cost * s["entry"] / s["risk"], status=res["status"],
                   filled=res["status"] == "closed", outcome=res.get("outcome"), r=res.get("r", np.nan),
                   score=int(sum(bool(v) for v in fac.values())), **{f"f_{k}": bool(v) for k, v in fac.items()})
        rec.update({k: float(af[k]) for k in ALIGNED})
        setups.append(rec)
    lev = level_events(ctx, tf)
    return S, setups, lev, len(df)


def build_dataset(series, budget=dict(rows=450_000), seed=0, log=print):
    """series: list of dict(symbol, tf, asset_class, sub, df). Returns pooled training data."""
    rng = np.random.default_rng(seed)
    per = max(400, int(budget["rows"] / max(1, len(series))))
    race, touch, dirs, tups, tdns, setups, levels, info = [], [], [], [], [], [], [], []
    t0 = time.time()
    for k, sr in enumerate(series):
        sid = f"{sr['symbol']}|{sr['tf']}"
        try:
            S, st, lev, nbar = series_dataset(sr["df"], sr["tf"], sr["asset_class"], sr["sub"], sid, rng, per)
        except Exception as e:                                   # one bad series must not stop training
            log(f"  ⚠️ {sid}: {type(e).__name__}: {e}")
            continue
        if S is not None:
            race.append(S["race"])
            touch.append(S["touch"])
            dirs.append(S["dir"])
            tups.append(S["t_up"])
            tdns.append(S["t_dn"])
        setups.extend(st)
        for typ, res in lev:
            levels.append((sr["asset_class"], TF_GROUP[sr["tf"]], typ, res))
        info.append(dict(series=sid, asset_class=sr["asset_class"], tf=sr["tf"], candles=nbar,
                         samples=0 if S is None else len(S["race"]), setups=len(st), level_events=len(lev)))
        log(f"  [{k + 1}/{len(series)}] {sid:22s} {nbar:>8,} candles → {info[-1]['samples']:>6,} samples, "
            f"{len(st):>5,} setups ({time.time() - t0:.0f}s)")
    D = dict(race=pd.concat(race, ignore_index=True), touch=pd.concat(touch, ignore_index=True),
             dir=pd.concat(dirs, ignore_index=True), t_up=np.concatenate(tups), t_dn=np.concatenate(tdns),
             setups=pd.DataFrame(setups),
             levels=pd.DataFrame(levels, columns=["asset_class", "tf_group", "type", "result"]),
             info=pd.DataFrame(info))
    return D


# ───────────────────────────── models ─────────────────────────────
def _hgb(cols, mono=None, max_iter=300):
    from sklearn.ensemble import HistGradientBoostingClassifier
    cst = None
    if mono:
        cst = [int(mono.get(c, 0)) for c in cols]
    return HistGradientBoostingClassifier(max_iter=max_iter, learning_rate=0.08, max_leaf_nodes=31,
                                          min_samples_leaf=400, l2_regularization=1.0, early_stopping=True,
                                          validation_fraction=0.1, n_iter_no_change=15, random_state=0,
                                          monotonic_cst=cst)


MONO_TP = {"a_atr": -1, "b_atr": 1, "implied": 1, "log_rr": -1}
MONO_SL = {"a_atr": 1, "b_atr": -1, "implied": -1, "log_rr": 1}
MONO_TOUCH = {"d_atr": -1}


def _fit(X, y, cols, mono=None):
    m = _hgb(cols, mono)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        m.fit(X[cols].to_numpy(dtype=np.float32), np.asarray(y).astype(int))
    m.feature_names_ = list(cols)
    return m


def _p(m, X):
    return m.predict_proba(X[m.feature_names_].to_numpy(dtype=np.float32))


def choose_k(t_up, t_dn, target=1 / 3):
    neither = ((t_up >= H_DIR) & (t_dn >= H_DIR)).mean(0)
    j = int(np.argmin(np.abs(neither - target)))
    return float(K_GRID[j]), j


def dir_labels(t_up, t_dn, j):
    u, d = t_up[:, j], t_dn[:, j]
    return np.where((u < H_DIR) & (u < d), 0, np.where((d < H_DIR) & (d < u), 1, 2))


def _brier(p, y):
    return float(np.mean((np.asarray(p) - np.asarray(y)) ** 2))


def _logloss(p, y):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    y = np.asarray(y, dtype=float)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def _auc(p, y):
    from sklearn.metrics import roc_auc_score
    y = np.asarray(y).astype(int)
    return float(roc_auc_score(y, p)) if 0 < y.sum() < len(y) else float("nan")


def calibration(p, y, bins=10):
    p, y = np.asarray(p, dtype=float), np.asarray(y, dtype=float)
    q = np.unique(np.quantile(p, np.linspace(0, 1, bins + 1)))
    if len(q) < 3:
        return pd.DataFrame(columns=["predicted", "actual", "n"])
    k = np.clip(np.searchsorted(q, p, side="right") - 1, 0, len(q) - 2)
    t = pd.DataFrame({"k": k, "p": p, "y": y}).groupby("k").agg(predicted=("p", "mean"), actual=("y", "mean"),
                                                                 n=("y", "size"))
    return t.reset_index(drop=True).round(4)


def fit_direction_cal(P, y):
    """Isotonic calibration, one curve per outcome: up, down and sideways (range). Fitted on held-out predictions.

    Why: the raw shares are too spread out (the model is too confident at the top). The curves pull them back to the
    rate that actually happened. They cannot create skill that the model does not have."""
    from sklearn.isotonic import IsotonicRegression
    cal = {}
    for i, name in enumerate(("up", "down", "sideways")):
        cal[name] = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0).fit(
            np.asarray(P[:, i], dtype=float), (np.asarray(y) == i).astype(float))
    return cal


def _cal_all(cal, P):
    """Calibrated up / down / sideways for each row (each curve's own output), and the same three scaled to 100%."""
    raw = np.stack([cal["up"].predict(P[:, 0]), cal["down"].predict(P[:, 1]), cal["sideways"].predict(P[:, 2])], 1)
    tot = raw.sum(1, keepdims=True)
    return raw, raw / np.where(tot > 0, tot, 1.0)


def _ll3(P, y):
    return float(-np.mean(np.log(np.clip(P[np.arange(len(y)), y], 1e-6, 1))))


def _f(x):
    return None if x is None else float(x)


def direction_view(cal, raw, direction_card):
    """What the app's bull/bear bar shows. `cal` is the brain's calibration (None for brains saved before it existed).
    `direction_card` is the model card's direction block. Its test numbers travel with each view, so the app can say
    how much skill sits behind it.
    Range uses the model's own sideways share: on data it never trained on, the calibrated version was further off at
    the top end, so the raw share is what the app shows. Lean uses the calibration, which helped in testing."""
    if not cal:
        return dict(calibrated=False)
    dc = direction_card or {}
    prior = dc.get("prior") or [None, None, None]
    raw_arr = np.asarray(raw, dtype=float).reshape(1, 3)
    _, lean = _cal_all(cal, raw_arr)
    return dict(calibrated=True,
                range=dict(p=_f(raw_arr[0, 2]), base=_f(prior[2]), skill_auc=_f(dc.get("auc_neither"))),
                lean=dict(up=_f(lean[0, 0]), down=_f(lean[0, 1]), sideways=_f(lean[0, 2]),
                          skill_auc=_f(dc.get("auc_up_vs_down"))))


def block_keys(X, size=2 * H_RACE):
    """Independent-ish blocks: per series, consecutive stretches of 200 candles (a trade lasts up to 100)."""
    return X["sid"].astype(str).to_numpy() + "|" + (np.asarray(X["bar"]) // size).astype(str)


def block_t(values, keys):
    """t-stat of the mean, treating each block of candles as one observation (trades overlap in time)."""
    s = pd.Series(np.asarray(values, dtype=float)).groupby(np.asarray(keys)).mean()
    if len(s) < 3 or s.std() == 0:
        return float("nan")
    return float(s.mean() / s.std() * np.sqrt(len(s)))


def ev_r(p_tp, p_sl, rr, cost_r):
    """Expected R of a trade (timeouts counted as 0R) after costs."""
    return p_tp * rr - p_sl - cost_r


def train(D, log=print):
    """Walk-forward evaluation (train on the first 70% of every series, test on the last 30%), then final
    models on everything. Returns the bundle: models + model card + experience tables."""
    t0 = time.time()
    R, T, Dd = D["race"], D["touch"], D["dir"]
    k_dir, kj = choose_k(D["t_up"][~Dd["test"].to_numpy()], D["t_dn"][~Dd["test"].to_numpy()])
    Dd = Dd.assign(y=dir_labels(D["t_up"], D["t_dn"], kj))
    card = dict(rows=dict(race=len(R), touch=len(T), direction=len(Dd)), k_dir=k_dir)
    tr, te = ~R["test"].to_numpy(), R["test"].to_numpy()
    log(f"Evaluating on the last 30% of every series ({te.sum():,} race / {T['test'].sum():,} touch rows)...")
    m_tp = _fit(R[tr], R["y_tp"][tr], RACE_COLS, MONO_TP)
    m_sl = _fit(R[tr], R["y_sl"][tr], RACE_COLS, MONO_SL)
    m_geo = _fit(R[tr], R["y_tp"][tr], GEO_BASE, MONO_TP)
    Rt = R[te]
    p_tp, p_sl, p_geo = _p(m_tp, Rt)[:, 1], _p(m_sl, Rt)[:, 1], _p(m_geo, Rt)[:, 1]
    p_rw = Rt["b_atr"] / (Rt["a_atr"] + Rt["b_atr"])
    y = Rt["y_tp"].to_numpy().astype(float)
    rr = Rt["a_atr"] / Rt["b_atr"]
    net = Rt["R"] - Rt["cost_r"]
    blocks = block_keys(Rt)
    ev_m, ev_g = ev_r(p_tp, p_sl, rr, Rt["cost_r"]), ev_r(p_geo, 1 - p_geo, rr, Rt["cost_r"])
    rows = []
    for name, mask in (("every trade", np.ones(len(Rt), bool)), ("model EV > 0", ev_m > 0),
                       ("model EV > +0.10R", ev_m > 0.10), ("geometry-only EV > 0", ev_g > 0)):
        rows.append(dict(selection=name, trades=int(mask.sum()), avg_net_R=float(net[mask].mean()) if mask.any() else np.nan,
                         t_stat=block_t(net[mask], blocks[mask]) if mask.sum() > 30 else np.nan))
    card["race"] = dict(brier_model=_brier(p_tp, y), brier_geometry=_brier(p_geo, y), brier_random_walk=_brier(p_rw, y),
                        logloss_model=_logloss(p_tp, y), logloss_geometry=_logloss(p_geo, y),
                        auc_model=_auc(p_tp, y), auc_geometry=_auc(p_geo, y),
                        skill_vs_geometry=1 - _brier(p_tp, y) / _brier(p_geo, y),
                        sl_brier_model=_brier(p_sl, Rt["y_sl"]), calibration=calibration(p_tp, y),
                        ev_test=pd.DataFrame(rows))
    by = []
    for (cls, tf_), g in Rt.assign(p=p_tp, pg=p_geo).groupby(["asset_class", "tf"]):
        if len(g) >= 300:
            yy = g["y_tp"].astype(float)
            by.append(dict(asset_class=cls, tf=tf_, rows=len(g), win_rate=float(yy.mean()),
                           skill_vs_geometry=1 - _brier(g["p"], yy) / _brier(g["pg"], yy)))
    card["race_by_market"] = pd.DataFrame(by)
    trT, teT = ~T["test"].to_numpy(), T["test"].to_numpy()
    m_touch = _fit(T[trT], T["y"][trT], TOUCH_COLS, MONO_TOUCH)
    m_tbase = _fit(T[trT], T["y"][trT], TOUCH_BASE, MONO_TOUCH)
    Tt = T[teT]
    pt, pb = _p(m_touch, Tt)[:, 1], _p(m_tbase, Tt)[:, 1]
    yt = Tt["y"].to_numpy().astype(float)
    card["touch"] = dict(brier_model=_brier(pt, yt), brier_distance_only=_brier(pb, yt), auc_model=_auc(pt, yt),
                         auc_distance_only=_auc(pb, yt), skill_vs_distance=1 - _brier(pt, yt) / _brier(pb, yt),
                         calibration=calibration(pt, yt))
    trD, teD = ~Dd["test"].to_numpy(), Dd["test"].to_numpy()
    m_dir = _fit(Dd[trD], Dd["y"][trD], DIR_COLS)
    Dt = Dd[teD]
    pdir = _p(m_dir, Dt)
    yd = Dt["y"].to_numpy()
    prior = np.bincount(Dd["y"][trD], minlength=3) / trD.sum()
    ll = lambda P: float(-np.mean(np.log(np.clip(P[np.arange(len(yd)), yd], 1e-6, 1))))
    card["direction"] = dict(k_atr=k_dir, horizon=H_DIR, prior=prior.round(4).tolist(),
                             logloss_model=ll(pdir), logloss_prior=ll(np.tile(prior, (len(yd), 1))),
                             auc_up=_auc(pdir[:, 0], yd == 0), auc_down=_auc(pdir[:, 1], yd == 1),
                             auc_neither=_auc(pdir[:, 2], yd == 2), calibration_up=calibration(pdir[:, 0], yd == 0))
    card["direction"]["skill_vs_prior"] = 1 - card["direction"]["logloss_model"] / card["direction"]["logloss_prior"]
    resolved = yd != 2                                            # pure direction skill: up vs down only
    card["direction"]["auc_up_vs_down"] = _auc(pdir[resolved, 0] / (pdir[resolved, 0] + pdir[resolved, 1]),
                                               yd[resolved] == 0)
    # Calibration for the bull/bear bar. Fit on the first half of every series' test period, then check on the second
    # half, which the curves never saw. The card records both, so the result can be judged, not assumed.
    fit_side = (Dt["bar"] <= Dt.groupby("sid")["bar"].transform("median")).to_numpy()
    cal = fit_direction_cal(pdir[fit_side], yd[fit_side])
    ev_P, ev_y = pdir[~fit_side], yd[~fit_side]
    cal_raw, cal_lean = _cal_all(cal, ev_P)
    card["direction_cal"] = dict(
        fit_rows=int(fit_side.sum()), eval_rows=int((~fit_side).sum()),
        logloss_raw=_ll3(ev_P, ev_y), logloss_cal=_ll3(cal_lean, ev_y),
        up_before=calibration(ev_P[:, 0], ev_y == 0), up_after=calibration(cal_raw[:, 0], ev_y == 0),
        sideways_before=calibration(ev_P[:, 2], ev_y == 2), sideways_after=calibration(cal_raw[:, 2], ev_y == 2))
    # setup evidence (test period only, predictions from the walk-forward models)
    S = D["setups"]
    ev_rows, grade_rows = [], []
    if len(S):
        St = S[S["test"]].copy()
        if len(St):
            St["p_tp"] = _p(m_tp, St.assign(implied=St["b_atr"] / (St["a_atr"] + St["b_atr"]),
                                             log_rr=np.log(St["a_atr"] / St["b_atr"])))[:, 1]
            St["p_fill"] = _p(m_touch, flip_aligned(St[ALIGNED]).assign(d_atr=St["dist_atr"].to_numpy()))[:, 1]
            St["grade"] = St["score"].map(grade_of)
            St["win"] = (St["outcome"] == "tp").astype(float)
            St["blk"] = block_keys(St)
            for keys, g in [(("ALL",), St)] + [((f,), g) for f, g in St.groupby("family")]:
                f_ = g[g["filled"]]
                ev_rows.append(dict(family=keys[0], setups=len(g), fill_rate=float(g["filled"].mean()),
                                    predicted_fill=float(g["p_fill"].mean()), filled=len(f_),
                                    win_rate=float(f_["win"].mean()) if len(f_) else np.nan,
                                    predicted_win=float(f_["p_tp"].mean()) if len(f_) else np.nan,
                                    excess_win=float((f_["win"] - f_["p_tp"]).mean()) if len(f_) else np.nan,
                                    excess_t=block_t(f_["win"] - f_["p_tp"], f_["blk"]) if len(f_) > 30 else np.nan,
                                    avg_net_R=float(f_["r"].mean()) if len(f_) else np.nan,
                                    t_stat=block_t(f_["r"], f_["blk"]) if len(f_) > 30 else np.nan))
            for (fam, gr), g in St[St["filled"]].groupby(["family", "grade"]):
                grade_rows.append(dict(family=fam, grade=gr, trades=len(g), win_rate=float(g["win"].mean()),
                                       predicted_win=float(g["p_tp"].mean()),
                                       excess_win=float((g["win"] - g["p_tp"]).mean()),
                                       avg_net_R=float(g["r"].mean()),
                                       t_stat=block_t(g["r"], g["blk"]) if len(g) > 30 else np.nan))
    card["setups"] = pd.DataFrame(ev_rows)
    card["setup_grades"] = pd.DataFrame(grade_rows)
    log(f"  evaluation done ({time.time() - t0:.0f}s). Training the final models on all data...")
    final = dict(p_tp=_fit(R, R["y_tp"], RACE_COLS, MONO_TP), p_sl=_fit(R, R["y_sl"], RACE_COLS, MONO_SL),
                 p_touch=_fit(T, T["y"], TOUCH_COLS, MONO_TOUCH), p_dir=_fit(Dd, Dd["y"], DIR_COLS))
    L = D["levels"]
    lev = pd.DataFrame()
    if len(L):
        lev = (L.groupby(["asset_class", "tf_group", "type"])["result"].value_counts(normalize=False)
               .unstack(fill_value=0).reset_index())
        for col in ("break", "reject", "unresolved"):
            if col not in lev:
                lev[col] = 0
        lev["n"] = lev[["break", "reject", "unresolved"]].sum(1)
        for col in ("break", "reject", "unresolved"):
            lev[f"p_{col}"] = lev[col] / lev["n"]
    setup_tab = pd.DataFrame()
    if len(S):
        Sf = S[S["filled"]].assign(win=lambda x: (x["outcome"] == "tp").astype(float),
                                   tf_group=lambda x: x["tf"].map(TF_GROUP), grade=lambda x: x["score"].map(grade_of))
        setup_tab = (Sf.groupby(["family", "asset_class", "tf_group", "grade"])
                     .agg(trades=("win", "size"), win_rate=("win", "mean"), avg_net_R=("r", "mean")).reset_index())
    fill_ratio = 1.0
    if len(card["setups"]) and card["setups"]["predicted_fill"].iloc[0] > 0:
        fill_ratio = float(np.clip(card["setups"]["fill_rate"].iloc[0] / card["setups"]["predicted_fill"].iloc[0],
                                   0.5, 1.2))
    card["fill_ratio"] = fill_ratio
    card["seconds"] = round(time.time() - t0, 1)
    log(f"  done in {card['seconds']:.0f}s")
    return dict(version=VERSION, created=str(pd.Timestamp.now(tz="UTC")), models=final, card=card,
                k_dir=k_dir, fill_ratio=fill_ratio, level_stats=lev, setup_stats=setup_tab, info=D["info"],
                direction_cal=cal, cols=dict(race=RACE_COLS, touch=TOUCH_COLS, dir=DIR_COLS))


def save(bundle, path):
    import joblib
    joblib.dump(bundle, path, compress=3)


def load(path):
    import joblib
    return joblib.load(path)


# ───────────────────────────── live predictions ─────────────────────────────
class Brain:
    """Thin wrapper the analyst uses: probabilities for a feature row (see raw_features)."""

    untrained = False

    def __init__(self, bundle):
        self.b = bundle
        self.m = bundle["models"]
        self.k_dir = bundle["k_dir"]
        # heads trained in Notebook 10 (None in older brains): liquidity (liquidity_head.py), strategies
        # (strategy_head.py)
        self.liq = bundle.get("liquidity_head")
        self.strat = bundle.get("strategy_head")

    def race(self, row, sides, a_atr, b_atr):
        sides, a_atr, b_atr = np.broadcast_arrays(np.atleast_1d(np.asarray(sides, dtype=float)),
                                                  np.atleast_1d(np.asarray(a_atr, dtype=float)),
                                                  np.atleast_1d(np.asarray(b_atr, dtype=float)))
        # stay inside the geometry the model was trained on (stops 0.5–4 ATR, R:R 0.5–5)
        rr = np.clip(np.maximum(a_atr, 1e-9) / np.maximum(b_atr, 1e-9), RR_GRID.min(), RR_GRID.max())
        b_atr = np.clip(b_atr, B_GRID.min(), B_GRID.max())
        a_atr = b_atr * rr
        X = align_row(row, sides).assign(a_atr=a_atr, b_atr=b_atr, implied=b_atr / (a_atr + b_atr),
                                         log_rr=np.log(a_atr / b_atr))
        p_tp, p_sl = _p(self.m["p_tp"], X)[:, 1], _p(self.m["p_sl"], X)[:, 1]
        tot = p_tp + p_sl
        scale = np.where(tot > 1, 1 / tot, 1.0)
        p_tp, p_sl = p_tp * scale, p_sl * scale
        return p_tp, p_sl, np.clip(1 - p_tp - p_sl, 0, 1)

    def touch(self, row, sides, d_atr):
        sides, d_atr = np.broadcast_arrays(np.atleast_1d(np.asarray(sides, dtype=float)),
                                           np.atleast_1d(np.asarray(d_atr, dtype=float)))
        X = align_row(row, sides).assign(d_atr=np.clip(np.abs(d_atr), 0.0, 12.0))
        p = _p(self.m["p_touch"], X)[:, 1]
        return np.where(np.abs(d_atr) <= 0.02, 1.0, p)

    def direction(self, row):
        X = align_row(row, [1.0])
        return _p(self.m["p_dir"], X)[0]

    def direction_bar(self, row, raw=None):
        """The bull/bear bar's numbers: a calibrated range view and a calibrated lean view (see direction_view)."""
        raw = self.direction(row) if raw is None else raw
        return direction_view(self.b.get("direction_cal"), raw, self.b.get("card", {}).get("direction"))

    def expected_range(self, row, q=0.15):
        """Distances (ATR) up and down that price reaches with probability q within 24 candles."""
        grid = np.round(np.arange(0.1, 12.01, 0.05), 3)
        out = {}
        for side, name in ((1.0, "up"), (-1.0, "down")):
            p = self.touch(row, np.full(len(grid), side), grid)
            p = np.minimum.accumulate(p)
            k = np.flatnonzero(p <= q)
            out[name] = float(grid[k[0]]) if len(k) else float(grid[-1])
        return out


class FallbackBrain:
    """Random-walk approximations, used only when no trained bundle is loaded (flagged as 'untrained')."""
    k_dir = 3.0
    untrained = True
    liq = None
    strat = None

    def __init__(self):
        self.b = dict(card={}, level_stats=pd.DataFrame(), setup_stats=pd.DataFrame(), fill_ratio=0.85)

    def race(self, row, sides, a_atr, b_atr):
        a_atr, b_atr = np.broadcast_arrays(np.atleast_1d(np.asarray(a_atr, dtype=float)),
                                           np.atleast_1d(np.asarray(b_atr, dtype=float)))
        p_to = np.clip((a_atr + b_atr) ** 2 / 400, 0, 0.6)          # rough share of trades still open after 100 candles
        p_tp = (1 - p_to) * b_atr / (a_atr + b_atr)
        return p_tp, (1 - p_to) * a_atr / (a_atr + b_atr), p_to

    def touch(self, row, sides, d_atr):
        from math import erf, sqrt
        d = np.abs(np.atleast_1d(np.asarray(d_atr, dtype=float)))
        z = d / (0.8 * np.sqrt(H_TOUCH))
        return np.array([min(1.0, 2 * (1 - 0.5 * (1 + erf(x / sqrt(2))))) for x in z])

    def direction(self, row):
        return np.array([1 / 3, 1 / 3, 1 / 3])

    def direction_bar(self, row, raw=None):
        return dict(calibrated=False)          # no trained brain: the app shows no bar numbers

    def expected_range(self, row, q=0.15):
        from statistics import NormalDist
        d = NormalDist().inv_cdf(1 - q / 2) * 0.8 * np.sqrt(H_TOUCH)
        return {"up": float(d), "down": float(d)}


In [ ]:
%%writefile strategies.py
"""strategies.py: the trading strategies a person can ask the analyst to use.

A person picks one strategy or several (a scalper, a fixed-time trader and a swing trader want different things), and
the analyst reads the chart with those strategies only. Every strategy here is a set of rules that can be checked on
a closed candle, so the same code serves three jobs:

  * the live analysis: does the strategy fire on the newest closed candle, and where would its stop and target go
  * the record: a backtest on the candles already loaded, after costs, so the analyst knows how the strategy has been
    doing on this market and timeframe (the "fit"), and weighs strategies that disagree by their record
  * the overlay: the strategy's past trades marked on the chart, like a strategy report

Rules use only what was known at the close of the signal candle. Trades enter at the next candle's open. If a candle
touches both the stop and the target, the stop is assumed to come first. Fixed-time strategies are judged at expiry:
the close N candles later is above (UP) or below (DOWN) the entry close.

The liquidity guard (guard_stop) moves a stop past the swing lows/highs and equal lows/highs that sit just beyond it, by
the typical depth of a stop hunt on this chart (sweep_depth). A stop placed a few ticks under an obvious low is where
stop hunts reach, which is how many plans get stopped out just before price goes their way.

Research tool, not financial advice."""
import math

import numpy as np
import pandas as pd

import structure_engine as se

FT_PAYOUT = 0.85            # a typical fixed-time payout: a win pays 85% of the stake, a loss loses the stake

GROUPS = [
    ("smc", "Smart money & liquidity", "ICT and SMC concepts: liquidity sweeps, structure shifts, order blocks, FVGs."),
    ("price_action", "Price action", "Support and resistance, breakouts and candle patterns, without indicators."),
    ("trend", "Trend following", "Indicator strategies that buy strength and sell weakness."),
    ("reversion", "Mean reversion", "Indicator strategies that fade stretched moves back to the average."),
    ("breakout", "Breakout & volatility", "Range and volatility breakouts."),
    ("scalp", "Scalping", "Fast entries on 1–15 minute charts with tight stops and quick targets."),
    ("fixed_time", "Fixed time", "Up/down calls with an expiry, for fixed-time (binary-style) platforms."),
]

# drawing layers of the analysis that matter to each kind of strategy (the chart shows only these for the chosen ones)
L_SMC = ["structure", "liquidity", "orderblocks", "fvg", "fib", "range", "session", "period"]
L_PA = ["structure", "zones", "trendlines", "patterns", "candles", "period", "round"]
L_IND = ["structure", "zones"]


def _s(id, name, group, how, wait, best, tfs, markets, layers, rr=2.0, hold=48, stop="swing", ft=False, expiry=3,
       indicators=(), min_bars=120, intraday=False):
    return dict(id=id, name=name, group=group, how=how, wait_for=wait, best_for=best, timeframes=list(tfs),
                markets=list(markets), layers=list(layers), rr=rr, hold=hold, stop=stop, fixed_time=ft,
                expiry=expiry, indicators=list(indicators), min_bars=min_bars, intraday=intraday)


ALL_MKT = ("fx", "crypto", "gold", "index", "stock")
CATALOG = [
    # ── smart money & liquidity
    _s("ict_sweep_mss", "Liquidity sweep + structure shift", "smc",
       "Price takes the stops above a high (or below a low), closes back, then breaks structure the other way. Enter on "
       "the shift, stop beyond the sweep's wick, target the liquidity on the other side.",
       "a sweep of a swing high or low followed by a break of structure the other way",
       "Tends to work best on liquid markets in the London and New York killzones: EURUSD, GBPUSD, XAUUSD, NAS100, "
       "US30, BTC and ETH. Noisy on thin altcoins.", ("5m", "15m", "1h", "4h"), ALL_MKT, L_SMC, rr=2.5),
    _s("ote_pullback", "OTE pullback (61.8–78.6%)", "smc",
       "After a break of structure, wait for price to retrace into the 61.8–78.6% zone of the leg (ICT's optimal trade "
       "entry) and turn back with the trend. Stop beyond the leg's origin.",
       "a pullback into the 61.8–78.6% zone of the last impulse, with a candle turning back with the trend",
       "Trending markets: gold, indices, major FX pairs and BTC on 15m–4h.", ("15m", "1h", "4h", "1D"), ALL_MKT,
       L_SMC, rr=2.5),
    _s("ob_retest", "Order block retest", "smc",
       "The last opposite candle before a move that broke structure is an order block. Buy (sell) the first return to "
       "it when price rejects from it. Stop beyond the block.",
       "price returning to the order block left by the last break of structure",
       "FX majors, gold and indices on 15m–4h; crypto majors on 1h–4h.", ("15m", "1h", "4h"), ALL_MKT, L_SMC, rr=2.5),
    _s("fvg_fill", "Fair value gap continuation", "smc",
       "In a trend, price often comes back to fill part of an imbalance (FVG) before continuing. Buy (sell) the "
       "rejection from an open FVG in the trend's direction.",
       "a pullback into an open FVG in the trend's direction",
       "Fast-moving markets: NAS100, gold, BTC/ETH, GBP pairs.", ("5m", "15m", "1h", "4h"), ALL_MKT, L_SMC, rr=2.0),
    _s("turtle_soup", "Turtle soup (false breakout)", "smc",
       "Price breaks the last 20 candles' high (low), fails to hold it and closes back inside: the breakout traders' "
       "stops become fuel the other way. Stop beyond the wick.",
       "a wick beyond the 20-candle high or low that closes back inside",
       "Ranging FX pairs (EURUSD, AUDUSD, EURGBP) and gold; weaker in strong crypto trends.",
       ("5m", "15m", "1h", "4h"), ALL_MKT, ["liquidity", "structure", "period", "session"], rr=2.0),
    _s("silver_bullet", "ICT silver bullet", "smc",
       "Inside the silver-bullet hours (03–04, 10–11 and 14–15 New York time), after a sweep, the first fair value gap "
       "in the new direction is the entry. Stop beyond the gap's first candle.",
       "a fair value gap forming inside a silver-bullet hour after a liquidity sweep",
       "Index futures and CFDs (NAS100, US30, SPX), gold, EURUSD/GBPUSD on 1m–15m.", ("1m", "5m", "15m"),
       ("fx", "gold", "index", "crypto"), L_SMC, rr=2.0, hold=24, intraday=True),
    _s("power_of_3", "Power of 3 (Asia sweep)", "smc",
       "Accumulation in the Asian session, manipulation in London (a sweep of the Asian high or low) and distribution "
       "the other way. Buy a London/NY sweep of the Asian low that closes back inside (sell the high).",
       "a London or New York sweep of the Asian range that closes back inside it",
       "FX majors (EURUSD, GBPUSD), gold and indices on 5m–1h. Less useful on 24/7 crypto.", ("5m", "15m", "30m", "1h"),
       ("fx", "gold", "index"), L_SMC, rr=2.5, hold=36, intraday=True),
    _s("premium_discount", "Premium / discount", "smc",
       "Buy only in the discount half of the range in an uptrend (sell only in the premium half in a downtrend), when a "
       "candle closes back above the previous candle's high (below its low).",
       "price in the discount half of the range with a candle turning up (premium half and turning down for sells)",
       "Any liquid market on 15m–1D.", ("15m", "1h", "4h", "1D"), ALL_MKT, L_SMC, rr=2.5),
    _s("liquidity_draw", "Draw on liquidity", "smc",
       "Trade toward the nearest pool of untaken stops (equal highs or lows) after structure breaks toward it. The "
       "target is the pool.",
       "a break of structure toward an untaken pool of equal highs or lows at least 1.5R away",
       "Markets with clean swings: EURUSD, GBPUSD, XAUUSD, NAS100, BTC.", ("15m", "1h", "4h"), ALL_MKT, L_SMC, rr=2.0),
    _s("breaker_block", "Breaker block", "smc",
       "An order block that fails (price closes through it) often flips to the other side. Buy the retest of a broken "
       "bearish block from above (sell the retest of a broken bullish block from below).",
       "price retesting an order block it has just broken through",
       "Gold, indices and FX majors on 15m–4h.", ("15m", "1h", "4h"), ALL_MKT, L_SMC, rr=2.0),
    # ── price action
    _s("sr_bounce", "Support / resistance bounce", "price_action",
       "Buy a rejection from a support level that held before (sell from resistance): a long lower wick or a bullish "
       "engulfing candle at the level.",
       "a rejection candle at a level that held before",
       "Ranging markets and the higher timeframes of any market.", ("15m", "1h", "4h", "1D"), ALL_MKT, L_PA, rr=2.0),
    _s("breakout_retest", "Breakout and retest", "price_action",
       "A close beyond the 20-candle high (low), then the first retest of the broken level from the other side.",
       "a retest of a level price has just closed beyond",
       "Trending markets: indices, gold, BTC/ETH, GBPJPY.", ("15m", "1h", "4h", "1D"), ALL_MKT, L_PA, rr=2.0),
    _s("chart_patterns", "Double top / bottom", "price_action",
       "Two lows (highs) at the same price, then a close beyond the neckline between them. Target the measured move.",
       "a close beyond the neckline of a double bottom or top",
       "Any market on 1h–1D; reversal patterns work best after an extended move.", ("1h", "4h", "1D"), ALL_MKT, L_PA,
       rr=2.0, hold=60),
    _s("engulfing_trend", "Engulfing in the trend", "price_action",
       "In an uptrend (EMA 50 above EMA 200), buy a bullish engulfing candle near the EMA 20/50 pullback (sells "
       "mirrored).",
       "a pullback to the EMA 20/50 that ends with an engulfing candle",
       "Trending FX pairs, gold, indices and crypto majors on 1h–1D.", ("1h", "4h", "1D"), ALL_MKT,
       L_PA, rr=2.0, indicators=("ema20", "ema50", "ema200")),
    _s("inside_bar", "Inside bar breakout", "price_action",
       "A candle inside the previous one is a pause. Buy a close above the mother candle's high in an uptrend (sell "
       "below its low in a downtrend).",
       "an inside bar followed by a close beyond the mother candle",
       "Daily and 4h charts of FX majors, gold and indices.", ("1h", "4h", "1D"), ALL_MKT, L_PA, rr=2.0),
    _s("outside_bar", "Outside bar", "price_action",
       "A candle that engulfs the previous one's whole range and closes strong shows a shift of control. Trade in the "
       "direction it closes.",
       "a candle that takes out both the previous high and low and closes strong",
       "Volatile markets on 1h–1D.", ("1h", "4h", "1D"), ALL_MKT, L_PA, rr=1.5),
    _s("pin_bar", "Pin bar rejection", "price_action",
       "A long wick (at least twice the body) poking below the recent low (above the recent high) shows rejection.",
       "a pin bar at the edge of the recent range",
       "Gold, FX majors and indices on 1h–1D.", ("1h", "4h", "1D"), ALL_MKT, L_PA, rr=2.0),
    # ── trend following (classic indicator strategies)
    _s("supertrend", "Supertrend", "trend",
       "Supertrend (10, 3) flips: buy when price closes above the line, sell when it closes below.",
       "the Supertrend line flipping", "Trending markets: crypto, indices and gold on 1h–1D.",
       ("15m", "1h", "4h", "1D"), ALL_MKT, L_IND, rr=2.0, stop="indicator", indicators=("supertrend",)),
    _s("psar", "Parabolic SAR", "trend",
       "Parabolic SAR dots flip from above price to below it (buy) or from below to above (sell).",
       "the Parabolic SAR flipping sides", "Smooth trends: indices, gold, JPY pairs on 1h–1D.",
       ("15m", "1h", "4h", "1D"), ALL_MKT, L_IND, rr=2.0, stop="indicator", indicators=("psar",)),
    _s("two_ma_cross", "Two moving averages crossover", "trend",
       "EMA 9 crosses above EMA 21 (buy) or below it (sell).", "EMA 9 crossing EMA 21",
       "Trending markets on 15m–4h.", ("5m", "15m", "1h", "4h"), ALL_MKT, L_IND, rr=2.0, indicators=("ema9", "ema21")),
    _s("ma_cross", "Moving average crossover", "trend",
       "Price closes above the 20 SMA (buy) or below it (sell).", "a close across the 20 SMA",
       "Strong trends on 1h–1D; whipsaws in ranges.", ("1h", "4h", "1D"), ALL_MKT, L_IND, rr=2.0,
       indicators=("sma20",)),
    _s("macd_cross", "MACD", "trend",
       "The MACD histogram crosses above zero (buy) or below zero (sell).", "the MACD histogram crossing zero",
       "Trending FX and crypto on 1h–1D.", ("15m", "1h", "4h", "1D"), ALL_MKT, L_IND, rr=2.0, indicators=("macd",)),
    _s("momentum", "Momentum", "trend",
       "12-candle momentum crosses above zero and keeps rising (buy); below zero and falling (sell).",
       "momentum crossing zero", "Indices and crypto on 1h–1D.", ("1h", "4h", "1D"), ALL_MKT, L_IND, rr=2.0),
    _s("tech_ratings", "Technical ratings", "trend",
       "Moving averages and oscillators vote buy, sell or neutral. Buy when the rating turns to buy, sell when it turns "
       "to sell.", "the overall technical rating turning", "Any market on 1h–1D.", ("1h", "4h", "1D"), ALL_MKT, L_IND,
       rr=2.0),
    _s("bar_updn", "Bar up / down", "trend",
       "A candle that opens above the last close and closes up (buy), or opens below and closes down (sell).",
       "a gap-and-go candle", "Stocks and indices with session gaps; noisy elsewhere.", ("15m", "1h", "4h", "1D"),
       ALL_MKT, L_IND, rr=1.5, stop="atr"),
    _s("consecutive", "Consecutive up / down", "trend",
       "Three closes in a row higher (buy) or lower (sell).", "three closes in a row the same way",
       "Momentum markets; filter by the higher timeframe.", ("15m", "1h", "4h", "1D"), ALL_MKT, L_IND, rr=1.5,
       stop="atr"),
    _s("greedy", "Gap continuation", "trend",
       "A candle that opens beyond the previous candle's high (low) and closes in the same direction.",
       "an opening gap that holds", "Stocks and indices; rare in 24/7 markets.", ("1h", "4h", "1D"),
       ("stock", "index"), L_IND, rr=1.5, stop="atr"),
    # ── mean reversion
    _s("rsi_strategy", "RSI", "reversion",
       "RSI (14) crosses back above 30 (buy) or back below 70 (sell).", "RSI leaving oversold or overbought",
       "Ranging FX crosses (EURGBP, AUDNZD, EURCHF) and range-bound crypto on 15m–4h.", ("15m", "1h", "4h"), ALL_MKT,
       ["zones", "structure"], rr=1.5, indicators=("rsi",)),
    _s("bb_strategy", "Bollinger Bands", "reversion",
       "A close back inside the lower band after closing below it (buy), or back inside the upper band (sell). Target "
       "the middle band.", "a close back inside a Bollinger band",
       "Ranging markets: EURUSD in Asia, EURGBP, AUDNZD, range-bound crypto.", ("5m", "15m", "1h", "4h"), ALL_MKT,
       ["zones"], rr=1.5, stop="atr", indicators=("bb",)),
    _s("stoch_slow", "Stochastic slow", "reversion",
       "%K crosses above %D below 20 (buy) or below %D above 80 (sell).", "a Stochastic cross in an extreme",
       "Ranges and pullbacks in a trend on 15m–4h.", ("15m", "1h", "4h"), ALL_MKT, ["zones"], rr=1.5,
       indicators=("stoch",)),
    _s("pivot_ext", "Pivot reversal (extension)", "reversion",
       "A confirmed swing low (4 candles each side) is bought, a confirmed swing high sold.",
       "a new swing low or high being confirmed", "Range-bound markets on 1h–1D.", ("1h", "4h", "1D"), ALL_MKT,
       ["structure", "zones"], rr=1.5),
    # ── breakout & volatility
    _s("channel_breakout", "Channel breakout", "breakout",
       "A close above the highest high of the last 20 candles (buy) or below the lowest low (sell).",
       "a close beyond the 20-candle channel", "Trending markets: gold, indices, crypto, JPY pairs.",
       ("15m", "1h", "4h", "1D"), ALL_MKT, ["zones", "structure"], rr=2.0),
    _s("price_channel", "Price channel (55)", "breakout",
       "A close beyond the 55-candle high or low: the slower, trend-following breakout (the turtles' system 2).",
       "a close beyond the 55-candle channel", "Long trends: commodities, gold, indices, crypto on 4h–1D.",
       ("1h", "4h", "1D"), ALL_MKT, ["zones", "structure"], rr=2.5, hold=80),
    _s("pivot_reversal", "Pivot breakout", "breakout",
       "A close above the last confirmed swing high (buy) or below the last swing low (sell). Stop at the opposite "
       "swing.", "a close beyond the last swing high or low", "Trending markets on 15m–4h.", ("15m", "1h", "4h"),
       ALL_MKT, ["structure", "liquidity"], rr=2.0),
    _s("keltner", "Keltner channels", "breakout",
       "A close above the upper Keltner channel (EMA 20 + 2 ATR) buys; below the lower one sells.",
       "a close beyond a Keltner channel", "Volatile trending markets on 15m–4h.", ("15m", "1h", "4h"), ALL_MKT,
       ["zones"], rr=2.0, indicators=("keltner",)),
    _s("adx_breakout", "ADX breakout", "breakout",
       "After a quiet stretch (ADX below 20), a close beyond the 20-candle box starts a trend.",
       "a quiet market (ADX < 20) breaking its box", "FX majors and indices on 15m–4h.", ("15m", "1h", "4h"),
       ALL_MKT, ["zones", "structure"], rr=2.0),
    _s("vol_expansion", "Volatility expansion close", "breakout",
       "A close more than 0.75 ATR beyond the previous close starts a move; trade with it.",
       "a large expansion candle", "News-driven markets: gold, indices, GBP pairs, crypto.", ("5m", "15m", "1h", "4h"),
       ALL_MKT, ["zones"], rr=1.5, stop="atr"),
    # ── scalping
    _s("scalp_ema_vwap", "EMA + VWAP scalp", "scalp",
       "EMA 9 above EMA 21 and price above VWAP: buy the first touch of the EMA 9 that closes back up (sells "
       "mirrored). Tight stop below the last few candles, quick 1.5R target.",
       "a touch of EMA 9 in the direction of EMA 21 and VWAP",
       "EURUSD, GBPUSD, XAUUSD, NAS100 and BTC/ETH on 1m–15m, in the London and New York sessions.",
       ("1m", "5m", "15m"), ALL_MKT, ["liquidity", "session"], rr=1.5, hold=20, indicators=("ema9", "ema21", "vwap"),
       intraday=True),
    _s("scalp_london", "S.C.A.L.P. London sweep", "scalp",
       "In the London killzone (02:00–05:00 New York): a liquidity sweep, a structure break the other way, entry only "
       "from the discount half (longs) or premium half (shorts).",
       "a London-killzone sweep followed by a structure break",
       "GBPUSD, EURUSD, GBPJPY and gold on 1m–15m.", ("1m", "5m", "15m"), ("fx", "gold", "index"), L_SMC, rr=3.0,
       hold=24, stop="atr", intraday=True),
    _s("scalp_range", "Range fade scalp", "scalp",
       "When the market is quiet (ADX below 20), fade closes back inside the Bollinger Bands toward the middle band.",
       "a quiet market closing back inside a band",
       "EURUSD and GBPUSD in the Asian session, EURGBP, and sideways crypto on 1m–5m.", ("1m", "5m", "15m"),
       ALL_MKT, ["zones", "session"], rr=1.2, hold=15, stop="atr", indicators=("bb",), intraday=True),
    _s("scalp_burst", "Momentum burst scalp", "scalp",
       "Inside a killzone, a candle larger than 1.8 ATR closing near its extreme on high volume: ride the continuation "
       "with a stop at the candle's middle.",
       "a large, high-volume candle in a killzone",
       "Gold, NAS100, US30, GBPUSD and BTC on 1m–5m at the session opens.", ("1m", "5m", "15m"), ALL_MKT,
       ["session", "liquidity"], rr=1.5, hold=12, intraday=True),
    # ── fixed time (expiry trades)
    _s("ft_trend", "Fixed time · trend pullback", "fixed_time",
       "EMA 8 above EMA 21 above EMA 50: after a red candle, a green candle closing above the previous high calls UP "
       "for the next 3 candles (DOWN mirrored).",
       "a pullback candle followed by a strong candle with the trend",
       "Trending EURUSD, GBPUSD, AUDUSD and BTC on 1m–5m.", ("1m", "5m", "15m"), ALL_MKT, ["structure", "session"],
       ft=True, expiry=3, indicators=("ema8", "ema21", "ema50"), intraday=True),
    _s("ft_reversal", "Fixed time · exhaustion reversal", "fixed_time",
       "RSI (7) below 20 and a candle rejecting the 20-candle low with a long lower wick calls UP for 3 candles "
       "(DOWN mirrored at the high).",
       "an RSI extreme with a rejection wick at the edge of the range",
       "Ranging EURUSD, GBPUSD and EURGBP on 1m–5m, outside news.", ("1m", "5m", "15m"), ALL_MKT, ["zones", "session"],
       ft=True, expiry=3, indicators=("rsi7",), intraday=True),
    _s("ft_level", "Fixed time · level touch", "fixed_time",
       "Price touches the last swing low (high) and closes back away from it: UP (DOWN) for 2 candles.",
       "a touch of the last swing level that closes away from it",
       "EURUSD, GBPUSD and USDJPY on 1m–5m.", ("1m", "5m", "15m"), ALL_MKT, ["structure", "zones"], ft=True, expiry=2,
       intraday=True),
    _s("ft_momentum", "Fixed time · momentum", "fixed_time",
       "Three candles the same colour with growing bodies, closing beyond the Bollinger middle: same direction for 2 "
       "candles.", "three growing candles in a row",
       "News-free sessions of EURUSD, GBPUSD and crypto majors on 1m–5m.", ("1m", "5m"), ALL_MKT, ["session"], ft=True,
       expiry=2, intraday=True),
]
BY_ID = {s["id"]: s for s in CATALOG}
DEFAULT_SET = ["ict_sweep_mss", "ote_pullback", "ob_retest", "sr_bounce", "breakout_retest"]
# the analyst's own setups (guru_analyst._candidate_plans) and the strategy each belongs to
GURU_SETUP = {"pullback": "ote_pullback", "sweep": "ict_sweep_mss", "breakout": "breakout_retest",
              "pattern": "chart_patterns"}


def clean_ids(ids):
    """The known strategy ids in `ids`, in catalog order, without repeats."""
    want = {str(i) for i in (ids or [])}
    return [s["id"] for s in CATALOG if s["id"] in want]


def public_catalog():
    keys = ("id", "name", "group", "how", "wait_for", "best_for", "timeframes", "markets", "fixed_time", "expiry", "rr",
            "intraday")
    return dict(groups=[dict(id=g, name=n, about=a) for g, n, a in GROUPS],
                strategies=[{k: s[k] for k in keys} for s in CATALOG], default=DEFAULT_SET)


def layers_for(ids):
    out = {"plan", "price", "user", "strategy", "mtf"}
    for i in clean_ids(ids):
        out.update(BY_ID[i]["layers"])
    return sorted(out)


# ───────────────────────────── indicator helpers (numpy, known at each close) ─────────────────────────────
def _sma(x, n):
    return pd.Series(x).rolling(n, min_periods=n).mean().to_numpy()


def _ema(x, n):
    return pd.Series(x).ewm(span=n, adjust=False, min_periods=n).mean().to_numpy()


def _rma(x, n):
    return pd.Series(x).ewm(alpha=1 / n, adjust=False, min_periods=n).mean().to_numpy()


def _std(x, n):
    return pd.Series(x).rolling(n, min_periods=n).std(ddof=0).to_numpy()


def _rsi(c, n):
    d = np.diff(c, prepend=c[0])
    up = _rma(np.clip(d, 0, None), n)
    dn = _rma(np.clip(-d, 0, None), n)
    with np.errstate(divide="ignore", invalid="ignore"):
        rs = up / dn
        out = 100 - 100 / (1 + rs)
    out = np.where(dn == 0, 100.0, out)
    return np.where(np.isnan(up), 50.0, out)


def _prev(a, k=1, fill=np.nan):
    a = np.asarray(a, dtype=float)
    if k <= 0:
        return a
    return np.r_[np.full(k, fill), a[:-k]]


def _cross_up(a, b):
    pa, pb = _prev(a), _prev(b)
    return np.nan_to_num((a > b) & (pa <= pb), nan=0).astype(bool)


def _cross_dn(a, b):
    pa, pb = _prev(a), _prev(b)
    return np.nan_to_num((a < b) & (pa >= pb), nan=0).astype(bool)


def _recent(mask, k):
    if k <= 1:
        return np.asarray(mask, bool)
    return pd.Series(np.asarray(mask, float)).rolling(k, min_periods=1).max().to_numpy() > 0


def _hh(h, n):
    """Highest high of the n candles before this one (this one excluded)."""
    return pd.Series(h).rolling(n, min_periods=n).max().shift(1).to_numpy()


def _ll(l_, n):
    return pd.Series(l_).rolling(n, min_periods=n).min().shift(1).to_numpy()


def _true(x):
    return np.nan_to_num(np.asarray(x, dtype=float), nan=0.0) > 0


class Ctx:
    """Everything the strategies read from one candle frame, computed once and cached."""

    def __init__(self, df, tf, A=None, asset_class=None):
        if "atr" not in df.columns:
            df = se.add_atr(df[["open", "high", "low", "close", "volume"]].astype(float).copy())
        self.df, self.tf, self.asset_class = df, tf, asset_class
        self.o, self.h, self.l, self.c = (df[k].to_numpy(dtype=float) for k in ("open", "high", "low", "close"))
        self.v = df["volume"].to_numpy(dtype=float)
        self.atr = df["atr"].bfill().to_numpy(dtype=float)
        self.n = len(df)
        if A is None:
            piv = se.find_pivots(df, 3, 3)
            events, sweeps, trend = se.detect_structure_breaks(df, piv)
            A = dict(pivots=piv, events=events, sweeps=sweeps, trend=trend, fvgs=se.find_fvgs(df))
        self.A = A
        self.trend = np.asarray(A["trend"], dtype=int)
        try:
            ny = df.index.tz_convert("America/New_York")
            self.ny_hour = np.asarray(ny.hour + ny.minute / 60.0, dtype=float)
            self.ny_day = np.asarray(ny.normalize().asi8)
        except (TypeError, AttributeError):
            self.ny_hour = np.full(self.n, 12.0)
            self.ny_day = np.zeros(self.n, dtype=np.int64)
        self._c = {}

    def get(self, key, fn):
        if key not in self._c:
            self._c[key] = fn()
        return self._c[key]

    def ema(self, n):
        return self.get(("ema", n), lambda: _ema(self.c, n))

    def sma(self, n):
        return self.get(("sma", n), lambda: _sma(self.c, n))

    def rsi(self, n=14):
        return self.get(("rsi", n), lambda: _rsi(self.c, n))

    def atr_n(self, n):
        def f():
            pc = _prev(self.c)
            tr = np.nanmax(np.vstack([self.h - self.l, np.abs(self.h - pc), np.abs(self.l - pc)]), axis=0)
            return _rma(tr, n)
        return self.get(("atr", n), f)

    def bb(self, n=20, k=2.0):
        def f():
            m, s = _sma(self.c, n), _std(self.c, n)
            return m - k * s, m, m + k * s
        return self.get(("bb", n, k), f)

    def adx(self, n=14):
        def f():
            up, dn = np.diff(self.h, prepend=self.h[0]), -np.diff(self.l, prepend=self.l[0])
            plus = np.where((up > dn) & (up > 0), up, 0.0)
            minus = np.where((dn > up) & (dn > 0), dn, 0.0)
            atr = self.atr_n(n)
            with np.errstate(divide="ignore", invalid="ignore"):
                pdi = 100 * _rma(plus, n) / atr
                mdi = 100 * _rma(minus, n) / atr
                dx = 100 * np.abs(pdi - mdi) / (pdi + mdi)
            return np.nan_to_num(_rma(np.nan_to_num(dx), n)), np.nan_to_num(pdi), np.nan_to_num(mdi)
        return self.get(("adx", n), f)

    def macd(self, f=12, s=26, sig=9):
        def g():
            m = _ema(self.c, f) - _ema(self.c, s)
            sg = _ema(np.nan_to_num(m), sig)
            sg = np.where(np.isnan(m), np.nan, sg)
            return m, sg, m - sg
        return self.get(("macd", f, s, sig), g)

    def stoch(self, k=14, d=3, smooth=3):
        def f():
            hh = pd.Series(self.h).rolling(k, min_periods=k).max().to_numpy()
            ll = pd.Series(self.l).rolling(k, min_periods=k).min().to_numpy()
            with np.errstate(divide="ignore", invalid="ignore"):
                raw = np.where(hh > ll, (self.c - ll) / (hh - ll) * 100, 50.0)
            kk = _sma(raw, smooth)
            return kk, _sma(kk, d)
        return self.get(("stoch", k, d, smooth), f)

    def supertrend(self, n=10, mult=3.0):
        def f():
            atr = self.atr_n(n)
            hl2 = (self.h + self.l) / 2
            up_b, dn_b = hl2 - mult * atr, hl2 + mult * atr
            line = np.full(self.n, np.nan)
            dirn = np.zeros(self.n, dtype=int)
            fu, fd, d = np.nan, np.nan, 1
            for i in range(self.n):
                if np.isnan(atr[i]):
                    continue
                pc = self.c[i - 1] if i else self.c[i]
                fu = up_b[i] if np.isnan(fu) or pc < fu else max(up_b[i], fu)
                fd = dn_b[i] if np.isnan(fd) or pc > fd else min(dn_b[i], fd)
                if d == 1 and self.c[i] < fu:
                    d = -1
                elif d == -1 and self.c[i] > fd:
                    d = 1
                dirn[i] = d
                line[i] = fu if d == 1 else fd
            return line, dirn
        return self.get(("st", n, mult), f)

    def psar(self, step=0.02, mx=0.2):
        def f():
            n = self.n
            sar = np.full(n, np.nan)
            dirn = np.zeros(n, dtype=int)
            if n < 3:
                return sar, dirn
            d = 1 if self.c[1] >= self.c[0] else -1
            ep = self.h[0] if d == 1 else self.l[0]
            s = self.l[0] if d == 1 else self.h[0]
            af = step
            for i in range(1, n):
                s = s + af * (ep - s)
                if d == 1:
                    s = min(s, self.l[i - 1], self.l[i - 2] if i >= 2 else self.l[i - 1])
                    if self.l[i] < s:
                        d, s, ep, af = -1, ep, self.l[i], step
                    elif self.h[i] > ep:
                        ep, af = self.h[i], min(mx, af + step)
                else:
                    s = max(s, self.h[i - 1], self.h[i - 2] if i >= 2 else self.h[i - 1])
                    if self.h[i] > s:
                        d, s, ep, af = 1, ep, self.h[i], step
                    elif self.l[i] < ep:
                        ep, af = self.l[i], min(mx, af + step)
                sar[i], dirn[i] = s, d
            return sar, dirn
        return self.get(("psar", step, mx), f)

    def vwap(self):
        def f():
            tp = (self.h + self.l + self.c) / 3
            v = np.where(self.v > 0, self.v, 1.0)
            day = pd.Series(self.df.index.floor("1D").asi8)
            num = pd.Series(tp * v).groupby(day.values).cumsum().to_numpy()
            den = pd.Series(v).groupby(day.values).cumsum().to_numpy()
            return num / den
        return self.get("vwap", f)

    def pivots(self, left=3, right=3):
        return self.get(("piv", left, right), lambda: se.find_pivots(self.df, left, right))

    def last_pivot(self, kind, left=3, right=3):
        """(price, idx) of the latest pivot of `kind` (high/low) confirmed by each candle's close."""
        def f():
            pr = np.full(self.n, np.nan)
            ix = np.full(self.n, -1, dtype=int)
            for p in self.pivots(left, right):
                if p["type"] == kind and p["confirmed_idx"] < self.n:
                    pr[p["confirmed_idx"]] = p["price"]
                    ix[p["confirmed_idx"]] = p["idx"]
            s_pr = pd.Series(pr).ffill().to_numpy()
            s_ix = pd.Series(np.where(ix >= 0, ix, np.nan)).ffill().fillna(-1).to_numpy().astype(int)
            return s_pr, s_ix
        return self.get(("lastpiv", kind, left, right), f)

    def asia(self):
        """The Asian range (20:00–24:00 New York, the evening before) high and low, known from midnight on."""
        def f():
            hi = np.full(self.n, np.nan)
            lo = np.full(self.n, np.nan)
            day, hr = self.ny_day, self.ny_hour
            cur_hi = cur_lo = np.nan
            build_hi = build_lo = np.nan
            for i in range(self.n):
                if i and day[i] != day[i - 1]:            # a new New York day: last evening's range is the Asian range
                    cur_hi, cur_lo = build_hi, build_lo
                    build_hi = build_lo = np.nan
                if hr[i] >= 20:
                    build_hi = self.h[i] if np.isnan(build_hi) else max(build_hi, self.h[i])
                    build_lo = self.l[i] if np.isnan(build_lo) else min(build_lo, self.l[i])
                hi[i], lo[i] = cur_hi, cur_lo
            return hi, lo
        return self.get("asia", f)

    def sweep_depth(self):
        """How far past a swing high or low stop hunts usually reach on this chart, in ATR: the 80th percentile of the
        wicks beyond swept swings (0.35 ATR when there are too few sweeps to tell)."""
        def f():
            d = [abs(s["wick"] - s["level"]) / max(self.atr[s["idx"]], 1e-12) for s in self.A.get("sweeps", [])
                 if 0 <= s["idx"] < self.n]
            if len(d) < 8:
                return 0.35
            return float(np.clip(np.percentile(d, 80), 0.12, 1.5))
        return self.get("sweep_depth", f)

    def pool_table(self):
        """Every confirmed swing as arrays: side (+1 high, -1 low), price, candle, the candle it became known, and the
        first candle that traded through it (n while none has). Computed once, so liquidity lookups are fast."""
        def f():
            piv = self.pivots()
            m = len(piv)
            side = np.empty(m, dtype=int)
            price = np.empty(m, dtype=float)
            idx = np.empty(m, dtype=int)
            conf = np.empty(m, dtype=int)
            taken = np.full(m, self.n, dtype=int)
            for k, p in enumerate(piv):
                sd, q, ix = (1 if p["type"] == "high" else -1), float(p["price"]), int(p["idx"])
                side[k], price[k], idx[k], conf[k] = sd, q, ix, int(p["confirmed_idx"])
                arr = self.h if sd == 1 else self.l
                start = ix + 1
                for span in (256, self.n):                  # most swings are taken soon; search near first
                    seg = arr[start:start + span]
                    hit = np.flatnonzero(seg > q) if sd == 1 else np.flatnonzero(seg < q)
                    if len(hit):
                        taken[k] = start + int(hit[0])
                        break
                    start += span
                    if start >= self.n:
                        break
            return dict(side=side, price=price, idx=idx, conf=conf, taken=taken)
        return self.get("pool_table", f)

    def pools(self, i, side, look=150):
        """Untaken liquidity at candle i: confirmed swing lows (side=-1) or highs (side=1) no candle has traded through
        since, within the last `look` candles. Equal lows/highs are among them (two swings at one price)."""
        T = self.pool_table()
        if not len(T["side"]):
            return []
        m = (T["side"] == side) & (T["conf"] <= i) & (T["idx"] >= max(0, i - look)) & (T["taken"] > i)
        return [float(x) for x in T["price"][m]]


# ───────────────────────────── signals ─────────────────────────────
def _zero(n):
    return np.zeros(n, dtype=bool)


def _sig(ctx, long, short, sl_l=None, sl_s=None, tp_l=None, tp_s=None):
    return dict(long=np.asarray(long, bool), short=np.asarray(short, bool), sl_long=sl_l, sl_short=sl_s,
                tp_long=tp_l, tp_short=tp_s)


def _events_arrays(ctx):
    def f():
        bull = np.zeros(ctx.n, bool)
        bear = np.zeros(ctx.n, bool)
        for e in ctx.A["events"]:
            if 0 <= e["idx"] < ctx.n:
                (bull if e["direction"] == "bullish" else bear)[e["idx"]] = True
        return bull, bear
    return ctx.get("events_arr", f)


def sig_ict_sweep_mss(ctx):
    n, L, H, atr = ctx.n, ctx.l, ctx.h, ctx.atr
    lng, sht = _zero(n), _zero(n)
    sl_l, sl_s = np.full(n, np.nan), np.full(n, np.nan)
    sweeps = sorted(ctx.A["sweeps"], key=lambda s: s["idx"])
    for e in ctx.A["events"]:
        i = e["idx"]
        if i >= n:
            continue
        want = "sell-side" if e["direction"] == "bullish" else "buy-side"
        prior = [s for s in sweeps if s["side"] == want and 0 <= i - s["idx"] <= 8]
        if not prior:
            continue
        s = prior[-1]
        if e["direction"] == "bullish":
            lng[i] = True
            sl_l[i] = float(L[s["idx"]:i + 1].min()) - 0.1 * atr[i]
        else:
            sht[i] = True
            sl_s[i] = float(H[s["idx"]:i + 1].max()) + 0.1 * atr[i]
    return _sig(ctx, lng, sht, sl_l, sl_s)


def sig_ote_pullback(ctx):
    n, L, H, C, O, atr = ctx.n, ctx.l, ctx.h, ctx.c, ctx.o, ctx.atr
    lng, sht = _zero(n), _zero(n)
    sl_l, sl_s = np.full(n, np.nan), np.full(n, np.nan)
    tp_l, tp_s = np.full(n, np.nan), np.full(n, np.nan)
    evs = sorted(ctx.A["events"], key=lambda e: e["idx"])
    for k, e in enumerate(evs):
        i0 = e["idx"]
        end = min(n, evs[k + 1]["idx"] if k + 1 < len(evs) else n, i0 + 60)
        if e["direction"] == "bullish":
            origin = float(L[max(0, e["from_idx"]):i0 + 1].min())
            ext = H[i0]
            for i in range(i0 + 1, end):
                ext = max(ext, H[i - 1])
                leg = ext - origin
                if leg < 1.5 * atr[i] or C[i] < origin:
                    if C[i] < origin:
                        break
                    continue
                top, bot = ext - 0.618 * leg, ext - 0.786 * leg
                if L[i] <= top and C[i] > bot and C[i] > O[i] and C[i] < ext:
                    lng[i], sl_l[i], tp_l[i] = True, origin - 0.1 * atr[i], ext + 0.272 * leg
                    break
        else:
            origin = float(H[max(0, e["from_idx"]):i0 + 1].max())
            ext = L[i0]
            for i in range(i0 + 1, end):
                ext = min(ext, L[i - 1])
                leg = origin - ext
                if leg < 1.5 * atr[i] or C[i] > origin:
                    if C[i] > origin:
                        break
                    continue
                top, bot = ext + 0.786 * leg, ext + 0.618 * leg
                if H[i] >= bot and C[i] < top and C[i] < O[i] and C[i] > ext:
                    sht[i], sl_s[i], tp_s[i] = True, origin + 0.1 * atr[i], ext - 0.272 * leg
                    break
    return _sig(ctx, lng, sht, sl_l, sl_s, tp_l, tp_s)


def _order_blocks(ctx):
    """(direction, idx, bottom, top, event_idx) for the order block behind every break of structure."""
    def f():
        out = []
        O, C, L, H = ctx.o, ctx.c, ctx.l, ctx.h
        for e in ctx.A["events"]:
            i0 = e["idx"]
            bull = e["direction"] == "bullish"
            for j in range(i0 - 1, max(-1, i0 - 12), -1):
                if (bull and C[j] < O[j]) or (not bull and C[j] > O[j]):
                    out.append((1 if bull else -1, j, float(L[j]), float(H[j]), i0))
                    break
        return out
    return ctx.get("obs", f)


def sig_ob_retest(ctx):
    n, L, H, C, O, atr = ctx.n, ctx.l, ctx.h, ctx.c, ctx.o, ctx.atr
    lng, sht = _zero(n), _zero(n)
    sl_l, sl_s = np.full(n, np.nan), np.full(n, np.nan)
    for d, j, bot, top, i0 in _order_blocks(ctx):
        for i in range(i0 + 1, min(n, i0 + 60)):
            if d == 1:
                if C[i] < bot:
                    break
                if L[i] <= top and C[i] > O[i] and C[i] >= bot:
                    lng[i], sl_l[i] = True, bot - 0.1 * atr[i]
                    break
            else:
                if C[i] > top:
                    break
                if H[i] >= bot and C[i] < O[i] and C[i] <= top:
                    sht[i], sl_s[i] = True, top + 0.1 * atr[i]
                    break
    return _sig(ctx, lng, sht, sl_l, sl_s)


def sig_breaker_block(ctx):
    n, L, H, C, atr = ctx.n, ctx.l, ctx.h, ctx.c, ctx.atr
    lng, sht = _zero(n), _zero(n)
    sl_l, sl_s = np.full(n, np.nan), np.full(n, np.nan)
    for d, j, bot, top, i0 in _order_blocks(ctx):
        broke = None
        for i in range(i0 + 1, min(n, i0 + 80)):
            if broke is None:
                if d == -1 and C[i] > top:            # a bearish block closed through upward → bullish breaker
                    broke = i
                elif d == 1 and C[i] < bot:
                    broke = i
                continue
            if i - broke > 30:
                break
            if d == -1:
                if C[i] < bot:
                    break
                if L[i] <= top and C[i] > top:
                    lng[i], sl_l[i] = True, bot - 0.1 * atr[i]
                    break
            else:
                if C[i] > top:
                    break
                if H[i] >= bot and C[i] < bot:
                    sht[i], sl_s[i] = True, top + 0.1 * atr[i]
                    break
    return _sig(ctx, lng, sht, sl_l, sl_s)


def sig_fvg_fill(ctx):
    n, L, H, C, O = ctx.n, ctx.l, ctx.h, ctx.c, ctx.o
    lng, sht = _zero(n), _zero(n)
    sl_l, sl_s = np.full(n, np.nan), np.full(n, np.nan)
    tr = ctx.trend
    for g in ctx.A.get("fvgs", []):
        i0 = g["idx"] + 2
        mid = (g["top"] + g["bottom"]) / 2
        for i in range(i0, min(n, i0 + 40)):
            if g["direction"] == "bullish":
                if C[i] < g["bottom"]:
                    break
                if tr[i] == 1 and L[i] <= g["top"] and C[i] > mid and C[i] > O[i]:
                    lng[i] = True
                    sl_l[i] = min(g["bottom"], float(L[max(0, i - 5):i + 1].min())) - 0.15 * ctx.atr[i]
                    break
            else:
                if C[i] > g["top"]:
                    break
                if tr[i] == -1 and H[i] >= g["bottom"] and C[i] < mid and C[i] < O[i]:
                    sht[i] = True
                    sl_s[i] = max(g["top"], float(H[max(0, i - 5):i + 1].max())) + 0.15 * ctx.atr[i]
                    break
    return _sig(ctx, lng, sht, sl_l, sl_s)


def sig_turtle_soup(ctx):
    L, H, C, atr = ctx.l, ctx.h, ctx.c, ctx.atr
    ll, hh = _ll(L, 20), _hh(H, 20)
    rng = np.maximum(H - L, 1e-12)
    lng = _true((L < ll) & (C > ll) & ((C - L) / rng >= 0.5))
    sht = _true((H > hh) & (C < hh) & ((H - C) / rng >= 0.5))
    return _sig(ctx, lng, sht, L - 0.1 * atr, H + 0.1 * atr)


def sig_silver_bullet(ctx):
    n, L, H, atr, hr = ctx.n, ctx.l, ctx.h, ctx.atr, ctx.ny_hour
    win = ((hr >= 3) & (hr < 4)) | ((hr >= 10) & (hr < 11)) | ((hr >= 14) & (hr < 15))
    ssl, bsl = _zero(n), _zero(n)
    for s in ctx.A["sweeps"]:
        if 0 <= s["idx"] < n:
            (ssl if s["side"] == "sell-side" else bsl)[s["idx"]] = True
    ssl, bsl = _recent(ssl, 20), _recent(bsl, 20)
    bull_fvg = (L > _prev(H, 2)) & ((L - _prev(H, 2)) >= 0.2 * atr)
    bear_fvg = (H < _prev(L, 2)) & ((_prev(L, 2) - H) >= 0.2 * atr)
    lng = _true(win & ssl & bull_fvg)
    sht = _true(win & bsl & bear_fvg)
    return _sig(ctx, lng, sht, _prev(L, 2) - 0.1 * atr, _prev(H, 2) + 0.1 * atr)


def sig_power_of_3(ctx):
    L, H, C, atr, hr = ctx.l, ctx.h, ctx.c, ctx.atr, ctx.ny_hour
    ahi, alo = ctx.asia()
    win = ((hr >= 2) & (hr < 5)) | ((hr >= 7) & (hr < 10))
    lng = _true(win & (L < alo) & (C > alo))
    sht = _true(win & (H > ahi) & (C < ahi))
    return _sig(ctx, lng, sht, L - 0.15 * atr, H + 0.15 * atr, ahi, alo)


def sig_premium_discount(ctx):
    L, H, C, atr = ctx.l, ctx.h, ctx.c, ctx.atr
    hh = pd.Series(H).rolling(50, min_periods=50).max().to_numpy()
    ll = pd.Series(L).rolling(50, min_periods=50).min().to_numpy()
    eq = (hh + ll) / 2
    tr = ctx.trend
    lng = _true((tr == 1) & (C < eq) & (C > _prev(H)))
    sht = _true((tr == -1) & (C > eq) & (C < _prev(L)))
    return _sig(ctx, lng, sht, ll - 0.1 * atr, hh + 0.1 * atr, hh, ll)


def sig_liquidity_draw(ctx):
    n, C = ctx.n, ctx.c
    bull, bear = _events_arrays(ctx)
    lng, sht = _zero(n), _zero(n)
    tp_l, tp_s = np.full(n, np.nan), np.full(n, np.nan)
    piv = [p for p in ctx.pivots()]
    highs = [p for p in piv if p["type"] == "high"]
    lows = [p for p in piv if p["type"] == "low"]
    for i in np.flatnonzero(bull | bear):
        a = ctx.atr[i]
        if bull[i]:
            pairs = [(x, y) for x, y in zip(highs, highs[1:]) if y["confirmed_idx"] <= i and abs(x["price"] - y["price"])
                     <= 0.15 * a and max(x["price"], y["price"]) > C[i] + a]
            pools = [max(x["price"], y["price"]) for x, y in pairs if ctx.h[y["idx"] + 1:i + 1].max(initial=-np.inf)
                     <= max(x["price"], y["price"])]
            if pools:
                lng[i], tp_l[i] = True, min(pools)
        if bear[i]:
            pairs = [(x, y) for x, y in zip(lows, lows[1:]) if y["confirmed_idx"] <= i and abs(x["price"] - y["price"])
                     <= 0.15 * a and min(x["price"], y["price"]) < C[i] - a]
            pools = [min(x["price"], y["price"]) for x, y in pairs if ctx.l[y["idx"] + 1:i + 1].min(initial=np.inf)
                     >= min(x["price"], y["price"])]
            if pools:
                sht[i], tp_s[i] = True, max(pools)
    return _sig(ctx, lng, sht, tp_l=tp_l, tp_s=tp_s)


def sig_sr_bounce(ctx):
    L, H, C, O, atr = ctx.l, ctx.h, ctx.c, ctx.o, ctx.atr
    pl, pli = ctx.last_pivot("low")
    ph, phi = ctx.last_pivot("high")
    idx = np.arange(ctx.n)
    rng = np.maximum(H - L, 1e-12)
    body_lo, body_hi = np.minimum(O, C), np.maximum(O, C)
    pin_up = (body_lo - L) / rng >= 0.5
    pin_dn = (H - body_hi) / rng >= 0.5
    eng_up = (C > O) & (_prev(C) < _prev(O)) & (C >= _prev(O))
    eng_dn = (C < O) & (_prev(C) > _prev(O)) & (C <= _prev(O))
    lng = _true((idx - pli >= 5) & (L <= pl + 0.15 * atr) & (L >= pl - 0.6 * atr) & (C > pl) & (pin_up | eng_up))
    sht = _true((idx - phi >= 5) & (H >= ph - 0.15 * atr) & (H <= ph + 0.6 * atr) & (C < ph) & (pin_dn | eng_dn))
    return _sig(ctx, lng, sht, np.minimum(L, pl) - 0.25 * atr, np.maximum(H, ph) + 0.25 * atr)


def sig_breakout_retest(ctx):
    n, L, H, C, atr = ctx.n, ctx.l, ctx.h, ctx.c, ctx.atr
    hh, ll = _hh(H, 20), _ll(L, 20)
    up = _true(C > hh)
    dn = _true(C < ll)
    lng, sht = _zero(n), _zero(n)
    sl_l, sl_s = np.full(n, np.nan), np.full(n, np.nan)
    for b in np.flatnonzero(up):
        lvl = hh[b]
        for i in range(b + 1, min(n, b + 11)):
            if C[i] < lvl - 0.5 * atr[i]:
                break
            if L[i] <= lvl + 0.2 * atr[i] and C[i] > lvl:
                lng[i], sl_l[i] = True, lvl - 1.0 * atr[i]
                break
    for b in np.flatnonzero(dn):
        lvl = ll[b]
        for i in range(b + 1, min(n, b + 11)):
            if C[i] > lvl + 0.5 * atr[i]:
                break
            if H[i] >= lvl - 0.2 * atr[i] and C[i] < lvl:
                sht[i], sl_s[i] = True, lvl + 1.0 * atr[i]
                break
    return _sig(ctx, lng, sht, sl_l, sl_s)


def sig_chart_patterns(ctx):
    n, L, H, C, atr = ctx.n, ctx.l, ctx.h, ctx.c, ctx.atr
    lng, sht = _zero(n), _zero(n)
    sl_l, sl_s = np.full(n, np.nan), np.full(n, np.nan)
    tp_l, tp_s = np.full(n, np.nan), np.full(n, np.nan)
    piv = ctx.pivots(4, 4)
    lows = [p for p in piv if p["type"] == "low"]
    highs = [p for p in piv if p["type"] == "high"]
    for a, b in zip(lows, lows[1:]):
        if b["idx"] - a["idx"] < 5 or abs(a["price"] - b["price"]) > 0.3 * atr[b["idx"]]:
            continue
        neck = float(H[a["idx"]:b["idx"] + 1].max())
        bottom = min(a["price"], b["price"])
        for i in range(b["confirmed_idx"], min(n, b["confirmed_idx"] + 40)):
            if L[i] < bottom - 0.3 * atr[i]:
                break
            if C[i] > neck:
                lng[i], sl_l[i], tp_l[i] = True, bottom - 0.15 * atr[i], neck + (neck - bottom)
                break
    for a, b in zip(highs, highs[1:]):
        if b["idx"] - a["idx"] < 5 or abs(a["price"] - b["price"]) > 0.3 * atr[b["idx"]]:
            continue
        neck = float(L[a["idx"]:b["idx"] + 1].min())
        top = max(a["price"], b["price"])
        for i in range(b["confirmed_idx"], min(n, b["confirmed_idx"] + 40)):
            if H[i] > top + 0.3 * atr[i]:
                break
            if C[i] < neck:
                sht[i], sl_s[i], tp_s[i] = True, top + 0.15 * atr[i], neck - (top - neck)
                break
    return _sig(ctx, lng, sht, sl_l, sl_s, tp_l, tp_s)


def sig_engulfing_trend(ctx):
    L, H, C, O, atr = ctx.l, ctx.h, ctx.c, ctx.o, ctx.atr
    e20, e50, e200 = ctx.ema(20), ctx.ema(50), ctx.ema(200)
    eng_up = (C > O) & (_prev(C) < _prev(O)) & (C >= _prev(O)) & (O <= _prev(C))
    eng_dn = (C < O) & (_prev(C) > _prev(O)) & (C <= _prev(O)) & (O >= _prev(C))
    near_up = np.minimum(np.abs(L - e20), np.abs(L - e50)) <= 0.8 * atr
    near_dn = np.minimum(np.abs(H - e20), np.abs(H - e50)) <= 0.8 * atr
    lng = _true(eng_up & (e50 > e200) & near_up)
    sht = _true(eng_dn & (e50 < e200) & near_dn)
    return _sig(ctx, lng, sht, np.minimum(L, _prev(L)) - 0.15 * atr, np.maximum(H, _prev(H)) + 0.15 * atr)


def sig_inside_bar(ctx):
    L, H, C, atr = ctx.l, ctx.h, ctx.c, ctx.atr
    inside = (_prev(H) < _prev(H, 2)) & (_prev(L) > _prev(L, 2))
    e50 = ctx.ema(50)
    lng = _true(inside & (C > _prev(H, 2)) & (C > e50))
    sht = _true(inside & (C < _prev(L, 2)) & (C < e50))
    return _sig(ctx, lng, sht, _prev(L, 2) - 0.1 * atr, _prev(H, 2) + 0.1 * atr)


def sig_outside_bar(ctx):
    L, H, C, O, atr = ctx.l, ctx.h, ctx.c, ctx.o, ctx.atr
    outside = (H > _prev(H)) & (L < _prev(L))
    rng = np.maximum(H - L, 1e-12)
    lng = _true(outside & (C > O) & ((C - L) / rng >= 0.7))
    sht = _true(outside & (C < O) & ((H - C) / rng >= 0.7))
    return _sig(ctx, lng, sht, L - 0.1 * atr, H + 0.1 * atr)


def sig_pin_bar(ctx):
    L, H, C, O, atr = ctx.l, ctx.h, ctx.c, ctx.o, ctx.atr
    rng = np.maximum(H - L, 1e-12)
    body = np.maximum(np.abs(C - O), 1e-12)
    lo_w = np.minimum(O, C) - L
    up_w = H - np.maximum(O, C)
    ll, hh = _ll(L, 20), _hh(H, 20)
    lng = _true((lo_w >= 2 * body) & (lo_w / rng >= 0.6) & (L <= ll + 0.3 * atr))
    sht = _true((up_w >= 2 * body) & (up_w / rng >= 0.6) & (H >= hh - 0.3 * atr))
    return _sig(ctx, lng, sht, L - 0.15 * atr, H + 0.15 * atr)


def sig_supertrend(ctx):
    line, d = ctx.supertrend()
    pd_ = _prev(d.astype(float))
    lng = _true((d == 1) & (pd_ == -1))
    sht = _true((d == -1) & (pd_ == 1))
    return _sig(ctx, lng, sht, line - 0.1 * ctx.atr, line + 0.1 * ctx.atr)


def sig_psar(ctx):
    sar, d = ctx.psar()
    pd_ = _prev(d.astype(float))
    lng = _true((d == 1) & (pd_ == -1))
    sht = _true((d == -1) & (pd_ == 1))
    return _sig(ctx, lng, sht, sar - 0.1 * ctx.atr, sar + 0.1 * ctx.atr)


def sig_two_ma_cross(ctx):
    f, s = ctx.ema(9), ctx.ema(21)
    return _sig(ctx, _cross_up(f, s), _cross_dn(f, s))


def sig_ma_cross(ctx):
    m = ctx.sma(20)
    return _sig(ctx, _cross_up(ctx.c, m), _cross_dn(ctx.c, m))


def sig_macd_cross(ctx):
    _, _, hist = ctx.macd()
    z = np.zeros(ctx.n)
    return _sig(ctx, _cross_up(hist, z), _cross_dn(hist, z))


def sig_momentum(ctx):
    mom = ctx.c - _prev(ctx.c, 12)
    z = np.zeros(ctx.n)
    rising = mom > _prev(mom)
    return _sig(ctx, _cross_up(mom, z) & _true(rising), _cross_dn(mom, z) & _true(~rising & ~np.isnan(mom)))


def _rating(ctx):
    def f():
        c = ctx.c
        votes = []
        for nn in (10, 20, 30, 50, 100, 200):
            for m in (ctx.sma(nn), ctx.ema(nn)):
                votes.append(np.where(np.isnan(m), 0, np.sign(c - m)))
        r = ctx.rsi(14)
        votes.append(np.where(r < 30, 1, np.where(r > 70, -1, 0)))
        k, d = ctx.stoch()
        votes.append(np.where((k < 20) & (k > d), 1, np.where((k > 80) & (k < d), -1, 0)))
        m, s, _ = ctx.macd()
        votes.append(np.nan_to_num(np.sign(m - s)))
        mom = c - _prev(c, 10)
        votes.append(np.nan_to_num(np.sign(mom - _prev(mom))))
        return np.mean(np.vstack(votes), axis=0)
    return ctx.get("rating", f)


def sig_tech_ratings(ctx):
    r = _rating(ctx)
    pr = _prev(r)
    return _sig(ctx, _true((r > 0.1) & (pr <= 0.1)), _true((r < -0.1) & (pr >= -0.1)))


def sig_bar_updn(ctx):
    C, O = ctx.c, ctx.o
    return _sig(ctx, _true((C > O) & (O > _prev(C))), _true((C < O) & (O < _prev(C))))


def sig_consecutive(ctx):
    C = ctx.c
    up = (C > _prev(C)) & (_prev(C) > _prev(C, 2)) & (_prev(C, 2) > _prev(C, 3))
    dn = (C < _prev(C)) & (_prev(C) < _prev(C, 2)) & (_prev(C, 2) < _prev(C, 3))
    up4 = up & ~((_prev(C) > _prev(C, 2)) & (_prev(C, 2) > _prev(C, 3)) & (_prev(C, 3) > _prev(C, 4)))
    dn4 = dn & ~((_prev(C) < _prev(C, 2)) & (_prev(C, 2) < _prev(C, 3)) & (_prev(C, 3) < _prev(C, 4)))
    return _sig(ctx, _true(up4), _true(dn4))


def sig_greedy(ctx):
    C, O, H, L = ctx.c, ctx.o, ctx.h, ctx.l
    return _sig(ctx, _true((O > _prev(H)) & (C > O)), _true((O < _prev(L)) & (C < O)))


def sig_rsi_strategy(ctx):
    r = ctx.rsi(14)
    return _sig(ctx, _cross_up(r, np.full(ctx.n, 30.0)), _cross_dn(r, np.full(ctx.n, 70.0)))


def sig_bb_strategy(ctx):
    lo, mid, up = ctx.bb()
    C = ctx.c
    lng = _true((_prev(C) < _prev(lo)) & (C > lo))
    sht = _true((_prev(C) > _prev(up)) & (C < up))
    return _sig(ctx, lng, sht, tp_l=mid, tp_s=mid)


def sig_stoch_slow(ctx):
    k, d = ctx.stoch()
    lng = _cross_up(k, d) & _true(k < 25)
    sht = _cross_dn(k, d) & _true(k > 75)
    return _sig(ctx, lng, sht)


def sig_pivot_ext(ctx):
    n = ctx.n
    lng, sht = _zero(n), _zero(n)
    sl_l, sl_s = np.full(n, np.nan), np.full(n, np.nan)
    for p in ctx.pivots(4, 2):
        i = p["confirmed_idx"]
        if i >= n:
            continue
        if p["type"] == "low":
            lng[i], sl_l[i] = True, p["price"] - 0.15 * ctx.atr[i]
        else:
            sht[i], sl_s[i] = True, p["price"] + 0.15 * ctx.atr[i]
    return _sig(ctx, lng, sht, sl_l, sl_s)


def sig_channel_breakout(ctx):
    hh, ll = _hh(ctx.h, 20), _ll(ctx.l, 20)
    return _sig(ctx, _cross_up(ctx.c, hh), _cross_dn(ctx.c, ll), _ll(ctx.l, 10), _hh(ctx.h, 10))


def sig_price_channel(ctx):
    hh, ll = _hh(ctx.h, 55), _ll(ctx.l, 55)
    return _sig(ctx, _cross_up(ctx.c, hh), _cross_dn(ctx.c, ll), _ll(ctx.l, 20), _hh(ctx.h, 20))


def sig_pivot_reversal(ctx):
    ph, _ = ctx.last_pivot("high", 4, 2)
    pl, _ = ctx.last_pivot("low", 4, 2)
    return _sig(ctx, _cross_up(ctx.c, ph), _cross_dn(ctx.c, pl), pl - 0.1 * ctx.atr, ph + 0.1 * ctx.atr)


def sig_keltner(ctx):
    m = ctx.ema(20)
    a = ctx.atr_n(10)
    up, lo = m + 2 * a, m - 2 * a
    return _sig(ctx, _cross_up(ctx.c, up), _cross_dn(ctx.c, lo), m - 0.5 * a, m + 0.5 * a)


def sig_adx_breakout(ctx):
    adx, _, _ = ctx.adx()
    quiet = pd.Series(adx < 20).rolling(5, min_periods=5).min().to_numpy() > 0
    quiet = _prev(quiet.astype(float)) > 0
    hh, ll = _hh(ctx.h, 20), _ll(ctx.l, 20)
    return _sig(ctx, _true(quiet & (ctx.c > hh)), _true(quiet & (ctx.c < ll)), ll, hh)


def sig_vol_expansion(ctx):
    C, atr = ctx.c, ctx.atr
    return _sig(ctx, _true(C > _prev(C) + 0.75 * _prev(atr)), _true(C < _prev(C) - 0.75 * _prev(atr)))


def sig_scalp_ema_vwap(ctx):
    L, H, C, O, atr = ctx.l, ctx.h, ctx.c, ctx.o, ctx.atr
    e9, e21, vw = ctx.ema(9), ctx.ema(21), ctx.vwap()
    lng = _true((e9 > e21) & (C > vw) & (L <= e9 + 0.1 * atr) & (C > e9) & (C > O))
    sht = _true((e9 < e21) & (C < vw) & (H >= e9 - 0.1 * atr) & (C < e9) & (C < O))
    lo4 = pd.Series(L).rolling(4, min_periods=1).min().to_numpy()
    hi4 = pd.Series(H).rolling(4, min_periods=1).max().to_numpy()
    return _sig(ctx, lng, sht, np.minimum(lo4, C - 0.8 * atr) - 0.1 * atr, np.maximum(hi4, C + 0.8 * atr) + 0.1 * atr)


def sig_scalp_london(ctx):
    n, hr = ctx.n, ctx.ny_hour
    london = (hr >= 2) & (hr < 5)
    ssl, bsl = _zero(n), _zero(n)
    for s in ctx.A["sweeps"]:
        if 0 <= s["idx"] < n:
            (ssl if s["side"] == "sell-side" else bsl)[s["idx"]] = True
    bull, bear = _events_arrays(ctx)
    hh = pd.Series(ctx.h).rolling(50, min_periods=50).max().to_numpy()
    ll = pd.Series(ctx.l).rolling(50, min_periods=50).min().to_numpy()
    mid = (hh + ll) / 2
    lng = _true(london & _recent(ssl, 6) & _recent(bull, 3) & (ctx.c < mid))
    sht = _true(london & _recent(bsl, 6) & _recent(bear, 3) & (ctx.c > mid))
    return _sig(ctx, lng, sht)


def sig_scalp_range(ctx):
    adx, _, _ = ctx.adx()
    lo, mid, up = ctx.bb()
    C = ctx.c
    quiet = adx < 20
    lng = _true(quiet & (_prev(C) < _prev(lo)) & (C > lo))
    sht = _true(quiet & (_prev(C) > _prev(up)) & (C < up))
    return _sig(ctx, lng, sht, tp_l=mid, tp_s=mid)


def sig_scalp_burst(ctx):
    L, H, C, O, V, atr, hr = ctx.l, ctx.h, ctx.c, ctx.o, ctx.v, ctx.atr, ctx.ny_hour
    kz = ((hr >= 2) & (hr < 5)) | ((hr >= 7) & (hr < 10))
    rng = np.maximum(H - L, 1e-12)
    vavg = _prev(pd.Series(V).rolling(20, min_periods=20).mean().to_numpy())
    big = (rng > 1.8 * _prev(atr)) & ((V > 1.5 * vavg) | (np.nan_to_num(vavg) == 0))
    lng = _true(kz & big & (C > O) & ((C - L) / rng >= 0.8))
    sht = _true(kz & big & (C < O) & ((H - C) / rng >= 0.8))
    midc = (H + L) / 2
    return _sig(ctx, lng, sht, midc, midc)


def sig_ft_trend(ctx):
    C, O, H, L = ctx.c, ctx.o, ctx.h, ctx.l
    e8, e21, e50 = ctx.ema(8), ctx.ema(21), ctx.ema(50)
    up_stack = (e8 > e21) & (e21 > e50)
    dn_stack = (e8 < e21) & (e21 < e50)
    lng = _true(up_stack & (_prev(C) < _prev(O)) & (C > O) & (C > _prev(H)))
    sht = _true(dn_stack & (_prev(C) > _prev(O)) & (C < O) & (C < _prev(L)))
    return _sig(ctx, lng, sht)


def sig_ft_reversal(ctx):
    C, O, H, L = ctx.c, ctx.o, ctx.h, ctx.l
    r = ctx.rsi(7)
    rng = np.maximum(H - L, 1e-12)
    ll, hh = _ll(L, 20), _hh(H, 20)
    lng = _true((r < 20) & (L <= ll) & ((np.minimum(O, C) - L) / rng >= 0.5))
    sht = _true((r > 80) & (H >= hh) & ((H - np.maximum(O, C)) / rng >= 0.5))
    return _sig(ctx, lng, sht)


def sig_ft_level(ctx):
    C, H, L, atr = ctx.c, ctx.h, ctx.l, ctx.atr
    pl, pli = ctx.last_pivot("low")
    ph, phi = ctx.last_pivot("high")
    idx = np.arange(ctx.n)
    lng = _true((idx - pli >= 5) & (np.abs(L - pl) <= 0.2 * atr) & (C > pl + 0.3 * atr))
    sht = _true((idx - phi >= 5) & (np.abs(H - ph) <= 0.2 * atr) & (C < ph - 0.3 * atr))
    return _sig(ctx, lng, sht)


def sig_ft_momentum(ctx):
    C, O = ctx.c, ctx.o
    b = C - O
    _, mid, _ = ctx.bb()
    up = (b > 0) & (_prev(b) > 0) & (_prev(b, 2) > 0) & (b > _prev(b)) & (_prev(b) > _prev(b, 2)) & (C > mid)
    dn = (b < 0) & (_prev(b) < 0) & (_prev(b, 2) < 0) & (b < _prev(b)) & (_prev(b) < _prev(b, 2)) & (C < mid)
    return _sig(ctx, _true(up), _true(dn))


SIGNALS = {k[4:]: v for k, v in globals().items() if k.startswith("sig_") and callable(v)}
assert set(SIGNALS) == set(BY_ID), sorted(set(BY_ID) ^ set(SIGNALS))


def signals(ctx, sid):
    """The strategy's signals on every candle (cached on the context)."""
    def f():
        s = BY_ID[sid]
        out = SIGNALS[sid](ctx)
        warm = max(s["min_bars"], 60)
        out["long"][:warm] = False
        out["short"][:warm] = False
        if s["intraday"] and ctx.tf in ("1D", "3D", "1W", "1M"):
            out["long"][:] = False
            out["short"][:] = False
        return out
    return ctx.get(("sig", sid), f)


# ───────────────────────────── stops, the liquidity guard, targets ─────────────────────────────
def default_stop(ctx, sid, i, d, entry):
    """The strategy's own stop for a trade signalled on candle i."""
    s = BY_ID[sid]
    sg = signals(ctx, sid)
    arr = sg["sl_long"] if d == 1 else sg["sl_short"]
    a = ctx.atr[i]
    stop = None
    if arr is not None and i < len(arr) and np.isfinite(arr[i]):
        stop = float(arr[i])
    elif s["stop"] == "atr":
        stop = entry - d * 1.5 * a
    else:                                               # the last swing beyond entry (structure stop)
        lb = 10
        if d == 1:
            stop = float(ctx.l[max(0, i - lb + 1):i + 1].min()) - 0.2 * a
        else:
            stop = float(ctx.h[max(0, i - lb + 1):i + 1].max()) + 0.2 * a
    if d * (entry - stop) < 0.5 * a:                    # never tighter than half an ATR (noise)
        stop = entry - d * 0.5 * a
    cap = 1.6 if s["group"] == "scalp" else 4.0         # a scalp is a tight-stop trade; nothing gets a huge stop
    if d * (entry - stop) > cap * a:
        stop = entry - d * cap * a
    return stop


def guard_stop(ctx, i, d, entry, stop, depth=None):
    """Move a stop past the untaken swing lows/highs (and equal lows/highs) that sit just beyond it or between it and
    the entry, by the typical depth of a stop hunt on this chart. Never moves a stop closer.
    Returns (stop, pool) where pool is the liquidity level the stop was moved past (None if it was fine)."""
    a = ctx.atr[i]
    depth = ctx.sweep_depth() if depth is None else depth
    buf = (depth + 0.05) * a
    pools = ctx.pools(i, -d)
    if d == 1:
        cand = [p for p in pools if stop - buf - 0.5 * a <= p < entry - 0.15 * a]
        if not cand:
            return stop, None
        p = min(cand)
        new = p - buf
        return (min(stop, new), p) if new < stop else (stop, None)
    cand = [p for p in pools if entry + 0.15 * a < p <= stop + buf + 0.5 * a]
    if not cand:
        return stop, None
    p = max(cand)
    new = p + buf
    return (max(stop, new), p) if new > stop else (stop, None)


def default_target(ctx, sid, i, d, entry, stop):
    s = BY_ID[sid]
    sg = signals(ctx, sid)
    arr = sg["tp_long"] if d == 1 else sg["tp_short"]
    risk = abs(entry - stop)
    if arr is not None and i < len(arr) and np.isfinite(arr[i]) and d * (arr[i] - entry) >= 0.8 * risk:
        return float(arr[i])
    return entry + d * s["rr"] * risk


# ───────────────────────────── the record (backtest) ─────────────────────────────
def trade_at(ctx, sid, i, d, cost_frac=0.0, slip_frac=0.0, guard=True, depth=None):
    """One trade of a price strategy signalled on candle i in direction d, run the app's way: entry at the next
    candle's open, the strategy's stop (moved past nearby liquidity by the guard), its target, the stop first when a
    candle touches both, and a close at the time limit. Returns the trade, or None when the signal is skipped (no
    room for a stop, or costs too large for the risk)."""
    s = BY_ID[sid]
    O, H, L, C = ctx.o, ctx.h, ctx.l, ctx.c
    n = ctx.n
    e = i + 1
    if e >= n:
        return None
    entry = float(O[e])
    stop = default_stop(ctx, sid, i, d, float(C[i]))
    if d * (entry - stop) <= 0:
        return None
    pool = None
    if guard:
        stop, pool = guard_stop(ctx, i, d, entry, stop, ctx.sweep_depth() if depth is None else depth)
    risk = d * (entry - stop)
    if risk < 0.25 * ctx.atr[i] or cost_frac * entry > 0.5 * risk:
        return None
    target = default_target(ctx, sid, i, d, entry, stop)
    if d * (target - entry) <= 0:
        return None
    last = min(n - 1, e + s["hold"] - 1)
    lo, hi = L[e:last + 1], H[e:last + 1]
    hs = np.flatnonzero(lo <= stop) if d == 1 else np.flatnonzero(hi >= stop)
    ht = np.flatnonzero(hi >= target) if d == 1 else np.flatnonzero(lo <= target)
    ks = int(hs[0]) if len(hs) else None
    kt = int(ht[0]) if len(ht) else None
    times = ctx.df.index
    base = dict(side="long" if d == 1 else "short", i=int(i), entry_i=int(e), t=str(times[e]), entry=entry,
                stop=float(stop), target=float(target), guarded=pool is not None)
    if ks is not None and (kt is None or ks <= kt):           # the stop first, also when one candle touches both
        j = e + ks
        gap = (O[j] < stop) if d == 1 else (O[j] > stop)
        exit_px, reason = (O[j] if gap else stop) - d * slip_frac * entry, "stop"
    elif kt is not None:
        j = e + kt
        exit_px, reason = target, "target"
    else:
        if last >= n - 1 and n - 1 - e < s["hold"] - 1:
            return dict(base, open=True)
        j = last
        exit_px, reason = float(C[last]), "time"
    r = d * (exit_px - entry) / risk - cost_frac * entry / risk
    return dict(base, open=False, exit_i=int(j), exit=float(exit_px), reason=reason, r=float(r), bars=int(j - e + 1))


def backtest(ctx, sid, cost_frac=0.0, slip_frac=0.0, guard=True, start=0, overlap=False):
    """Trades the strategy would have taken on this frame. Price strategies return R multiples after costs;
    fixed-time strategies return wins and losses at expiry. overlap=True takes every signal, also while an earlier
    trade is still running (used to train the strategy model); the app's record takes one trade at a time."""
    s = BY_ID[sid]
    sg = signals(ctx, sid)
    lng, sht = sg["long"], sg["short"]
    C = ctx.c
    n = ctx.n
    times = ctx.df.index
    trades = []
    i = max(start, 0)
    depth = ctx.sweep_depth()
    if s["fixed_time"]:
        N = s["expiry"]
        while i < n - 1:
            d = 1 if lng[i] else -1 if sht[i] else 0
            if d == 0:
                i += 1
                continue
            if i + N >= n:
                if not overlap:
                    trades.append(dict(open=True, side="up" if d == 1 else "down", i=int(i), t=str(times[i]),
                                       entry=float(C[i]), expiry=N))
                break
            win = d * (C[i + N] - C[i]) > 0
            trades.append(dict(open=False, side="up" if d == 1 else "down", i=int(i), t=str(times[i]),
                               entry=float(C[i]), exit=float(C[i + N]), expiry=N, win=bool(win),
                               r=float(FT_PAYOUT if win else -1.0)))
            i += 1 if overlap else N
        return trades
    while i < n - 1:
        d = 1 if lng[i] else -1 if sht[i] else 0
        if d == 0:
            i += 1
            continue
        t = trade_at(ctx, sid, i, d, cost_frac, slip_frac, guard, depth)
        if t is None:
            i += 1
            continue
        if t["open"]:
            if not overlap:
                trades.append(t)
                break
            i += 1
            continue
        trades.append(t)
        i = i + 1 if overlap else t["exit_i"] + 1
    return trades


def record(trades, fixed_time=False):
    """Win rate, average R, profit factor ... of a list of closed trades, and a plain verdict."""
    closed = [t for t in trades if not t.get("open")]
    n = len(closed)
    if n == 0:
        return dict(trades=0, verdict="no trades on the loaded candles")
    rs = np.array([t["r"] for t in closed], dtype=float)
    wins = rs[rs > 0]
    losses = rs[rs <= 0]
    win_rate = float((rs > 0).mean())
    out = dict(trades=n, win_rate=win_rate, avg_r=float(rs.mean()), total_r=float(rs.sum()),
               profit_factor=float(wins.sum() / -losses.sum()) if losses.sum() < 0 else None)
    eq = np.cumsum(rs)
    out["max_drawdown_r"] = float((eq - np.maximum.accumulate(np.r_[0.0, eq])[1:]).min())
    sd = float(rs.std(ddof=1)) if n > 1 else 0.0
    out["t_stat"] = float(rs.mean() / sd * math.sqrt(n)) if sd > 0 else None
    last = rs[-20:]
    out["recent_win_rate"] = float((last > 0).mean())
    out["recent_avg_r"] = float(last.mean())
    if fixed_time:
        be = 1 / (1 + FT_PAYOUT)
        out.update(breakeven=be, payout=FT_PAYOUT, edge=win_rate - be)
        out["verdict"] = ("too few calls to judge (need 30+)" if n < 30 else
                          f"wins {win_rate * 100:.0f}% vs {be * 100:.0f}% needed at an {FT_PAYOUT * 100:.0f}% payout: "
                          + ("an edge so far" if win_rate > be + 0.02 else "no edge after the payout"))
    else:
        out["stops"] = int(sum(1 for t in closed if t["reason"] == "stop"))
        out["targets"] = int(sum(1 for t in closed if t["reason"] == "target"))
        out["verdict"] = ("too few trades to judge (need 30+)" if n < 30 else
                          "positive and steady" if (out["t_stat"] or 0) >= 2 and out["avg_r"] > 0 else
                          "positive, but it could be luck" if out["avg_r"] > 0 else "loses after costs here")
    return out


def score_of(rec):
    """One number for how much to trust a strategy on this chart, from its record: around 0 = no edge,
    positive = an edge, shrunk toward 0 when there are few trades."""
    if not rec or not rec.get("trades"):
        return 0.0
    n = rec["trades"]
    shrink = n / (n + 30.0)
    if "edge" in rec:
        return float(np.clip(rec["edge"] * 4, -1, 1) * shrink)
    return float(np.clip(rec["avg_r"], -1, 1) * shrink)


# ───────────────────────────── live: the newest closed candle ─────────────────────────────
def live(ctx, sid, price=None, cost_frac=0.0, max_age=3):
    """What the strategy says now: a signal on one of the last `max_age` closed candles that is still valid (stop not
    hit, price not already more than 0.6R past the entry), with its entry, stop and target; else what to wait for."""
    s = BY_ID[sid]
    sg = signals(ctx, sid)
    n = ctx.n
    price = float(ctx.c[-1] if price is None else price)
    out = dict(id=sid, name=s["name"], group=s["group"], fixed_time=s["fixed_time"], signal=None,
               wait_for=s["wait_for"])
    for i in range(n - 1, max(-1, n - 1 - max_age), -1):
        d = 1 if sg["long"][i] else -1 if sg["short"][i] else 0
        if d == 0:
            continue
        age = n - 1 - i
        if s["fixed_time"]:
            if age > 0:
                continue                               # a fixed-time call is only good on the candle that made it
            out.update(signal="up" if d == 1 else "down", dir=d, idx=int(i), age=0, entry=float(ctx.c[i]),
                       expiry=s["expiry"])
            return out
        ref = float(ctx.c[i])
        stop = default_stop(ctx, sid, i, d, ref)
        stop, pool = guard_stop(ctx, i, d, ref, stop)
        risk = d * (ref - stop)
        if risk <= 0:
            continue
        seg_l = ctx.l[i + 1:] if i + 1 < n else np.array([])
        seg_h = ctx.h[i + 1:] if i + 1 < n else np.array([])
        if len(seg_l) and ((d == 1 and seg_l.min() <= stop) or (d == -1 and seg_h.max() >= stop)):
            continue                                    # stopped out already
        if d * (price - ref) > 0.6 * risk:
            out.update(missed=dict(dir=d, age=age, entry=ref))
            continue                                    # it ran without us: don't chase
        target = default_target(ctx, sid, i, d, ref, stop)
        out.update(signal="long" if d == 1 else "short", dir=d, idx=int(i), age=age, entry=ref, sl=float(stop),
                   tp=float(target), guarded_past=None if pool is None else float(pool),
                   chase=bool(d * (price - ref) > 0.15 * risk))
        return out
    # no live signal: the last one, for context
    for i in range(n - 1, max(-1, n - 200), -1):
        d = 1 if sg["long"][i] else -1 if sg["short"][i] else 0
        if d:
            out["last"] = dict(dir=d, age=int(n - 1 - i), price=float(ctx.c[i]))
            break
    return out


def fit(ctx, ids=None, cost_frac=0.0, slip_frac=0.0, guard=True):
    """Each strategy's record on this frame, best first."""
    rows = []
    for sid in clean_ids(ids) if ids else [s["id"] for s in CATALOG]:
        s = BY_ID[sid]
        if s["intraday"] and ctx.tf in ("1D", "3D", "1W", "1M"):
            continue
        try:
            tr = backtest(ctx, sid, cost_frac, slip_frac, guard=guard)
        except Exception as e:                          # one broken strategy must not break the rest
            rows.append(dict(id=sid, name=s["name"], group=s["group"], error=f"{type(e).__name__}: {e}"[:160]))
            continue
        rec = record(tr, s["fixed_time"])
        rows.append(dict(id=sid, name=s["name"], group=s["group"], fixed_time=s["fixed_time"], score=score_of(rec),
                         **rec))
    rows.sort(key=lambda r: -(r.get("score") or -9))
    return rows


def markers(ctx, sid, trades, show_from=0):
    """The strategy's trades as chart drawings (layer 'strategy'), like a strategy report on the chart."""
    s = BY_ID[sid]
    out = []
    ts = ctx.df.index
    up, dn = "#26a69a", "#ef5350"
    for t in trades:
        if t.get("open"):
            continue
        i = t.get("entry_i", t.get("i"))
        if i is None or i < show_from:
            continue
        d = 1 if t["side"] in ("long", "up") else -1
        win = t["r"] > 0
        out.append(dict(kind="marker", i=int(i), t=str(ts[i]), price=t["entry"], symbol="triangle-up" if d == 1 else "triangle-down",
                        color=up if d == 1 else dn, text=f"{s['name']} {'buy' if d == 1 else 'sell'}",
                        why=f"{s['name']}: {s['how']}"))
        j = t.get("exit_i", (t.get("i") or 0) + t.get("expiry", 0))
        if j is not None and j < ctx.n:
            out.append(dict(kind="line", i1=int(i), i2=int(j), t1=str(ts[i]), t2=str(ts[j]), p1=t["entry"], p2=t["exit"],
                            color="rgba(38,166,154,0.8)" if win else "rgba(239,83,80,0.8)", dash="dot",
                            text=(f"{t['r']:+.1f}R" if not s["fixed_time"] else ("win" if win else "loss")),
                            why=(f"Exit: {t.get('reason', 'expiry')} after {t.get('bars', t.get('expiry'))} candles, "
                                 f"{t['r']:+.2f}R after costs." if not s["fixed_time"] else
                                 f"Expiry after {t['expiry']} candles: {'won' if win else 'lost'}.")))
    return out


def indicator_lines(ctx, ids, start):
    """The indicator lines the chosen strategies use, from candle `start` on: {name: (label, color, values)}."""
    out = {}
    want = set()
    for sid in clean_ids(ids):
        want.update(BY_ID[sid]["indicators"])
    sl = slice(start, None)
    col = {"ema8": "#80deea", "ema9": "#4fc3f7", "ema20": "#ffffff", "ema21": "#ffb74d", "ema50": "#ffd54f",
           "ema200": "#9575cd", "sma20": "#90caf9", "vwap": "#ff9800"}
    for k in ("ema8", "ema9", "ema20", "ema21", "ema50", "ema200"):
        if k in want:
            out[k] = (k.upper().replace("EMA", "EMA "), col[k], ctx.ema(int(k[3:]))[sl])
    if "sma20" in want:
        out["sma20"] = ("SMA 20", col["sma20"], ctx.sma(20)[sl])
    if "vwap" in want:
        out["vwap"] = ("VWAP", col["vwap"], ctx.vwap()[sl])
    if "bb" in want:
        lo, mid, up = ctx.bb()
        out["bb_up"] = ("BB upper", "rgba(144,202,249,0.8)", up[sl])
        out["bb_mid"] = ("BB basis", "rgba(255,183,77,0.8)", mid[sl])
        out["bb_lo"] = ("BB lower", "rgba(144,202,249,0.8)", lo[sl])
    if "keltner" in want:
        m, a = ctx.ema(20), ctx.atr_n(10)
        out["kc_up"] = ("Keltner upper", "rgba(77,208,225,0.8)", (m + 2 * a)[sl])
        out["kc_lo"] = ("Keltner lower", "rgba(77,208,225,0.8)", (m - 2 * a)[sl])
    if "supertrend" in want:
        line, d = ctx.supertrend()
        out["st_up"] = ("Supertrend", "#26a69a", np.where(d == 1, line, np.nan)[sl])
        out["st_dn"] = ("Supertrend", "#ef5350", np.where(d == -1, line, np.nan)[sl])
    if "psar" in want:
        sar, _ = ctx.psar()
        out["psar"] = ("Parabolic SAR", "#ffb74d", sar[sl])
    return out


In [ ]:
%%writefile liquidity_head.py
"""liquidity_head.py: the analyst's experience of liquidity, learned from years of candles on many markets.

Stops sit just past obvious highs and lows, and price often runs those stops before it moves. This head learns, for
every pool of liquidity near price, three things that decide where a trade goes in and where its stop goes:

  * sweep     P(price trades through the pool within the next 24 candles)
  * reverse   given it does: P(the sweep closes back inside within 3 candles and then runs 1 ATR the other way
              before closing beyond the pool again), i.e. a stop hunt that turns, not a breakout
  * depth     how far past the pool such a hunt reaches (the 80th percentile, in ATR), for this pool, on this market,
              in this context: a stop placed beyond that is past where hunts usually finish

Pools are the ones a trader marks: untaken swing highs and lows (equal highs/lows flagged), the previous New York
day's high and low (intraday charts) and the Asian range high and low (1h and below). Everything is point-in-time:
a pool is only used once it was known, and only while no candle had traded through it.

Each model is checked on the last 30% of every series (never seen in training) against a simple baseline: the
distance alone for sweeps, the historical rate for reversals, and the chart's own 80th percentile (what the app used
before) for depth. The app uses a model only if it beats its baseline there (the `use` flags).

Research tool, not financial advice."""
import math
import time
import warnings

import numpy as np
import pandas as pd

import guru_model as gm
import market_data as md
import strategies as st
import structure_engine as se

VERSION = "1.0"
H = 24                  # candles ahead: is the pool taken within this many?
BACK = 3                # a hunt closes back inside within this many candles of first trading through
MOVE = 1.0              # ATR: a reversal runs this far back the other way before a close beyond the pool again
MAX_D = 8.0             # pools further than this (ATR) are left out
PER_SIDE = 3            # nearest pools per side
LOOK = 150              # swings older than this many candles are left out (the app's guard uses the same)
DEPTH_Q = 0.8           # the depth quantile the stop buffer is built from
TYPES = ("swing", "pdhl", "asia")

POOL_COLS = ["d_atr", "log_age", "is_equal", "n_equal", "n_tests", "rank", "opp_d_atr", "typ_pdhl", "typ_asia"]
COLS = POOL_COLS + gm.ALIGNED
BASE_COLS = ["d_atr", "tf_log"] + gm.CLASS_COLS


# ───────────────────────────── pools (point-in-time) ─────────────────────────────
def session_levels(ctx, tf):
    """For every candle: the previous New York day's high and low (charts below 1D) and the Asian range high and low
    (1h and below), the candle each became known, and whether price had already traded through it since."""
    n = ctx.n
    nan = np.full(n, np.nan)
    out = dict(pdh=nan.copy(), pdl=nan.copy(), asia_hi=nan.copy(), asia_lo=nan.copy(), day_start=np.zeros(n, int))
    if md.TF_MIN[tf] >= 1440:
        return out
    day = ctx.ny_day
    new = np.r_[True, day[1:] != day[:-1]]
    starts = np.flatnonzero(new)
    ds = np.repeat(starts, np.diff(np.r_[starts, n]))
    out["day_start"] = ds
    # previous day's high / low, known from the first candle of the next day
    dh = pd.Series(ctx.h).groupby(day).transform("max").to_numpy()
    dl = pd.Series(ctx.l).groupby(day).transform("min").to_numpy()
    prev_h, prev_l = np.full(n, np.nan), np.full(n, np.nan)
    for k in range(1, len(starts)):
        a, b = starts[k], (starts[k + 1] if k + 1 < len(starts) else n)
        prev_h[a:b], prev_l[a:b] = dh[starts[k - 1]], dl[starts[k - 1]]
    out["pdh"], out["pdl"] = prev_h, prev_l
    if md.TF_MIN[tf] <= 60:
        out["asia_hi"], out["asia_lo"] = ctx.asia()
    out["day_hi"] = pd.Series(ctx.h).groupby(day).cummax().to_numpy()      # running high / low since the day began
    out["day_lo"] = pd.Series(ctx.l).groupby(day).cummin().to_numpy()
    return out


def pools_at(ctx, S, i, price, a):
    """The untaken pools around candle i: list of dicts (price, side +1 above / -1 below, type, known_from, ...)."""
    T = ctx.pool_table()
    out = []
    if len(T["side"]):
        m = (T["conf"] <= i) & (T["idx"] >= max(0, i - LOOK)) & (T["taken"] > i)
        for k in np.flatnonzero(m):
            q, sd = float(T["price"][k]), int(T["side"][k])
            if sd * (q - price) <= 0:
                continue
            out.append(dict(price=q, side=sd, type="swing", known=int(T["idx"][k])))
    if S.get("day_hi") is not None:
        ds = int(S["day_start"][i])
        for key, sd, typ in (("pdh", 1, "pdhl"), ("pdl", -1, "pdhl"), ("asia_hi", 1, "asia"), ("asia_lo", -1, "asia")):
            q = S[key][i]
            if not np.isfinite(q) or sd * (q - price) <= 0:
                continue
            ran = S["day_hi"][i] if sd == 1 else S["day_lo"][i]
            if sd * (ran - q) > 0:                       # already traded through today
                continue
            out.append(dict(price=float(q), side=sd, type=typ, known=ds))
    return out


def _features(ctx, pools, i, price, a):
    """The pool features for the nearest PER_SIDE pools on each side (within MAX_D ATR)."""
    rows = []
    near = {1: [], -1: []}
    for p in pools:
        p["d_atr"] = p["side"] * (p["price"] - price) / a
        if 0 < p["d_atr"] <= MAX_D:
            near[p["side"]].append(p)
    best = {sd: min((p["d_atr"] for p in near[sd]), default=MAX_D + 2) for sd in (1, -1)}
    swings = {sd: np.array([p["price"] for p in near[sd] if p["type"] == "swing"]) for sd in (1, -1)}
    for sd in (1, -1):
        lst = sorted(near[sd], key=lambda p: p["d_atr"])[:PER_SIDE]
        for rank, p in enumerate(lst):
            sw = swings[sd]
            n_eq = int((np.abs(sw - p["price"]) <= 0.15 * a).sum()) - (1 if p["type"] == "swing" else 0) if len(sw) else 0
            lo = max(p["known"] + 1, i - LOOK)
            seg = ctx.h[lo:i + 1] if sd == 1 else ctx.l[lo:i + 1]
            tests = int((sd * (seg - p["price"]) >= -0.15 * a).sum()) if len(seg) else 0
            rows.append(dict(p, d_atr=float(p["d_atr"]), log_age=math.log1p(max(0, i - p["known"])),
                             is_equal=float(n_eq > 0), n_equal=float(n_eq), n_tests=float(tests), rank=float(rank),
                             opp_d_atr=float(min(best[-sd], MAX_D + 2)), typ_pdhl=float(p["type"] == "pdhl"),
                             typ_asia=float(p["type"] == "asia")))
    return rows


def _outcome(ctx, i, q, sd, a):
    """What happened to a pool after candle i: swept within H candles, then a reversal or not, and the hunt's depth."""
    n = ctx.n
    h, l, c = ctx.h, ctx.l, ctx.c
    seg = h[i + 1:i + 1 + H] if sd == 1 else l[i + 1:i + 1 + H]
    hit = np.flatnonzero(seg > q) if sd == 1 else np.flatnonzero(seg < q)
    if not len(hit):
        return 0, np.nan, np.nan
    j = i + 1 + int(hit[0])
    back = None
    for k in range(j, min(n, j + BACK)):
        if sd * (c[k] - q) <= 0:
            back = k
            break
    if back is None:
        return 1, 0.0, np.nan                            # a breakout: it held beyond the pool
    ext = (h[j:back + 1].max() - q) if sd == 1 else (q - l[j:back + 1].min())
    depth = float(ext / a)
    goal = q - sd * MOVE * a
    rev = 0.0
    for k in range(back + 1, min(n, back + 1 + H)):
        if (sd == 1 and l[k] <= goal) or (sd == -1 and h[k] >= goal):
            rev = 1.0
            break
        if sd * (c[k] - q) > 0:                          # closed beyond the pool again: the hunt became a breakout
            break
    return 1, rev, depth


# ───────────────────────────── training data ─────────────────────────────
def series_rows(df, tf, asset_class, sid, rng, cap, test_frac=0.3):
    """Pool rows (features + labels) for one market and timeframe."""
    df = df if "atr" in df.columns else se.add_atr(df[["open", "high", "low", "close", "volume"]].astype(float).copy())
    htf = gm.htf_frames_from(df, tf, asset_class)
    F, _ = gm.raw_features(df, tf, asset_class, htf=htf)
    ctx = st.Ctx(df, tf, None, asset_class)
    S = session_levels(ctx, tf)
    n = ctx.n
    lo, hi = min(gm.WARMUP, max(150, n // 5)), n - H - BACK - 2
    if hi - lo < 200:
        return None
    split = int(n * (1 - test_frac))
    stride = max(1, int(math.ceil((hi - lo) * 4 / max(cap, 1))))
    bars = np.arange(lo + int(rng.integers(0, stride)), hi, stride)
    bars = bars[(bars >= split) | (bars < split - H - BACK - H)]          # purge: no label crosses the split
    rows = []
    for i in bars:
        a = ctx.atr[i]
        if not np.isfinite(a) or a <= 0:
            continue
        price = float(ctx.c[i])
        for p in _features(ctx, pools_at(ctx, S, i, price, a), i, price, a):
            y, rev, depth = _outcome(ctx, i, p["price"], p["side"], a)
            rows.append(dict({k: p[k] for k in POOL_COLS}, side=p["side"], type=p["type"], bar=int(i), y_sweep=y,
                             y_rev=rev, depth=depth))
    if not rows:
        return None
    R = pd.DataFrame(rows)
    al = gm.align(F.iloc[R["bar"].to_numpy()], R["side"].to_numpy()).reset_index(drop=True)
    R = pd.concat([R.reset_index(drop=True), al], axis=1)
    R["test"] = R["bar"] >= split
    R["sid"], R["tf"], R["asset_class"] = sid, tf, asset_class
    R["tf_group"] = gm.TF_GROUP[tf]
    return R


def build(series, budget=300_000, seed=1, log=print):
    """series: list of dict(symbol, tf, asset_class, df). Returns one DataFrame of pool rows."""
    rng = np.random.default_rng(seed)
    cap = max(500, int(budget / max(1, len(series))))
    out, t0 = [], time.time()
    for k, sr in enumerate(series):
        sid = f"{sr['symbol']}|{sr['tf']}"
        try:
            R = series_rows(sr["df"], sr["tf"], sr["asset_class"], sid, rng, cap)
        except Exception as e:                                   # one bad series must not stop training
            log(f"  warning {sid}: {type(e).__name__}: {e}")
            continue
        if R is not None:
            out.append(R)
        log(f"  [{k + 1}/{len(series)}] {sid:22s} {0 if R is None else len(R):>7,} pool rows ({time.time() - t0:.0f}s)")
    if not out:
        raise RuntimeError("No liquidity rows: too little data.")
    D = pd.concat(out, ignore_index=True)
    return D.astype({c: np.float32 for c in COLS if D[c].dtype == np.float64})


# ───────────────────────────── models ─────────────────────────────
def _clf(cols, mono=None):
    from sklearn.ensemble import HistGradientBoostingClassifier
    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08, max_leaf_nodes=31, min_samples_leaf=300,
                                          l2_regularization=1.0, early_stopping=True, validation_fraction=0.1,
                                          n_iter_no_change=15, random_state=0,
                                          monotonic_cst=[int((mono or {}).get(c, 0)) for c in cols])


def _reg(q):
    from sklearn.ensemble import HistGradientBoostingRegressor
    return HistGradientBoostingRegressor(loss="quantile", quantile=q, max_iter=250, learning_rate=0.06,
                                         max_leaf_nodes=31, min_samples_leaf=200, l2_regularization=1.0,
                                         early_stopping=True, validation_fraction=0.1, n_iter_no_change=15,
                                         random_state=0)


def _fit(model, X, y, cols):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model.fit(X[cols].to_numpy(dtype=np.float32), np.asarray(y))
    model.feature_names_ = list(cols)
    return model


def _pp(m, X):
    return m.predict_proba(X[m.feature_names_].to_numpy(dtype=np.float32))[:, 1]


def _pr(m, X):
    return m.predict(X[m.feature_names_].to_numpy(dtype=np.float32))


def _chart_depth(d):
    """The app's old rule, per series: the 80th percentile of past hunts' depth (0.35 ATR with fewer than 8)."""
    d = d.dropna()
    return 0.35 if len(d) < 8 else float(np.clip(np.percentile(d, 80), 0.12, 1.5))


def train(D, log=print):
    """Walk-forward test (first 70% of every series trains, last 30% tests), the `use` gates, then final models."""
    t0 = time.time()
    tr, te = ~D["test"].to_numpy(), D["test"].to_numpy()
    Dtr, Dte = D[tr], D[te]
    card = dict(rows=dict(all=len(D), train=int(tr.sum()), test=int(te.sum()),
                          swept=int((D["y_sweep"] == 1).sum()), hunts=int(D["depth"].notna().sum()),
                          series=int(D["sid"].nunique())))
    log(f"Liquidity: {len(D):,} pool rows from {card['rows']['series']} series ({te.sum():,} for testing)")
    # sweep
    m_sw = _fit(_clf(COLS, {"d_atr": -1}), Dtr, Dtr["y_sweep"].astype(int), COLS)
    m_b = _fit(_clf(BASE_COLS, {"d_atr": -1}), Dtr, Dtr["y_sweep"].astype(int), BASE_COLS)
    y = Dte["y_sweep"].to_numpy(dtype=float)
    p, pb_ = _pp(m_sw, Dte), _pp(m_b, Dte)
    sweep = dict(rate_test=float(y.mean()), brier_model=gm._brier(p, y), brier_distance_only=gm._brier(pb_, y),
                 auc_model=gm._auc(p, y), auc_distance_only=gm._auc(pb_, y), calibration=gm.calibration(p, y))
    sweep["skill_vs_distance"] = 1 - sweep["brier_model"] / sweep["brier_distance_only"]
    by = []
    for typ, g in Dte.assign(p=p, pb=pb_).groupby("type"):
        yy = g["y_sweep"].astype(float)
        by.append(dict(type=typ, rows=len(g), rate=float(yy.mean()), auc=gm._auc(g["p"], yy),
                       skill_vs_distance=1 - gm._brier(g["p"], yy) / gm._brier(g["pb"], yy)))
    sweep["by_type"] = pd.DataFrame(by)
    card["sweep"] = sweep
    # reversal after a sweep
    S_tr, S_te = Dtr[Dtr["y_sweep"] == 1], Dte[Dte["y_sweep"] == 1]
    rev = dict(rows_train=len(S_tr), rows_test=len(S_te))
    m_rv = None
    if len(S_tr) >= 500 and len(S_te) >= 200:
        m_rv = _fit(_clf(COLS), S_tr, S_tr["y_rev"].astype(int), COLS)
        rate = S_tr.groupby(["type", "tf_group"])["y_rev"].mean()
        yr = S_te["y_rev"].to_numpy(dtype=float)
        pr = _pp(m_rv, S_te)
        base = np.array([rate.get((a_, b_), S_tr["y_rev"].mean()) for a_, b_ in zip(S_te["type"], S_te["tf_group"])])
        rev.update(rate_test=float(yr.mean()), brier_model=gm._brier(pr, yr), brier_rate=gm._brier(base, yr),
                   auc_model=gm._auc(pr, yr), calibration=gm.calibration(pr, yr))
        rev["skill_vs_rate"] = 1 - rev["brier_model"] / rev["brier_rate"]
        top = pr >= np.quantile(pr, 0.75)
        rev["top_quarter_rate"] = float(yr[top].mean())
    card["reverse"] = rev
    # depth of hunts (the stop buffer)
    H_tr, H_te = S_tr[S_tr["depth"].notna()], S_te[S_te["depth"].notna()]
    dep = dict(rows_train=len(H_tr), rows_test=len(H_te), quantile=DEPTH_Q)
    m_dp, shift = None, 0.0
    if len(H_tr) >= 500 and len(H_te) >= 200:
        # fit on the earlier 80% of the training hunts, then shift the line so 80% of the later 20% fall under it
        # (a calibrated buffer); then judge it on the test period against the chart rule
        late = (H_tr.groupby("sid")["bar"].rank(pct=True) > 0.8).to_numpy()
        m_cal = _fit(_reg(DEPTH_Q), H_tr[~late], H_tr["depth"][~late].astype(float), COLS)
        res = H_tr["depth"][late].to_numpy(dtype=float) - _pr(m_cal, H_tr[late])
        shift = float(np.quantile(res, DEPTH_Q)) if late.sum() >= 50 else 0.0
        m_dp = _fit(_reg(DEPTH_Q), H_tr, H_tr["depth"].astype(float), COLS)
        pd_ = np.clip(_pr(m_dp, H_te) + shift, 0.12, 1.5)
        chart = H_tr.groupby("sid")["depth"].apply(_chart_depth)
        cb = np.array([chart.get(s_, 0.35) for s_ in H_te["sid"]])
        yd = H_te["depth"].to_numpy(dtype=float)
        dep.update(coverage_model=float((yd <= pd_).mean()), width_model=float(pd_.mean()),
                   coverage_chart_rule=float((yd <= cb).mean()), width_chart_rule=float(cb.mean()),
                   median_depth=float(np.median(yd)), shift=shift)
    card["depth"] = dep
    # which parts the app should use: only what beat its baseline on data it never saw
    use = dict(
        sweep=bool(sweep["skill_vs_distance"] > 0.005 and sweep["auc_model"] > sweep["auc_distance_only"]),
        reverse=bool(m_rv is not None and rev.get("skill_vs_rate", 0) > 0.0 and rev.get("auc_model", 0) > 0.53),
        depth=bool(m_dp is not None and (
            abs(dep["coverage_model"] - DEPTH_Q) <= 0.03 and dep["width_model"] < dep["width_chart_rule"]
            or dep["coverage_model"] >= dep["coverage_chart_rule"] + 0.03
            and dep["width_model"] <= dep["width_chart_rule"] * 1.1)))
    card["use"] = use
    log(f"  sweep AUC {sweep['auc_model']:.3f} (distance only {sweep['auc_distance_only']:.3f}), skill "
        f"{sweep['skill_vs_distance']:+.3f} | reversal AUC {rev.get('auc_model', float('nan')):.3f} | depth coverage "
        f"{dep.get('coverage_model', float('nan')):.2f} at {dep.get('width_model', float('nan')):.2f} ATR vs chart rule "
        f"{dep.get('coverage_chart_rule', float('nan')):.2f} at {dep.get('width_chart_rule', float('nan')):.2f} ATR")
    log(f"  the app will use: {', '.join(k for k, v in use.items() if v) or 'none of the models (tables only)'}")
    # experience table, all data
    Sw = D[D["y_sweep"] == 1]
    tab = (D.groupby(["asset_class", "tf_group", "type"]).agg(pools=("y_sweep", "size"), p_sweep=("y_sweep", "mean"))
           .reset_index())
    if len(Sw):
        t2 = Sw.groupby(["asset_class", "tf_group", "type"]).agg(
            sweeps=("y_rev", "size"), p_reverse=("y_rev", "mean"),
            depth_q50=("depth", lambda x: float(np.nanquantile(x, 0.5)) if x.notna().any() else np.nan),
            depth_q80=("depth", lambda x: float(np.nanquantile(x, 0.8)) if x.notna().any() else np.nan),
            depth_q90=("depth", lambda x: float(np.nanquantile(x, 0.9)) if x.notna().any() else np.nan)).reset_index()
        tab = tab.merge(t2, on=["asset_class", "tf_group", "type"], how="left")
    # final models on everything
    models = dict(sweep=_fit(_clf(COLS, {"d_atr": -1}), D, D["y_sweep"].astype(int), COLS))
    Sall = D[D["y_sweep"] == 1]
    if m_rv is not None:
        models["reverse"] = _fit(_clf(COLS), Sall, Sall["y_rev"].astype(int), COLS)
    Hall = Sall[Sall["depth"].notna()]
    if m_dp is not None:
        models["depth"] = _fit(_reg(DEPTH_Q), Hall, Hall["depth"].astype(float), COLS)
    card["seconds"] = round(time.time() - t0, 1)
    return dict(version=VERSION, created=str(pd.Timestamp.now(tz="UTC")), models=models, cols=COLS, card=card,
                use=use, table=tab, depth_shift=shift,
                params=dict(H=H, BACK=BACK, MOVE=MOVE, MAX_D=MAX_D, LOOK=LOOK, DEPTH_Q=DEPTH_Q))


# ───────────────────────────── live ─────────────────────────────
def assess(head, ctx, row, tf, price=None):
    """The pools around the newest candle with the head's numbers. Returns a list of dicts sorted by distance:
    price, side (+1 above / -1 below), type, d_atr, p_sweep, p_reverse, depth_atr (each None when the head isn't
    used for it). None when there is no head."""
    if not head:
        return None
    i = ctx.n - 1
    a = float(ctx.atr[i])
    if not np.isfinite(a) or a <= 0:
        return None
    price = float(ctx.c[i] if price is None else price)
    S = session_levels(ctx, tf)
    rows = _features(ctx, pools_at(ctx, S, i, price, a), i, price, a)
    if not rows:
        return []
    R = pd.DataFrame(rows)
    al = gm.align_row(row, R["side"].to_numpy(dtype=float))
    X = pd.concat([R[POOL_COLS].reset_index(drop=True), al], axis=1)
    use, m = head.get("use", {}), head["models"]
    p_sw = _pp(m["sweep"], X) if use.get("sweep") and "sweep" in m else None
    p_rv = _pp(m["reverse"], X) if use.get("reverse") and "reverse" in m else None
    dp = np.clip(_pr(m["depth"], X) + head.get("depth_shift", 0.0), 0.12, 1.5) \
        if use.get("depth") and "depth" in m else None
    out = []
    for k, r in enumerate(rows):
        out.append(dict(price=float(r["price"]), side=int(r["side"]), type=r["type"], d_atr=float(r["d_atr"]),
                        equal=bool(r["is_equal"]), tests=int(r["n_tests"]),
                        p_sweep=None if p_sw is None else float(p_sw[k]),
                        p_reverse=None if p_rv is None else float(p_rv[k]),
                        depth_atr=None if dp is None else float(dp[k])))
    out.sort(key=lambda r: r["d_atr"])
    return out


def type_word(r):
    return {"pdhl": ("previous day high" if r["side"] == 1 else "previous day low"),
            "asia": ("Asian range high" if r["side"] == 1 else "Asian range low")}.get(
        r["type"], ("equal highs" if r["side"] == 1 else "equal lows") if r.get("equal") else
        ("swing high" if r["side"] == 1 else "swing low"))


In [ ]:
%%writefile strategy_head.py
"""strategy_head.py: the analyst's experience of every strategy in strategies.py, learned from years of candles.

Every signal of every strategy (smart money and ICT setups, price action, the indicator strategies, scalping and
fixed-time calls) is run on every market and timeframe exactly the way the app runs it: entry at the next candle's
open, the strategy's own stop moved past nearby liquidity by the guard, its target, costs and slippage for that market,
the stop first when one candle touches both. From those trades it learns:

  * the model   P(the trade ends in profit) for a signal, from the market's context at the signal candle (structure,
                trend on the higher timeframes, momentum, volatility, liquidity, session), the trade's geometry, the
                strategy itself, and how that strategy had been doing on that market just before (known at the time)
  * evidence    every strategy's results by market type and timeframe, and per market: what tends to work where

Tested on the last 30% of every series (never seen in training), against each strategy's own historical rate. The
app uses the per-signal odds only if they beat that rate there (the `use` flag); the evidence tables are always shown.

Research tool, not financial advice."""
import math
import time
import warnings

import numpy as np
import pandas as pd

import guru_model as gm
import strategies as st
import structure_engine as se

VERSION = "1.0"
IDS = [s["id"] for s in st.CATALOG]
GROUP_IDS = [g[0] for g in st.GROUPS]
SID_CODE = {sid: k for k, sid in enumerate(IDS)}
GROUP_CODE = {g: k for k, g in enumerate(GROUP_IDS)}
RECENT = 30                     # the strategy's last 30 closed trades on the market (known at the signal) as context
GEO = ["b_atr", "a_atr", "log_rr", "guarded", "is_ft", "expiry"]
REC = ["recent_wr", "recent_r", "recent_n"]
CAT = ["sid_code", "group_code"]
COLS = gm.ALIGNED + GEO + REC + CAT
DAILY_UP = ("1D", "3D", "1W", "1M")


def _group_of(sid):
    return st.BY_ID[sid]["group"]


# ───────────────────────────── trades ─────────────────────────────
def strategy_trades(ctx, sid, cost, slip, cap=None, rng=None):
    """Every signal of one strategy on one frame, run the app's way (signals may overlap). cap: at most this many
    signals (spread over the whole frame) to keep huge frames quick."""
    s = st.BY_ID[sid]
    if s["intraday"] and ctx.tf in DAILY_UP:
        return []
    sg = st.signals(ctx, sid)
    lng, sht = sg["long"], sg["short"]
    idx = np.flatnonzero(lng | sht)
    idx = idx[idx < ctx.n - 2]
    if cap and len(idx) > cap:
        idx = np.sort((rng or np.random.default_rng(0)).choice(idx, cap, replace=False))
    depth = ctx.sweep_depth()
    out = []
    C = ctx.c
    for i in idx:
        d = 1 if lng[i] else -1
        a = ctx.atr[i]
        if not np.isfinite(a) or a <= 0:
            continue
        if s["fixed_time"]:
            N = s["expiry"]
            if i + N >= ctx.n:
                continue
            win = d * (C[i + N] - C[i]) > 0
            out.append(dict(bar=int(i), side=d, exit_i=int(i + N), r=float(st.FT_PAYOUT if win else -1.0),
                            win=bool(win), b_atr=0.0, a_atr=0.0, log_rr=0.0, guarded=0.0, is_ft=1.0,
                            expiry=float(N)))
            continue
        t = st.trade_at(ctx, sid, i, d, cost, slip, True, depth)
        if t is None or t["open"]:
            continue
        b = abs(t["entry"] - t["stop"]) / a
        aa = abs(t["target"] - t["entry"]) / a
        out.append(dict(bar=int(i), side=d, exit_i=int(t["exit_i"]), r=float(t["r"]), win=bool(t["r"] > 0),
                        b_atr=float(b), a_atr=float(aa), log_rr=float(math.log(max(aa, 1e-6) / max(b, 1e-6))),
                        guarded=float(t["guarded"]), is_ft=0.0, expiry=0.0))
    return out


def recent_stats(trades, at):
    """The strategy's last RECENT closed trades before candle `at` (by exit): win rate, average R and how many."""
    if not trades:
        return np.nan, np.nan, 0.0
    ex = np.array([t["exit_i"] for t in trades])
    order = np.argsort(ex, kind="stable")
    ex = ex[order]
    w = np.array([t["win"] for t in trades], dtype=float)[order]
    r = np.array([t["r"] for t in trades], dtype=float)[order]
    k = int(np.searchsorted(ex, at, side="left"))
    lo = max(0, k - RECENT)
    nn = k - lo
    if nn < 5:
        return np.nan, np.nan, float(nn)
    return float(w[lo:k].mean()), float(r[lo:k].mean()), float(nn)


def _recent_arrays(trades, bars):
    """recent_stats for many signal candles at once."""
    if not trades:
        z = np.full(len(bars), np.nan)
        return z, z.copy(), np.zeros(len(bars))
    ex = np.array([t["exit_i"] for t in trades])
    order = np.argsort(ex, kind="stable")
    ex = ex[order]
    cw = np.r_[0.0, np.cumsum(np.array([t["win"] for t in trades], dtype=float)[order])]
    cr = np.r_[0.0, np.cumsum(np.array([t["r"] for t in trades], dtype=float)[order])]
    k = np.searchsorted(ex, np.asarray(bars), side="left")
    lo = np.maximum(0, k - RECENT)
    nn = (k - lo).astype(float)
    with np.errstate(invalid="ignore", divide="ignore"):
        wr = np.where(nn >= 5, (cw[k] - cw[lo]) / nn, np.nan)
        rr = np.where(nn >= 5, (cr[k] - cr[lo]) / nn, np.nan)
    return wr, rr, nn


def series_rows(df, tf, asset_class, sub, symbol, rng, cap_per_strategy, test_frac=0.3, all_cap=4000):
    """Trades of every strategy on one market and timeframe, with the features at each signal candle."""
    df = df if "atr" in df.columns else se.add_atr(df[["open", "high", "low", "close", "volume"]].astype(float).copy())
    htf = gm.htf_frames_from(df, tf, asset_class)
    F, _ = gm.raw_features(df, tf, asset_class, htf=htf)
    ctx = st.Ctx(df, tf, None, asset_class)
    cost, slip = gm.cost_of(sub)
    n = ctx.n
    split = int(n * (1 - test_frac))
    sid_series = f"{symbol}|{tf}"
    frames = []
    for sid in IDS:
        try:
            tr = strategy_trades(ctx, sid, cost, slip, cap=all_cap, rng=rng)
        except Exception:                                # one broken strategy must not stop the rest
            continue
        if not tr:
            continue
        T = pd.DataFrame(tr)
        T["recent_wr"], T["recent_r"], T["recent_n"] = _recent_arrays(tr, T["bar"].to_numpy())
        T = T[(T["bar"] >= split) | (T["exit_i"] < split)]          # purge: no training trade runs into the test
        if len(T) > cap_per_strategy:
            T = T.sample(cap_per_strategy, random_state=int(rng.integers(0, 2 ** 31 - 1))).sort_values("bar")
        T["sid"] = sid
        frames.append(T)
    if not frames:
        return None
    R = pd.concat(frames, ignore_index=True)
    al = gm.align(F.iloc[R["bar"].to_numpy()], R["side"].to_numpy()).reset_index(drop=True)
    R = pd.concat([R.reset_index(drop=True), al], axis=1)
    R["sid_code"] = R["sid"].map(SID_CODE).astype(float)
    R["group_code"] = R["sid"].map(lambda x: GROUP_CODE[_group_of(x)]).astype(float)
    R["test"] = R["bar"] >= split
    R["series"], R["symbol"], R["tf"], R["asset_class"] = sid_series, symbol, tf, asset_class
    R["tf_group"] = gm.TF_GROUP[tf]
    R["t"] = df.index[R["bar"].to_numpy()]
    return R


def build(series, budget=600_000, seed=2, log=print):
    """series: list of dict(symbol, tf, asset_class, sub, df). Returns one DataFrame of strategy trades."""
    rng = np.random.default_rng(seed)
    per_series = max(1000, int(budget / max(1, len(series))))
    cap = max(60, int(per_series / max(1, len(IDS) * 0.6)))
    out, t0 = [], time.time()
    for k, sr in enumerate(series):
        try:
            R = series_rows(sr["df"], sr["tf"], sr["asset_class"], sr["sub"], sr["symbol"], rng, cap)
        except Exception as e:
            log(f"  warning {sr['symbol']}|{sr['tf']}: {type(e).__name__}: {e}")
            continue
        if R is not None:
            out.append(R)
        log(f"  [{k + 1}/{len(series)}] {sr['symbol'] + '|' + sr['tf']:22s} {0 if R is None else len(R):>7,} trades "
            f"from {0 if R is None else R['sid'].nunique()} strategies ({time.time() - t0:.0f}s)")
    if not out:
        raise RuntimeError("No strategy trades: too little data.")
    D = pd.concat(out, ignore_index=True)
    return D.astype({c: np.float32 for c in COLS if D[c].dtype == np.float64})


# ───────────────────────────── model ─────────────────────────────
def _clf():
    from sklearn.ensemble import HistGradientBoostingClassifier
    return HistGradientBoostingClassifier(max_iter=400, learning_rate=0.06, max_leaf_nodes=31, min_samples_leaf=300,
                                          l2_regularization=1.0, early_stopping=True, validation_fraction=0.1,
                                          n_iter_no_change=20, random_state=0,
                                          categorical_features=[COLS.index(c) for c in CAT])


def _fit(X, y):
    m = _clf()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        m.fit(X[COLS].to_numpy(dtype=np.float32), np.asarray(y).astype(int))
    m.feature_names_ = list(COLS)
    return m


def _pp(m, X):
    return m.predict_proba(X[m.feature_names_].to_numpy(dtype=np.float32))[:, 1]


def payoff(D):
    """Average winning and losing R for each strategy (what a win and a loss are worth)."""
    out = {}
    for sid, g in D.groupby("sid"):
        w, l_ = g.loc[g["r"] > 0, "r"], g.loc[g["r"] <= 0, "r"]
        out[sid] = dict(win_r=float(w.mean()) if len(w) else 1.0, loss_r=float(-l_.mean()) if len(l_) else 1.0)
    return out


def ev_of(p, sid, pay):
    q = pay.get(sid, dict(win_r=1.0, loss_r=1.0))
    return p * q["win_r"] - (1 - p) * q["loss_r"]


def _t(x):
    x = np.asarray(x, dtype=float)
    if len(x) < 3 or x.std(ddof=1) == 0:
        return np.nan
    return float(x.mean() / x.std(ddof=1) * np.sqrt(len(x)))


def _blocks(D):
    return D["series"].astype(str).to_numpy() + "|" + (D["bar"].to_numpy() // 200).astype(str)


def evidence(D, keys):
    rows = []
    blk = _blocks(D)
    D = D.assign(_blk=blk)
    for k, g in D.groupby(keys):
        rows.append(dict(zip(keys, k if isinstance(k, tuple) else (k,)), trades=len(g), win_rate=float(g["win"].mean()),
                         avg_r=float(g["r"].mean()), total_r=float(g["r"].sum()),
                         t_stat=gm.block_t(g["r"], g["_blk"]) if len(g) > 30 else np.nan))
    return pd.DataFrame(rows)


def train(D, log=print):
    """Walk-forward test, the `use` gate, evidence tables, then the final model on everything."""
    t0 = time.time()
    D = D.assign(win=D["win"].astype(bool))
    tr, te = ~D["test"].to_numpy(), D["test"].to_numpy()
    Dtr, Dte = D[tr], D[te]
    card = dict(rows=dict(all=len(D), train=int(tr.sum()), test=int(te.sum()), strategies=int(D["sid"].nunique()),
                          series=int(D["series"].nunique())))
    log(f"Strategies: {len(D):,} trades of {card['rows']['strategies']} strategies on {card['rows']['series']} series "
        f"({te.sum():,} for testing)")
    m = _fit(Dtr, Dtr["win"])
    pay = payoff(Dtr)
    rate = Dtr.groupby(["sid", "tf_group"])["win"].mean()
    rate_sid = Dtr.groupby("sid")["win"].mean()
    p = _pp(m, Dte)
    y = Dte["win"].to_numpy(dtype=float)
    base = np.array([rate.get((s_, g_), rate_sid.get(s_, Dtr["win"].mean())) for s_, g_ in
                     zip(Dte["sid"], Dte["tf_group"])])
    ev = np.array([ev_of(pi, s_, pay) for pi, s_ in zip(p, Dte["sid"])])
    r = Dte["r"].to_numpy(dtype=float)
    blk = _blocks(Dte)
    sel = []
    q50, q75 = np.quantile(ev, 0.5), np.quantile(ev, 0.75)
    for name, mask in (("every signal", np.ones(len(Dte), bool)), ("model EV > 0", ev > 0),
                       ("top half by model", ev >= q50), ("top quarter by model", ev >= q75)):
        sel.append(dict(selection=name, trades=int(mask.sum()), win_rate=float(y[mask].mean()) if mask.any() else np.nan,
                        avg_net_R=float(r[mask].mean()) if mask.any() else np.nan,
                        t_stat=gm.block_t(r[mask], blk[mask]) if mask.sum() > 30 else np.nan))
    card["model"] = dict(auc_model=gm._auc(p, y), auc_rate=gm._auc(base, y), brier_model=gm._brier(p, y),
                         brier_rate=gm._brier(base, y), calibration=gm.calibration(p, y), selection=pd.DataFrame(sel))
    card["model"]["skill_vs_rate"] = 1 - card["model"]["brier_model"] / card["model"]["brier_rate"]
    per = []
    Dte_ = Dte.assign(p=p, ev=ev, _blk=blk)
    for sid, g in Dte_.groupby("sid"):
        top = g["ev"] >= g["ev"].median()
        per.append(dict(strategy=sid, name=st.BY_ID[sid]["name"], group=_group_of(sid), trades=len(g),
                        win_rate=float(g["win"].mean()), avg_r=float(g["r"].mean()),
                        t_stat=gm.block_t(g["r"], g["_blk"]) if len(g) > 30 else np.nan,
                        auc=gm._auc(g["p"], g["win"].astype(float)) if g["win"].nunique() > 1 else np.nan,
                        top_half_avg_r=float(g.loc[top, "r"].mean()) if top.any() else np.nan))
    card["per_strategy_test"] = pd.DataFrame(per).sort_values("avg_r", ascending=False).reset_index(drop=True)
    use = bool(card["model"]["auc_model"] > max(0.52, card["model"]["auc_rate"]) and card["model"]["skill_vs_rate"] > 0)
    card["use"] = dict(odds=use)
    log(f"  AUC {card['model']['auc_model']:.3f} (strategy rate alone {card['model']['auc_rate']:.3f}), Brier skill "
        f"{card['model']['skill_vs_rate']:+.3f} | the app will {'use' if use else 'NOT use'} the per-signal odds")
    # evidence on all data: what works where
    ev_class = evidence(D, ["sid", "asset_class", "tf_group"])
    ev_market = evidence(D, ["sid", "symbol", "tf"])
    m_all = _fit(D, D["win"])
    card["seconds"] = round(time.time() - t0, 1)
    return dict(version=VERSION, created=str(pd.Timestamp.now(tz="UTC")), model=m_all, cols=COLS, ids=IDS,
                card=card, use=dict(odds=use), payoff=payoff(D), evidence_class=ev_class, evidence_market=ev_market)


# ───────────────────────────── live ─────────────────────────────
def _live_recent(ctx, sid, cost, slip):
    def f():
        tr = strategy_trades(ctx, sid, cost, slip, cap=1500)
        return recent_stats(tr, ctx.n)
    return ctx.get(("head_recent", sid, round(cost, 8)), f)


def score(head, ctx, row, sid, d, entry, sl, tp, cost=0.0, slip=0.0):
    """The head's view of one live signal: P(profit), the expected R from it, and the win rate it needs.
    None when there is no head, the strategy is unknown to it, or its odds aren't used."""
    if not head or sid not in SID_CODE or not head.get("use", {}).get("odds"):
        return None
    s = st.BY_ID[sid]
    i = ctx.n - 1
    a = float(ctx.atr[i])
    if not np.isfinite(a) or a <= 0:
        return None
    if s["fixed_time"]:
        geo = dict(b_atr=0.0, a_atr=0.0, log_rr=0.0, guarded=0.0, is_ft=1.0, expiry=float(s["expiry"]))
    else:
        b, aa = abs(entry - sl) / a, abs(tp - entry) / a
        if b <= 0 or aa <= 0:
            return None
        geo = dict(b_atr=b, a_atr=aa, log_rr=math.log(aa / b), guarded=0.0, is_ft=0.0, expiry=0.0)
    wr, rr, nn = _live_recent(ctx, sid, cost, slip)
    X = gm.align_row(row, [float(d)]).assign(**geo, recent_wr=wr, recent_r=rr, recent_n=nn,
                                              sid_code=float(SID_CODE[sid]),
                                              group_code=float(GROUP_CODE[s["group"]]))
    p = float(_pp(head["model"], X)[0])
    pay = head["payoff"].get(sid, dict(win_r=1.0, loss_r=1.0))
    need = pay["loss_r"] / (pay["win_r"] + pay["loss_r"]) if (pay["win_r"] + pay["loss_r"]) > 0 else 0.5
    return dict(p_win=p, ev=float(ev_of(p, sid, head["payoff"])), needed=float(need))


def evidence_for(head, sid, asset_class, tf):
    """The strategy's history on this kind of market and timeframe group (all data), or None."""
    if not head:
        return None
    E = head.get("evidence_class")
    if E is None or not len(E):
        return None
    g = E[(E["sid"] == sid) & (E["asset_class"] == asset_class) & (E["tf_group"] == gm.TF_GROUP.get(tf, ""))]
    if not len(g):
        return None
    r = g.iloc[0]
    return dict(trades=int(r["trades"]), win_rate=float(r["win_rate"]), avg_r=float(r["avg_r"]),
                t_stat=None if pd.isna(r["t_stat"]) else float(r["t_stat"]))


def best_for(head, symbol=None, tf=None, k=5, min_trades=40):
    """Strategies that did best on this market and timeframe in the training history (after costs)."""
    if not head:
        return []
    E = head.get("evidence_market")
    if E is None or not len(E):
        return []
    g = E[(E["symbol"] == symbol) & (E["tf"] == tf) & (E["trades"] >= min_trades)]
    g = g.sort_values("avg_r", ascending=False).head(k)
    return [dict(id=r["sid"], name=st.BY_ID[r["sid"]]["name"] if r["sid"] in st.BY_ID else r["sid"],
                 trades=int(r["trades"]), win_rate=float(r["win_rate"]), avg_r=float(r["avg_r"]),
                 t_stat=None if pd.isna(r["t_stat"]) else float(r["t_stat"])) for _, r in g.iterrows()]


In [ ]:
%%writefile playbook.py
"""playbook.py: how the analyst uses the strategies a person picked.

  * plans: the analyst's own setups (OTE pullback, sweep → shift, reactions at levels, breakouts, patterns) belong to
    strategies in strategies.py; rule-based strategies add plans of their own. Only the chosen strategies' plans count.
  * the liquidity guard: every stop goes past the liquidity sitting just beyond it, by the depth stop hunts usually
    reach on this chart, and an entry right above (below) an obvious pool is flagged: wait for the sweep first.
  * records: each chosen strategy's backtest on the loaded candles. Strategies that disagree are weighed by their
    record and by the higher timeframes; a close split is "no trade".
  * the timeframe matrix: trend, last break, location and the chosen strategies' signals on every timeframe.
  * confidence: plain, modest wording. A plan is a scenario with odds, never a certainty.
  * fixed time: an up/down call with an expiry for fixed-time traders, with its record against the payout's break-even.

Research tool, not financial advice."""
import numpy as np

import guru_model as gm
import liquidity_head as lh
import market_data as md
import strategies as st
import strategy_head as sh
import structure_engine as se

# lower timeframe used to fine-tune entries, and the matrix shown for each chart timeframe
LOWER = {"1m": None, "5m": "1m", "15m": "5m", "30m": "5m", "1h": "15m", "4h": "1h", "1D": "4h", "3D": "1D",
         "1W": "1D", "1M": "1W"}
TF_WEIGHT = {"1M": 3.0, "1W": 3.0, "3D": 2.5, "1D": 2.5, "4h": 2.0, "1h": 1.5, "30m": 1.2, "15m": 1.0, "5m": 0.8,
             "1m": 0.6}


def chosen(ids):
    """The strategies the person picked (catalog ids), or None for the analyst's own mix."""
    out = st.clean_ids(ids)
    return out or None


def strategy_of(p):
    """Which strategy one of the analyst's own plans belongs to."""
    if p.get("strategy"):
        return p["strategy"]
    fam = p.get("family")
    if fam in st.GURU_SETUP:
        return st.GURU_SETUP[fam]
    setup = (p.get("setup") or "").lower()
    if setup.startswith("breakout"):
        return "breakout_retest"
    if "reaction at" in setup:
        if "breaker" in setup:
            return "breaker_block"
        if " ob" in setup or "order block" in setup:
            return "ob_retest"
        if "fvg" in setup:
            return "fvg_fill"
        return "sr_bounce"
    if "breakout" in setup:
        return "chart_patterns"
    return None


def tag(plans):
    for p in plans:
        sid = strategy_of(p)
        p["strategy"] = sid
        p["strategy_name"] = st.BY_ID[sid]["name"] if sid in st.BY_ID else None
    return plans


def rule_plans(X, ids):
    """Plans from the chosen strategies' rules (signals on the last few closed candles that are still valid)."""
    import guru_analyst as ga
    if not ids:
        return []
    ctx, f, atr, price = X["sctx"], X["f"], X["atr"], X["price"]
    out = []
    for sid in ids:
        s = st.BY_ID[sid]
        if s["fixed_time"]:
            continue
        lv = st.live(ctx, sid, price=price)
        X.setdefault("live", {})[sid] = lv
        if not lv.get("signal"):
            continue
        d = lv["dir"]
        side = "long" if d == 1 else "short"
        entry = lv["entry"]
        near = abs(price - entry) <= 0.15 * abs(entry - lv["sl"])
        kind = "market" if near else "limit"
        if kind == "market":
            entry = price
        tps = ga._targets(X["clusters"], entry, lv["sl"], d, atr, extra=[(lv["tp"], f"{s['name']} target")])
        when = "on the last closed candle" if lv["age"] == 0 else f"{lv['age']} candle{'s' if lv['age'] > 1 else ''} ago"
        why = [f"{s['name']} signalled a {'buy' if d == 1 else 'sell'} {when}: {s['how']}"]
        if kind == "limit":
            why.append(f"price has moved since: the plan waits for a return to {f.p(entry)} instead of chasing")
        p = ga._plan(side, kind, f"{s['name']}: {'buy' if d == 1 else 'sell'} signal", entry, lv["sl"], tps, why,
                     f"a close {'below' if d == 1 else 'above'} {f.p(lv['sl'])}", min(gm.EXPIRY, s["hold"]),
                     family=None, extra=dict(strategy=sid, strategy_name=s["name"], signal_age=lv["age"]))
        if p:
            out.append(p)
    return out


def guard(p, X):
    """The liquidity guard for one plan: the stop moves past liquidity just beyond it (never closer), and an entry with
    an untaken pool right under (over) it gets a 'wait for the sweep' note. Returns the plan (rebuilt if moved)."""
    import guru_analyst as ga
    ctx, f, atr = X["sctx"], X["f"], X["atr"]
    i = ctx.n - 1
    d, entry, sl = p["dir"], p["entry"], p["sl"]
    depth, learned = stop_depth(X, d, sl)
    new_sl, pool = st.guard_stop(ctx, i, d, entry, sl, depth)
    # the analyst's key levels too (zones, equal highs/lows, previous day/week levels): a stop a few ticks past one of
    # them is the first thing a sweep takes, so it goes past the level by the same stop-hunt depth
    buf = (depth + 0.05) * atr
    lv = [c["price"] for c in X.get("clusters") or []
          if d * (entry - c["price"]) > 0.15 * atr and -0.5 * atr <= d * (c["price"] - new_sl) < buf]
    if lv:
        far = min(lv) if d == 1 else max(lv)
        cand = far - d * buf
        if d * (new_sl - cand) > 0 and abs(cand - entry) <= 2.5 * abs(sl - entry):
            new_sl, pool = cand, far
    keep = {k: p[k] for k in ("zone", "strategy", "strategy_name", "signal_age", "family") if k in p}
    if pool is not None and abs(new_sl - sl) > 1e-12:
        moved = abs(new_sl - entry) / max(abs(sl - entry), 1e-12)
        why = p["why"] + [f"stop moved past the {'sell' if d == 1 else 'buy'}-side liquidity at {f.p(pool)}: stop hunts "
                          + (f"at pools like this usually reach {depth:.2f} ATR past them (learned from years of "
                             f"markets)" if learned else f"on this chart usually reach {depth:.2f} ATR past a swing")
                          + ", so a stop just beyond that is where they finish"]
        q = ga._plan(p["side"], p["type"], p["setup"], entry, new_sl,
                     ga._targets(X["clusters"], entry, new_sl, d, atr, extra=list(zip(p["tps"], p["tp_labels"]))),
                     why, p["invalidation"], p["expiry"], keep.get("family"), keep)
        if q is not None:
            q["liquidity_guard"] = dict(pool=float(pool), moved_from=float(sl), depth_atr=float(depth),
                                        wider=float(moved), learned=bool(learned))
            p = q
    # an obvious pool just past the entry on the stop side: price often takes it before going our way
    pools = ctx.pools(i, -d)
    close = [q for q in pools if 0 < d * (entry - q) <= 0.6 * atr and d * (q - p["sl"]) > 0]
    if close:
        lvl = max(close) if d == 1 else min(close)
        odds = pool_odds(X, lvl, -d)
        ps = odds.get("p_sweep") if odds else None
        if ps is None or ps >= 0.35:                      # the liquidity head can rule out a pool unlikely to be hit
            extra = ""
            if ps is not None:
                extra = f" There's about a {round(ps * 100)}% chance it's taken within {LIQ_H} candles"
                pr = odds.get("p_reverse")
                extra += (f", and after sweeps of pools like it price reversed {round(pr * 100)}% of the time." if pr
                          is not None else ".")
            p["sweep_first"] = dict(level=float(lvl), entry_after=float(lvl - d * 0.5 * depth * atr), p_sweep=ps,
                                    p_reverse=(odds or {}).get("p_reverse"), note=(
                f"{'Sell' if d == 1 else 'Buy'}-side liquidity at {f.p(lvl)} sits just {'under' if d == 1 else 'over'} "
                f"the entry. Price often takes it first.{extra} Safer: wait for a sweep of {f.p(lvl)} and a close back "
                f"{'above' if d == 1 else 'below'} it, or enter nearer {f.p(lvl - d * 0.5 * depth * atr)}."))
    return p


LIQ_H = 24


def pool_odds(X, level, side):
    """The liquidity head's numbers for the pool at `level` (side +1 above, -1 below price), or None."""
    rows = X.get("liq") or []
    best = None
    for r in rows:
        if r["side"] == side and abs(r["price"] - level) <= 0.05 * X["atr"]:
            if best is None or abs(r["price"] - level) < abs(best["price"] - level):
                best = r
    return best


def stop_depth(X, d, sl):
    """How far past a pool to put a stop (ATR): the liquidity head's learned hunt depth for the pool nearest the
    stop, when the head is trained and passed its test; else this chart's own 80th percentile. (depth, learned?)"""
    rows = [r for r in (X.get("liq") or []) if r["side"] == -d and r.get("depth_atr") is not None]
    if rows:
        r = min(rows, key=lambda r: abs(r["price"] - sl))
        if abs(r["price"] - sl) <= 1.5 * X["atr"]:
            return float(np.clip(r["depth_atr"], 0.12, 1.5)), True
    return float(X["depth"]), False


def records(X, ids):
    """Each chosen strategy's record on the loaded candles (after this market's costs), with its trades."""
    out = {}
    if not ids:
        return out
    ctx = X["sctx"]
    for sid in ids:
        s = st.BY_ID[sid]
        if s["intraday"] and X["tf"] in ("1D", "3D", "1W", "1M"):
            out[sid] = dict(record=dict(trades=0, verdict="needs an intraday chart (1m–4h)"), trades=[], score=0.0)
            continue
        try:
            tr = st.backtest(ctx, sid, X["cost"], X["slip"], guard=True)
        except Exception as e:
            out[sid] = dict(record=dict(trades=0, verdict=f"could not test ({type(e).__name__})"), trades=[], score=0.0)
            continue
        rec = st.record(tr, s["fixed_time"])
        out[sid] = dict(record=rec, trades=tr, score=st.score_of(rec))
    return out


def rank_bonus(p, X):
    r = (X.get("records") or {}).get(p.get("strategy"))
    bonus = 2.0 * (r["score"] if r else 0.0)
    h = p.get("head")
    if h:                                                 # years of similar signals, not just this chart's candles
        bonus += 3.0 * float(np.clip(h["ev"], -0.5, 0.5))
    return bonus


def _htf_agree(X, d):
    vals = []
    for h in X["ladder"][:2]:
        hs = X["htf"].get(h)
        if hs:
            vals.append(1.0 if hs["trend"] == d else -1.0 if hs["trend"] == -d else 0.0)
    return float(np.mean(vals)) if vals else 0.0


def conflict(plans, X):
    """Do the chosen strategies point opposite ways right now? Each strategy with a live plan votes for its side,
    weighted by its record on this chart and by the higher timeframes."""
    if not X.get("chosen"):
        return None
    price, atr = X["price"], X["atr"]
    votes = {}
    for p in plans:
        sid = p.get("strategy")
        live_now = (p.get("signal_age") is not None and p["signal_age"] <= 3) or p["type"] == "market" or \
            abs(p["entry"] - price) <= 1.0 * atr
        if not sid or not live_now:
            continue
        rec = (X.get("records") or {}).get(sid) or {}
        h = p.get("head") or {}
        w = max(0.2, 1.0 + rec.get("score", 0.0) + 0.5 * _htf_agree(X, p["dir"])
                + float(np.clip(h.get("ev", 0.0), -0.5, 0.5)))
        votes.setdefault(sid, {})[p["dir"]] = max(votes.get(sid, {}).get(p["dir"], 0.0), w)
    long_w = sum(v.get(1, 0.0) for v in votes.values())
    short_w = sum(v.get(-1, 0.0) for v in votes.values())
    if not long_w or not short_w:
        return dict(split=False, long=long_w, short=short_w,
                    side=1 if long_w > short_w else -1 if short_w > long_w else 0)
    margin = abs(long_w - short_w) / (long_w + short_w)
    side = 1 if long_w > short_w else -1
    names = lambda d: [st.BY_ID[s]["name"] for s, v in votes.items() if v.get(d)]
    return dict(split=margin < 0.25, margin=float(margin), long=float(long_w), short=float(short_w), side=side,
                buyers=names(1), sellers=names(-1),
                note=(f"Your strategies disagree: {', '.join(names(1))} say buy, {', '.join(names(-1))} say sell. "
                      + ("The split is close, so this is a time to stand aside." if margin < 0.25 else
                         f"The {'buy' if side == 1 else 'sell'} side is stronger once each strategy's record on this "
                         f"chart and the higher timeframes are weighed, but treat it with extra care.")))


def confidence(p, X):
    """How much weight a plan deserves, in plain words. Never 'certain': the best is 'good'."""
    if p is None:
        return dict(level="none", score=0.0, text="No plan worth taking right now.", reasons=[])
    j = min(1, len(p["tps"]) - 1)
    pts, reasons = 0.0, []
    g = {"A+": 2.0, "A": 1.5, "B": 1.0, "C": 0.3, "D": 0.0}.get(p["grade"], 0.0)
    pts += g
    edge = p["p_tp"][j] - p["breakeven"][j]
    pts += float(np.clip(edge * 10, -1.5, 1.5))
    reasons.append(f"similar setups reached TP{j + 1} before the stop about {round(p['p_tp'][j] * 100)}% of the time "
                   f"(break-even needs {round(p['breakeven'][j] * 100)}%)")
    a = X.get("mtf_alignment")
    if a is not None:
        pts += float(np.clip(a * p["dir"], -1, 1))
        if a * p["dir"] > 0.3:
            reasons.append("the timeframes mostly agree")
        elif a * p["dir"] < -0.3:
            reasons.append("the bigger timeframes lean the other way")
    rec = (X.get("records") or {}).get(p.get("strategy"))
    if rec and rec["record"].get("trades", 0) >= 15:
        pts += float(np.clip(rec["score"] * 2, -1, 1))
        r = rec["record"]
        reasons.append(f"{p.get('strategy_name')} on this chart: {r['trades']} trades, {r['win_rate'] * 100:.0f}% won, "
                       f"{r['avg_r']:+.2f}R on average")
    h = p.get("head")
    if h:
        pts += float(np.clip((h["p_win"] - h["needed"]) * 6, -1.0, 1.0))
        reasons.append(f"across years of markets, {p.get('strategy_name') or 'this strategy'}'s signals like this one "
                       f"ended in profit (at its own target or time limit) about {round(h['p_win'] * 100)}% of the time "
                       f"(needs about {round(h['needed'] * 100)}%)")
    if p.get("sweep_first"):
        pts -= 0.5
        ps = p["sweep_first"].get("p_sweep")
        reasons.append("liquidity right under the entry may be taken first"
                       + (f" ({round(ps * 100)}% chance within {LIQ_H} candles)" if ps is not None else ""))
    if X.get("brain") is not None and getattr(X["brain"], "untrained", False):
        pts = min(pts, 2.0)
        reasons.append("the odds model isn't trained yet, so the odds are rough")
    level = "good" if pts >= 3.5 else "moderate" if pts >= 2.0 else "low"
    word = {"good": "Good", "moderate": "Moderate", "low": "Low"}[level]
    return dict(level=level, score=round(float(pts), 2), reasons=reasons,
                text=f"{word} confidence. This is a scenario, not a certainty: " + "; ".join(reasons[:3]) + ".")


def soften(verdict, p, X, conf, clash):
    """Modest wording on top of the analyst's verdict: a possible trade, its confidence, and what can change it."""
    v = dict(verdict)
    v["confidence"] = conf
    v["tf_note"] = (f"Analysis can change with every new {X['tf']} candle and differ on other timeframes. Recheck after "
                    f"the next candle closes, and check the timeframe table before entering.")
    ids = X.get("chosen")
    if p is None and ids:
        ft = X.get("fixed_time")
        price_ids = [i for i in ids if not st.BY_ID[i]["fixed_time"]]
        if ft and ft.get("direction") and (ft.get("edge") or 0) > 0.01:
            v.update(action=f"Fixed time: {ft['direction']} for {ft['expiry']} candles", trade=False, side=None,
                     headline=ft["text"], fixed_time_call=True)
        elif not price_ids:
            v.update(action="NO CALL (wait)", headline=(ft or {}).get("text") or "No fixed-time call right now.")
        else:
            waits = [st.BY_ID[i]["wait_for"] for i in price_ids][:3]
            v.update(action="NO TRADE (wait)", headline="None of your strategies has a setup on this chart right now. "
                     "Wait for " + "; or ".join(waits) + ".")
        return v
    if clash and clash.get("split"):
        v.update(action="NO TRADE (strategies disagree)", trade=False, side=None, conflict=clash,
                 headline=clash["note"])
        return v
    if clash and clash.get("sellers") and clash.get("buyers"):
        v["conflict"] = clash
    # the grade never says more than the confidence does: an "A" with low confidence reads as a promise
    cap = {"low": "C", "moderate": "B"}.get(conf.get("level"))
    order = ["A+", "A", "B", "C", "D"]
    if cap and p is not None:
        for obj in (v, p):
            g = obj.get("grade")
            if g in order and order.index(g) < order.index(cap):
                obj["grade"] = cap
    if v.get("trade") and p is not None:
        word = "BUY" if p["dir"] == 1 else "SELL"
        how = {"market": "now (market)", "limit": f"limit at {X['f'].p(p['entry'])}",
               "stop": f"stop order at {X['f'].p(p['entry'])}"}[p["type"]]
        v["action"] = f"Possible {word} · {how}"
        v["headline"] = (f"Possible {word.lower()} {how} · {conf['level']} confidence · {p['setup']}"
                         + (f" ({p['strategy_name']})" if p.get("strategy_name") else ""))
    return v


def how_to_enter(p, X):
    """Step-by-step: how to take this plan without getting caught by the usual traps."""
    if p is None:
        return []
    f = X["f"]
    d = p["dir"]
    w = "buy" if d == 1 else "sell"
    steps = []
    if p["type"] == "market":
        steps.append(f"Wait for the current {X['tf']} candle to close {'above' if d == 1 else 'below'} "
                     f"{f.p(p['entry'])} before you {w}; a signal on a candle that hasn't closed can disappear.")
    elif p["type"] == "limit":
        steps.append(f"Place a {w} limit at {f.p(p['entry'])}. If price never comes back there, there is no trade: "
                     f"don't chase it.")
    else:
        steps.append(f"Place a {w} stop order at {f.p(p['entry'])}: it only fills if price breaks through.")
    if p.get("sweep_first"):
        steps.append(p["sweep_first"]["note"])
    steps.append(f"Stop loss at {f.p(p['sl'])}, set at the same time as the entry. "
                 + ("It sits past the nearby liquidity so a stop hunt is less likely to take you out. "
                    if p.get("liquidity_guard") else "")
                 + "Size the position so this stop costs no more than your risk per trade (1% is common).")
    tps = p["tps"]
    steps.append(f"Take part profit at TP1 {f.p(tps[0])}"
                 + (f", move the stop to break-even, and let the rest run to TP2 {f.p(tps[1])}" if len(tps) > 1 else "")
                 + ".")
    steps.append(f"Cancel the idea if {p['invalidation']}, or if nothing has filled within {p['expiry']} candles.")
    if X["intraday"] and X.get("news_high"):
        steps.append("High-impact news is close: wait until it has passed.")
    return steps


def _frame_row(tf, df, A, ids, price=None, asset_class=None):
    """One row of the timeframe table: trend, last break, location and signals."""
    n = len(df)
    t = int(A["trend"][-1])
    ev = A["events"][-1] if A["events"] else None
    c = df["close"].to_numpy(dtype=float)
    e50 = st._ema(c, 50)
    e200 = st._ema(c, 200)
    rsi = float(st._rsi(c, 14)[-1])
    hi = float(df["high"].tail(100).max())
    lo = float(df["low"].tail(100).min())
    pos = (c[-1] - lo) / (hi - lo) if hi > lo else 0.5
    ema_bias = 0
    if np.isfinite(e50[-1]) and np.isfinite(e200[-1]):
        ema_bias = 1 if c[-1] > e50[-1] > e200[-1] else -1 if c[-1] < e50[-1] < e200[-1] else 0
    bias = float(np.clip(0.6 * t + 0.4 * ema_bias, -1, 1))
    sigs = []
    if ids:
        ctx = st.Ctx(df, tf, A, asset_class)
        for sid in ids:
            s = st.BY_ID[sid]
            if s["intraday"] and tf in ("1D", "3D", "1W", "1M"):
                continue
            lv = st.live(ctx, sid, max_age=2)
            if lv.get("signal"):
                sigs.append(dict(id=sid, name=s["name"], dir=lv["dir"], age=lv.get("age", 0)))
    return dict(tf=tf, trend=t, trend_word={1: "up", -1: "down", 0: "unclear"}[t], ema_bias=ema_bias,
                bias=round(bias, 2), rsi=round(rsi, 1), zone="premium" if pos > 0.55 else "discount" if pos < 0.45 else
                "middle", range_pos=round(float(pos), 2),
                last_event=None if ev is None else dict(kind=ev["kind"], direction=ev["direction"], level=float(ev["level"]),
                                                        bars_ago=int(n - 1 - ev["idx"])),
                signals=sigs)


def mtf(X, symbol, HT, asof=None, ids=None):
    """The timeframe table: every timeframe from the higher ones the analyst loaded to one lower timeframe."""
    rows = []
    tf = X["tf"]
    for h in X["ladder"][::-1]:
        v = HT.get(h)
        if v is None:
            continue
        try:
            rows.append(_frame_row(h, v["df"], v["A"], ids, asset_class=X["asset_class"]))
        except Exception:
            continue
    try:
        rows.append(dict(_frame_row(tf, X["df"], X["A"], ids, asset_class=X["asset_class"]), current=True))
    except Exception:
        pass
    low = LOWER.get(tf)
    if low:
        try:
            Cl = md.get_candles(symbol, low, bars=600, end=asof, min_bars=60)
            dl, Al = se.analyze(Cl["df"])
            rows.append(dict(_frame_row(low, dl, Al, ids, asset_class=X["asset_class"]), lower=True))
        except Exception:
            pass
    rows.sort(key=lambda r: -md.TF_MIN[r["tf"]])
    if not rows:
        return dict(rows=[], alignment=0.0, text="")
    w = np.array([TF_WEIGHT.get(r["tf"], 1.0) for r in rows])
    b = np.array([r["bias"] for r in rows])
    align = float((w * b).sum() / w.sum())
    word = ("aligned up" if align > 0.45 else "leaning up" if align > 0.15 else "aligned down" if align < -0.45 else
            "leaning down" if align < -0.15 else "mixed")
    ups = [r["tf"] for r in rows if r["bias"] > 0.2]
    dns = [r["tf"] for r in rows if r["bias"] < -0.2]
    text = (f"Timeframes {word}: up on {', '.join(ups) or 'none'}; down on {', '.join(dns) or 'none'}."
            + (" Trade with the higher timeframes, or stand aside when they fight." if word == "mixed" else ""))
    return dict(rows=rows, alignment=round(align, 2), word=word, text=text)


def fixed_time(X, ids):
    """An up/down call with an expiry for fixed-time traders, from the chosen fixed-time strategies."""
    ft = [s for s in (ids or []) if st.BY_ID[s]["fixed_time"]]
    if not ft:
        return None
    ctx = X["sctx"]
    mins = md.TF_MIN[X["tf"]]
    calls = []
    for sid in ft:
        s = st.BY_ID[sid]
        lv = st.live(ctx, sid)
        rec = (X.get("records") or {}).get(sid, {}).get("record") or {}
        n = rec.get("trades", 0)
        wr = rec.get("win_rate", 0.5)
        p = (wr * n + 0.5 * 30) / (n + 30)                       # shrunk toward a coin flip when there are few calls
        head = None
        if lv.get("signal"):
            head = sh.score(getattr(X.get("brain"), "strat", None), ctx, X["row"], sid, lv["dir"], lv["entry"],
                            lv["entry"], lv["entry"], X["cost"], X["slip"])
            if head:                                          # years of history weigh more than one chart's candles
                p = 0.7 * head["p_win"] + 0.3 * p
        calls.append(dict(id=sid, name=s["name"], signal=lv.get("signal"), dir=lv.get("dir"), expiry=s["expiry"],
                          expiry_minutes=s["expiry"] * mins, p_win=float(p), record=rec,
                          last=lv.get("last"), head=head))
    live_calls = [c for c in calls if c["signal"]]
    be = 1 / (1 + st.FT_PAYOUT)
    out = dict(breakeven=be, payout=st.FT_PAYOUT, calls=calls, tf=X["tf"])
    dirs = {c["dir"] for c in live_calls}
    skip = []
    if X["intraday"] and X.get("news_high"):
        skip.append("high-impact news is close")
    ph = X.get("phase") or {}
    if ph.get("adx", 0) > 45:
        skip.append("the move is very stretched (ADX above 45)")
    if not live_calls:
        out.update(direction=None, text="No fixed-time call on the last closed candle. Wait for one of your fixed-time "
                                        "setups: " + "; ".join(st.BY_ID[c['id']]['wait_for'] for c in calls) + ".")
    elif len(dirs) > 1:
        out.update(direction=None, text="Your fixed-time strategies disagree on this candle: skip it.")
    else:
        best = max(live_calls, key=lambda c: c["p_win"])
        d = best["dir"]
        edge = best["p_win"] - be
        out.update(direction="UP" if d == 1 else "DOWN", expiry=best["expiry"], expiry_minutes=best["expiry_minutes"],
                   p_win=best["p_win"], edge=edge, strategy=best["name"],
                   text=(f"{'UP' if d == 1 else 'DOWN'} for {best['expiry']} candles ({best['expiry_minutes']} min) from "
                         f"{best['name']}. "
                         f"{'Across years of markets and on this chart' if best.get('head') else 'On this chart'} "
                         f"such calls won about {round(best['p_win'] * 100)}% vs "
                         f"{round(be * 100)}% needed at an {round(st.FT_PAYOUT * 100)}% payout"
                         + (": skip it, there's no edge." if edge <= 0.01 else ".")))
        if edge <= 0.01:
            skip.append("its record here doesn't beat the payout's break-even")
    out["skip_if"] = skip
    return out


def liquidity_map(X):
    """The liquidity around price: the nearest untaken pools on each side, how deep stop hunts usually reach, and the
    side price is most likely drawn to next. With the liquidity head (Notebook 10) every pool also gets its odds of
    being swept within 24 candles and of reversing after the sweep, and the draw is the side most likely taken first."""
    ctx, f, price = X["sctx"], X["f"], X["price"]
    i = ctx.n - 1
    above = sorted(p for p in ctx.pools(i, 1) if p > price)[:4]
    below = sorted((p for p in ctx.pools(i, -1) if p < price), reverse=True)[:4]
    pools = [c for c in X["clusters"] if any(k in " ".join(c["labels"]).lower() for k in ("equal", "liquidity", "pdh",
                                                                                          "pdl", "asian"))]
    up = min((c for c in pools if c["side"] == "above"), key=lambda c: c["dist_atr"], default=None)
    dn = max((c for c in pools if c["side"] == "below"), key=lambda c: c["dist_atr"], default=None)
    lean = X.get("mtf_alignment", 0.0) or 0.0
    draw = None
    if up and dn:
        su = (up.get("strength", 1) + 2 * max(lean, 0)) / max(abs(up["dist_atr"]), 0.3)
        sd = (dn.get("strength", 1) + 2 * max(-lean, 0)) / max(abs(dn["dist_atr"]), 0.3)
        draw = "above" if su > sd else "below"
    elif up or dn:
        draw = "above" if up else "below"
    tgt = up if draw == "above" else dn if draw == "below" else None
    draw_level = None if tgt is None else float(tgt["price"])
    draw_labels = None if tgt is None else tgt["labels"][:3]
    learned = [r for r in (X.get("liq") or []) if r.get("p_sweep") is not None]
    rows, odds_text = [], ""
    if learned:
        rows = [dict(price=r["price"], side="above" if r["side"] == 1 else "below", kind=lh.type_word(r),
                     d_atr=round(r["d_atr"], 2), p_sweep=r["p_sweep"], p_reverse=r.get("p_reverse"),
                     depth_atr=r.get("depth_atr")) for r in learned]
        first = max(learned, key=lambda r: r["p_sweep"])
        if first["p_sweep"] >= 0.25:
            draw = "above" if first["side"] == 1 else "below"
            draw_level, draw_labels = float(first["price"]), [lh.type_word(first)]
            odds_text = (f"Most likely taken first: the {lh.type_word(first)} at {f.p(first['price'])} "
                         f"({round(first['p_sweep'] * 100)}% within {LIQ_H} candles"
                         + (f"; after sweeps there, price reversed {round(first['p_reverse'] * 100)}% of the time"
                            if first.get("p_reverse") is not None else "") + "). ")
        else:
            odds_text = (f"No pool near price is likely to be taken within {LIQ_H} candles (the best chance is "
                         f"{round(first['p_sweep'] * 100)}%). ")
    return dict(depth_atr=float(X["depth"]), above=[float(p) for p in above], below=[float(p) for p in below],
                draw=draw, draw_level=draw_level, draw_labels=draw_labels, pools=rows, learned=bool(learned),
                text=(f"Stop hunts on this chart usually reach {X['depth']:.2f} ATR past a swing. "
                      + (f"Nearest buy-side liquidity {f.p(above[0])}. " if above else "")
                      + (f"Nearest sell-side liquidity {f.p(below[0])}. " if below else "")
                      + (odds_text or (f"Price looks drawn to the liquidity {'above' if draw == 'above' else 'below'} at "
                                       f"{f.p(tgt['price'])} ({', '.join(tgt['labels'][:2])})." if tgt else ""))))


def whatif(X, price, brain=None):
    """'If I enter here, now': a buy and a sell from `price` on the latest candle, each with a stop past the nearest
    liquidity, targets at the next levels, the odds of reaching them first, and which side (if either) is better."""
    import guru_analyst as ga
    brain = brain or X["brain"]
    ctx, f, atr, now = X["sctx"], X["f"], X["atr"], X["price"]
    i = ctx.n - 1
    depth = X["depth"]
    sides = []
    for d in (1, -1):
        pools = [q for q in ctx.pools(i, -d) if d * (price - q) > 0.25 * atr]
        if pools:
            near = max(pools) if d == 1 else min(pools)
            sl = near - d * (depth + 0.05) * atr if d * (price - near) <= 3.5 * atr else price - d * 1.5 * atr
        else:
            sl = price - d * 1.5 * atr
        if d * (price - sl) < 0.5 * atr:
            sl = price - d * 0.5 * atr
        kind = "market" if abs(price - now) <= 0.05 * atr else ("limit" if d * (now - price) > 0 else "stop")
        tps = ga._targets(X["clusters"], price, sl, d, atr)
        p = ga._plan("long" if d == 1 else "short", kind, f"{'Buy' if d == 1 else 'Sell'} from {f.p(price)}", price, sl,
                     tps, [f"you picked {f.p(price)} on the latest candle"], f"price reaching {f.p(sl)}", gm.EXPIRY)
        if p is None:
            continue
        p = guard(p, X)
        p = ga.evaluate_plan(p, X, brain)
        j = min(1, len(p["tps"]) - 1)
        warn = []
        opp = [q for q in ctx.pools(i, d) if 0 < d * (q - price) <= 0.6 * atr]
        if opp:
            lvl = min(opp) if d == 1 else max(opp)
            warn.append(f"{'Buy' if d == 1 else 'Sell'}-side liquidity at {f.p(lvl)} is only "
                        f"{abs(lvl - price) / atr:.2f} ATR away: price often tags it and turns, so the room to move is "
                        f"small.")
        if p.get("sweep_first"):
            warn.append(p["sweep_first"]["note"])
        a = X.get("mtf_alignment", 0.0) or 0.0
        if a * d < -0.3:
            warn.append(f"The bigger timeframes lean {'down' if d == 1 else 'up'}: this is a counter-trend "
                        f"{'buy' if d == 1 else 'sell'}.")
        for c in p["checklist"]:
            if not c["ok"] and c["w"] < 0:
                warn.append(c["text"][0].upper() + c["text"][1:] + ".")
        sides.append(dict(side=p["side"], dir=d, type=kind, entry=float(price), sl=float(p["sl"]), tps=p["tps"],
                          tp_labels=p["tp_labels"], rr=p["rr"], p_tp=p["p_tp"], breakeven=p["breakeven"], ev=p["ev"],
                          p_fill=p["p_fill"], grade=p["grade"], score=p["score"], checklist=p["checklist"],
                          liquidity_guard=p.get("liquidity_guard"), sweep_first=p.get("sweep_first"), warnings=warn[:5],
                          ev_main=float(p["ev"][j]), confidence=confidence(dict(p, strategy=None), X)))
    best = None
    if sides:
        ranked = sorted(sides, key=lambda s: -(s["ev_main"] + 0.15 * s["score"]))
        top = ranked[0]
        if top["ev_main"] > 0.02 and top["grade"] in ("A+", "A", "B"):
            best = top["side"]
    if best:
        b = next(s for s in sides if s["side"] == best)
        text = (f"From {f.p(price)}, a {'buy' if best == 'long' else 'sell'} has the better odds: grade {b['grade']}, "
                f"{round(b['p_tp'][min(1, len(b['tps']) - 1)] * 100)}% to reach TP{min(2, len(b['tps']))} first. Still "
                f"a scenario, not a certainty.")
    else:
        text = (f"From {f.p(price)}, neither a buy nor a sell has an edge right now. Better to wait for one of your "
                f"strategies' setups.")
    return dict(price=float(price), now=float(now), sides=sides, better=best, text=text,
                p_up=X["prob"]["p_up"], p_down=X["prob"]["p_down"], horizon=X["prob"]["horizon"],
                tf_note=f"Only for the latest {X['tf']} candle. It changes as price moves.")


In [ ]:
%%writefile guru_analyst.py
"""guru_analyst.py: the analyst. analyze("BTCUSDT", "1h") returns a full top-down analysis:

  * multi-timeframe bias, structure (HH/HL/LH/LL, BOS/CHoCH), market phase
  * key levels with their probability of being reached, and how such levels historically react
  * liquidity (BSL/SSL, equal highs/lows, sweeps), FVGs, order blocks, breakers, OTE, premium/discount
  * trendlines, channels, triangles/wedges, double tops/bottoms, head & shoulders, RSI divergences,
    volume profile, VWAP, sessions & ICT killzones, previous day/week/month levels, round numbers
  * trade plans (EP / SL / TP1-3) graded with a confluence checklist, their probabilities from the trained
    models, the historical evidence for that setup type, position size, leverage and liquidation price
  * a written report in the long step-by-step format (pro and beginner versions), and a JSON list of
    drawings the web app can draw on the chart one by one.

Research tool, not financial advice. Probabilities are calibrated estimates, not promises.
"""
import math
import time

import numpy as np
import pandas as pd

import guru_levels as gl
import guru_model as gm
import liquidity_head as lh
import market_data as md
import playbook as pb
import strategy_head as sh
import strategies as st
import structure_engine as se

BARS = {"1m": 1500, "5m": 1500, "15m": 1500, "30m": 1500, "1h": 1500, "4h": 1500, "1D": 1500, "3D": 800,
        "1W": 600, "1M": 300}
HTF_BARS = 600
SHOW_BARS = 300
USD_LIKE = {"USD", "USDT", "USDC", "FDUSD", "BUSD", "TUSD", "DAI"}
SPECS = {"fx": dict(unit="units", lot=100_000, min_lot=0.01, step=0.01),
         "gold": dict(unit="oz", lot=100, min_lot=0.01, step=0.01),
         "silver": dict(unit="oz", lot=5000, min_lot=0.01, step=0.01),
         "metal": dict(unit="units", lot=1, min_lot=0.01, step=0.01),
         "index": dict(unit="index units", lot=1, min_lot=0.01, step=0.01),
         "energy": dict(unit="barrels", lot=1000, min_lot=0.01, step=0.01),
         "commodity": dict(unit="units", lot=1, min_lot=0.01, step=0.01),
         "stock": dict(unit="shares", lot=1, min_lot=1, step=1),
         "other": dict(unit="units", lot=1, min_lot=0.01, step=0.01),
         "crypto": dict(unit="coins", lot=None, min_notional=5.0)}
MAX_LEV = {"fx_major": 500, "fx_cross": 500, "fx_exotic": 100, "gold": 400, "silver": 200, "metal": 100,
           "index": 200, "energy": 100, "commodity": 50, "crypto_major": 125, "crypto_alt": 50, "stock": 20,
           "other": 20}
MMR = {"crypto_major": 0.005, "crypto_alt": 0.015}
STD_LEV = (1, 2, 3, 5, 8, 10, 12, 15, 20, 25, 30, 40, 50, 75, 100, 125)
COL = dict(up="#26a69a", down="#ef5350", text="#d1d4dc", choch="#f5c542", liq="#b388ff", sup="rgba(38,166,154,0.16)",
           res="rgba(239,83,80,0.16)", mid="rgba(209,212,220,0.10)", fvg_bull="rgba(41,98,255,0.20)",
           fvg_bear="rgba(255,152,0,0.20)", ob_bull="rgba(0,188,212,0.20)", ob_bear="rgba(233,30,99,0.20)",
           fib="#ffb74d", ote="rgba(255,183,77,0.13)", zig="rgba(209,212,220,0.45)", period="#90caf9",
           round="rgba(209,212,220,0.25)", trend="#4dd0e1", pattern="#ce93d8", user="#ffeb3b", vwap="#ff9800",
           ema20="#ffffff", ema50="#ffd54f", ema200="#7e57c2", poc="#f06292")
_STATE = {"brain": None}


def use_brain(bundle_or_brain):
    """Load the trained models (bundle dict, path or Brain) for every later analysis."""
    if isinstance(bundle_or_brain, str):
        bundle_or_brain = gm.load(bundle_or_brain)
    if isinstance(bundle_or_brain, dict):
        bundle_or_brain = gm.Brain(bundle_or_brain)
    _STATE["brain"] = bundle_or_brain
    return bundle_or_brain


# ───────────────────────────── formatting ─────────────────────────────
class Fmt:
    def __init__(self, dec):
        self.dec = int(dec)

    def p(self, x):
        if x is None or not np.isfinite(x):
            return "—"
        return f"{x:,.{self.dec}f}" if abs(x) >= 1000 else f"{x:.{self.dec}f}"

    def r(self, x):
        return "—" if x is None or not np.isfinite(x) else f"{x:.2f}".rstrip("0").rstrip(".")


def pct(x, d=0):
    return "—" if x is None or not np.isfinite(x) else f"{100 * x:.{d}f}%"


def money(x):
    if x is None or not np.isfinite(x):
        return "—"
    return f"${x:,.2f}" if abs(x) < 100_000 else f"${x:,.0f}"


def qty(x):
    if not np.isfinite(x) or x == 0:
        return "0"
    mag = math.floor(math.log10(abs(x)))
    dec = max(0, 3 - mag) if mag < 3 else 0
    s = f"{x:,.{min(dec, 8)}f}"
    return s.rstrip("0").rstrip(".") if "." in s else s


def level_history(d_, lo, hi, atr, horizon=24):
    """How price behaved at a level/zone in the loaded history: tests, holds (1 ATR rejection first) and breaks
    (a close 0.25 ATR through first). A new test needs 5 candles away from the level."""
    h, l, c = d_["high"].to_numpy(), d_["low"].to_numpy(), d_["close"].to_numpy()
    tol = 0.1 * atr
    touch = np.flatnonzero((l <= hi + tol) & (h >= lo - tol))
    tests, held, broke, last = 0, 0, 0, -10
    for i in touch:
        if i - last < 5 or i < 1 or i >= len(c) - 1:
            last = i if i - last < 5 else last
            continue
        last = i
        from_above = c[i - 1] > hi
        seg = slice(i, min(len(c), i + horizon))
        if from_above:
            brk = np.flatnonzero(c[seg] < lo - 0.25 * atr)
            rej = np.flatnonzero(h[seg] > hi + atr)
        else:
            brk = np.flatnonzero(c[seg] > hi + 0.25 * atr)
            rej = np.flatnonzero(l[seg] < lo - atr)
        tests += 1
        tb = brk[0] if len(brk) else None
        tr = rej[0] if len(rej) else None
        if tb is not None and (tr is None or tb <= tr):
            broke += 1
        elif tr is not None:
            held += 1
    return dict(tests=tests, held=held, broke=broke)


def ago(bars, tf):
    if bars is None:
        return ""
    if bars <= 0:
        return "on the last closed candle"
    m = bars * md.TF_MIN[tf]
    if m < 120:
        span = f"{m} min"
    elif m < 2880:
        span = f"{m / 60:.0f} h"
    elif m < 60 * 24 * 60:
        span = f"{m / 1440:.0f} days"
    else:
        span = f"{m / 43830:.0f} months"
    return f"{bars} candle{'s' if bars != 1 else ''} ago (~{span})"


def _safe(fn, default, notes, label):
    try:
        return fn()
    except Exception as e:                                   # one broken tool must not kill the analysis
        notes.append(f"{label} skipped ({type(e).__name__}: {str(e)[:80]})")
        return default


# ───────────────────────────── data & context ─────────────────────────────
def quote_to_usd(C, end=None):
    q, b = C["quote"], C["base"]
    if q in USD_LIKE:
        return 1.0, None
    if q == "USX":                                                  # quoted in US cents
        return 0.01, None
    if b in USD_LIKE:
        return 1.0 / C["price"], None
    order = ((f"{q}USD", False), (f"USD{q}", True)) if q in ("EUR", "GBP", "AUD", "NZD") else \
        ((f"USD{q}", True), (f"{q}USD", False))
    for sym, inv in order + ((f"{q}USDT", False),):
        try:
            p = md.get_candles(sym, "1D", bars=10, min_bars=2, end=end)["price"]
            return (1.0 / p if inv else p), None
        except Exception:
            continue
    return 1.0, f"couldn't convert {q} to USD: sizes below assume 1 {q} = $1"


def _htf_summary(h, d, A, brain, asset_class, frames):
    t = int(A["trend"][-1])
    ev = A["events"][-1] if A["events"] else None
    n = len(d)
    dr = gl.dealing_range(d, A)
    ph = gl.market_phase(d, A)
    out = dict(tf=h, trend=t, trend_word={1: "bullish", -1: "bearish", 0: "no clear trend"}[t],
               phase=ph["label"], last_event=None if ev is None else dict(kind=ev["kind"], direction=ev["direction"],
                                                                         level=ev["level"], bars_ago=n - 1 - ev["idx"]),
               swings=[s["label"] for s in A["swings"][-6:]], zone=None if dr is None else dr["zone"],
               range_pos=None if dr is None else dr["pos"])
    try:
        lad = gm.LADDER[h]
        F, _ = gm.raw_features(d, h, asset_class, htf={k: frames[k] for k in lad[:2] if k in frames})
        p = brain.direction(F.iloc[-1])
        out["p_up"], out["p_down"], out["p_flat"] = map(float, p)
    except Exception:
        pass
    return out


BULL_KW = ("support", "bullish", "demand", "sell-side", "pdl", "pwl", "pml", "range low", "uptrend line", "swing low",
           "equal lows")
BEAR_KW = ("resistance", "bearish", "supply", "buy-side", "pdh", "pwh", "pmh", "range high", "downtrend line",
           "swing high", "equal highs")


def polarity(label):
    """+1 = a buyer's level (support, demand, sell-side liquidity), -1 = a seller's level, 0 = neutral."""
    lab = label.lower()
    b, s = any(k in lab for k in BULL_KW), any(k in lab for k in BEAR_KW)
    return 1 if b and not s else (-1 if s and not b else 0)


def collect_levels(price, atr, items, merge_atr=0.2):
    """Merge levels within 0.2 ATR into one cluster; confluence = how many different things sit there."""
    items = [x for x in items if x.get("price") is not None and np.isfinite(x["price"])]
    items.sort(key=lambda x: x["price"])
    clusters = []
    for it in items:
        if clusters and it["price"] - clusters[-1]["hi"] <= merge_atr * atr:
            c = clusters[-1]
            c["items"].append(it)
            c["hi"] = max(c["hi"], it["price"])
        else:
            clusters.append(dict(items=[it], lo=it["price"], hi=it["price"]))
    out = []
    for c in clusters:
        its = c["items"]
        w = np.array([max(0.5, x.get("weight", 1.0)) for x in its])
        px = float(np.average([x["price"] for x in its], weights=w))
        kinds = []
        for x in its:
            if x["label"] not in kinds:
                kinds.append(x["label"])
        pol = float(sum(polarity(x["label"]) * max(0.5, x.get("weight", 1.0)) for x in its))
        out.append(dict(price=px, lo=c["lo"], hi=c["hi"], labels=kinds, types=sorted({x["type"] for x in its}),
                        strength=float(w.sum()), polarity=pol, side="above" if px > price else "below",
                        dist_atr=(px - price) / atr, start_idx=min(x.get("idx", 10 ** 9) for x in its)))
    return out


def reaction_stats(brain, asset_class, tf, typ):
    tab = brain.b.get("level_stats") if hasattr(brain, "b") else None
    if tab is None or len(tab) == 0:
        return None
    g = gm.TF_GROUP[tf]
    rows = tab[(tab["asset_class"] == asset_class) & (tab["tf_group"] == g)]
    if not len(rows):
        rows = tab[tab["tf_group"] == g]
    if not len(rows):
        return None

    def agg(t):
        r = rows[rows["type"] == t]
        if not len(r):
            return None
        n = float(r["n"].sum())
        return dict(n=int(n), p_break=float(r["break"].sum() / n), p_reject=float(r["reject"].sum() / n)) if n >= 30 else None

    s, base = agg(typ), agg("random")
    return None if s is None else dict(s, base_break=None if base is None else base["p_break"],
                                       base_reject=None if base is None else base["p_reject"])


# ───────────────────────────── plans ─────────────────────────────
def _targets(clusters, entry, sl, d, atr, extra=()):
    """TP1 = first level ≥ 1R, TP2 = first level ≥ 2R (else a fixed 2R), TP3 = the next one ≥ 3R (optional).
    Targets sit 3% of an ATR before the level, so they fill when the level is merely tagged."""
    risk = abs(entry - sl)
    # a target must be on the far side of the trade and be something the other side defends / liquidity rests at
    cands = [(c["price"], " + ".join(c["labels"][:2])) for c in clusters
             if d * (c["price"] - entry) > 0.3 * risk and d * c.get("polarity", 0) <= 0]
    cands += [(p, lab) for p, lab in extra if p is not None and d * (p - entry) > 0.3 * risk]
    cands = [(p, lab) for p, lab in cands if p > 0]                    # never a target below zero
    cands.sort(key=lambda x: d * (x[0] - entry))
    tps = []
    for need, max_atr, fallback in ((1.0, 8.0, 1.5), (2.0, 10.0, 2.0), (3.0, 14.0, None)):
        pick = None
        last = tps[-1][0] if tps else entry
        for p, lab in cands:
            rr = d * (p - entry) / risk
            if rr >= need and abs(p - entry) <= max_atr * atr and d * (p - last) > 0.5 * risk:
                pick = (p - d * 0.03 * atr, lab)
                break
        if pick is None and fallback is not None:
            r_ = max(fallback, d * (last - entry) / risk + 0.75)
            if r_ * risk <= max_atr * atr:
                pick = (entry + d * r_ * risk, f"{r_:.1f}R (no level closer)")
        if pick is not None and pick[0] > 0:
            tps.append(pick)
    return tps


def _plan(side, kind, setup, entry, sl, tps, why, invalid, expiry, family=None, extra=None):
    d = 1 if side == "long" else -1
    if not (d * (entry - sl) > 0) or not tps:
        return None
    tps = [t for t in tps if d * (t[0] - entry) > 0]
    if not tps:
        return None
    risk = abs(entry - sl)
    p = dict(side=side, dir=d, type=kind, setup=setup, family=family, entry=float(entry), sl=float(sl),
             tps=[float(t[0]) for t in tps], tp_labels=[t[1] for t in tps], risk=float(risk),
             rr=[abs(t[0] - entry) / risk for t in tps], why=list(why), invalidation=invalid, expiry=int(expiry))
    if extra:
        p.update(extra)
    return p


def make_plans(X):
    """Trade plans for both directions, from the chosen strategies only (all of the analyst's own setups when none
    were chosen). Stops are widened to ≥ 0.5 ATR and ≥ 8× round-trip costs, then moved past nearby liquidity (the
    liquidity guard); if costs make that impossible (e.g. 1m charts), the unwidened plans are kept and flagged, so
    EP/SL/TP are always stated."""
    ids = X.get("chosen")
    raw = pb.tag(_candidate_plans(X))
    if not raw:
        raw = pb.tag(_candidate_plans(X, max_trig_atr=60.0))
    if ids:
        raw = [p for p in raw if p.get("strategy") in ids]
        have = {p["strategy"] for p in raw}
        raw += [p for p in pb.rule_plans(X, ids) if p["strategy"] not in have or p.get("signal_age", 9) <= 1]
    out = _finalize_plans(raw, X)
    if not out:
        X["notes"].append("costs are too large relative to normal stops on this timeframe: plans shown with tight "
                          "stops are for reference only")
        out = _finalize_plans(raw, X, widen=False)
    return out


def _candidate_plans(X, max_trig_atr=5.0):
    """Candidate trade plans for both directions from everything the analysis found."""
    price, atr, clusters, A, d_ = X["price"], X["atr"], X["clusters"], X["A"], X["df"]
    plans = []
    liq = A["liquidity"]
    leg = X["leg"]
    above = [c for c in clusters if c["side"] == "above"]
    below = [c for c in clusters if c["side"] == "below"]
    # 1) trend pullback into OTE (ICT optimal trade entry)
    if leg is not None and leg["bars_old"] <= 150:
        d = leg["direction"]
        side = "long" if d == 1 else "short"
        L, ext, origin = leg["length"], leg["extreme"], leg["origin"]
        z_near, z_far = ext - d * 0.618 * L, ext - d * 0.786 * L
        if d * (price - origin) > 0.2 * atr and d * (price - z_far) > 0:
            inside = d * (price - z_near) <= 0
            entry = price if inside else z_near
            sl = origin - d * 0.1 * atr
            ext_lv = [(ext, "leg extreme (old high)" if d == 1 else "leg extreme (old low)"),
                      (origin + d * 1.272 * L, "1.272 extension"), (origin + d * 1.618 * L, "1.618 extension")]
            tps = _targets(clusters, entry, sl, d, atr, extra=ext_lv)
            p = _plan(side, "market" if inside else "limit", "Trend pullback into OTE (61.8–78.6% retracement)",
                      entry, sl, tps,
                      [f"{leg['event_kind']} {'up' if d == 1 else 'down'} {leg['bars_old']} candles ago started a "
                       f"{'bullish' if d == 1 else 'bearish'} leg of {L / atr:.1f} ATR",
                       "OTE = where ICT/SMC traders join the trend after a retracement"],
                      f"a close {'below' if d == 1 else 'above'} the leg origin {X['f'].p(origin)}", gm.EXPIRY,
                      family="pullback", extra=dict(zone=(min(z_near, z_far), max(z_near, z_far))))
            if p:
                plans.append(p)
    # 2) liquidity sweep → MSS → FVG (ICT / SMC reversal), from the same point-in-time detector as training
    for s in X["setups"][::-1][:6]:
        if s["family"] != "sweep" or s["idx"] < len(d_) - 1 - gm.EXPIRY:
            continue
        res = gm.simulate(d_, s, X["cost"], X["slip"])
        if res["status"] != "pending":
            continue
        d = s["dir"]
        if d * (price - s["entry"]) <= 0:
            continue
        tps = _targets(clusters, s["entry"], s["sl"], d, atr, extra=[(s["tp"], "liquidity target")])
        p = _plan("long" if d == 1 else "short", "limit", "Liquidity sweep → market structure shift → FVG entry",
                  s["entry"], s["sl"], tps,
                  [f"{'sell' if d == 1 else 'buy'}-side liquidity was swept {len(d_) - 1 - s['sweep_idx']} candles ago,"
                   f" then a {s['event_kind']} {'up' if d == 1 else 'down'} left an FVG "
                   f"{X['f'].p(s['fvg_bottom'])}–{X['f'].p(s['fvg_top'])}"],
                  f"price trading beyond the sweep extreme ({X['f'].p(s['sl'])})",
                  gm.EXPIRY - (len(d_) - 1 - s["idx"]), family="sweep",
                  extra=dict(zone=(s["fvg_bottom"], s["fvg_top"])))
        if p:
            plans.append(p)
        break
    # 3) reaction at the nearest demand / supply (zone, order block, FVG, HTF zone)
    for d, pool in ((1, below), (-1, above)):
        cands = [c for c in pool if abs(c["dist_atr"]) <= 4.0 and d * c.get("polarity", 0) > 0 and
                 any(t in c["types"] for t in ("zone", "ob", "fvg", "htf_zone", "user")) and
                 (c["hi"] - c["lo"]) <= 2.5 * atr]
        if not cands:
            continue
        c = max(cands, key=lambda c: c["strength"] - 0.3 * abs(c["dist_atr"]))
        if d == 1:
            entry, sl = c["hi"], c["lo"] - 0.15 * atr
        else:
            entry, sl = c["lo"], c["hi"] + 0.15 * atr
        if abs(entry - sl) < 0.3 * atr:
            sl = entry - d * 0.5 * atr
        if d * (price - entry) < 0:
            continue
        tps = _targets(clusters, entry, sl, d, atr)
        p = _plan("long" if d == 1 else "short", "limit", f"Reaction at {'demand' if d == 1 else 'supply'}: "
                  + " + ".join(c["labels"][:3]), entry, sl, tps,
                  [f"{len(c['labels'])} thing{'s' if len(c['labels']) > 1 else ''} line up at "
                   f"{X['f'].p(c['lo'])}–{X['f'].p(c['hi'])}: " + ", ".join(c["labels"][:4])],
                  f"a close {'below' if d == 1 else 'above'} {X['f'].p(c['lo'] if d == 1 else c['hi'])}", gm.EXPIRY,
                  extra=dict(zone=(c["lo"], c["hi"])))
        if p:
            plans.append(p)
    # 4) breakout of the nearest untaken swing (stop order = the BOS trigger)
    for d in (1, -1):
        trig = liq["buy_side"][0] if d == 1 and liq["buy_side"] else (
            liq["sell_side"][0] if d == -1 and liq["sell_side"] else None)
        opp = liq["sell_side"][0] if d == 1 and liq["sell_side"] else (
            liq["buy_side"][0] if d == -1 and liq["buy_side"] else None)
        if trig is None or abs(trig["price"] - price) > max_trig_atr * atr:
            continue
        entry = trig["price"] + d * 0.1 * atr
        sl = (opp["price"] - d * 0.1 * atr) if opp is not None else entry - d * 1.5 * atr
        if abs(entry - sl) > 2.0 * atr:                                # structure too far: volatility stop
            sl = entry - d * 1.5 * atr
        tps = _targets(clusters, entry, sl, d, atr)
        p = _plan("long" if d == 1 else "short", "stop", f"Breakout: {'buy' if d == 1 else 'sell'} stop beyond the "
                  f"swing {'high' if d == 1 else 'low'} (BOS trigger)", entry, sl, tps,
                  [f"a close beyond {X['f'].p(trig['price'])} would be a "
                   f"{'BOS' if X['trend'] == d else 'CHoCH'} {'up' if d == 1 else 'down'}"],
                  f"price back {'below' if d == 1 else 'above'} {X['f'].p(sl)}", gm.EXPIRY)
        if p:
            plans.append(p)
    # 5) chart patterns
    for pat in X["patterns"]:
        if pat["status"] != "forming" or pat.get("bias", 0) == 0:
            continue
        d = pat["bias"]
        if pat["kind"] == "lines":
            trig = pat["breakout_up"] if d == 1 else pat["breakout_down"]
            tgt = pat["target_up"] if d == 1 else pat["target_down"]
            stop_ref = pat["breakout_down"] if d == 1 else pat["breakout_up"]
        else:
            trig, tgt = pat["trigger"], pat.get("target_up" if d == 1 else "target_down")
            stop_ref = pat["invalidation"]
        if tgt is None or abs(trig - price) > 4 * atr:
            continue
        entry = trig + d * 0.1 * atr
        sl = stop_ref - d * 0.1 * atr
        if not d * (entry - sl) > 0.3 * atr or abs(entry - sl) > 5 * atr:
            continue
        tps = _targets(clusters, entry, sl, d, atr, extra=[(tgt, f"{pat['name'].lower()} measured target")])
        p = _plan("long" if d == 1 else "short", "stop", f"{pat['name']} breakout", entry, sl, tps,
                  [f"{pat['name']} ({'bullish' if d == 1 else 'bearish'} pattern), measured move to {X['f'].p(tgt)}"],
                  f"price back inside the pattern beyond {X['f'].p(sl)}", gm.EXPIRY)
        if p:
            plans.append(p)
    return plans


def _finalize_plans(plans, X, widen=True):
    price, atr, clusters = X["price"], X["atr"], X["clusters"]
    min_risk = max(0.5 * atr, 8 * X["cost"] * price) if widen else 0.25 * atr   # costs ≤ 12.5% of the risk
    out = []
    for p in plans:
        if p["type"] == "stop" and p["dir"] * (price - p["entry"]) >= 0:
            # the breakout is already happening on the forming candle: don't chase, plan the retest instead
            trig = p["entry"] - p["dir"] * 0.1 * atr
            side_w = ("buy the retest of the broken swing high" if p["dir"] == 1 else
                      "sell the retest of the broken swing low")
            p = dict(p, type="limit", entry=float(trig),
                     setup=f"Breakout retest: {side_w}" if p["setup"].startswith("Breakout:") else p["setup"] + " (retest)",
                     why=p["why"] + ["price is already through the trigger: wait for the candle to close beyond it and "
                                     "buy/sell the retest instead of chasing"])
            p["risk"] = abs(p["entry"] - p["sl"])
            if not p["dir"] * (p["entry"] - p["sl"]) > 0:
                continue
            p["rr"] = [abs(t - p["entry"]) / p["risk"] for t in p["tps"]]
        if p["risk"] < min_risk:
            d = p["dir"]
            sl = p["entry"] - d * min_risk
            q = _plan(p["side"], p["type"], p["setup"], p["entry"], sl,
                      _targets(clusters, p["entry"], sl, d, atr, extra=list(zip(p["tps"], p["tp_labels"]))),
                      p["why"] + [f"stop widened to {min_risk / atr:.2f} ATR so noise and costs don't decide the trade"],
                      p["invalidation"], p["expiry"], p.get("family"),
                      {k: p[k] for k in ("zone", "strategy", "strategy_name", "signal_age") if k in p})
            if q is None:
                continue
            p = q
        if X.get("sctx") is not None:
            p = pb.guard(p, X)
        out.append(p)
    return out


def evaluate_plan(p, X, brain):
    """Probabilities, costs, confluence checklist and grade for one plan."""
    price, atr, row = X["price"], X["atr"], X["row"]
    d, entry, sl = p["dir"], p["entry"], p["sl"]
    risk = abs(entry - sl)
    b_atr = risk / atr
    a_atrs = np.array([abs(t - entry) / atr for t in p["tps"]])
    p_tp, p_sl, p_to = brain.race(row, np.full(len(a_atrs), float(d)), a_atrs, np.full(len(a_atrs), b_atr))
    dist = abs(price - entry) / atr
    if p["type"] == "market" or dist < 0.05:
        p_fill = 1.0
    elif p["type"] == "limit":
        p_fill = float(brain.touch(row, [-d], [dist])[0]) * float(getattr(brain, "b", {}).get("fill_ratio", 1.0) or 1.0)
    else:
        p_fill = float(brain.touch(row, [d], [dist])[0])
    cost_r = X["cost"] * entry / risk
    rr = np.array(p["rr"])
    be = (1 + cost_r) / (1 + rr)
    ev = p_tp * rr - p_sl - cost_r
    p.update(p_fill=min(1.0, p_fill), p_tp=p_tp.tolist(), p_sl=p_sl.tolist(), p_open=p_to.tolist(),
             breakeven=be.tolist(), ev=ev.tolist(), cost_r=float(cost_r), risk_atr=float(b_atr),
             chase_limit=float((p["tps"][min(1, len(p["tps"]) - 1)] + 1.5 * sl) / 2.5))
    # confluence checklist
    chk = []

    def add(ok, text, w, bad=0.0):
        chk.append(dict(ok=bool(ok), text=text, w=w if ok else bad))

    for k, (h, w, bad) in enumerate(zip(X["ladder"][:2], (1.5, 1.0), (-1.0, -0.5))):
        hs = X["htf"].get(h)
        if hs is None:
            continue
        add(hs["trend"] == d, f"{h} trend {'agrees' if hs['trend'] == d else 'is ' + hs['trend_word']}", w,
            bad if hs["trend"] == -d else 0.0)
    add(X["trend"] == d, f"{X['tf']} structure {'agrees' if X['trend'] == d else 'does not agree'}", 1.0)
    dr = X["dr"]
    if dr is not None:
        good = (entry <= dr["eq"]) if d == 1 else (entry >= dr["eq"])
        add(good, f"entry in {'discount' if d == 1 else 'premium'} (below/above 50% of the range)" if good else
            f"entry in {'premium' if d == 1 else 'discount'}: buying high / selling low in the range", 1.0)
    near = [c for c in X["clusters"] if c["lo"] - 0.3 * atr <= entry <= c["hi"] + 0.3 * atr]
    if p["type"] == "stop":
        room = [c for c in X["clusters"] if d * (c["price"] - entry) > 0.1 * atr and d * c.get("polarity", 0) < 0
                and abs(c["price"] - entry) < abs(entry - sl)]
        add(not room, "room to run after the breakout (no opposing level within 1R)" if not room else
            f"opposing level only {abs(room[0]['price'] - entry) / atr:.1f} ATR past the trigger", 1.0, -0.5)
    else:
        good = sorted({lab for c in near for lab in c["labels"] if polarity(lab) in (0, d)} -
                      {"round number", "daily open", "weekly open", "monthly open"})[:4]
        bad = sorted({lab for c in near for lab in c["labels"] if polarity(lab) == -d})[:2]
        if good and not bad:
            add(True, "entry at a key level: " + ", ".join(good), 1.5 if len(good) >= 2 else 1.0)
        elif bad:
            add(False, f"entry is into {'supply / resistance' if d == 1 else 'demand / support'}: " + ", ".join(bad),
                0.0, -0.5)
        else:
            add(False, "entry not at a key level", 0.0)
    sw = [s for s in X["A"]["sweeps"] if s["side"] == ("sell-side" if d == 1 else "buy-side")
          and len(X["df"]) - 1 - s["idx"] <= 30]
    add(bool(sw), f"{'sell' if d == 1 else 'buy'}-side liquidity swept recently (stop hunt done)" if sw else
        "no recent liquidity sweep in your favour", 1.0)
    tgt_liq = any("liquidity" in lab.lower() or "equal" in lab.lower() or lab.startswith(("PDH", "PDL", "PWH", "PWL"))
                  for lab in p["tp_labels"][:2])
    add(tgt_liq, "target is resting liquidity (draw on liquidity)" if tgt_liq else "target isn't a liquidity pool", 0.5)
    rsi = X["rsi"]
    mom_ok = (40 <= rsi <= 72) if d == 1 else (28 <= rsi <= 60)
    divs = [v for v in X["divs"] if v["bias"] == d]
    add(mom_ok or bool(divs), f"momentum fits (RSI {rsi:.0f}{', ' + divs[0]['name'].lower() if divs else ''})"
        if (mom_ok or divs) else f"RSI {rsi:.0f} is stretched {'up' if d == 1 else 'down'}", 0.5)
    pats = [q for q in X["patterns"] if q.get("bias_now") == d]
    if pats:
        add(True, f"pattern agrees: {pats[0]['name'].lower()}", 0.5)
    if X["intraday"] and X["asset_class"] != "crypto":
        s = X["session"]
        ok = bool([k for k in s.get("active", []) if "killzone" in k or "New York" in k]) or (
            s.get("minutes_to_next") is not None and s["minutes_to_next"] <= 60)
        add(ok, "inside / just before a London or New York killzone" if ok else
            f"outside the killzones (next: {s.get('next_killzone')} in {s.get('minutes_to_next')} min)", 0.5)
    rr2 = p["rr"][1] if len(p["rr"]) > 1 else p["rr"][0]
    add(rr2 >= 2.5, f"reward:risk {rr2:.1f} to TP2" + (" ≥ 2.5" if rr2 >= 2.5 else ""), 0.5)
    first = p["tps"][0]
    block = [c for c in X["clusters"] if c["strength"] >= 2.5 and d * (c["price"] - entry) > 0.1 * atr
             and d * (first - c["price"]) > 0.6 * abs(first - entry)]
    add(not block, "clear path to TP1" if not block else f"strong level in the way at {X['f'].p(block[0]['price'])}",
        0.5, -0.5)
    add(cost_r <= 0.10, f"costs {pct(cost_r)} of the risk" + (" (fine)" if cost_r <= 0.10 else
                                                             " (too high for this timeframe)"), 0.5,
        -1.0 if cost_r > 0.25 else 0.0)
    score = float(sum(c["w"] for c in chk))
    grade = "A+" if score >= 8 else "A" if score >= 6.5 else "B" if score >= 5 else "C" if score >= 3.5 else "D"
    ny = X["df"].index[-1].tz_convert("America/New_York")
    hf = gm.hist_factors(gm.align_row(row, [d]).iloc[0], rr2, X["tf"], ny.hour + ny.minute / 60)
    hgrade = gm.grade_of(sum(bool(v) for v in hf.values()))
    p.update(checklist=chk, score=score, grade=grade, hist_grade=hgrade, evidence=_evidence(p, X, brain, hgrade))
    return p


def _evidence(p, X, brain, hgrade):
    tab = brain.b.get("setup_stats") if hasattr(brain, "b") else None
    if p.get("family") is None or tab is None or len(tab) == 0:
        return None
    g = gm.TF_GROUP[X["tf"]]
    r = tab[(tab["family"] == p["family"]) & (tab["asset_class"] == X["asset_class"]) & (tab["tf_group"] == g)]
    scope = f"{X['asset_class']} {g}"
    if r["trades"].sum() < 50:
        r = tab[(tab["family"] == p["family"]) & (tab["tf_group"] == g)]
        scope = f"all markets, {g}"
    if r["trades"].sum() < 30:
        return None
    rg = r[r["grade"] == hgrade]
    n = float(r["trades"].sum())
    out = dict(scope=scope, trades=int(n), win_rate=float((r["win_rate"] * r["trades"]).sum() / n),
               avg_R=float((r["avg_net_R"] * r["trades"]).sum() / n), grade=hgrade)
    if rg["trades"].sum() >= 30:
        m = float(rg["trades"].sum())
        out.update(grade_trades=int(m), grade_win=float((rg["win_rate"] * rg["trades"]).sum() / m),
                   grade_R=float((rg["avg_net_R"] * rg["trades"]).sum() / m))
    return out


def size_position(p, X, account, risk_pct, leverage=None):
    """Units/lots for a fixed % risk, notional, fees, leverage band and (crypto) liquidation price."""
    C = X["C"]
    entry, sl, d = p["entry"], p["sl"], p["dir"]
    q2usd, warn = X["q2usd"]
    risk_amt = account * risk_pct / 100.0
    per_unit = abs(entry - sl) * q2usd
    units = risk_amt / per_unit
    notional = units * entry * q2usd
    cls, sub = C["asset_class"], C["sub"]
    spec = SPECS["crypto"] if cls == "crypto" else SPECS.get(sub if sub in ("gold", "silver") else cls, SPECS["other"])
    out = dict(account=account, risk_pct=risk_pct, risk_amount=risk_amt, units=units, unit=spec["unit"],
               notional=notional, fees=X["cost"] * notional, warn=warn)
    if cls == "crypto":
        min_units = spec["min_notional"] / (entry * q2usd)
        out.update(min_units=min_units, too_small=units < min_units)
    else:
        lots = units / spec["lot"]
        step = spec["step"]
        lots_r = math.floor(lots / step + 1e-9) * step
        min_units = spec["min_lot"] * spec["lot"]
        out.update(lot_size=spec["lot"], lots=lots, lots_rounded=lots_r, min_units=min_units,
                   too_small=lots_r < spec["min_lot"], risk_at_rounded=lots_r * spec["lot"] * per_unit)
    out["min_capital"] = min_units * per_unit / (risk_pct / 100.0)
    lev_needed = notional / account
    out["leverage_needed"] = lev_needed
    maxlev = MAX_LEV.get(sub, 20)
    if cls == "crypto":
        mmr = MMR.get(sub, 0.01)
        sl_frac = abs(entry - sl) / entry
        safe = max(1, min(maxlev, int(1.0 / (1.5 * sl_frac + mmr))))
        want = max(1.0, lev_needed, notional / (0.25 * account))
        rec = next((L for L in STD_LEV if want <= L <= safe), None)
        if rec is None:
            rec = max([L for L in STD_LEV if L <= safe] or [1])
        liq = lambda L: entry * (1 - 1.0 / L + mmr) if d == 1 else entry * (1 + 1.0 / L - mmr)
        out.update(mode="isolated", mmr=mmr, max_safe_leverage=safe, leverage=rec, margin=notional / rec,
                   margin_pct=notional / rec / account, liquidation=liq(rec),
                   liq_beyond_stop=(liq(rec) < sl) if d == 1 else (liq(rec) > sl))
        if leverage:
            L = float(leverage)
            out["user_leverage"] = dict(leverage=L, margin=notional / L, liquidation=liq(L),
                                        safe=(liq(L) < sl) if d == 1 else (liq(L) > sl))
    else:
        out.update(mode="cross (CFD/forex)", broker_max_leverage=maxlev,
                   margin_at={f"1:{L}": notional / L for L in (30, 100, 500) if L <= max(500, maxlev)},
                   leverage=max(1, math.ceil(lev_needed)))
    return out


# ───────────────────────────── user drawings ─────────────────────────────
def analyze_user_drawings(items, X, brain):
    """The user's own levels, zones and trendlines: where they are now, how often price reacted there, odds."""
    out, levels = [], []
    d_, atr, price = X["df"], X["atr"], X["price"]
    h, l, c = d_["high"].to_numpy(), d_["low"].to_numpy(), d_["close"].to_numpy()
    piv = X["A"]["pivots"]
    ts = md.epoch_s(d_.index)
    for k, it in enumerate(items or []):
        it = dict(it)
        typ = it.get("type", "hline")
        lab = it.get("label") or it.get("text") or f"your {typ}"
        res = dict(it, id=it.get("id", f"user{k}"))
        try:
            if typ in ("hline", "level"):
                lv = float(it["price"])
                lo = hi = lv
            elif typ == "zone":
                lo, hi = sorted([float(it["bottom"]), float(it["top"])])
                lv = (lo + hi) / 2
            elif typ == "trendline":
                t1, t2, p1, p2 = float(it["t1"]), float(it["t2"]), float(it["p1"]), float(it["p2"])
                slope = (p2 - p1) / (t2 - t1) if t2 != t1 else 0.0
                lv = p1 + slope * (ts[-1] - t1)
                lo = hi = lv
                line = p1 + slope * (ts - t1)
                tol = 0.25 * atr
                tch = [q for q in piv if abs(q["price"] - line[q["idx"]]) <= tol and ts[q["idx"]] >= min(t1, t2)]
                side_now = np.sign(c[-1] - line[-1])
                cross = np.flatnonzero(np.sign(c - line)[-30:] != side_now)
                res.update(value_now=float(lv), touches=len(tch),
                           status="broken recently" if len(cross) else
                           ("price above it" if side_now > 0 else "price below it"))
            else:
                out.append(res)
                continue
            tol = 0.25 * atr
            turns = len([q for q in piv if lo - tol <= q["price"] <= hi + tol])
            hist = level_history(d_, lo, hi, atr) if typ != "trendline" else dict(tests=0, held=0, broke=0)
            dist = (lv - price) / atr
            side = 1.0 if lv > price else -1.0
            p_touch = float(brain.touch(X["row"], [side], [abs(dist)])[0])
            conf = [cl for cl in X["clusters"] if cl["lo"] - 0.3 * atr <= hi and cl["hi"] + 0.3 * atr >= lo
                    and "user" not in cl["types"]]
            res.update(level=float(lv), distance_atr=float(dist), p_touch_24=p_touch, past_turns=turns,
                       tests=hist["tests"], held=hist["held"], broke=hist["broke"],
                       confluence=sorted({x for cl in conf for x in cl["labels"]})[:5],
                       role="resistance" if lv > price else "support")
            levels.append(dict(price=lo, label=f"{lab}", type="user", weight=2.0, idx=0))
            if hi != lo:
                levels.append(dict(price=hi, label=f"{lab}", type="user", weight=2.0, idx=0))
        except Exception as e:
            res["error"] = f"couldn't read this drawing: {e}"
        out.append(res)
    return out, levels


# ───────────────────────────── main ─────────────────────────────
def analyze(symbol, tf="1h", account=1000.0, risk_pct=1.0, mode="pro", user_drawings=None, brain=None,
            leverage=None, now=None, show_bars=SHOW_BARS, data=None, asof=None, strategies=None):
    """Full analysis of `symbol` on timeframe `tf` (1m 5m 15m 30m 1h 4h 1D 3D 1W 1M).
    `asof` (a replay): the analysis is run as it would have been at that moment. Every candle it reads, on every
    timeframe, had closed by then, and the clock is that moment.
    `strategies` (catalog ids from strategies.py): the analysis uses only these strategies' setups and draws what
    they use. None = the analyst's own mix of setups."""
    t_start = time.time()
    brain = brain or _STATE["brain"] or gm.FallbackBrain()
    tf = md.norm_tf(tf)
    now = asof or now or md.utcnow()
    notes = []
    C = data or md.get_candles(symbol, tf, bars=BARS[tf])
    need = 60 if tf in ("3D", "1W", "1M") else 120
    if len(C["df"]) < need:
        alt = {"1M": "1W", "1W": "1D", "3D": "1D"}.get(tf, "a higher timeframe")
        raise ValueError(f"Only {len(C['df'])} closed {tf} candles for {C['symbol']} (need {need}). "
                         f"Try {alt}.")
    dfe, A = se.analyze(C["df"])
    n = len(dfe)
    ladder = gm.LADDER[tf]
    HT = {}
    for h in ladder:
        try:
            Ch = md.get_candles(symbol, h, bars=HTF_BARS, end=asof)
            if len(Ch["df"]) >= 40:
                dh, Ah = se.analyze(Ch["df"])
                HT[h] = dict(C=Ch, df=dh, A=Ah)
        except Exception as e:
            notes.append(f"{h} candles unavailable ({str(e)[:80]})")
    daily = HT["1D"]["C"] if "1D" in HT else None
    if daily is None and md.TF_MIN[tf] < 43830:
        daily = _safe(lambda: md.get_candles(symbol, "1D", bars=400, min_bars=5, end=asof), None, notes, "daily candles")
    intraday = md.TF_MIN[tf] < 1440
    frames = {h: v["C"]["df"] for h, v in HT.items()}
    F, ctx = gm.raw_features(dfe, tf, C["asset_class"], htf=frames,
                             daily=daily["df"] if (daily is not None and intraday) else None)
    ctx["fvgs"] = A["fvgs"]
    row = F.iloc[-1]
    price, atr = float(C["price"]), float(dfe["atr"].iloc[-1])
    f = Fmt(C["decimals"])
    cost, slip = gm.cost_of(C["sub"])
    X = dict(C=C, df=dfe, A=A, tf=tf, ladder=ladder, price=price, atr=atr, f=f, row=row, cost=cost, slip=slip,
             asset_class=C["asset_class"], intraday=intraday, trend=int(A["trend"][-1]), brain=brain)
    # ── tools
    X["htf"] = {h: _safe(lambda h=h: _htf_summary(h, HT[h]["df"], HT[h]["A"], brain, C["asset_class"], frames),
                         None, notes, f"{h} summary") for h in HT}
    X["htf"] = {k: v for k, v in X["htf"].items() if v is not None}
    X["phase"] = _safe(lambda: gl.market_phase(dfe, A), dict(label="?", notes=[], adx=0, er=0, atr_ratio=1, bbw_pct=.5),
                       notes, "market phase")
    X["dr"] = _safe(lambda: gl.dealing_range(dfe, A), None, notes, "dealing range")
    X["leg"] = _safe(lambda: gl.impulse_leg(dfe, A), None, notes, "impulse leg")
    X["obs"] = _safe(lambda: gl.order_blocks(dfe, A), [], notes, "order blocks")
    tl = _safe(lambda: gl.trendlines(dfe, A), ([], [], []), notes, "trendlines")
    X["lines"] = tl[0]
    X["patterns"] = _safe(lambda: gl.chart_patterns(dfe, A, tl[1], tl[2]), [], notes, "chart patterns")
    rs = gl.rsi(dfe["close"])
    X["rsi"] = float(rs.iloc[-1])
    X["divs"] = _safe(lambda: gl.rsi_divergences(dfe, A, rs), [], notes, "divergences")
    X["vp"] = _safe(lambda: gl.volume_profile(dfe), None, notes, "volume profile")
    X["emas"] = {k: gl.ema(dfe["close"], k) for k in (20, 50, 200)}
    X["vwap"] = _safe(lambda: gl.vwap_daily(dfe), None, notes, "VWAP") if md.TF_MIN[tf] <= 60 else None
    X["session"] = gl.sessions(dfe, now) if md.TF_MIN[tf] <= 60 else {}
    X["period"] = _safe(lambda: gl.period_levels(daily["df"], daily["forming"]), {}, notes, "period levels") \
        if daily is not None else {}
    d_atr = float(daily["df"]["high"].sub(daily["df"]["low"]).tail(20).mean()) if daily is not None else \
        atr * math.sqrt(1440 / md.TF_MIN[tf])
    X["rounds"] = gl.round_levels(price, d_atr)
    X["setups"] = _safe(lambda: gm.find_setups(ctx, cost), [], notes, "setup detector")
    X["q2usd"] = quote_to_usd(C, asof)
    # ── key levels
    items = []
    add = lambda price_, label, typ, w=1.0, idx=0: items.append(dict(price=float(price_), label=label, type=typ,
                                                                     weight=w, idx=int(idx)))
    for z in A["zones"]:
        lab = f"{tf} {'support' if z['role'] == 'support' else 'resistance' if z['role'] == 'resistance' else 'S/R'} zone ×{z['touches']}"
        add(z["bottom"], lab, "zone", 0.5 + 0.25 * z["touches"], z["first_idx"])
        add(z["top"], lab, "zone", 0.5 + 0.25 * z["touches"], z["first_idx"])
    for h, v in HT.items():
        for z in v["A"]["zones"][:4]:
            lab = (f"{h} support zone" if z["top"] < price else f"{h} resistance zone" if z["bottom"] > price
                   else f"{h} zone (price inside)")
            add(z["bottom"], lab, "htf_zone", 1.5)
            add(z["top"], lab, "htf_zone", 1.5)
        for s in v["A"]["liquidity"]["buy_side"][:2]:
            add(s["price"], f"{h} swing high (buy-side liquidity)", "htf_liq", 1.5)
        for s in v["A"]["liquidity"]["sell_side"][:2]:
            add(s["price"], f"{h} swing low (sell-side liquidity)", "htf_liq", 1.5)
    for ob in X["obs"]:
        lab = (f"{'bullish' if ob['role'] == 'demand' else 'bearish'} breaker" if ob["status"] == "breaker" else
               f"{'bullish' if ob['direction'] == 'bullish' else 'bearish'} OB ({ob['status']})")
        add(ob["bottom"], lab, "ob", 1.2, ob["idx"])
        add(ob["top"], lab, "ob", 1.2, ob["idx"])
    open_fvg = sorted([g for g in A["fvgs"] if g["filled_idx"] is None],
                      key=lambda g: abs((g["top"] + g["bottom"]) / 2 - price))[:6]
    for g in open_fvg:
        lab = f"{g['direction']} FVG"
        add(g["bottom"], lab, "fvg", 1.0, g["idx"])
        add(g["top"], lab, "fvg", 1.0, g["idx"])
    for s in A["liquidity"]["buy_side"][:4]:
        add(s["price"], "buy-side liquidity (swing high)", "liq", 1.0, s["idx"])
    for s in A["liquidity"]["sell_side"][:4]:
        add(s["price"], "sell-side liquidity (swing low)", "liq", 1.0, s["idx"])
    for pl in A["liquidity"]["pools"]:
        if not pl["taken"] and n - 1 - pl["idx_b"] <= 300:
            add(pl["level"], "equal highs (liquidity)" if pl["kind"] == "EQH" else "equal lows (liquidity)", "liq",
                1.5, pl["idx_a"])
    P = X["period"]
    for k, lab in (("pdh", "PDH (previous day high)"), ("pdl", "PDL (previous day low)"),
                   ("pwh", "PWH (previous week high)"), ("pwl", "PWL (previous week low)"),
                   ("pmh", "PMH (previous month high)"), ("pml", "PML (previous month low)")):
        if P.get(k) is not None and (intraday or k[1] in "wm"):
            add(P[k], lab, "period", 1.5 if k[1] != "d" else 1.2)
    for k, lab in (("day_open", "daily open"), ("week_open", "weekly open"), ("month_open", "monthly open")):
        if P.get(k) is not None and (intraday or k != "day_open"):
            add(P[k], lab, "open", 0.6)
    S_ = X["session"]
    if S_.get("asia"):
        add(S_["asia"]["high"], "Asian range high", "session", 0.8, S_["asia"]["i1"])
        add(S_["asia"]["low"], "Asian range low", "session", 0.8, S_["asia"]["i1"])
    if S_.get("midnight_open") is not None and X["asset_class"] != "crypto":
        add(S_["midnight_open"], "NY midnight open", "open", 0.6)
    for r_ in X["rounds"]:
        add(r_["price"], "round number", "round", 0.5)
    add(float(X["emas"][200].iloc[-1]), "EMA 200", "ema", 0.6)
    if X["vwap"] is not None:
        add(float(X["vwap"].iloc[-1]), "VWAP", "vwap", 0.6)
    if X["vp"] is not None:
        add(X["vp"]["poc"], "volume POC", "vp", 0.8)
    if X["dr"] is not None:
        add(X["dr"]["eq"], "equilibrium (50% of range)", "eq", 0.5)
    for L in X["lines"]:
        add(L["p2"], L["name"].lower(), "trendline", 1.0)
    X["user"], ulev = analyze_user_drawings(user_drawings, dict(X, clusters=[]), brain)
    items.extend(ulev)
    clusters = collect_levels(price, atr, items)
    # probability for every level (only those within 12 ATR are meaningful in 24 candles)
    near = [c for c in clusters if abs(c["dist_atr"]) <= 12]
    if near:
        pr = brain.touch(row, [1.0 if c["dist_atr"] > 0 else -1.0 for c in near], [abs(c["dist_atr"]) for c in near])
        for c, p_ in zip(near, pr):
            c["p_touch"] = float(p_)
    for c in clusters:
        typ = None
        labs = " ".join(c["labels"]).lower()
        if "equal highs" in labs:
            typ = "eqh"
        elif "equal lows" in labs:
            typ = "eql"
        elif "pdh" in labs:
            typ = "pdh"
        elif "pdl" in labs:
            typ = "pdl"
        elif "buy-side" in labs:
            typ = "swing_high"
        elif "sell-side" in labs:
            typ = "swing_low"
        elif "fvg" in labs:
            typ = "fvg_bull" if c["side"] == "below" else "fvg_bear"
        c["reaction"] = reaction_stats(brain, X["asset_class"], tf, typ) if typ else None
    X["clusters"] = clusters
    X["user"], _ = analyze_user_drawings(user_drawings, X, brain)
    # ── strategies: the chosen ones, their record here, the liquidity guard's sweep depth, the timeframe table
    X["chosen"] = pb.chosen(strategies)
    X["sctx"] = st.Ctx(dfe, tf, A, C["asset_class"])
    X["depth"] = X["sctx"].sweep_depth()
    # the liquidity head (Notebook 10): every pool near price with its odds of being swept, of reversing after a
    # sweep, and how deep hunts reach there. None without a trained head.
    X["liq"] = _safe(lambda: lh.assess(getattr(brain, "liq", None), X["sctx"], row, tf, price), None, notes,
                     "liquidity model")
    X["records"] = pb.records(X, X["chosen"])
    X["mtf"] = _safe(lambda: pb.mtf(X, symbol, HT, asof, X["chosen"]), dict(rows=[], alignment=0.0, text=""), notes,
                     "timeframe table")
    X["mtf_alignment"] = X["mtf"].get("alignment", 0.0)
    # ── probabilities
    pdir = brain.direction(row)
    rng = brain.expected_range(row, 0.15)
    rng50 = brain.expected_range(row, 0.5)
    k = float(brain.k_dir)
    X["prob"] = dict(p_up=float(pdir[0]), p_down=float(pdir[1]), p_flat=float(pdir[2]), k_atr=k,
                     up_level=price + k * atr, down_level=price - k * atr, horizon=gm.H_TOUCH,
                     range70=(price - rng["down"] * atr, price + rng["up"] * atr),
                     range50=(price - rng50["down"] * atr, price + rng50["up"] * atr))
    X["prob"]["bar"] = brain.direction_bar(row, raw=pdir)     # the bull/bear bar: range and lean views
    # ── plans
    X["notes"] = notes
    plans = [evaluate_plan(p, X, brain) for p in make_plans(X)]
    plans = [p for p in plans if p["rr"][0] >= 0.8]
    # the strategy head (Notebook 10): how signals like this one did across years of markets
    for p in plans:
        if p.get("strategy"):
            p["head"] = _safe(lambda p=p: sh.score(getattr(brain, "strat", None), X["sctx"], row, p["strategy"],
                                                   p["dir"], p["entry"], p["sl"], p["tps"][0], cost, slip),
                              None, notes, "strategy model")
    best = {}
    for p in plans:
        key = p["side"]
        j = min(1, len(p["ev"]) - 1)
        val = p["score"] + 4 * float(np.clip(p["ev"][j], -0.5, 0.5)) * max(0.3, p["p_fill"])
        val += pb.rank_bonus(p, X)
        p["rank"] = val
        if key not in best or val > best[key]["rank"]:
            best[key] = p
    order = sorted(best.values(), key=lambda p: -p["rank"])
    primary = order[0] if order else None
    alternative = order[1] if len(order) > 1 else None
    for p in plans:
        p["sizing"] = size_position(p, X, account, risk_pct, leverage)
    X["plans"], X["primary"], X["alternative"] = plans, primary, alternative
    X["conflict"] = pb.conflict(plans, X)
    X["fixed_time"] = _safe(lambda: pb.fixed_time(X, X["chosen"]), None, notes, "fixed-time view")
    X["verdict"] = pb.soften(_verdict(primary, X), primary, X, pb.confidence(primary, X), X["conflict"])
    X["liq_map"] = _safe(lambda: pb.liquidity_map(X), None, notes, "liquidity map")
    X["skill"] = model_skill(brain)
    X["notes"] = notes
    result = dict(
        symbol=C["symbol"], query=C["query"], tf=tf, tf_label=md.TF_LABEL[tf], source=C["source"],
        asset_class=C["asset_class"], sub=C["sub"], time=str(dfe.index[-1]), now=str(now), price=price,
        decimals=C["decimals"], atr=atr, data_age_minutes=C["age_minutes"], forming=C["forming"] is not None,
        verdict=X["verdict"], plan=_public_plan(primary, f), alternative=_public_plan(alternative, f),
        plans=[_public_plan(p, f) for p in sorted(plans, key=lambda p: -p["rank"])],
        timeframes=[dict(X["htf"][h]) for h in ladder if h in X["htf"]],
        structure=dict(trend=X["trend"], phase=X["phase"], dealing_range=X["dr"], leg=X["leg"],
                       swings=[dict(label=s["label"], price=s["price"]) for s in A["swings"][-8:]],
                       last_event=None if not A["events"] else {k_: A["events"][-1][k_] for k_ in
                                                                ("kind", "direction", "level")}),
        levels=[{k_: v for k_, v in c.items()} for c in clusters if abs(c["dist_atr"]) <= 15],
        patterns=[{k_: v for k_, v in q.items() if k_ not in ("support", "resistance")} for q in X["patterns"]],
        divergences=X["divs"], session=X["session"], period_levels=X["period"], rsi=X["rsi"],
        probabilities=X["prob"], model=X["skill"], user_drawings=X["user"], notes=notes,
        strategies=_strategies_block(X), mtf=X["mtf"], fixed_time=X["fixed_time"], liquidity=X["liq_map"],
        how_to_enter=pb.how_to_enter(primary, X), layers_scope=st.layers_for(X["chosen"]) if X["chosen"] else None,
        disclaimer="Research tool, not financial advice. Probabilities are calibrated estimates from history; "
                   "markets change. Analysis may differ on other timeframes and changes with every new candle. "
                   "Paper-trade first and never risk money you can't afford to lose.")
    result["report_pro"] = report(X, beginner=False)
    result["report_beginner"] = report(X, beginner=True)
    result["report"] = result["report_beginner"] if mode == "beginner" else result["report_pro"]
    cd = chart_data(X, show_bars)
    result["candles"], result["drawings"], result["indicators"] = cd["candles"], cd["drawings"], cd["indicators"]
    result["view"] = cd["view"]
    result["seconds"] = round(time.time() - t_start, 2)
    ctxs = _STATE.setdefault("ctx", {})
    ctxs[(result["symbol"], tf)] = X                                # kept for review_trade()
    while len(ctxs) > 20:
        ctxs.pop(next(iter(ctxs)))
    return result


def jsonable(x):
    """Plain-Python copy of an analysis (numpy types → int/float/list), ready for json.dumps / a web app."""
    if isinstance(x, dict):
        return {str(k): jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [jsonable(v) for v in x]
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating, float)):
        v = float(x)
        return v if math.isfinite(v) else None
    if isinstance(x, np.ndarray):
        return [jsonable(v) for v in x.tolist()]
    if isinstance(x, (np.bool_,)):
        return bool(x)
    if isinstance(x, (pd.Timestamp,)):
        return str(x)
    if isinstance(x, pd.DataFrame):
        return x.to_dict("records")
    return x


def _strategies_block(X):
    """The chosen strategies: what each says now, and its record on the loaded candles."""
    ids = X.get("chosen")
    if not ids:
        return dict(mode="auto", chosen=[], items=[], note="The analyst's own mix of setups (no strategy chosen).")
    items = []
    for sid in ids:
        s = st.BY_ID[sid]
        rec = (X.get("records") or {}).get(sid) or {}
        lv = (X.get("live") or {}).get(sid) or st.live(X["sctx"], sid, price=X["price"])
        plan = next((p for p in X.get("plans") or [] if p.get("strategy") == sid), None)
        items.append(dict(id=sid, name=s["name"], group=s["group"], fixed_time=s["fixed_time"],
                          signal=lv.get("signal"), age=lv.get("age"), last=lv.get("last"), missed=lv.get("missed"),
                          wait_for=s["wait_for"], best_for=s["best_for"], record=rec.get("record"),
                          score=rec.get("score", 0.0), has_plan=plan is not None,
                          suits_tf=X["tf"] in s["timeframes"], suits_market=_market_key(X) in s["markets"],
                          history=sh.evidence_for(getattr(X.get("brain"), "strat", None), sid, X["asset_class"],
                                                  X["tf"]),
                          head=(plan or {}).get("head")))
    return dict(mode="chosen", chosen=ids, items=items, conflict=X.get("conflict"))


def _market_key(X):
    ac = X.get("asset_class") or ""
    sub = (X["C"].get("sub") or "") if X.get("C") is not None else ""
    if sub.startswith("fx"):
        return "fx"
    if sub in ("gold", "silver", "metal"):
        return "gold"
    if ac in ("crypto", "stock", "index"):
        return ac
    return "fx" if ac == "fx" else ac


def _public_plan(p, f):
    if p is None:
        return None
    keep = ("side", "type", "setup", "family", "entry", "sl", "tps", "tp_labels", "rr", "p_fill", "p_tp", "p_sl",
            "p_open", "breakeven", "ev", "cost_r", "risk_atr", "chase_limit", "score", "grade", "hist_grade",
            "evidence", "checklist", "why", "invalidation", "expiry", "sizing", "zone", "strategy", "strategy_name",
            "signal_age", "liquidity_guard", "sweep_first", "head")
    out = {k: p.get(k) for k in keep}
    out["text"] = (f"{'BUY' if p['dir'] == 1 else 'SELL'} {p['type'].upper()} EP {f.p(p['entry'])} | SL {f.p(p['sl'])} | "
                   + " | ".join(f"TP{i + 1} {f.p(t)}" for i, t in enumerate(p["tps"])))
    return out


def model_skill(brain):
    card = brain.b.get("card", {}) if hasattr(brain, "b") else {}
    if not card:
        return dict(trained=False, summary="untrained: probabilities are random-walk approximations")
    ev = card["race"]["ev_test"].set_index("selection")
    pick = ev.loc["model EV > 0"] if "model EV > 0" in ev.index else None
    geo = ev.loc["geometry-only EV > 0"] if "geometry-only EV > 0" in ev.index else None
    skill = card["race"]["skill_vs_geometry"]
    # an edge must come from the model's reading of the chart, not from stop/target geometry alone
    # (geometry can't beat a fair market; if it seems to, the fill assumptions are too kind)
    proven = (pick is not None and pick["trades"] >= 300 and pick["avg_net_R"] > 0 and (pick["t_stat"] or 0) >= 3
              and skill >= 0.01 and (geo is None or pick["avg_net_R"] > (geo["avg_net_R"] or 0) + 0.02))
    d = card["direction"]
    return dict(trained=True, created=brain.b.get("created"), rows=card["rows"],
                race_skill=card["race"]["skill_vs_geometry"], touch_skill=card["touch"]["skill_vs_distance"],
                direction_auc=d.get("auc_up_vs_down"), direction_skill=d["skill_vs_prior"],
                model_trades=None if pick is None else int(pick["trades"]),
                model_avg_R=None if pick is None else float(pick["avg_net_R"]),
                model_t=None if pick is None else float(pick["t_stat"]), edge_proven=bool(proven),
                geometry_avg_R=None if geo is None else float(geo["avg_net_R"]))


def _verdict(p, X):
    f = X["f"]
    if p is None:
        return dict(action="NO TRADE", grade="—", headline="No valid plan: structure is unclear and no level offers "
                    "a sensible stop and target. Wait for a break of structure.")
    word = "BUY" if p["dir"] == 1 else "SELL"
    j = min(1, len(p["tps"]) - 1)
    gross = p["ev"][j] + p["cost_r"]
    untrained = getattr(X.get("brain"), "untrained", False)
    odds_ok = untrained or (gross >= -0.05 and p["ev"][j] >= -0.15)
    trade = p["grade"] in ("A+", "A", "B") and odds_ok and p["cost_r"] <= 0.30
    why_not = None
    if p["grade"] in ("A+", "A", "B") and not trade:
        why_not = ("costs are too large for this timeframe" if p["cost_r"] > 0.30 else
                   f"the odds don't support it (EV {p['ev'][j]:+.2f}R)")
    if trade:
        act = f"{word} NOW (market)" if p["type"] == "market" else f"{word} {p['type'].upper()}"
    else:
        act = "NO TRADE (wait)"
    j = min(1, len(p["tps"]) - 1)
    if trade:
        head = f"{act} · grade {p['grade']} ({p['score']:.1f}/10) · {p['setup']}"
    elif why_not:
        head = (f"NO TRADE: the best idea is grade {p['grade']} on the chart, but {why_not}. Conditional plan: {word} "
                f"{p['type']} {f.p(p['entry'])} only if conditions improve")
    else:
        head = (f"NO TRADE: the best idea is only grade {p['grade']} ({p['score']:.1f}/10). Conditional plan: {word} "
                f"{p['type']} {f.p(p['entry'])} only if conditions improve")
    return dict(action=act, side=p["side"] if trade else None, grade=p["grade"], score=p["score"], headline=head,
                trade=trade, ep=p["entry"], sl=p["sl"], tp=p["tps"][j], tps=p["tps"])


# ───────────────────────────── the written report ─────────────────────────────
GLOSSARY = {
    "BOS": "Break of structure: a candle closes beyond the last swing high (up) or low (down) in the trend's direction.",
    "CHoCH": "Change of character: the first break against the trend, an early sign it may be turning.",
    "HH/HL": "Higher high / higher low: the stair-step of an uptrend. LH/LL (lower high / lower low) = downtrend.",
    "liquidity": "Clusters of stop orders above swing highs (buy-side) and below swing lows (sell-side). Price "
                 "is often drawn to them.",
    "sweep": "A wick through a swing high/low that closes back inside: stops were taken, then price reversed.",
    "FVG": "Fair value gap: a 3-candle imbalance where candle 1 and candle 3 don't overlap. Price often returns to it.",
    "OB": "Order block: the last opposite-colour candle before a strong move that broke structure.",
    "breaker": "An order block that failed (price closed through it); it often flips to the opposite role.",
    "OTE": "Optimal trade entry: the 62–79% retracement of the last impulse (ICT).",
    "premium/discount": "Above 50% of the current range = premium (expensive), below = discount (cheap).",
    "killzone": "The busiest hours: London 02:00–05:00 and New York 07:00–10:00 (New York time).",
    "ATR": "Average true range: the typical size of one candle. Distances are measured in ATRs.",
    "R:R": "Reward-to-risk: how many times your risk the target pays (2R = twice what you risk).",
    "EV": "Expected value: the average result per trade if this exact situation repeated many times, in R.",
    "break-even": "The win rate you need for a strategy with this R:R and costs to stop losing money.",
    "liquidation": "With leverage, the price where the exchange closes your position for you. It must be beyond your stop.",
    "PDH/PDL": "Previous day high / low (PWH/PWL = previous week, PMH/PML = previous month).",
    "EQH/EQL": "Equal highs / equal lows: obvious double tops/bottoms where stops pile up.",
}


def _lvl_line(c, f, tf, beginner, last_close=None, price=None):
    dist = f"{abs(c['dist_atr']):.1f} ATR {'above' if c['dist_atr'] > 0 else 'below'}"
    s = f"{f.p(c['price'])}: " + " + ".join(c["labels"][:4]) + f" ({dist}"
    if last_close is not None and min(last_close, price) < c["price"] < max(last_close, price):
        s += ", price is trading through it right now"
    if c.get("p_touch") is not None:
        s += f", {pct(c['p_touch'])} chance to be reached in 24 candles"
    s += ")"
    r = c.get("reaction")
    if r:
        s += (f". Historically when reached: breaks {pct(r['p_break'])} / rejects {pct(r['p_reject'])}"
              + (f" (a random level: {pct(r['base_break'])} / {pct(r['base_reject'])})" if r.get("base_break") is not None
                 else ""))
    return s


def report(X, beginner=False):
    f, tf, C, price, atr = X["f"], X["tf"], X["C"], X["price"], X["atr"]
    A, p, alt, v = X["A"], X["primary"], X["alternative"], X["verdict"]
    L = []
    w = L.append
    t_last = X["df"].index[-1]
    w(f"━━━ {C['symbol']} · {tf} · last closed candle {t_last:%Y-%m-%d %H:%M} UTC · price {f.p(price)} ━━━")
    w(f"VERDICT: {v['headline']}")
    if p is not None:
        tps = " · ".join(f"TP{i + 1} {f.p(t)}" for i, t in enumerate(p["tps"]))
        w(f"EP {f.p(p['entry'])} · SL {f.p(p['sl'])} · {tps}")
        rrs = " / ".join(f"{r:.1f}" for r in p["rr"])
        j = min(1, len(p["tps"]) - 1)
        w(f"R:R {rrs} · P(TP{j + 1} before SL) {pct(p['p_tp'][j])} vs {pct(p['breakeven'][j])} needed to break even"
          + ("" if p["type"] == "market" else f" · fill chance {pct(p['p_fill'])} within {p['expiry']} candles"))
        sz = p["sizing"]
        lev = (f"leverage {sz['leverage']}x isolated, margin {money(sz['margin'])}, liquidation {f.p(sz['liquidation'])}"
               if sz.get("mode") == "isolated" else f"effective leverage {sz['leverage_needed']:.1f}x")
        w(f"Size: {_size_txt(sz)} risking {money(sz['risk_amount'])} ({sz['risk_pct']:g}%) · {lev}")
    w("")
    # STEP 1
    w("STEP 1 — BIG PICTURE (higher timeframes)")
    if not X["htf"]:
        w("• No higher-timeframe data: reading this chart on its own.")
    for h in X["ladder"]:
        hs = X["htf"].get(h)
        if hs is None:
            continue
        ev = hs["last_event"]
        evs = "" if ev is None else (f"; last {ev['kind']} {'up' if ev['direction'] == 'bullish' else 'down'} through "
                                     f"{f.p(ev['level'])} {ago(ev['bars_ago'], h)}")
        zone = "" if hs["zone"] is None else f"; price in {hs['zone']} ({pct(hs['range_pos'])} of its range)"
        probs = "" if "p_up" not in hs else (f"; next 24 candles: ↑ {pct(hs['p_up'])} · ↓ {pct(hs['p_down'])} · "
                                             f"↔ {pct(hs['p_flat'])}")
        w(f"• {h}: {hs['trend_word'].upper()} ({hs['phase'].lower()}){evs}{zone}{probs}")
    w("• " + _top_down(X))
    w("")
    # STEP 2
    w(f"STEP 2 — STRUCTURE ON {tf}")
    tw = {1: "BULLISH (higher highs & higher lows)", -1: "BEARISH (lower highs & lower lows)",
          0: "UNCLEAR (no break of structure yet)"}[X["trend"]]
    w(f"• Trend: {tw}. Recent swings: {' → '.join(s['label'] + ' ' + f.p(s['price']) for s in A['swings'][-5:])}")
    if A["events"]:
        e = A["events"][-1]
        w(f"• Last break: {e['kind']} {'up' if e['direction'] == 'bullish' else 'down'} through {f.p(e['level'])} "
          f"{ago(len(X['df']) - 1 - e['idx'], tf)}"
          + (" (a CHoCH is an early reversal signal: it needs follow-through)" if e["kind"] == "CHoCH" else ""))
    ph = X["phase"]
    w(f"• Phase: {ph['label']} (ADX {ph['adx']:.0f}, efficiency {ph['er']:.2f})" +
      ("" if not ph["notes"] else "; " + "; ".join(ph["notes"])))
    dr = X["dr"]
    if dr is not None:
        w(f"• Dealing range {f.p(dr['bottom'])} – {f.p(dr['top'])}: price at {pct(dr['pos'])} → {dr['zone'].upper()}. "
          f"Equilibrium {f.p(dr['eq'])}. OTE for longs {f.p(dr['ote_long'][0])}–{f.p(dr['ote_long'][1])}, "
          f"for shorts {f.p(dr['ote_short'][0])}–{f.p(dr['ote_short'][1])}.")
    leg = X["leg"]
    if leg is not None:
        r = leg["retracements"]
        w(f"• Impulse leg {f.p(leg['origin'])} → {f.p(leg['extreme'])} ({leg['length'] / atr:.1f} ATR): 50% {f.p(r['0.5'])}"
          f", 61.8% {f.p(r['0.618'])}, 70.5% {f.p(r['0.705'])}, 78.6% {f.p(r['0.786'])}; extensions 1.272 "
          f"{f.p(leg['extensions']['1.272'])}, 1.618 {f.p(leg['extensions']['1.618'])}")
    if beginner:
        w("  What this means: trends move in steps. In an uptrend each pullback should hold above the last low; "
          "a close below it (CHoCH) is the first warning that buyers are losing control.")
    w("")
    # STEP 3
    w("STEP 3 — KEY LEVELS & LIQUIDITY")
    at = [c for c in X["clusters"] if abs(c["dist_atr"]) < 0.15]
    ab = sorted([c for c in X["clusters"] if c["dist_atr"] >= 0.15], key=lambda c: c["price"])[:5]
    be = sorted([c for c in X["clusters"] if c["dist_atr"] <= -0.15], key=lambda c: -c["price"])[:5]
    if at:
        w("At price right now: " + "; ".join(f"{f.p(c['price'])} " + " + ".join(c["labels"][:3]) for c in at))
    w("Above (resistance / buy-side liquidity):")
    for c in ab:
        w("  • " + _lvl_line(c, f, tf, beginner, float(X["df"]["close"].iloc[-1]), price))
    if not ab:
        w("  • nothing marked nearby: price is at the top of everything in view (price discovery)")
    w("Below (support / sell-side liquidity):")
    for c in be:
        w("  • " + _lvl_line(c, f, tf, beginner, float(X["df"]["close"].iloc[-1]), price))
    if not be:
        w("  • nothing marked nearby")
    sw = A["sweeps"][-1] if A["sweeps"] else None
    if sw is not None and len(X["df"]) - 1 - sw["idx"] <= 50:
        w(f"• Last liquidity sweep: {sw['side']} at {f.p(sw['level'])} {ago(len(X['df']) - 1 - sw['idx'], tf)} "
          f"(wick to {f.p(sw['wick'])}, closed back inside) → "
          + ("bullish hint (sell stops taken)" if sw["side"] == "sell-side" else "bearish hint (buy stops taken)"))
    obs = [o for o in X["obs"] if o["status"] != "breaker"][:3]
    if obs:
        w("• Order blocks: " + "; ".join(f"{o['direction']} {f.p(o['bottom'])}–{f.p(o['top'])} ({o['status']})" for o in obs))
    brk = [o for o in X["obs"] if o["status"] == "breaker"][:2]
    if brk:
        w("• Breakers (failed blocks, role flipped): " + "; ".join(
            f"{f.p(o['bottom'])}–{f.p(o['top'])} now {o['role']}" for o in brk))
    gaps = sorted([g for g in A["fvgs"] if g["filled_idx"] is None],
                  key=lambda g: abs((g["top"] + g["bottom"]) / 2 - price))[:3]
    if gaps:
        w("• Open FVGs: " + "; ".join(f"{g['direction']} {f.p(g['bottom'])}–{f.p(g['top'])}" for g in gaps))
    if X["user"]:
        w("• Your drawings:")
        for u in X["user"]:
            if "level" in u:
                w(f"  • {u.get('label') or u.get('type')}: {f.p(u['level'])} ({u['role']}, {abs(u['distance_atr']):.1f} ATR "
                  f"away, {pct(u['p_touch_24'])} to be reached in 24 candles). "
                  + (f"Tested {u['tests']}× in the loaded history: held {u['held']}, broke {u['broke']}"
                     if u.get("tests") else f"Swing points there: {u['past_turns']}")
                  + (f"; lines up with {', '.join(u['confluence'][:3])}" if u["confluence"] else
                     "; no other level confirms it") +
                  (f"; trendline: {u['status']}, {u['touches']} touches" if u.get("type") == "trendline" else ""))
            elif u.get("type") == "note":
                w(f"  • note: {u.get('text', '')}")
    w("")
    # STEP 4
    w("STEP 4 — PRICE ACTION, MOMENTUM & PATTERNS")
    recent = [q for q in A["patterns"] if q["idx"] >= len(X["df"]) - 3 and q["bias"] != "neutral"]
    if recent:
        w("• Last candles: " + ", ".join(f"{q['name']} ({ago(len(X['df']) - 1 - q['idx'], tf)})" for q in recent[-3:]))
    else:
        w("• Last candles: no strong reversal/continuation pattern on the last 3 candles")
    rsi = X["rsi"]
    rs_txt = "overbought (>70)" if rsi > 70 else "oversold (<30)" if rsi < 30 else (
        "bullish side (>50)" if rsi >= 50 else "bearish side (<50)")
    w(f"• RSI(14) {rsi:.0f}: {rs_txt}" + ("" if not X["divs"] else "; " + "; ".join(
        f"{dv['name']} ({ago(dv['bars_ago'], tf)})" for dv in X["divs"])))
    e20, e50, e200 = (float(X["emas"][k].iloc[-1]) for k in (20, 50, 200))
    stack = ("bullish stack (price > EMA20 > EMA50 > EMA200)" if price > e20 > e50 > e200 else
             "bearish stack (price < EMA20 < EMA50 < EMA200)" if price < e20 < e50 < e200 else "mixed (no clean stack)")
    w(f"• EMAs 20/50/200 {f.p(e20)} / {f.p(e50)} / {f.p(e200)}: {stack}")
    for q in X["patterns"]:
        if q["kind"] == "lines":
            w(f"• {q['name']} ({q['status']}): breakout ≥ {f.p(q['breakout_up'])} → {f.p(q['target_up'])}; "
              f"breakdown ≤ {f.p(q['breakout_down'])} → {f.p(q['target_down'])}")
        else:
            tgt = q.get("target_up", q.get("target_down"))
            w(f"• {q['name']} ({q['status']}): trigger {f.p(q['trigger'])}, measured target {f.p(tgt)}, "
              f"invalid beyond {f.p(q['invalidation'])}")
    for L_ in X["lines"]:
        w(f"• {L_['name']}: {L_['touches']} touches, now at {f.p(L_['p2'])} ({L_['status']})"
          + (" with a parallel channel line" if "channel_offset" in L_ else ""))
    if X["vp"] is not None:
        w(f"• Volume profile (last {min(300, len(X['df']))} candles): POC {f.p(X['vp']['poc'])}, value area "
          f"{f.p(X['vp']['val'])}–{f.p(X['vp']['vah'])}" + (" (price inside value: rotation more likely)"
                                                            if X['vp']['val'] <= price <= X['vp']['vah'] else
                                                            " (price outside value: trend/acceptance test)"))
    w("")
    # STEP 5
    w("STEP 5 — VOLATILITY & TIMING")
    pr = X["prob"]
    w(f"• ATR(14) {f.p(atr)} ({pct(atr / price, 2)} of price); volatility {X['phase']['atr_ratio']:.1f}× its usual level")
    w(f"• Expected range, next 24 candles: ≈70% {f.p(pr['range70'][0])} – {f.p(pr['range70'][1])}; typical (50%) "
      f"{f.p(pr['range50'][0])} – {f.p(pr['range50'][1])}")
    s = X["session"]
    if s:
        act = ", ".join(s["active"]) if s.get("active") else "no killzone active"
        w(f"• New York time {s['ny_time']} ({s['ny_weekday']}): {act}" + (
            f"; silver bullet window: {', '.join(s['silver_bullet'])}" if s.get("silver_bullet") else "") +
          (f"; next: {s['next_killzone']} in {s['minutes_to_next'] // 60}h {s['minutes_to_next'] % 60:02d}m"
           if s.get("next_killzone") else ""))
        if s.get("asia"):
            a_ = s["asia"]
            w(f"• Asian range {f.p(a_['low'])} – {f.p(a_['high'])}: high {'swept' if a_['high_swept'] else 'intact'}, "
              f"low {'swept' if a_['low_swept'] else 'intact'}")
        if C["asset_class"] == "crypto":
            w("• Crypto trades 24/7, but volume still peaks in the London/New York overlap; weekends are thinner")
    if C["age_minutes"] is not None and C["age_minutes"] > 3 * md.TF_MIN[tf] + 60 * 24:
        w(f"• Data note: the last closed candle is {C['age_minutes'] / 60:.0f} h old (market closed or data delayed)")
    w("")
    # STEP 6
    w("STEP 6 — PROBABILITIES (trained on many markets; out-of-sample calibrated)")
    w(f"• Next 24 candles, first move of ±{pr['k_atr']:.2f} ATR: ↑ to {f.p(pr['up_level'])} {pct(pr['p_up'])} · "
      f"↓ to {f.p(pr['down_level'])} {pct(pr['p_down'])} · neither (range) {pct(pr['p_flat'])}")
    sk = X["skill"]
    if sk.get("trained"):
        dauc = sk.get("direction_auc")
        dtxt = (f"direction skill AUC {dauc:.3f} (0.50 = coin flip, 0.55+ = useful)" if dauc is not None and
                np.isfinite(dauc) else f"direction skill {pct(sk['direction_skill'], 1)} better than base rates")
        w(f"• How much to trust them: {dtxt}; level/range odds beat distance-only by {pct(sk['touch_skill'], 1)}; "
          f"trade odds beat stop/target geometry alone by {pct(sk['race_skill'], 1)}.")
        if sk.get("model_trades") is not None:
            w(f"• Out-of-sample test of the model's own picks: {sk['model_trades']:,} trades, average "
              f"{sk['model_avg_R']:+.3f}R after costs (t = {sk['model_t']:.1f}) → "
              + ("EDGE SHOWN (still paper-trade it first)" if sk["edge_proven"] else
                 "NO PROVEN EDGE: treat the odds as honest estimates, not a money machine"
                 + (" (the picks look profitable, but no better than stop/target geometry alone, so that's a "
                    "backtest effect, not skill)" if (sk["model_avg_R"] or 0) > 0 and (sk["model_t"] or 0) >= 3 else "")))
    else:
        w("• Model not trained: odds below are random-walk approximations")
    w("")
    # STEP 7
    if p is not None:
        w("STEP 7 — TRADE PLAN" + ("" if v["trade"] else " (CONDITIONAL: only if conditions improve)"))
        w(f"• Setup: {p['setup']}. " + " ".join(x[0].upper() + x[1:] + ("" if x.endswith(".") else ".") for x in p["why"]))
        zone = p.get("zone")
        ztxt = "" if not zone else f" (zone {f.p(zone[0])}–{f.p(zone[1])})"
        w(f"• Entry: {'BUY' if p['dir'] == 1 else 'SELL'} {p['type'].upper()} {f.p(p['entry'])}{ztxt}"
          + (f". Don't chase beyond {f.p(p['chase_limit'])}: R:R to TP2 falls below 1.5 there" if p["type"] != "stop"
             else ""))
        w(f"• Stop loss: {f.p(p['sl'])} ({p['risk_atr']:.1f} ATR). The idea is wrong on {p['invalidation']}.")
        for i, (t, lab) in enumerate(zip(p["tps"], p["tp_labels"])):
            mgmt = (" → take 50% and move the stop to break-even" if i == 0 and len(p["tps"]) > 1 else
                    " → runner" if i == 2 else "")
            w(f"• TP{i + 1}: {f.p(t)} ({lab}; {p['rr'][i]:.1f}R): P(hit before SL) {pct(p['p_tp'][i])} vs "
              f"{pct(p['breakeven'][i])} break-even, EV {p['ev'][i]:+.2f}R{mgmt}")
        if p["type"] != "market":
            w(f"• Order valid {p['expiry']} candles (~{_dur(p['expiry'], tf)}); fill chance {pct(p['p_fill'])}. Cancel it "
              f"if price reaches TP1 first.")
        if p["cost_r"] > 0.15:
            w(f"• ⚠️ Costs eat {pct(p['cost_r'])} of the risk on this timeframe: use a higher timeframe or a bigger stop.")
        ev = p.get("evidence")
        if ev:
            w(f"• History of this setup ({ev['scope']}): {ev['trades']:,} trades, won {pct(ev['win_rate'])}, average "
              f"{ev['avg_R']:+.2f}R after costs" + (f"; grade-{ev['grade']} ones: {ev['grade_trades']:,} trades, "
                                                    f"{ev['grade_R']:+.2f}R" if "grade_trades" in ev else ""))
        w(f"• Checklist (grade {p['grade']}, {p['score']:.1f}/10):")
        for c in p["checklist"]:
            w(f"   {'✓' if c['ok'] else '✗'} {c['text']}")
        w("")
        w("STEP 8 — POSITION SIZE & LEVERAGE")
        for line in _sizing_lines(p, f):
            w("• " + line)
        w("")
    w("STEP 9 — ALTERNATIVE SCENARIO" if alt is not None else "STEP 9 — WHAT WOULD CHANGE THE PICTURE")
    if alt is not None:
        tps = " · ".join(f"TP{i + 1} {f.p(t)}" for i, t in enumerate(alt["tps"]))
        w(f"• {'BUY' if alt['dir'] == 1 else 'SELL'} {alt['type'].upper()} EP {f.p(alt['entry'])} · SL {f.p(alt['sl'])} · "
          f"{tps} (grade {alt['grade']}, {alt['setup'].lower()}). Valid only on {_trigger_text(alt, X)}.")
    liq = A["liquidity"]
    if liq["buy_side"]:
        w(f"• A close above {f.p(liq['buy_side'][0]['price'])} = "
          f"{'BOS up (trend continues)' if X['trend'] == 1 else 'CHoCH up (bullish shift)'}")
    if liq["sell_side"]:
        w(f"• A close below {f.p(liq['sell_side'][0]['price'])} = "
          f"{'BOS down (trend continues)' if X['trend'] == -1 else 'CHoCH down (bearish shift)'}")
    w("")
    w("GURU'S TAKE")
    w(_guru_take(X))
    w("")
    w("RISK RULES")
    for r_ in _advice(X):
        w("• " + r_)
    if beginner:
        import re
        body = "\n".join(L)

        def present(term):
            first = term.split("/")[0]
            if first.isupper() or first in ("CHoCH", "R:R"):
                return re.search(r"(?<![A-Za-z])" + re.escape(first) + r"(?![A-Za-z])", body) is not None
            return re.search(r"\b" + re.escape(first.lower()), body.lower()) is not None

        used = [k for k in GLOSSARY if present(k)]
        if used:
            w("")
            w("WORDS USED")
            for k in used:
                w(f"• {k}: {GLOSSARY[k]}")
    if X["notes"]:
        w("")
        w("Notes: " + "; ".join(X["notes"]))
    w("")
    w("Research tool, not financial advice. Paper-trade first.")
    return "\n".join(L)


def _dur(bars, tf):
    m = bars * md.TF_MIN[tf]
    return f"{m} min" if m < 120 else (f"{m / 60:.0f} h" if m < 2880 else f"{m / 1440:.0f} days")


def _size_txt(sz):
    if sz.get("too_small"):
        return (f"below the broker minimum size (you'd need ≥ {money(sz['min_capital'])} at {sz['risk_pct']:g}% risk; "
                f"exact size {qty(sz['units'])} {sz['unit']})")
    if "lots" in sz:
        u = sz["lots_rounded"] * sz["lot_size"]
        return (f"{sz['lots_rounded']:.2f} lots ({qty(u)} {sz['unit']}, notional "
                f"{money(sz['notional'] * u / sz['units'] if sz['units'] else 0)})")
    return f"{qty(sz['units'])} {sz['unit']} (notional {money(sz['notional'])})"


def _sizing_lines(p, f):
    sz = p["sizing"]
    out = []
    if sz.get("warn"):
        out.append("⚠️ " + sz["warn"])
    out.append(f"Account {money(sz['account'])}, risk {sz['risk_pct']:g}% = {money(sz['risk_amount'])} → "
               f"{_size_txt(sz)}; round-trip costs ≈ {money(sz['fees'])}")
    if sz.get("too_small"):
        out.append(f"⚠️ That's below the minimum order size: you'd need at least {money(sz['min_capital'])} in the "
                   f"account to take this trade at {sz['risk_pct']:g}% risk (or accept more risk, which I don't advise).")
    elif "lots" in sz:
        out.append(f"Rounded to {sz['lots_rounded']:.2f} lots, the risk is {money(sz['risk_at_rounded'])}. Minimum "
                   f"capital for this trade at {sz['risk_pct']:g}% risk: {money(sz['min_capital'])}.")
    if sz.get("mode") == "isolated":
        out.append(f"Leverage: {sz['leverage']}x isolated → margin {money(sz['margin'])} ({pct(sz['margin_pct'])} of the "
                   f"account), liquidation ≈ {f.p(sz['liquidation'])}: "
                   + ("your stop is hit first ✓" if sz["liq_beyond_stop"] else "⚠️ liquidation comes before the stop"))
        out.append(f"Max safe leverage {sz['max_safe_leverage']}x (liquidation stays beyond the stop with a buffer). "
                   f"More leverage doesn't raise the risk if the stop holds, it only locks up less margin. But "
                   f"it pushes liquidation closer, so never trade without the stop.")
        ul = sz.get("user_leverage")
        if ul:
            out.append(f"Your {ul['leverage']:g}x: margin {money(ul['margin'])}, liquidation {f.p(ul['liquidation'])} "
                       + ("(beyond the stop ✓)" if ul["safe"] else "⚠️ BEFORE your stop: lower the leverage"))
    else:
        m = sz["margin_at"]
        need = max(1, math.ceil(sz["leverage_needed"] * 2))
        nice = next((L for L in (10, 20, 30, 50, 100, 200, 500, 1000) if L >= need), need)
        out.append(f"This position is {sz['leverage_needed']:.1f}x your account. Margin needed: "
                   + ", ".join(f"{k} → {money(v)}" for k, v in m.items())
                   + f". Use an account leverage of at least 1:{nice} so this trade ties up ≤ 50% of your margin. "
                     "Leverage only sets the margin; your risk is fixed by the stop and the size.")
    return out


def _trigger_text(p, X):
    f = X["f"]
    if p["type"] == "stop":
        return f"a move through {f.p(p['entry'])}"
    if p["type"] == "limit":
        return f"a pullback to {f.p(p['entry'])} with the stop at {f.p(p['sl'])} still intact"
    return "current conditions"


def _top_down(X):
    ts = [X["htf"][h]["trend"] for h in X["ladder"] if h in X["htf"]]
    t = X["trend"]
    if ts and all(x == 1 for x in ts):
        base = "Higher timeframes all bullish → bias LONG; look for buys on pullbacks into discount."
    elif ts and all(x == -1 for x in ts):
        base = "Higher timeframes all bearish → bias SHORT; look for sells on rallies into premium."
    elif ts and ts[0] != 0 and (len(ts) < 2 or ts[0] == ts[1]):
        base = f"Higher-timeframe bias {'LONG' if ts[0] == 1 else 'SHORT'} ({X['ladder'][0]} and {X['ladder'][1] if len(ts) > 1 else ''} agree)."
    else:
        base = "Higher timeframes disagree → no strong bias; trade only A-grade setups or wait for alignment."
    if ts and t != 0 and ts[0] != 0 and t != ts[0]:
        base += f" {X['tf']} is moving against {X['ladder'][0]}: that's a pullback until it breaks structure."
    return base


def _guru_take(X):
    f, p, v, price = X["f"], X["primary"], X["verdict"], X["price"]
    dr, ts = X["dr"], [X["htf"][h]["trend"] for h in X["ladder"] if h in X["htf"]]
    bias = ts[0] if ts else X["trend"]
    who = {1: "Buyers are in control on the higher timeframe", -1: "Sellers are in control on the higher timeframe",
           0: "The higher timeframe is undecided"}[bias]
    parts = [who + "."]
    if dr is not None:
        parts.append(f"On the {X['tf']} price sits in {dr['zone']} ({pct(dr['pos'])} of the range), "
                     + ("so buying here means paying up." if dr["zone"] == "premium" and bias == 1 else
                        "so selling here means selling low." if dr["zone"] == "discount" and bias == -1 else
                        "a good place to look for entries in the trend's direction." if
                        (dr["zone"] == "discount" and bias == 1) or (dr["zone"] == "premium" and bias == -1) else
                        "nothing special about the location."))
    if p is not None:
        j = min(1, len(p["tps"]) - 1)
        if v["trade"]:
            act = ("I'd take it now" if p["type"] == "market" else
                   f"I'd let price come to me at {f.p(p['entry'])}" if p["type"] == "limit" else
                   f"I'd only join on a break through {f.p(p['entry'])}")
            parts.append(f"The best idea is a {p['side']}: {p['setup'].lower()}. {act}, stop {f.p(p['sl'])}, first target "
                         f"{f.p(p['tps'][0])}. If {p['invalidation']} happens, I'm wrong and I step aside.")
        else:
            parts.append(f"Nothing here is good enough. The best idea ({p['side']}, {p['setup'].lower()}) only scores "
                         f"{p['grade']}. Patience is a position: wait for the trigger in step 9.")
        edge = p["ev"][j]
        sk = X["skill"]
        parts.append(f"Edge check: {pct(p['p_tp'][j])} to reach TP{j + 1} vs {pct(p['breakeven'][j])} needed "
                     + ("→ positive on paper" if edge > 0 else "→ negative on paper")
                     + ("; the model's picks haven't proven an edge out-of-sample, so size small"
                        if sk.get("trained") and not sk.get("edge_proven") else "") + ".")
    return " ".join(parts)


def _advice(X):
    out = ["Risk 0.5–1% per trade. After 3 losses in a row, stop for the day.",
           "Put the stop in the order itself and never move it further away. Move it to break-even after TP1.",
           "Check the economic calendar: skip entries in the 15 minutes around high-impact news (CPI, NFP, FOMC)."]
    p = X["primary"]
    if p is not None:
        if p["cost_r"] > 0.15:
            out.insert(0, f"On {X['tf']} the costs are {pct(p['cost_r'])} of your risk. Lower timeframes look exciting "
                          "but fees decide the result: prefer 15m and above.")
        if any(X["htf"][h]["trend"] == -p["dir"] for h in X["ladder"][:1] if h in X["htf"]):
            out.insert(0, "This trade is against the higher-timeframe trend: half size at most.")
    if X["phase"]["atr_ratio"] >= 1.6:
        out.insert(0, "Volatility is unusually high: smaller size, wider stops, expect slippage.")
    if X["C"]["asset_class"] == "crypto":
        out.append("Crypto with leverage: use isolated margin, keep liquidation far beyond the stop, and keep "
                   "most of your capital off the exchange.")
    return out


# ───────────────────────────── drawings for the chart / web app ─────────────────────────────
CANDLE_WHY = {
    "Hammer": "A small body near the top with a long lower wick after a fall: sellers pushed down and buyers pushed back.",
    "Inverted Hammer": "A long upper wick after a fall: buyers tried to lift price; a bullish candle next confirms it.",
    "Bullish Engulfing": "A green body that covers the whole previous red body: buyers took over.",
    "Bearish Engulfing": "A red body that covers the whole previous green body: sellers took over.",
    "Bullish Harami": "A small green body inside the previous red one: the selling is pausing.",
    "Bearish Harami": "A small red body inside the previous green one: the buying is pausing.",
    "Piercing Line": "A green candle that closes above the middle of the previous red one: buyers fighting back.",
    "Dark Cloud Cover": "A red candle that closes below the middle of the previous green one: sellers fighting back.",
    "Morning Star": "Red, small, then a strong green candle: a three-candle bottoming pattern.",
    "Evening Star": "Green, small, then a strong red candle: a three-candle topping pattern.",
    "Three White Soldiers": "Three strong green candles in a row: steady buying.",
    "Three Black Crows": "Three strong red candles in a row: steady selling.",
    "Bullish Marubozu": "A green candle with almost no wicks: buyers in control the whole candle.",
    "Bearish Marubozu": "A red candle with almost no wicks: sellers in control the whole candle.",
    "Tweezer Bottom": "Two candles with the same low: that price was defended twice.",
    "Tweezer Top": "Two candles with the same high: that price was rejected twice.",
    "Hanging Man": "A hammer shape after a rise: a warning that sellers are stepping in.",
    "Shooting Star": "A long upper wick after a rise: buyers were rejected at the high.",
    "Doji": "Open and close almost equal: indecision.",
    "Inside Bar": "A candle inside the previous one's range: a pause before the next move.",
}


def _why_swing(s, sw, k, f):
    lab = s["label"]
    prev = [q for q in sw[:k] if q["type"] == s["type"]]
    before = f" than the previous swing {s['type']} at {f.p(prev[-1]['price'])}" if prev else ""
    meaning = {"HH": "Higher high: buyers pushed past the last high. Higher highs with higher lows make an uptrend.",
               "HL": "Higher low: sellers couldn't reach the last low. In an uptrend, higher lows are where buyers step in.",
               "LH": "Lower high: buyers couldn't reach the last high. Lower highs with lower lows make a downtrend.",
               "LL": "Lower low: sellers pushed past the last low.",
               "H": "The first swing high on the chart.", "L": "The first swing low on the chart."}.get(lab, "")
    word = {"HH": "higher", "HL": "higher", "LH": "lower", "LL": "lower"}.get(lab)
    return (f"A swing {s['type']} at {f.p(s['price'])}" + (f", {word}{before}" if word else "") + ". " + meaning)


def _why_event(e, d_, f, tf):
    up = e["direction"] == "bullish"
    when = d_.index[e["idx"]]
    base = (f"On the {tf} candle of {when:%d %b %H:%M} UTC, price closed {'above' if up else 'below'} the swing "
            f"{'high' if up else 'low'} at {f.p(e['level'])}.")
    if e["kind"] == "CHoCH":
        return (base + f" That was the first break against the {'down' if up else 'up'}trend: a change of character "
                       f"(CHoCH), an early sign the trend may be turning {'up' if up else 'down'}. It needs "
                       f"follow-through (a higher low and a BOS) to confirm.")
    return (base + f" It broke structure in the trend's direction (BOS): the {'up' if up else 'down'}trend continues "
                   f"while price stays {'above' if up else 'below'} the last {'higher low' if up else 'lower high'}.")


def _why_pattern(q, f):
    name = q["name"]
    low = name.lower()
    pts = ", ".join(f.p(x) for x in q.get("prices", [])[:5])
    if "double top" in low:
        core = "Two highs at about the same price with a dip between: buyers failed twice at the same place."
    elif "double bottom" in low:
        core = "Two lows at about the same price with a bounce between: sellers failed twice at the same place."
    elif "head" in low and "inverse" in low:
        core = "A low, a lower low (the head), then a higher low: selling pressure is weakening."
    elif "head" in low:
        core = "A high, a higher high (the head), then a lower high: buying pressure is weakening."
    elif "triangle" in low or "wedge" in low:
        core = "Price is squeezing between two converging lines; the breakout direction usually sets the next move."
    else:
        core = f"A {q.get('bias_word', '')} chart pattern."
    tgt = q.get("target_up") if q.get("bias", 0) == 1 else q.get("target_down")
    return (f"{core} Points: {pts}. It confirms on a close beyond {f.p(q['trigger'])}"
            + (f"; the measured move points to {f.p(tgt)}" if tgt else "") + ".")


def _why_target(label, p_tp):
    odds = "" if p_tp is None else f" Odds of reaching it before the stop: about {round(p_tp * 100)}%."
    if "no level closer" in label:
        return f"No level sits closer, so this target is a fixed {label.split(' ')[0]} of the risk.{odds}"
    if label.endswith(" target"):
        return f"The strategy's own target ({label[:-7]}).{odds}"
    return (f"Set just before {label}: a level the other side defends or where liquidity rests, placed a touch early "
            f"so it fills when the level is tagged.{odds}")


def _segments(xs, vals):
    """Split a series with gaps (NaN) into continuous runs of [x, y] points."""
    out, cur = [], []
    for x, v in zip(xs, vals):
        if v is None or not np.isfinite(v):
            if len(cur) > 1:
                out.append(cur)
            cur = []
        else:
            cur.append([int(x), float(v)])
    if len(cur) > 1:
        out.append(cur)
    return out


def _strategy_overlay(X, add, tx, s0, n):
    """For the chosen strategies: their indicator lines and their past trades on the visible candles."""
    ids = X.get("chosen")
    ctx = X.get("sctx")
    if not ids or ctx is None:
        return
    for key, (label, color, vals) in st.indicator_lines(ctx, ids, s0).items():
        xs = list(range(len(vals)))
        for seg in _segments(xs, vals):
            add("indicators", "path", 0, 2, points=seg, color=color, width=1.2, text=None, title=label,
                why=f"{label}: used by your strategy " + ", ".join(st.BY_ID[i]["name"] for i in ids
                                                                    if any(k in st.BY_ID[i]["indicators"] for k in
                                                                           (key, key.split("_")[0], key[:2])))
                + ".")
    for sid in ids[:3]:
        rec = (X.get("records") or {}).get(sid)
        if not rec:
            continue
        for m in st.markers(ctx, sid, rec["trades"][-40:], show_from=s0):
            if m["kind"] == "marker":
                add("strategy", "marker", 16, 2, x1=tx(m["i"]), y1=m["price"], symbol=m["symbol"], color=m["color"],
                    text=m["text"], title=m["text"], why=m["why"])
            else:
                add("strategy", "line", 16, 2, x1=tx(m["i1"]), x2=tx(m["i2"]), y1=m["p1"], y2=m["p2"], color=m["color"],
                    dash="dot", width=1.1, text=m["text"], label_at="end", title=m["text"], why=m["why"])


def chart_data(X, show_bars=SHOW_BARS):
    """Candles + drawing instructions (JSON-friendly) for the chart / web app.
    x = candle index in `candles` (the forming candle, if any, is the last one), t = epoch seconds.
    order = the sequence to animate drawings in; prio = 1 essential, 2 normal, 3 detail (lets a viewer pick how busy
    the chart is); extend = "right" for levels that run to the right edge."""
    d_, A, f, price, atr = X["df"], X["A"], X["f"], X["price"], X["atr"]
    n = len(d_)
    s0 = max(0, n - show_bars)
    view = d_.iloc[s0:]
    ts = md.epoch_s(view.index)
    candles = [[int(t), float(o), float(h), float(l), float(c), float(v)] for t, o, h, l, c, v in
               zip(ts, view["open"], view["high"], view["low"], view["close"], view["volume"])]
    forming = X["C"]["forming"]
    if forming is not None:
        candles.append([int(md.epoch_s(pd.DatetimeIndex([forming.name]))[0]), float(forming["open"]),
                        float(forming["high"]), float(forming["low"]), float(forming["close"]),
                        float(forming["volume"])])
    last_x = len(candles) - 1
    right = last_x + 18
    step = int(md.TF_MIN[X["tf"]] * 60)
    tx = lambda i: max(0, int(i) - s0)
    tt = lambda x: candles[min(max(0, x), last_x)][0] if x <= last_x else candles[last_x][0] + (x - last_x) * step
    rec = view.iloc[-120:]
    lo_v, hi_v = float(rec["low"].min()), float(rec["high"].max())
    pad = max((hi_v - lo_v) * 0.35, 2 * atr)
    vis = lambda a, b=None: (b if b is not None else a) >= lo_v - pad and a <= hi_v + pad
    D = []

    def add(layer, kind, order, prio=2, **kw):
        kw.update(layer=layer, kind=kind, order=order, prio=prio, id=f"{layer}-{len(D)}")
        for k in ("x1", "x2"):
            if k in kw and kw[k] is not None:
                kw[k.replace("x", "t")] = tt(kw[k])
        D.append(kw)

    sw = [s for s in A["swings"] if s["idx"] >= s0]
    if len(sw) >= 2:
        add("structure", "path", 1, 1, points=[[tx(s["idx"]), float(s["price"])] for s in sw], color=COL["zig"], width=1,
            title="Swing path", why="The zigzag joins the confirmed swing highs and lows (a swing high has 3 lower highs on "
                                   "each side). Its shape is the market structure: higher highs and higher lows are an "
                                   "uptrend, lower highs and lower lows a downtrend.")
    for k, s in enumerate(sw):
        col = COL["up"] if s["label"] in ("HH", "HL") else COL["down"] if s["label"] in ("LH", "LL") else COL["text"]
        add("structure", "label", 2, 1 if k >= len(sw) - 8 else 3, x1=tx(s["idx"]), y1=float(s["price"]),
            text=s["label"], color=col, anchor="above" if s["type"] == "high" else "below",
            title=f"{s['label']} at {f.p(s['price'])}", why=_why_swing(s, sw, k, f))
    evs = [e for e in A["events"] if e["idx"] >= s0]
    for k, e in enumerate(evs):
        col = COL["choch"] if e["kind"] == "CHoCH" else (COL["up"] if e["direction"] == "bullish" else COL["down"])
        add("structure", "line", 3, 1 if k >= len(evs) - 4 else 3, x1=tx(e["from_idx"]), x2=tx(e["idx"]),
            y1=e["level"], y2=e["level"], color=col, dash="dash" if e["kind"] == "BOS" else "dot", width=1.3,
            text=e["kind"], label_at="middle", anchor="above" if e["direction"] == "bullish" else "below",
            title=f"{e['kind']} {'up' if e['direction'] == 'bullish' else 'down'} at {f.p(e['level'])}",
            why=_why_event(e, d_, f, X["tf"]))
    for side in ("support", "resistance"):
        zs = sorted([z for z in A["zones"] if z["role"] in (side, "at zone") and vis(z["bottom"], z["top"])],
                    key=lambda z: abs((z["top"] + z["bottom"]) / 2 - price))
        for k, z in enumerate(zs[:3]):
            fill = COL["sup"] if z["role"] == "support" else COL["res"] if z["role"] == "resistance" else COL["mid"]
            add("zones", "box", 4, 2 if k < 2 else 3, x1=tx(z["first_idx"]), x2=right, y1=z["bottom"], y2=z["top"],
                fill=fill, text=f"{z['role'].capitalize()} ×{z['touches']}", extend="right",
                title=f"{z['role'].capitalize()} zone {f.p(z['bottom'])}–{f.p(z['top'])}",
                why=f"Price turned here {z['touches']} times (swing highs and lows clustered within 0.6 ATR). The more "
                    f"often a level held, the more traders watch it; it is {'support' if z['role'] == 'support' else 'resistance' if z['role'] == 'resistance' else 'where price is now'}. "
                    f"A close through it with momentum usually flips its role.")
    for k, ob in enumerate([o for o in X["obs"] if vis(o["bottom"], o["top"])]):
        bull = ob["role"] == "demand"
        add("orderblocks", "box", 5, 3 if ob["status"] == "breaker" or k >= 3 else 2, x1=tx(ob["idx"]), x2=right,
            y1=ob["bottom"], y2=ob["top"], fill=COL["ob_bull"] if bull else COL["ob_bear"], extend="right",
            text=("Breaker" if ob["status"] == "breaker" else "OB") + (" ▲" if bull else " ▼"),
            title=("Breaker block" if ob["status"] == "breaker" else "Order block") + (" (bullish)" if bull else " (bearish)"),
            why=(f"The last {'down' if bull else 'up'} candle before the move that broke structure "
                 f"{'up' if bull else 'down'} ({f.p(ob['bottom'])}–{f.p(ob['top'])}). Big orders filled here; when "
                 f"price returns, the rest of those orders often defend it, so it acts as "
                 f"{'demand' if bull else 'supply'}." if ob["status"] != "breaker" else
                 f"An order block price closed through. Failed blocks often flip: this one now acts as "
                 f"{'demand' if bull else 'supply'} from the other side."))
    gaps = sorted([g for g in A["fvgs"] if g["filled_idx"] is None and vis(g["bottom"], g["top"])],
                  key=lambda g: abs((g["top"] + g["bottom"]) / 2 - price))
    for k, g in enumerate(gaps[:5]):
        add("fvg", "box", 6, 2 if k < 3 else 3, x1=tx(g["idx"]), x2=right, y1=g["bottom"], y2=g["top"],
            fill=COL["fvg_bull"] if g["direction"] == "bullish" else COL["fvg_bear"], text="FVG", extend="right",
            title=f"{g['direction'].capitalize()} fair value gap {f.p(g['bottom'])}–{f.p(g['top'])}",
            why=f"Three candles where the first and third don't overlap: price moved so fast "
                f"{'up' if g['direction'] == 'bullish' else 'down'} that only one side traded. Markets often come back "
                f"to fill part of such gaps, and they often act as {'support' if g['direction'] == 'bullish' else 'resistance'} "
                f"when they do. It is still open (not filled).")
    liq = A["liquidity"]
    for tag, lvls in (("BSL", liq["buy_side"][:3]), ("SSL", liq["sell_side"][:3])):
        for k, s in enumerate(lvls):
            if vis(s["price"]):
                add("liquidity", "line", 7, 1 if k == 0 else 2, x1=tx(s["idx"]), x2=right, y1=s["price"],
                    y2=s["price"], color=COL["liq"], dash="dashdot", width=1, text=tag, extend="right",
                    title=f"{'Buy' if tag == 'BSL' else 'Sell'}-side liquidity {f.p(s['price'])}",
                    why=(f"An untaken swing {'high' if tag == 'BSL' else 'low'}. Traders put stop orders just "
                         f"{'above' if tag == 'BSL' else 'below'} swings, and breakout orders too, so a pool of orders "
                         f"rests here. Price is often drawn to it, and a wick through it that closes back is a sweep "
                         f"(stops taken). Stops placed just beyond such a level are the ones that get hunted."))
    for pl in liq["pools"]:
        if not pl["taken"] and pl["idx_a"] >= s0 and vis(pl["level"]):
            add("liquidity", "line", 7, 2, x1=tx(pl["idx_a"]), x2=tx(pl["idx_b"]), y1=pl["level"], y2=pl["level"],
                color=COL["liq"], dash="dot", width=1, text=pl["kind"], label_at="end",
                title=f"{'Equal highs' if pl['kind'] == 'EQH' else 'Equal lows'} {f.p(pl['level'])}",
                why=f"Two swing {'highs' if pl['kind'] == 'EQH' else 'lows'} at almost the same price (within 0.15 ATR). "
                    f"They look like strong {'resistance' if pl['kind'] == 'EQH' else 'support'}, so many stops sit "
                    f"just {'above' if pl['kind'] == 'EQH' else 'below'}: one of the most obvious liquidity pools. "
                    f"Smart-money traders expect it to be taken before a real move the other way.")
    for k, s in enumerate([s for s in A["sweeps"] if s["idx"] >= s0][-6:][::-1]):
        add("liquidity", "marker", 8, 2 if k < 3 else 3, x1=tx(s["idx"]), y1=s["wick"], symbol="x", color=COL["liq"],
            text=f"{s['side']} liquidity swept", title=f"{s['side'].capitalize()} sweep",
            why=f"A wick went {abs(s['wick'] - s['level']) / max(atr, 1e-12):.2f} ATR past the swing at {f.p(s['level'])} "
                f"and the candle closed back inside: the stops there were taken (a stop hunt), then price turned. "
                f"After a sweep, a break of structure the other way is the classic ICT reversal entry.")
    P = X["period"]
    for k, lab in (("pdh", "PDH"), ("pdl", "PDL"), ("pwh", "PWH"), ("pwl", "PWL"), ("pmh", "PMH"), ("pml", "PML")):
        if P.get(k) is not None and vis(P[k]) and (X["intraday"] or k[1] in "wm"):
            add("period", "hline", 9, 2, x1=0, x2=right, y1=P[k], y2=P[k], color=COL["period"], dash="dot", width=1,
                text=lab, extend="right", title=f"{lab} {f.p(P[k])}",
                why=f"The previous {'day' if k[1] == 'd' else 'week' if k[1] == 'w' else 'month'}'s "
                    f"{'high' if k[2] == 'h' else 'low'}. Every trader sees it, so stops and orders gather there: a "
                    f"key liquidity level and a common target or turning point.")
    for r_ in X["rounds"]:
        if vis(r_["price"]):
            add("round", "hline", 9, 3, x1=0, x2=right, y1=r_["price"], y2=r_["price"], color=COL["round"],
                dash="dot", width=0.8, text=f.p(r_["price"]), extend="right", title=f"Round number {f.p(r_['price'])}",
                why="A psychological round number. People place orders at round prices, so price often reacts there.")
    S_ = X["session"]
    if S_.get("asia") and S_["asia"]["i1"] >= s0:
        a_ = S_["asia"]
        add("session", "box", 10, 2, x1=tx(a_["i1"]), x2=tx(a_["i2"]), y1=a_["low"], y2=a_["high"],
            fill="rgba(144,202,249,0.08)", text="Asia", label_at="start", title="Asian range",
            why=f"The Asian session's range ({f.p(a_['low'])}–{f.p(a_['high'])}, 20:00–24:00 New York). London often "
                f"sweeps one side of it before the day's real move (the Power of 3: accumulation, manipulation, "
                f"distribution).")
    for L_ in X["lines"]:
        x1 = tx(L_["i1"])
        y1 = L_["p1"] + L_["slope"] * (s0 + x1 - L_["i1"]) if L_["i1"] < s0 else L_["p1"]
        y2 = L_["p1"] + L_["slope"] * (s0 + right - L_["i1"])
        add("trendlines", "line", 11, 2, x1=x1, x2=right, y1=float(y1), y2=float(y2), color=COL["trend"], width=1.4,
            text=L_["name"], title=L_["name"].capitalize(),
            why=f"A line through {L_.get('touches', 'several')} swing points that price respected. Until a candle closes "
                f"through it, it is likely to act as {'support' if 'up' in L_['name'].lower() else 'resistance'}.")
        if "channel_offset" in L_:
            add("trendlines", "line", 11, 3, x1=x1, x2=right, y1=float(y1 + L_["channel_offset"]),
                y2=float(y2 + L_["channel_offset"]), color=COL["trend"], width=1, dash="dash", text="channel",
                title="Channel line", why="The parallel line on the other side of the trendline: price has been moving "
                                          "between the two. Its edge is a natural place to take profit.")
    for q in X["patterns"]:
        if q["kind"] == "lines":
            for side in ("support", "resistance"):
                Lq = q[side]
                x1 = max(tx(Lq["i1"]), tx(q["start_idx"]))
                gi = s0 + x1
                add("patterns", "line", 12, 2, x1=x1, x2=last_x, y1=float(Lq["p1"] + Lq["slope"] * (gi - Lq["i1"])),
                    y2=float(Lq["p1"] + Lq["slope"] * (n - 1 - Lq["i1"])), color=COL["pattern"], width=1.4,
                    text=q["name"] if side == "resistance" else None, label_at="start",
                    title=f"{q['name']} ({side})", why=_why_pattern(q, f) if q.get("trigger") is not None else
                    f"The {side} line of a {q['name'].lower()}: price is squeezing between two lines. A close through "
                    f"either line usually starts the next move in that direction.")
        else:
            pts = [[tx(i), float(pp)] for i, pp in zip(q["points"], q["prices"])]
            add("patterns", "path", 12, 2, points=pts, color=COL["pattern"], width=1.2, dash="dot", text=q["name"],
                title=q["name"], why=_why_pattern(q, f))
            add("patterns", "hline", 12, 2, x1=pts[0][0], x2=right, y1=q["trigger"], y2=q["trigger"],
                color=COL["pattern"], dash="dash", width=1, text=f"{q['name']} trigger",
                title=f"{q['name']} trigger {f.p(q['trigger'])}",
                why=f"The {q['name'].lower()} only counts once a candle closes beyond {f.p(q['trigger'])} (the neckline "
                    f"or trigger). Before that it is a possible pattern, not a confirmed one.")
    leg = X["leg"]
    if leg is not None and leg["origin_idx"] >= s0:
        x1 = tx(min(leg["origin_idx"], leg["extreme_idx"]))
        for r_, v in leg["retracements"].items():
            if r_ in ("0.5", "0.618", "0.705", "0.786"):
                add("fib", "line", 13, 3, x1=x1, x2=right, y1=v, y2=v, color=COL["fib"], width=0.8, dash="dot",
                    text=f"{float(r_) * 100:.1f}%".replace(".0%", "%"), title=f"Fibonacci {float(r_) * 100:.1f}%",
                    why=f"The {float(r_) * 100:.1f}% retracement of the last impulse leg. Pullbacks often pause at 50–78.6% "
                        f"before the trend resumes.")
        r = leg["retracements"]
        add("fib", "box", 13, 2, x1=x1, x2=right, y1=min(r["0.618"], r["0.786"]), y2=max(r["0.618"], r["0.786"]),
            fill=COL["ote"], text="OTE", title="Optimal trade entry (61.8–78.6%)",
            why=f"The 61.8–78.6% retracement of the last impulse leg ({f.p(leg['origin'])} → {f.p(leg['extreme'])}). "
                f"ICT's optimal trade entry: where traders who missed the move join the trend at a discount "
                f"(premium for sells).")
    dr = X["dr"]
    if dr is not None:
        add("range", "hline", 14, 3, x1=tx(min(dr["top_idx"], dr["bottom_idx"])), x2=right, y1=dr["eq"], y2=dr["eq"],
            color="rgba(209,212,220,0.5)", dash="dash", width=0.8, text="EQ 50%", title="Equilibrium (50%)",
            why=f"The middle of the dealing range {f.p(dr['bottom'])}–{f.p(dr['top'])}. Above it is premium (expensive: "
                f"look to sell), below it discount (cheap: look to buy).")
    if X["vp"] is not None:
        for k, lab in (("poc", "POC"), ("vah", "VAH"), ("val", "VAL")):
            add("volume_profile", "hline", 15, 3, x1=tx(X["vp"]["start_idx"]), x2=right, y1=X["vp"][k],
                y2=X["vp"][k], color=COL["poc"], dash="dot" if k != "poc" else None, width=0.8, text=lab,
                title={"poc": "Point of control", "vah": "Value area high", "val": "Value area low"}[k],
                why={"poc": "The price with the most volume traded in the window: where buyers and sellers agreed most. "
                            "Price is often pulled back to it.",
                     "vah": "The top of the value area (where 70% of the volume traded). Above it price is 'expensive'.",
                     "val": "The bottom of the value area. Below it price is 'cheap'."}[k])
        add("volume_profile", "histogram", 15, 3, bins=X["vp"]["bins"], x_right=right, color="rgba(240,98,146,0.25)",
            title="Volume profile", why="How much volume traded at each price over the window. Thick parts are prices "
                                        "the market accepted; thin parts are where price moves quickly.")
    pats = [q for q in A["patterns"] if q["idx"] >= n - 30 and q["bias"] != "neutral"]
    for q in pats[-6:]:
        bull = q["bias"] == "bullish"
        i = q["idx"]
        y = float(d_["low"].iat[i] - 0.4 * atr) if bull else float(d_["high"].iat[i] + 0.4 * atr)
        add("candles", "marker", 16, 3 if i < n - 5 else 2, x1=tx(i), y1=y,
            symbol="triangle-up" if bull else "triangle-down", color=COL["up"] if bull else COL["down"], text=q["name"],
            title=q["name"], why=CANDLE_WHY.get(q["name"], f"A {q['bias']} candle pattern.") + " On its own a candle "
            "pattern is weak; it matters at a key level.")
    for dv in X["divs"]:
        if dv["i1"] >= s0:
            add("divergence", "line", 17, 2, x1=tx(dv["i1"]), x2=tx(dv["i2"]), y1=dv["p1"], y2=dv["p2"],
                color=COL["up"] if dv["bias"] == 1 else COL["down"], width=1.6, text=dv["name"], label_at="end",
                title=dv["name"],
                why=f"Price made a {'lower low' if dv['bias'] == 1 else 'higher high'} but RSI didn't: momentum is "
                    f"fading, an early warning the move may {'bottom' if dv['bias'] == 1 else 'top'}. It needs a "
                    f"structure break to confirm.")
    for k, (pl, dim) in enumerate(((X["primary"], False), (X["alternative"], True))):
        if pl is None:
            continue
        x1, x2 = (last_x + 2, last_x + 10) if not dim else (last_x + 11, right - 1)
        o, pr = 18 + k, 1 if not dim else 2
        tag = "plan" if not dim else "alternative"
        add("plan", "box", o, pr, x1=x1, x2=x2, y1=min(pl["entry"], pl["tps"][-1]), y2=max(pl["entry"], pl["tps"][-1]),
            fill="rgba(38,166,154,0.10)" if dim else "rgba(38,166,154,0.22)", plan=tag)
        add("plan", "box", o, pr, x1=x1, x2=x2, y1=min(pl["entry"], pl["sl"]), y2=max(pl["entry"], pl["sl"]),
            fill="rgba(239,83,80,0.10)" if dim else "rgba(239,83,80,0.22)", plan=tag)
        pre = "" if not dim else "alt "
        guard_txt = ""
        if pl.get("liquidity_guard"):
            g = pl["liquidity_guard"]
            guard_txt = (f" It was moved past the liquidity at {f.p(g['pool'])} (from {f.p(g['moved_from'])}): stop hunts "
                         f"here usually reach {g['depth_atr']:.2f} ATR past a swing.")
        add("plan", "line", o, pr, x1=x1, x2=x2, y1=pl["entry"], y2=pl["entry"], color=COL["text"], width=1.5,
            text=f"{pre}EP {f.p(pl['entry'])}", label_at="plan", plan=tag, copy=f.p(pl["entry"]),
            title=f"Entry {f.p(pl['entry'])} ({pl['type']})",
            why=f"{pl['setup']}. " + "; ".join(pl["why"][:2]) + "."
                + (" " + pl["sweep_first"]["note"] if pl.get("sweep_first") else ""))
        add("plan", "line", o, pr, x1=x1, x2=x2, y1=pl["sl"], y2=pl["sl"], color=COL["down"], width=1.5,
            text=f"{pre}SL {f.p(pl['sl'])}", label_at="plan", plan=tag, copy=f.p(pl["sl"]),
            title=f"Stop loss {f.p(pl['sl'])}",
            why=f"The idea is wrong if price gets here ({pl['invalidation']}). {abs(pl['entry'] - pl['sl']) / atr:.2f} ATR "
                f"from the entry.{guard_txt}")
        for i, t in enumerate(pl["tps"]):
            add("plan", "line", o, pr, x1=x1, x2=x2, y1=t, y2=t, color=COL["up"], width=1.2,
                text=f"{pre}TP{i + 1} {f.p(t)}", label_at="plan", plan=tag, copy=f.p(t),
                title=f"Take profit {i + 1}: {f.p(t)} ({pl['rr'][i]:.1f}R)",
                why=_why_target(pl["tp_labels"][i], pl["p_tp"][i] if pl.get("p_tp") else None))
    add("price", "hline", 20, 1, x1=0, x2=right, y1=price, y2=price, color=COL["text"], dash="dot", width=0.8,
        text=f.p(price), extend="right")
    _strategy_overlay(X, add, tx, s0, n)
    for u in X["user"]:
        if u.get("type") in ("hline", "level") and "level" in u:
            add("user", "hline", 21, 1, x1=0, x2=right, y1=u["level"], y2=u["level"], color=COL["user"], width=1.2,
                text=u.get("label") or "yours", extend="right")
        elif u.get("type") == "zone" and "level" in u:
            add("user", "box", 21, 1, x1=0, x2=right, y1=min(u["bottom"], u["top"]), y2=max(u["bottom"], u["top"]),
                fill="rgba(255,235,59,0.12)", text=u.get("label") or "your zone", extend="right")
        elif u.get("type") == "trendline" and "level" in u:
            t1, t2 = float(u["t1"]), float(u["t2"])
            slope = (float(u["p2"]) - float(u["p1"])) / (t2 - t1) if t2 != t1 else 0.0
            ya = float(u["p1"]) + slope * (candles[0][0] - t1)
            yb = float(u["p1"]) + slope * (tt(right) - t1)
            add("user", "line", 21, 1, x1=0, x2=right, y1=ya, y2=yb, color=COL["user"], width=1.2,
                text=u.get("label") or "your trendline")
        elif u.get("type") == "note" and u.get("t") is not None:
            x = int(np.searchsorted([c_[0] for c_ in candles], float(u["t"])))
            add("user", "label", 21, 1, x1=min(x, last_x), y1=float(u.get("price", price)), text=u.get("text", ""),
                color=COL["user"], anchor="above")
    ind = {"ema20": X["emas"][20].iloc[s0:].round(10).tolist(), "ema50": X["emas"][50].iloc[s0:].round(10).tolist(),
           "ema200": X["emas"][200].iloc[s0:].round(10).tolist(), "rsi": gl.rsi(d_["close"]).iloc[s0:].round(2).tolist()}
    if X["vwap"] is not None:
        ind["vwap"] = X["vwap"].iloc[s0:].round(10).tolist()
    focus = [lo_v, hi_v] + [v for pl in (X["primary"],) if pl for v in [pl["entry"], pl["sl"], pl["tps"][0]]]
    return dict(candles=candles, drawings=D, indicators=ind,
                view=dict(x_from=max(0, last_x - 150), x_to=right + 2, y_from=min(focus) - 0.6 * atr,
                          y_to=max(focus) + 0.6 * atr))


# ───────────────────────────── review the user's own trade ─────────────────────────────
def review_trade(symbol, tf, side, entry, sl, tp, tp2=None, account=1000.0, risk_pct=1.0, leverage=None,
                 analysis=None, brain=None, mode="pro"):
    """Grade a trade idea the user already has: odds, R:R, where the stop/target sit relative to liquidity and
    levels, position size and leverage, and concrete improvements."""
    brain = brain or _STATE["brain"] or gm.FallbackBrain()
    res = analysis or analyze(symbol, tf, account=account, risk_pct=risk_pct, brain=brain)
    X = _STATE.get("ctx", {}).get((res["symbol"], res["tf"]))
    if X is None:                                                      # rebuild the context from the analysis
        X = _context_for_review(symbol, tf, brain, res)
    d = 1 if str(side).lower() in ("long", "buy", "b", "1") else -1
    tps = [float(tp)] + ([float(tp2)] if tp2 is not None else [])
    if not d * (entry - sl) > 0:
        return dict(ok=False, text=f"The stop must be {'below' if d == 1 else 'above'} the entry for a "
                                   f"{'long' if d == 1 else 'short'}.")
    if not all(d * (t - entry) > 0 for t in tps):
        return dict(ok=False, text=f"The target must be {'above' if d == 1 else 'below'} the entry.")
    f, atr = X["f"], X["atr"]
    p = _plan("long" if d == 1 else "short", "market" if abs(entry - X["price"]) < 0.05 * atr else
              ("limit" if d * (X["price"] - entry) > 0 else "stop"), "Your trade", entry, sl,
              [(t, "your target") for t in tps], [], f"price reaching {f.p(sl)}", gm.EXPIRY)
    p = evaluate_plan(p, X, brain)
    p["sizing"] = size_position(p, X, account, risk_pct, leverage)
    issues, fixes = [], []
    risk = abs(entry - sl)
    if risk < 0.5 * atr:
        issues.append(f"Stop is only {risk / atr:.2f} ATR away: normal noise can hit it.")
        fixes.append(f"Give it at least 0.5–1 ATR ({f.p(entry - d * 0.75 * atr)}) or wait for a better entry.")
    pools = [c for c in X["clusters"] if ("liquidity" in " ".join(c["labels"]) or "equal" in " ".join(c["labels"]))]
    bad_sl = [c for c in pools if 0 <= d * (c["price"] - sl) <= 0.25 * atr]
    if bad_sl:
        lvl = bad_sl[0]["price"]
        better = lvl - d * 0.3 * atr
        issues.append(f"Your stop sits right at liquidity ({', '.join(bad_sl[0]['labels'][:2])} at {f.p(lvl)}): "
                      f"exactly where stop hunts happen.")
        fixes.append(f"Put the stop beyond the pool, e.g. {f.p(better)} (and size down to keep the same $ risk).")
    walls = [c for c in X["clusters"] if c["strength"] >= 2 and d * (c["price"] - entry) > 0.2 * atr
             and d * (tps[0] - c["price"]) > 0.15 * atr]
    if walls:
        w_ = walls[0] if d == 1 else walls[-1]
        issues.append(f"There's a strong level before your target: {', '.join(w_['labels'][:2])} at {f.p(w_['price'])}.")
        fixes.append(f"Take partial profit just before it ({f.p(w_['price'] - d * 0.05 * atr)}) or move the target there.")
    if p["rr"][0] < 1.5:
        issues.append(f"Reward:risk is only {p['rr'][0]:.2f}.")
        fixes.append("Aim for at least 1.5–2R, or skip it: low-R:R trades need a very high win rate.")
    for c in p["checklist"]:
        if not c["ok"] and c["w"] < 0 and not (walls and "in the way" in c["text"]):
            issues.append(c["text"][0].upper() + c["text"][1:] + ".")
    j = len(p["tps"]) - 1
    if risk < 0.5 * atr or p["rr"][j] > 5:
        issues.append("This geometry is outside what the model was trained on (stops 0.5–4 ATR, R:R up to 5): "
                      "the odds shown are for the nearest trained case and are probably optimistic.")
    verdict = ("GOOD TRADE" if p["grade"] in ("A+", "A") else "OK TRADE" if p["grade"] == "B" else
               "WEAK TRADE" if p["grade"] == "C" else "POOR TRADE")
    severe = risk < 0.4 * atr or p["cost_r"] > 0.20 or p["rr"][j] < 1.0 or p["ev"][j] < -0.15
    if severe and verdict in ("GOOD TRADE", "OK TRADE"):
        verdict = "WEAK TRADE"
    L = [f"YOUR TRADE: {'BUY' if d == 1 else 'SELL'} {f.p(entry)} · SL {f.p(sl)} · "
         + " · ".join(f"TP{i + 1} {f.p(t)}" for i, t in enumerate(p["tps"])),
         f"Verdict: {verdict} (grade {p['grade']}, {p['score']:.1f}/10) · R:R {p['rr'][j]:.2f} · "
         f"P(target before stop) {pct(p['p_tp'][j])} vs {pct(p['breakeven'][j])} break-even · EV {p['ev'][j]:+.2f}R",
         ""]
    L.append("Checklist:")
    L += [f"  {'✓' if c['ok'] else '✗'} {c['text']}" for c in p["checklist"]]
    if issues:
        L += ["", "Problems:"] + [f"  • {x}" for x in issues]
    if fixes:
        L += ["", "Improve it:"] + [f"  • {x}" for x in fixes]
    L += ["", "Size & leverage:"] + [f"  • {x}" for x in _sizing_lines(p, f)]
    return dict(ok=True, verdict=verdict, plan=_public_plan(p, f), issues=issues, fixes=fixes, text="\n".join(L))


def _context_for_review(symbol, tf, brain, res):
    """Minimal context rebuild for review_trade when only the public analysis dict is at hand."""
    tf = md.norm_tf(tf)
    C = md.get_candles(symbol, tf, bars=BARS[tf])
    dfe, A = se.analyze(C["df"])
    frames = {}
    for h in gm.LADDER[tf][:2]:
        try:
            frames[h] = md.get_candles(symbol, h, bars=HTF_BARS)["df"]
        except Exception:
            pass
    F, ctx = gm.raw_features(dfe, tf, C["asset_class"], htf=frames)
    cost, slip = gm.cost_of(C["sub"])
    X = dict(C=C, df=dfe, A=A, tf=tf, ladder=gm.LADDER[tf], price=float(C["price"]), atr=float(dfe["atr"].iloc[-1]),
             f=Fmt(C["decimals"]), row=F.iloc[-1], cost=cost, slip=slip, asset_class=C["asset_class"],
             intraday=md.TF_MIN[tf] < 1440, trend=int(A["trend"][-1]))
    X["htf"] = {t["tf"]: t for t in res.get("timeframes", [])}
    X["dr"] = gl.dealing_range(dfe, A)
    X["clusters"] = res.get("levels", [])
    X["rsi"] = float(res.get("rsi", 50))
    X["divs"] = res.get("divergences", [])
    X["patterns"] = res.get("patterns", [])
    X["session"] = res.get("session", {})
    X["q2usd"] = quote_to_usd(C)
    return X


In [ ]:
import os, sys, io, json, glob, time, subprocess, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
sys.path.insert(0, os.getcwd())         # the engine files written above live here
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

subprocess.run([sys.executable, "-m", "pip", "-q", "install", "dukascopy-python==4.0.1",
                "yfinance>=0.2.40", "joblib>=1.3"], check=True)

import sklearn
REQUIRED_SKLEARN = "1.6.1"               # the server's pin (backend/requirements.txt); the server refuses any other version
if sklearn.__version__ != REQUIRED_SKLEARN:
    raise RuntimeError(f"This Colab has scikit-learn {sklearn.__version__}, but the server needs {REQUIRED_SKLEARN}. "
                       "In a new cell run !pip install -q scikit-learn==1.6.1, then Runtime > Restart session, then Run all again.")
import joblib
import market_data as md
import guru_model as gm
import guru_analyst as ga
import strategies as st
import strategy_head as sh
import liquidity_head as lh
md.set_cache_dir(CACHE_DIR)
print("scikit-learn", sklearn.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)
print("Engine", gm.VERSION, "| strategies:", len(st.CATALOG), "| strategy head", sh.VERSION, "| liquidity head", lh.VERSION)


## 1. Market data

The same 30 markets as Notebooks 7 and 8: 15-minute, 1-hour, daily (4h and weekly are built from them), plus 5-minute and 1-minute history for the markets scalpers trade most (EURUSD, GBPUSD, USDJPY, gold, NAS100, US30, BTC, ETH, SOL). Each market comes from the first data feed that answers; for crypto, Dukascopy is the fallback. Everything is cached on Drive, so a second run is quick.

In [ ]:
MARKETS = {
    "quick": ["EURUSD", "GBPUSD", "USDJPY", "XAUUSD", "US500", "NAS100", "BTCUSDT", "ETHUSDT", "SOLUSDT", "XRPUSDT"],
    "standard": ["EURUSD", "GBPUSD", "USDJPY", "AUDUSD", "USDCAD", "USDCHF", "NZDUSD", "EURJPY", "GBPJPY", "EURGBP",
                 "XAUUSD", "XAGUSD", "US500", "NAS100", "US30", "GER40", "UK100", "JP225", "USOIL", "UKOIL",
                 "BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT", "DOGEUSDT", "ADAUSDT", "AVAXUSDT",
                 "LINKUSDT", "LTCUSDT"],
}
SCALP_MARKETS = {"quick": ["EURUSD", "XAUUSD", "BTCUSDT"],
                 "standard": ["EURUSD", "GBPUSD", "USDJPY", "XAUUSD", "NAS100", "US30", "BTCUSDT", "ETHUSDT", "SOLUSDT"]}
PRESETS = {
    "quick": dict(markets=MARKETS["quick"], years={"1h": 3, "15m": 1}, daily_from="2015-01-01",
                  scalp=SCALP_MARKETS["quick"], days_5m=60, days_1m=14, rows=200_000,
                  strategy_rows=250_000, liquidity_rows=150_000),
    "standard": dict(markets=MARKETS["standard"], years={"1h": 5, "15m": 2}, daily_from="2012-01-01",
                     scalp=SCALP_MARKETS["standard"], days_5m=180, days_1m=30, rows=450_000,
                     strategy_rows=900_000, liquidity_rows=600_000),
}
P = PRESETS[PRESET]


def load_history(sym, tf, years=None, start=None):
    """History for one market and timeframe. Crypto falls back to Dukascopy if no exchange feed answers."""
    tries = [sym]
    spec = md.parse_symbol(sym)
    if spec["asset_class"] == "crypto" and spec["base"] in md.DUKA_CRYPTO:
        tries.append(f"DUKASCOPY:{spec['base']}USD")
    for s in tries:
        try:
            return md.history(s, tf, years=years, start=start)
        except Exception as e:
            print(f"  warning {s} {tf}: {str(e)[:160]}")
    return None


DATA = {}
t0 = time.time()
for k, sym in enumerate(P["markets"]):
    print(f"[{k + 1}/{len(P['markets'])}] {sym}")
    for tf, yrs in P["years"].items():
        c = load_history(sym, tf, years=yrs)
        if c is not None:
            DATA[(sym, tf)] = c
            if tf == "1h":
                DATA[(sym, "4h")] = dict(c, tf="4h", df=md.resample(c["df"], "4h", "1h"))
    c = load_history(sym, "1D", start=P["daily_from"])
    if c is not None:
        DATA[(sym, "1D")] = c
        d = c["df"] if c["asset_class"] == "crypto" else md.trading_days(c["df"])
        DATA[(sym, "1W")] = dict(c, tf="1W", df=md.resample(d, "1W", "1D"))
    if SCALP_DATA and sym in P["scalp"]:
        for tf, days in (("5m", P["days_5m"]), ("1m", P["days_1m"])):
            c = load_history(sym, tf, years=days / 365.25)
            if c is not None:
                DATA[(sym, tf)] = c
COVERAGE = pd.DataFrame([dict(market=sym, tf=tf, source=c["source"], candles=len(c["df"]))
                         for (sym, tf), c in DATA.items()])
print(f"Loaded {len(DATA)} series ({COVERAGE['candles'].sum():,} candles) in {time.time() - t0:.0f}s")
display(COVERAGE.pivot_table(index="market", columns="tf", values="candles", aggfunc="sum")
        .reindex(columns=[t for t in md.TFS if t in set(COVERAGE["tf"])]).fillna(0).astype(int))


## 2. The main brain

The trade-odds, level and direction models the app already uses. With `REUSE_BRAIN = True` and a brain from Notebook 8 or 7 on your Drive, those models are kept as they are, and only the two new heads are trained. Otherwise they are trained here the same way as Notebook 7 (about 30 minutes more).

In [ ]:
BUNDLE = None
if REUSE_BRAIN:
    for path in PREVIOUS:
        if os.path.exists(path):
            BUNDLE = gm.load(path)
            print(f"Keeping the main models of {os.path.basename(path)} (trained {BUNDLE['created'][:16]})")
            break
if BUNDLE is not None and BUNDLE.get("sklearn") not in (None, sklearn.__version__):
    print(f"That brain was built with scikit-learn {BUNDLE.get('sklearn')}, not {sklearn.__version__}: retraining it.")
    BUNDLE = None
if BUNDLE is None:
    SERIES_MAIN = [dict(symbol=c["symbol"], tf=tf, asset_class=c["asset_class"], sub=c["sub"], df=c["df"])
                   for (sym, tf), c in DATA.items() if len(c["df"]) >= 400]
    if len(SERIES_MAIN) < 5:
        raise RuntimeError("Too few markets downloaded to train. Check the warnings above (internet or data source).")
    print(f"Training the main brain on {len(SERIES_MAIN)} series...")
    D = gm.build_dataset(SERIES_MAIN, budget=dict(rows=P["rows"]))
    BUNDLE = gm.train(D)
    del D


## 3. The strategy head

Every strategy, on every market and timeframe, run the app's way. The strategies that need an intraday chart are left out of daily and weekly charts. For each trade, the features at the signal candle are recorded with what happened next. A signal still running at the 70% mark of a market's history is left out of training, so no training trade overlaps the test period.

This is the longest part: roughly 15–35 minutes with the standard preset.

In [ ]:
SERIES = [dict(symbol=sym, tf=tf, asset_class=c["asset_class"], sub=c["sub"], df=c["df"])
          for (sym, tf), c in DATA.items() if len(c["df"]) >= 600]
print(f"{len(SERIES)} series for the heads")
t0 = time.time()
SD = sh.build(SERIES, budget=P["strategy_rows"])
print(f"{len(SD):,} strategy trades in {time.time() - t0:.0f}s")
STRATEGY_HEAD = sh.train(SD)


In [ ]:
c = STRATEGY_HEAD["card"]
print("== Does the model pick better signals than taking them all? (test period, after costs)")
print(c["model"]["selection"].round(3).to_string(index=False))
print()
print(f"AUC {c['model']['auc_model']:.3f} vs the strategy's own rate {c['model']['auc_rate']:.3f} | "
      f"Brier skill {c['model']['skill_vs_rate']:+.3f} | the app uses the per-signal odds: {c['use']['odds']}")
print()
print("== Each strategy on the test period (sorted by average R after costs)")
display(c["per_strategy_test"].round(3))


### What works where

The strategies with the best average result after costs on each market and timeframe, over the whole history (at least 40 trades). The app shows these in the strategy picker. A t-stat below about 2 means the result could easily be luck.

In [ ]:
E = STRATEGY_HEAD["evidence_market"]
best = (E[E["trades"] >= 40].sort_values("avg_r", ascending=False).groupby(["symbol", "tf"]).head(3)
        .sort_values(["symbol", "tf", "avg_r"], ascending=[True, True, False]))
best = best.assign(name=best["sid"].map(lambda s: st.BY_ID[s]["name"]))
display(best[["symbol", "tf", "name", "trades", "win_rate", "avg_r", "t_stat"]].round(3).reset_index(drop=True))
print("\nBy kind of market and timeframe group (all strategies):")
K = STRATEGY_HEAD["evidence_class"].assign(name=lambda x: x["sid"].map(lambda s: st.BY_ID[s]["name"]))
display(K.sort_values(["asset_class", "tf_group", "avg_r"], ascending=[True, True, False])
        .groupby(["asset_class", "tf_group"]).head(5)[["asset_class", "tf_group", "name", "trades", "win_rate", "avg_r", "t_stat"]]
        .round(3).reset_index(drop=True))


## 4. The liquidity head

For sampled candles of every market and timeframe: the nearest untaken pools above and below price (up to 3 each side, within 8 ATR), their features, and what happened next (swept or not within 24 candles; if swept, whether it closed back inside within 3 candles and ran 1 ATR the other way; how far past the pool the hunt reached).

In [ ]:
t0 = time.time()
LD = lh.build(SERIES, budget=P["liquidity_rows"])
print(f"{len(LD):,} pool rows in {time.time() - t0:.0f}s")
LIQUIDITY_HEAD = lh.train(LD)


In [ ]:
c = LIQUIDITY_HEAD["card"]
print("== Sweep: will the pool be taken within 24 candles?")
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in c["sweep"].items() if not isinstance(v, pd.DataFrame)})
display(c["sweep"]["by_type"].round(3))
print("Calibration (predicted vs what happened):")
display(c["sweep"]["calibration"])
print("\n== Reversal after a sweep")
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in c["reverse"].items() if not isinstance(v, pd.DataFrame)})
print("\n== Depth of stop hunts (the stop buffer): coverage = share of hunts the buffer would have survived")
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in c["depth"].items()})
print("\nThe app uses:", c["use"])
print("\n== Experience table: how often pools are taken, reverse, and how deep hunts go (ATR)")
display(LIQUIDITY_HEAD["table"].round(3))


## 5. A live check

The new brain reads one market now, with a few strategies chosen, so you can see what the app will show: the pools with their odds, the stops placed past liquidity, and each strategy's history.

In [ ]:
BUNDLE["strategy_head"] = STRATEGY_HEAD
BUNDLE["liquidity_head"] = LIQUIDITY_HEAD
BRAIN = ga.use_brain(BUNDLE)
try:
    res = ga.jsonable(ga.analyze("EURUSD", "1h", strategies=["ict_sweep_mss", "ote_pullback", "sr_bounce", "supertrend"]))
    print(res["verdict"]["action"], "|", res["verdict"]["headline"])
    print(res["liquidity"]["text"])
    if res["liquidity"].get("pools"):
        display(pd.DataFrame(res["liquidity"]["pools"]).round(3))
    for x in res["strategies"]["items"]:
        print(x["name"], "| history:", x.get("history"), "| this signal:", x.get("head"))
    if res["plan"]:
        print("Plan:", res["plan"]["text"], "| liquidity guard:", res["plan"].get("liquidity_guard"))
except Exception as e:
    print("Live check skipped (no live data from this Colab right now):", type(e).__name__, str(e)[:200])


## 6. Save and next step

In [ ]:
BUNDLE["preset"] = PRESET
BUNDLE["app_version"] = APP_VERSION
BUNDLE["sklearn"] = sklearn.__version__          # the server must run this exact version
BUNDLE["notebook"] = "trading_ai_10 · strategies and liquidity"
gm.save(BUNDLE, BRAIN_PATH)
size = os.path.getsize(BRAIN_PATH) / 1e6
print(f"Saved {BRAIN_PATH} ({size:.1f} MB)")
print()
print("What the app will use from this brain:")
print("  strategy odds per signal:", STRATEGY_HEAD["use"]["odds"], "(strategy history in the picker: always)")
print("  liquidity sweep odds:", LIQUIDITY_HEAD["use"]["sweep"])
print("  reversal after a sweep:", LIQUIDITY_HEAD["use"]["reverse"])
print("  learned stop depth past pools:", LIQUIDITY_HEAD["use"]["depth"])
print("\nSet the server's scikit-learn to exactly:", BUNDLE["sklearn"])


### Before you upload
1. Read sections 3 and 4. For the strategy head, the question is whether **top half / top quarter by model** beats **every signal** on the test period, with a t-stat that isn't near zero. For the liquidity head, the `use` flags say which parts beat their baseline on data they never saw.
2. Upload `guru_brain_v15.joblib`: sign in as the admin, open **Admin → System & model**, and choose the file. The page then lists which heads are on.
3. In the app, the strategy picker shows each strategy's history on the market being viewed, and analyses show the pools with their odds and the learned stop depth. A part whose `use` flag is off stays off.
4. Rerun this notebook every few months so the history stays current. Nothing changes in the app until you upload a new file.